In [ ]:
### The NDJSON file used in the code below was created with Nextclade CLI using the terminal commands:
# nextclade run -D ref_data -N EPI_ISL_400001_20300000_NNL_CFF.ndjson EPI_ISL_400001_20300000_NNL_CFF.fasta
#
# The GISAID sequences were downloaded manually, 10000 at a time (due to GISAID's asinine download limits), by entering EPI_ISL ranges for each 10000-sequence range
# from EPI_ISL_400001 to EPI_ISL_20300000. Because GISAID intentionally slows down user downloads, this took weeks to complete. 

In [ ]:
@time begin
    using Pkg
    Pkg.add("JSON3"); using JSON3
    Pkg.add("JLD2"); using JLD2
    Pkg.add("Dates"); using Dates
    Pkg.add("FASTX"); using FASTX
end
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
####################################################################################################################################
println(pwd()); cd("/Users/ryhisner"); println(pwd())
####################################################################################################################################

In [ ]:
### Essential functions and dictionary creation   | Runtime: 0 hr, 3 min, 48.70 sec (loading all 3 HQCS_qualified_seq files)
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
start = time()
#######################################################################################################################################
HQCS_qualified_seqs_5_1_5 = load("2026_01_06_all_private_muts_EPI_ISL_400001_20300000_maxAAmut5_maxRevs1_qcMax5_HQCS_qualified_seqs.jld2", "HQCS_qualified_seqs_5_1_5")
HQCS_qualified_seqs_10_1_30 = load("2026_01_06_all_private_muts_EPI_ISL_400001_20300000_maxAAmut10_maxRevs1_qcMax30_HQCS_qualified_seqs.jld2", "HQCS_qualified_seqs_10_1_30")
HQCS_qualified_seqs_90_5_8000 = load("2026_02_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_HQCS_qualified_seqs.jld2", "HQCS_qualified_seqs_90_5_8000")
seq_country_HQCS_90_5_8000 = load("2026_02_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_seq_country_HQCS.jld2", "seq_country_HQCS")
#######################################################################################################################################
#####################################################################################################################################
wuhan_ref_seq = "..."
ref_seq = wuhan_ref_seq
#####################################################################################################################################
######################################################################################################################################
clade_set_complete = Set(["recombinant", "19A", "19B", "20A", "20B", "20C", "20D", "20E", "20F", "20G", "20H", "20I", "20J", "21A", "21B", "21C", "21D", "21E", "21F", "21G", "21H", "21I", "21J", "21K", "21L", "21M", "22A", "22B", "22C", "22D", "22E", "22F", "23A", "23B", "23C", "23D", "23E", "23F", "23G", "23H", "23I", "24A", "24B", "24C", "24D", "24E", "24F", "24G", "24H", "24I", "25A", "25B", "25C", "25D", "25E", "25F", "25G", "25H", "25I"])
clade_arr_complete = ["recombinant", "19A", "19B", "20A", "20B", "20C", "20D", "20E", "20F", "20G", "20H", "20I", "20J", "21A", "21B", "21C", "21D", "21E", "21F", "21G", "21H", "21I", "21J", "21K", "21L", "21M", "22A", "22B", "22C", "22D", "22E", "22F", "23A", "23B", "23C", "23D", "23E", "23F", "23G", "23H", "23I", "24A", "24B", "24C", "24D", "24E", "24F", "24G", "24H", "24I", "25A", "25B", "25C", "25D", "25E", "25F", "25G", "25H", "25I"]
clade_to_pango = Dict("recombinant"=>"recombinant", "19A"=>"B", "19B"=>"A", "20A"=>"B.1", "20B"=>"B.1.1", "20C"=>"B.1", "20D"=>"B.1.1.1", "20E"=>"B.1.177", "20F"=>"D.2", "20G"=>"B.1.2", "20H"=>"B.1.351", "20I"=>"B.1.1.7", "20J"=>"P.1", "21A"=>"B.1.617.2", "21B"=>"B.1.617.1", "21C"=>"B.1.427", "21D"=>"B.1.525", "21E"=>"P.3", "21F"=>"B.1.526", "21G"=>"C.37", "21H"=>"B.1.621", "21I"=>"B.1.617.2", "21J"=>"B.1.617.2", "21K"=>"BA.1", "21L"=>"BA.2", "21M"=>"BA.3", "22A"=>"BA.4", "22B"=>"BA.5", "22C"=>"BA.2.12.1", "22D"=>"BA.2.75", "22E"=>"BQ.1", "22F"=>"XBB", "23A"=>"XBB.1.5", "23B"=>"XBB.1.16", "23C"=>"CH.1.1", "23D"=>"XBB.1.9", "23E"=>"XBB.2.3", "23F"=>"EG.5.1", "23G"=>"XBB.1.5.70", "23H"=>"HK.3", "23I"=>"BA.2.86", "24A"=>"JN.1", "24B"=>"JN.1.11.1", "24C"=>"KP.3", "24D"=>"XDV.1", "24E"=>"KP.3.1.1", "24F"=>"XEC", "24G"=>"KP.2.3", "24H"=>"LF.7", "24I"=>"MV.1", "25A"=>"LP.8.1", "25B"=>"NB.1.8.1", "25C"=>"XFG", "25D"=>"MC.10.2.1", "25E"=>"PY.1", "25F"=>"NW.1.2", "25G"=>"XFC", "25H"=>"XFJ", "25I"=>"BA.3.2")
######################################################################################################################################
EPI_ISL(n) = split(n, "|")[2]
country(n) = split(n, "/")[2]
sequence_date(n) = split(n, "|")[3]
seq_lab(n) = split(n, "/")[3]
US_state(n) = split(split(n, "/")[3], "-")[1]
mut_gene(n) = split(string(n), ":")[1]
######################################################################################################################################
######################################################################################################################################
AAsub_gene(n) = aa_gene_comprehensive_dict[n]
AAsub_gene_num(n) = [aa_gene_comprehensive_dict[n], aa_pos_comprehensive_dict[n]]
mut_num_pos_only(n) = aa_pos_comprehensive_dict[n]
AAsub_gene_num_pos_only(n) = [aa_gene_comprehensive_dict[n], aa_pos_comprehensive_dict[n]]
mut_gene_Dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "E"=>4, "M"=>5, "N"=>6, "ORF3a"=>7, "ORF6"=>8, "ORF7a"=>9, "ORF7b"=>10, "ORF8"=>11, "ORF9b"=>12)
#################################
AA_gene_sortKey_2(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n])
AA_gene_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_ct_sortKey1(n) = (1000÷mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_ct_sortKey2(n) = (n[2], AA_ct_sortKey1(n))
AA_gene_pos_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_gene_pos_sortKey_2(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n])
AA_ct_pos_sortKey1(n) = (1000÷mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_ct_pos_sortKey2(n) = (n[2], AA_ct_pos_sortKey1(n))
######################################################################################################################################
function AA_order_key(mut)
    gene = aa_gene_comprehensive_dict[mut]
    AApos = aa_pos_comprehensive_dict[mut]
    gene_pos = gene_print_order[gene]
    return (gene_pos, AApos)
end
#####################################################################################################################################
function pango_variant_sort_key(pango::String)
    dotparts = split(pango, ".")
    k1 = string(dotparts[1])
    k2 = 0
    k3 = 0
    k4 = 0
    if length(dotparts) ≥ 2
        k2 = parse(Int, string(dotparts[2]))
    end
    if length(dotparts) ≥ 3
        k3 = parse(Int, string(dotparts[3]))
    end
    if length(dotparts) ≥ 4
        k4 = parse(Int, string(dotparts[4]))
    end
    return (k1, k2, k3, k4)
end
######################################################################################################################################
gene_print_order = Dict{String, Int}("S"=>1, "N"=>2, "E"=>3, "M"=>4, "ORF3a"=>5, "ORF6"=>6, "ORF7a"=>7, "ORF7b"=>8, "ORF8"=>9, "ORF9b"=>10, "ORF1a"=>12, "ORF1b"=>13)
######################################################################################################################################
function pango_minus_X_fx(pango::String, minus::Int)
    unaliased = pango_to_pango_unaliased_v2[pango]
    dot_ct = count(".", unaliased)
    println(dot_ct)
    if dot_ct ≥ minus
        dotsplits = split(unaliased, ".")
        minus_X_unaliased = join(dotsplits[1:dot_ct+1-minus], ".")
        minus_X_pango = pango_unaliased_to_pango[minus_X_unaliased]
        println("$(pango), $(unaliased), minus-$(minus) = $(minus_X_unaliased)")
        return minus_X_pango
    else
        return pango
    end
end
####################################################################################
function pango_unaliased_minus_X_fx(unaliased::String, minus::Int)
    dot_ct = count(".", unaliased)
    if dot_ct ≥ minus 
        dotsplits = split(unaliased, ".")
        println(dotsplits)
        minus_X_unaliased = join(dotsplits[1:dot_ct+1-minus], ".")
        minus_X_pango = pango_unaliased_to_pango[minus_X_unaliased]
        println("$(unaliased), minus-$(minus) = $(minus_X_unaliased)")
        return minus_X
    else
        return minus_X_pango
    end
end
##########################################################################################################################################################################
function read_fasta(filepath::String)
    reader = FASTX.FASTA.Reader(open(filepath, "r"))
    fasta_in = [record for record in reader]
    close(reader)
    return[String(FASTX.FASTX.description(rec)) for rec in fasta_in],
    [uppercase(String(FASTX.FASTA.sequence(rec))) for rec in fasta_in]
end
##########################################################################################################################################################################
wuhan_ref_seq = "..."
ref = wuhan_ref_seq
refAA_ORF1a = "MESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLKDGTCGLVEVEKGVLPQLEQPYVFIKRSDARTAPHGHVMVELVAELEGIQYGRSGETLGVLVPHVGEIPVAYRKVLLRKNGNKGAGGHSYGADLKSFDLGDELGTDPYEDFQENWNTKHSSGVTRELMRELNGGAYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGGAPTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGIFGADPIHSLRVCVDTVRTNVYLAVFDKNLYDKLVSSFLEMKSEKQVEQKIAEIPKEEVKPFITESKPSVEQRKQDDKKIKACVEEVTTTLEETKFLTENLLLYIDINGNLHPDSATLVSDIDITFLKKDAPYIVGDVVQEGVLTAVVIPTKKAGGTTEMLAKALRKVPTDNYITTYPGQGLNGYTVEEAKTVLKKCKSAFYILPSIISNEKQEILGTVSWNLREMLAHAEETRKLMPVCVETKAIVSTIQRKYKGIKIQEGVVDYGARFYFYTSKTTVASLINTLNDLNETLVTMPLGYVTHGLNLEEAARYMRSLKVPATVSVSSPDAVTAYNGYLTSSSKTPEEHFIETISLAGSYKDWSYSGQSTQLGIEFLKRGDKSVYYTSNPTTFHLDGEVITFDNLKTLLSLREVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIKPHNSHEGKTFYVLPNDDTLRVEAFEYYHTTDPSFLGRYMSALNHTKKWKYPQVNGLTSIKWADNNCYLATALLTLQQIELKFNPPALQDAYYRARAGEAANFCALILAYCNKTVGELGDVRETMSYLFQHANLDSCKRVLNVVCKTCGQQQTTLKGVEAVMYMGTLSYEQFKKGVQIPCTCGKQATKYLVQQESPFVMMSAPPAQYELKHGTFTCASEYTGNYQCGHYKHITSKETLYCIDGALLTKSSEYKGPITDVFYKENSYTTTIKPVTYKLDGVVCTEIDPKLDNYYKKDNSYFTEQPIDLVPNQPYPNASFDNFKFVCDNIKFADDLNQLTGYKKPASRELKVTFFPDLNGDVVAIDYKHYTPSFKKGAKLLHKPIVWHVNNATNKATYKPNTWCIRCLWSTKPVETSNSFDVLKSEDAQGMDNLACEDLKPVSEEVVENPTIQKDVLECNVKTTEVVGDIILKPANNSLKITEEVGHTDLMAAYVDNSSLTIKKPNELSRVLGLKTLATHGLAAVNSVPWDTIANYAKPFLNKVVSTTTNIVTRCLNRVCTNYMPYFFTLLLQLCTFTRSTNSRIKASMPTTIAKNTVKSVGKFCLEASFNYLKSPNFSKLINIIIWFLLLSVCLGSLIYSTAALGVLMSNLGMPSYCTGYREGYLNSTNVTIATYCTGSIPCSVCLSGLDSLDTYPSLETIQITISSFKWDLTAFGLVAEWFLAYILFTRFFYVLGLAAIMQLFFSYFAVHFISNSWLMWLIINLVQMAPISAMVRMYIFFASFYYVWKSYVHVVDGCNSSTCMMCYKRNRATRVECTTIVNGVRRSFYVYANGGKGFCKLHNWNCVNCDTFCAGSTFISDEVARDLSLQFKRPINPTDQSSYIVDSVTVKNGSIHLYFDKAGQKTYERHSLSHFVNLDNLRANNTKGSLPINVIVFDGKSKCEESSAKSASVYYSQLMCQPILLLDQALVSDVGDSAEVAVKMFDAYVNTFSSTFNVPMEKLKTLVATAEAELAKNVSLDNVLSTFISAARQGFVDSDVETKDVVECLKLSHQSDIEVTGDSCNNYMLTYNKVENMTPRDLGACIDCSARHINAQVAKSHNIALIWNVKDFMSLSEQLRKQIRSAAKKNNLPFKLTCATTRQVVNVVTTKIALKGGKIVNNWLKQLIKVTLVFLFVAAIFYLITPVHVMSKHTDFSSEIIGYKAIDGGVTRDIASTDTCFANKHADFDTWFSQRGGSYTNDKACPLIAAVITREVGFVVPGLPGTILRTTNGDFLHFLPRVFSAVGNICYTPSKLIEYTDFATSACVLAAECTIFKDASGKPVPYCYDTNVLEGSVAYESLRPDTRYVLMDGSIIQFPNTYLEGSVRVVTTFDSEYCRHGTCERSEAGVCVSTSGRWVLNNDYYRSLPGVFCGVDAVNLLTNMFTPLIQPIGALDISASIVAGGIVAIVVTCLAYYFMRFRRAFGEYSHVVAFNTLLFLMSFTVLCLTPVYSFLPGVYSVIYLYLTFYLTNDVSFLAHIQWMVMFTPLVPFWITIAYIICISTKHFYWFFSNYLKRRVVFNGVSFSTFEEAALCTFLLNKEMYLKLRSDVLLPLTQYNRYLALYNKYKYFSGAMDTTSYREAACCHLAKALNDFSNSGSDVLYQPPQTSITSAVLQSGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQSAVKRTIKGTHHWLLLTILTSLLVLVQSTQWSLFFFLYENAFLPFAMGIIAMSAFAMMFVKHKHAFLCLFLLPSLATVAYFNMVYMPASWVMRIMTWLDMVDTSLSGFKLKDCVMYASAVVLLILMTARTVYDDGARRVWTLMNVLTLVYKVYYGNALDQAISMWALIISVTSNYSGVVTTVMFLARGIVFMCVEYCPIFFITGNTLQCIMLVYCFLGYFCTCYFGLFCLLNRYFRLTLGVYDYLVSTQEFRYMNSQGLLPPKNSIDAFKLNIKLLGVGGKPCIKVATVQSKMSDVKCTSVVLLSVLQQLRVESSSKLWAQCVQLHNDILLAKDTTEAFEKMVSLLSVLLSMQGAVDINKLCEEMLDNRATLQAIASEFSSLPSYAAFATAQEAYEQAVANGDSEVVLKKLKKSLNVAKSEFDRDAAMQRKLEKMADQAMTQMYKQARSEDKRAKVTSAMQTMLFTMLRKLDNDALNNIINNARDGCVPLNIIPLTTAAKLMVVIPDYNTYKNTCDGTTFTYASALWEIQQVVDADSKIVQLSEISMDNSPNLAWPLIVTALRANSAVKLQNNELSPVALRQMSCAAGTTQTACTDDNALAYYNTTKGGRFVLALLSDLQDLKWARFPKSDGTGTIYTELEPPCRFVTDTPKGPKVKYLYFIKGLNNLNRGMVLGSLAATVRLQAGNATEVPANSTVLSFCAFAVDAAKAYKDYLASGGQPITNCVKMLCTHTGTGQAITVTPEANMDQESFGGASCCLYCRCHIDHPNPKGFCDLKGKYVQIPTTCANDPVGFTLKNTVCTVCGMWKGYGCSCDQLREPMLQSADAQSFLNGFAV"
refAA_ORF1b = "NRVCGVSAARLTPCGTGTSTDVVYRAFDIYNDKVAGFAKFLKTNCCRFQEKDEDDNLIDSYFVVKRHTFSNYQHEETIYNLLKDCPAVAKHDFFKFRIDGDMVPHISRQRLTKYTMADLVYALRHFDEGNCDTLKEILVTYNCCDDDYFNKKDWYDFVENPDILRVYANLGERVRQALLKTVQFCDAMRNAGIVGVLTLDNQDLNGNWYDFGDFIQTTPGSGVPVVDSYYSLLMPILTLTRALTAESHVDTDLTKPYIKWDLLKYDFTEERLKLFDRYFKYWDQTYHPNCVNCLDDRCILHCANFNVLFSTVFPPTSFGPLVRKIFVDGVPFVVSTGYHFRELGVVHNQDVNLHSSRLSFKELLVYAADPAMHAASGNLLLDKRTTCFSVAALTNNVAFQTVKPGNFNKDFYDFAVSKGFFKEGSSVELKHFFFAQDGNAAISDYDYYRYNLPTMCDIRQLLFVVEVVDKYFDCYDGGCINANQVIVNNLDKSAGFPFNKWGKARLYYDSMSYEDQDALFAYTKRNVIPTITQMNLKYAISAKNRARTVAGVSICSTMTNRQFHQKLLKSIAATRGATVVIGTSKFYGGWHNMLKTVYSDVENPHLMGWDYPKCDRAMPNMLRIMASLVLARKHTTCCSLSHRFYRLANECAQVLSEMVMCGGSLYVKPGGTSSGDATTAYANSVFNICQAVTANVNALLSTDGNKIADKYVRNLQHRLYECLYRNRDVDTDFVNEFYAYLRKHFSMMILSDDAVVCFNSTYASQGLVASIKNFKSVLYYQNNVFMSEAKCWTETDLTKGPHEFCSQHTMLVKQGDDYVYLPYPDPSRILGAGCFVDDIVKTDGTLMIERFVSLAIDAYPLTKHPNQEYADVFHLYLQYIRKLHDELTGHMLDMYSVMLTNDNTSRYWEPEFYEAMYTPHTVLQAVGACVLCNSQTSLRCGACIRRPFLCCKCCYDHVISTSHKLVLSVNPYVCNAPGCDVTDVTQLYLGGMSYYCKSHKPPISFPLCANGQVFGLYKNTCVGSDNVTDFNAIATCDWTNAGDYILANTCTERLKLFAAETLKATEETFKLSYGIATVREVLSDRELHLSWEVGKPRPPLNRNYVFTGYRVTKNSKVQIGEYTFEKGDYGDAVVYRGTTTYKLNVGDYFVLTSHTVMPLSAPTLVPQEHYVRITGLYPTLNISDEFSSNVANYQKVGMQKYSTLQGPPGTGKSHFAIGLALYYPSARIVYTACSHAAVDALCEKALKYLPIDKCSRIIPARARVECFDKFKVNSTLEQYVFCTVNALPETTADIVVFDEISMATNYDLSVVNARLRAKHYVYIGDPAQLPAPRTLLTKGTLEPEYFNSVCRLMKTIGPDMFLGTCRRCPAEIVDTVSALVYDNKLKAHKDKSAQCFKMFYKGVITHDVSSAINRPQIGVVREFLTRNPAWRKAVFISPYNSQNAVASKILGLPTQTVDSSQGSEYDYVIFTQTTETAHSCNVNRFNVAITRAKVGILCIMSDRDLYDKLQFTSLEIPRRNVATLQAENVTGLFKDCSKVITGLHPTQAPTHLSVDTKFKTEGLCVDIPGIPKDMTYRRLISMMGFKMNYQVNGYPNMFITREEAIRHVRAWIGFDVEGCHATREAVGTNLPLQLGFSTGVNLVAVPTGYVDTPNNTDFSRVSAKPPPGDQFKHLIPLMYKGLPWNVVRIKIVQMLSDTLKNLSDRVVFVLWAHGFELTSMKYFVKIGPERTCCLCDRRATCFSTASDTYACWHHSIGFDYVYNPFMIDVQQWGFTGNLQSNHDLYCQVHGNAHVASCDAIMTRCLAVHECFVKRVDWTIEYPIIGDELKINAACRKVQHMVVKAALLADKFPVLHDIGNPKAIKCVPQADVEWKFYDAQPCSDKAYKIEELFYSYATHSDKFTDGVCLFWNCNVDRYPANSIVCRFDTRVLSNLNLPGCDGGSLYVNKHAFHTPAFDKSAFVNLKQLPFFYYSDSPCESHGKQVVSDIDYVPLKSATCITRCNLGGAVCRHHANEYRLYLDAYNMMISAGFSLWVYKQFDTYNLWNTFTRLQSLENVAFNVVNKGHFDGQQGEVPVSIINNTVYTKVDGVDVELFENKTTLPVNVAFELWAKRNIKPVPEVKILNNLGVDIAANTVIWDYKRDAPAHISTIGVCSMTDIAKKPTETICAPLTVFFDGRVDGQVDLFRNARNGVLITEGSVKGLQPSVGPKQASLNGVTLIGEAVKTQFNYYKKVDGVVQQLPETYFTQSRNLQEFKPRSQMEIDFLELAMDEFIERYKLEGYAFEHIVYGDFSHSQLGGLHLLIGLAKRFKESPFELEDFIPMDSTVKNYFITDAQTGSSKCVCSVIDLLLDDFVEIIKSQDLSVVSKVVKVTIDYTEISFMLWCKDGHVETFYPKLQSSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVAKYTQLCQYLNTLTLAVPYNMRVIHFGAGSDKGVAPGTAVLRQWLPTGTLLVDSDLNDFVSDADSTLIGDCATVHTANKWDLIISDMYDPKTKNVTKENDSKEGFFTYICGFIQQKLALGGSVAIKITEHSWNADLYKLMGHFAWWTAFVTNVNASSSEAFLIGCNYLGKPREQIDGYVMHANYIFWRNTNPIQLSSYSLFDMSKFPLKLRGTAVMSLKEGQINDMILSLLSKGRLIIRENNRVVISSDVLVNN*"
refAA_S = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKWPWYIWLGFIAGLIAIVMVTIMLCCMTSCCSCLKGCCSCGSCCKFDEDDSEPVLKGVKLHYT*"
refAA_ORF3a = "MDLFMRIFTIGTVTLKQGEIKDATPSDFVRATATIPIQASLPFGWLIVGVALLAVFQSASKIITLKKRWQLALSKGVHFVCNLLLLFVTVYSHLLLVAAGLEAPFLYLYALVYFLQSINFVRIIMRLWLCWKCRSKNPLLYDANYFLCWHTNCYDYCIPYNSVTSSIVITSGDGTTSPISEHDYQIGGYTEKWESGVKDCVVLHSYFTSDYYQLYSTQLSTDTGVEHVTFFIYNKIVDEPEEHVQIHTIDGSSGVVNPVMEPIYDEPTTTTSVPL*"
refAA_E = "MYSFVSEETGTLIVNSVLLFLAFVVFLLVTLAILTALRLCAYCCNIVNVSLVKPSFYVYSRVKNLNSSRVPDLLV*"
refAA_M = "MADSNGTITVEELKKLLEQWNLVIGFLFLTWICLLQFAYANRNRFLYIIKLIFLWLLWPVTLACFVLAAVYRINWITGGIAIAMACLVGLMWLSYFIASFRLFARTRSMWSFNPETNILLNVPLHGTILTRPLLESELVIGAVILRGHLRIAGHHLGRCDIKDLPKEITVATSRTLSYYKLGASQRVAGDSGFAAYSRYRIGNYKLNTDHSSSSDNIALLVQ*"
refAA_ORF6 = "MFHLVDFQVTIAEILLIIMRTFKVSIWNLDYIINLIIKNLSKSLTENKYSQLDEEQPMEID*"
refAA_ORF7a = "MKIILFLALITLATCELYHYQECVRGTTVLLKEPCSSGTYEGNSPFHPLADNKFALTCFSTQFAFACPDGVKHVYQLRARSVSPKLFIRQEEVQELYSPIFLIVAAIVFITLCFTLKRKTE*"
refAA_ORF7b = "MIELSLIDFYLCFLAFLLFLVLIMLIIFWFSLELQDHNETCHA*"
refAA_ORF8 = "MKFLVFLGIITTVAAFHQECSLQSCTQHQPYVVDDPCPIHFYSKWYIRVGARKSAPLIELCVDEAGSKSPIQYIDIGNYTVSCLPFTINCQEPKLGSLVVRCSFYEDFLEYHDVRVVLDFI*"
refAA_N = "MSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLPNNTASWFTALTQHGKEDLKFPRGQGVPINTNSSPDDQIGYYRRATRRIRGGDGKMKDLSPRWYFYYLGTGPEAGLPYGANKDGIIWVATEGALNTPKDHIGTRNPANNAAIVLQLPQGTTLPKGFYAEGSRGGSQASSRSSSRSRNSSRNSTPGSSRGTSPARMAGNGGDAALALLLLDRLNQLESKMSGKGQQQQGQTVTKKSAAEASKKPRQKRTATKAYNVTQAFGRRGPEQTQGNFGDQELIRQGTDYKHWPQIAQFAPSASAFFGMSRIGMEVTPSGTWLTYTGAIKLDDKDPNFKDQVILLNKHIDAYKTFPPTEPKKDKKKKADETQALPQRQKKQQTVTLLPAADLDDFSKQLQQSMSSADSTQA*"
refAA_ORF9b = "MDPKISEMHPALRLVDPQIQLAVTRMENAVGRDQNNVGPKVYPIILRLGSPLSLNMARKTLNSLEDKAFQLTPIAVQMTKLATTEELPDEFVVVTVK*"
######################################################################################################################################
#ref_AA_dict = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
######################################################################################################################################
gene_array = ["ORF1a", "ORF1b", "S", "ORF3a", "E", "M", "ORF6", "ORF7a", "ORF7b", "ORF8", "N", "ORF9b"]
gene_order_dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "ORF3a"=>4, "E"=>4, "M"=>6, "ORF6"=>7, "ORF7a"=>8, "ORF7b"=>9, "ORF8"=>10, "N"=>11, "ORF9b"=>12)
gene_order_dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "ORF3a"=>4, "E"=>4, "M"=>6, "ORF6"=>7, "ORF7a"=>8, "ORF7b"=>9, "ORF8"=>10, "N"=>11, "ORF9b"=>12)
gene_AA_sortKey(m) = (gene_order_dict[aa_gene_comprehensive_dict[m]], aa_pos_comprehensive_dict[m])
gene_AApos_sortKey(m) = (gene_order_dict[aa_gene_comprehensive_dict[m]], aa_pos_comprehensive_dict[m])
######################################################################################################################################
gene_AA_dict = Dict{String, String}()
gene_AA_dict["ORF1a"] = "MESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLKDGTCGLVEVEKGVLPQLEQPYVFIKRSDARTAPHGHVMVELVAELEGIQYGRSGETLGVLVPHVGEIPVAYRKVLLRKNGNKGAGGHSYGADLKSFDLGDELGTDPYEDFQENWNTKHSSGVTRELMRELNGGAYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGGAPTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGIFGADPIHSLRVCVDTVRTNVYLAVFDKNLYDKLVSSFLEMKSEKQVEQKIAEIPKEEVKPFITESKPSVEQRKQDDKKIKACVEEVTTTLEETKFLTENLLLYIDINGNLHPDSATLVSDIDITFLKKDAPYIVGDVVQEGVLTAVVIPTKKAGGTTEMLAKALRKVPTDNYITTYPGQGLNGYTVEEAKTVLKKCKSAFYILPSIISNEKQEILGTVSWNLREMLAHAEETRKLMPVCVETKAIVSTIQRKYKGIKIQEGVVDYGARFYFYTSKTTVASLINTLNDLNETLVTMPLGYVTHGLNLEEAARYMRSLKVPATVSVSSPDAVTAYNGYLTSSSKTPEEHFIETISLAGSYKDWSYSGQSTQLGIEFLKRGDKSVYYTSNPTTFHLDGEVITFDNLKTLLSLREVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIKPHNSHEGKTFYVLPNDDTLRVEAFEYYHTTDPSFLGRYMSALNHTKKWKYPQVNGLTSIKWADNNCYLATALLTLQQIELKFNPPALQDAYYRARAGEAANFCALILAYCNKTVGELGDVRETMSYLFQHANLDSCKRVLNVVCKTCGQQQTTLKGVEAVMYMGTLSYEQFKKGVQIPCTCGKQATKYLVQQESPFVMMSAPPAQYELKHGTFTCASEYTGNYQCGHYKHITSKETLYCIDGALLTKSSEYKGPITDVFYKENSYTTTIKPVTYKLDGVVCTEIDPKLDNYYKKDNSYFTEQPIDLVPNQPYPNASFDNFKFVCDNIKFADDLNQLTGYKKPASRELKVTFFPDLNGDVVAIDYKHYTPSFKKGAKLLHKPIVWHVNNATNKATYKPNTWCIRCLWSTKPVETSNSFDVLKSEDAQGMDNLACEDLKPVSEEVVENPTIQKDVLECNVKTTEVVGDIILKPANNSLKITEEVGHTDLMAAYVDNSSLTIKKPNELSRVLGLKTLATHGLAAVNSVPWDTIANYAKPFLNKVVSTTTNIVTRCLNRVCTNYMPYFFTLLLQLCTFTRSTNSRIKASMPTTIAKNTVKSVGKFCLEASFNYLKSPNFSKLINIIIWFLLLSVCLGSLIYSTAALGVLMSNLGMPSYCTGYREGYLNSTNVTIATYCTGSIPCSVCLSGLDSLDTYPSLETIQITISSFKWDLTAFGLVAEWFLAYILFTRFFYVLGLAAIMQLFFSYFAVHFISNSWLMWLIINLVQMAPISAMVRMYIFFASFYYVWKSYVHVVDGCNSSTCMMCYKRNRATRVECTTIVNGVRRSFYVYANGGKGFCKLHNWNCVNCDTFCAGSTFISDEVARDLSLQFKRPINPTDQSSYIVDSVTVKNGSIHLYFDKAGQKTYERHSLSHFVNLDNLRANNTKGSLPINVIVFDGKSKCEESSAKSASVYYSQLMCQPILLLDQALVSDVGDSAEVAVKMFDAYVNTFSSTFNVPMEKLKTLVATAEAELAKNVSLDNVLSTFISAARQGFVDSDVETKDVVECLKLSHQSDIEVTGDSCNNYMLTYNKVENMTPRDLGACIDCSARHINAQVAKSHNIALIWNVKDFMSLSEQLRKQIRSAAKKNNLPFKLTCATTRQVVNVVTTKIALKGGKIVNNWLKQLIKVTLVFLFVAAIFYLITPVHVMSKHTDFSSEIIGYKAIDGGVTRDIASTDTCFANKHADFDTWFSQRGGSYTNDKACPLIAAVITREVGFVVPGLPGTILRTTNGDFLHFLPRVFSAVGNICYTPSKLIEYTDFATSACVLAAECTIFKDASGKPVPYCYDTNVLEGSVAYESLRPDTRYVLMDGSIIQFPNTYLEGSVRVVTTFDSEYCRHGTCERSEAGVCVSTSGRWVLNNDYYRSLPGVFCGVDAVNLLTNMFTPLIQPIGALDISASIVAGGIVAIVVTCLAYYFMRFRRAFGEYSHVVAFNTLLFLMSFTVLCLTPVYSFLPGVYSVIYLYLTFYLTNDVSFLAHIQWMVMFTPLVPFWITIAYIICISTKHFYWFFSNYLKRRVVFNGVSFSTFEEAALCTFLLNKEMYLKLRSDVLLPLTQYNRYLALYNKYKYFSGAMDTTSYREAACCHLAKALNDFSNSGSDVLYQPPQTSITSAVLQSGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQSAVKRTIKGTHHWLLLTILTSLLVLVQSTQWSLFFFLYENAFLPFAMGIIAMSAFAMMFVKHKHAFLCLFLLPSLATVAYFNMVYMPASWVMRIMTWLDMVDTSLSGFKLKDCVMYASAVVLLILMTARTVYDDGARRVWTLMNVLTLVYKVYYGNALDQAISMWALIISVTSNYSGVVTTVMFLARGIVFMCVEYCPIFFITGNTLQCIMLVYCFLGYFCTCYFGLFCLLNRYFRLTLGVYDYLVSTQEFRYMNSQGLLPPKNSIDAFKLNIKLLGVGGKPCIKVATVQSKMSDVKCTSVVLLSVLQQLRVESSSKLWAQCVQLHNDILLAKDTTEAFEKMVSLLSVLLSMQGAVDINKLCEEMLDNRATLQAIASEFSSLPSYAAFATAQEAYEQAVANGDSEVVLKKLKKSLNVAKSEFDRDAAMQRKLEKMADQAMTQMYKQARSEDKRAKVTSAMQTMLFTMLRKLDNDALNNIINNARDGCVPLNIIPLTTAAKLMVVIPDYNTYKNTCDGTTFTYASALWEIQQVVDADSKIVQLSEISMDNSPNLAWPLIVTALRANSAVKLQNNELSPVALRQMSCAAGTTQTACTDDNALAYYNTTKGGRFVLALLSDLQDLKWARFPKSDGTGTIYTELEPPCRFVTDTPKGPKVKYLYFIKGLNNLNRGMVLGSLAATVRLQAGNATEVPANSTVLSFCAFAVDAAKAYKDYLASGGQPITNCVKMLCTHTGTGQAITVTPEANMDQESFGGASCCLYCRCHIDHPNPKGFCDLKGKYVQIPTTCANDPVGFTLKNTVCTVCGMWKGYGCSCDQLREPMLQSADAQSFLNGFAV*"
gene_AA_dict["ORF1b"] = "RVCGVSAARLTPCGTGTSTDVVYRAFDIYNDKVAGFAKFLKTNCCRFQEKDEDDNLIDSYFVVKRHTFSNYQHEETIYNLLKDCPAVAKHDFFKFRIDGDMVPHISRQRLTKYTMADLVYALRHFDEGNCDTLKEILVTYNCCDDDYFNKKDWYDFVENPDILRVYANLGERVRQALLKTVQFCDAMRNAGIVGVLTLDNQDLNGNWYDFGDFIQTTPGSGVPVVDSYYSLLMPILTLTRALTAESHVDTDLTKPYIKWDLLKYDFTEERLKLFDRYFKYWDQTYHPNCVNCLDDRCILHCANFNVLFSTVFPPTSFGPLVRKIFVDGVPFVVSTGYHFRELGVVHNQDVNLHSSRLSFKELLVYAADPAMHAASGNLLLDKRTTCFSVAALTNNVAFQTVKPGNFNKDFYDFAVSKGFFKEGSSVELKHFFFAQDGNAAISDYDYYRYNLPTMCDIRQLLFVVEVVDKYFDCYDGGCINANQVIVNNLDKSAGFPFNKWGKARLYYDSMSYEDQDALFAYTKRNVIPTITQMNLKYAISAKNRARTVAGVSICSTMTNRQFHQKLLKSIAATRGATVVIGTSKFYGGWHNMLKTVYSDVENPHLMGWDYPKCDRAMPNMLRIMASLVLARKHTTCCSLSHRFYRLANECAQVLSEMVMCGGSLYVKPGGTSSGDATTAYANSVFNICQAVTANVNALLSTDGNKIADKYVRNLQHRLYECLYRNRDVDTDFVNEFYAYLRKHFSMMILSDDAVVCFNSTYASQGLVASIKNFKSVLYYQNNVFMSEAKCWTETDLTKGPHEFCSQHTMLVKQGDDYVYLPYPDPSRILGAGCFVDDIVKTDGTLMIERFVSLAIDAYPLTKHPNQEYADVFHLYLQYIRKLHDELTGHMLDMYSVMLTNDNTSRYWEPEFYEAMYTPHTVLQAVGACVLCNSQTSLRCGACIRRPFLCCKCCYDHVISTSHKLVLSVNPYVCNAPGCDVTDVTQLYLGGMSYYCKSHKPPISFPLCANGQVFGLYKNTCVGSDNVTDFNAIATCDWTNAGDYILANTCTERLKLFAAETLKATEETFKLSYGIATVREVLSDRELHLSWEVGKPRPPLNRNYVFTGYRVTKNSKVQIGEYTFEKGDYGDAVVYRGTTTYKLNVGDYFVLTSHTVMPLSAPTLVPQEHYVRITGLYPTLNISDEFSSNVANYQKVGMQKYSTLQGPPGTGKSHFAIGLALYYPSARIVYTACSHAAVDALCEKALKYLPIDKCSRIIPARARVECFDKFKVNSTLEQYVFCTVNALPETTADIVVFDEISMATNYDLSVVNARLRAKHYVYIGDPAQLPAPRTLLTKGTLEPEYFNSVCRLMKTIGPDMFLGTCRRCPAEIVDTVSALVYDNKLKAHKDKSAQCFKMFYKGVITHDVSSAINRPQIGVVREFLTRNPAWRKAVFISPYNSQNAVASKILGLPTQTVDSSQGSEYDYVIFTQTTETAHSCNVNRFNVAITRAKVGILCIMSDRDLYDKLQFTSLEIPRRNVATLQAENVTGLFKDCSKVITGLHPTQAPTHLSVDTKFKTEGLCVDIPGIPKDMTYRRLISMMGFKMNYQVNGYPNMFITREEAIRHVRAWIGFDVEGCHATREAVGTNLPLQLGFSTGVNLVAVPTGYVDTPNNTDFSRVSAKPPPGDQFKHLIPLMYKGLPWNVVRIKIVQMLSDTLKNLSDRVVFVLWAHGFELTSMKYFVKIGPERTCCLCDRRATCFSTASDTYACWHHSIGFDYVYNPFMIDVQQWGFTGNLQSNHDLYCQVHGNAHVASCDAIMTRCLAVHECFVKRVDWTIEYPIIGDELKINAACRKVQHMVVKAALLADKFPVLHDIGNPKAIKCVPQADVEWKFYDAQPCSDKAYKIEELFYSYATHSDKFTDGVCLFWNCNVDRYPANSIVCRFDTRVLSNLNLPGCDGGSLYVNKHAFHTPAFDKSAFVNLKQLPFFYYSDSPCESHGKQVVSDIDYVPLKSATCITRCNLGGAVCRHHANEYRLYLDAYNMMISAGFSLWVYKQFDTYNLWNTFTRLQSLENVAFNVVNKGHFDGQQGEVPVSIINNTVYTKVDGVDVELFENKTTLPVNVAFELWAKRNIKPVPEVKILNNLGVDIAANTVIWDYKRDAPAHISTIGVCSMTDIAKKPTETICAPLTVFFDGRVDGQVDLFRNARNGVLITEGSVKGLQPSVGPKQASLNGVTLIGEAVKTQFNYYKKVDGVVQQLPETYFTQSRNLQEFKPRSQMEIDFLELAMDEFIERYKLEGYAFEHIVYGDFSHSQLGGLHLLIGLAKRFKESPFELEDFIPMDSTVKNYFITDAQTGSSKCVCSVIDLLLDDFVEIIKSQDLSVVSKVVKVTIDYTEISFMLWCKDGHVETFYPKLQSSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVAKYTQLCQYLNTLTLAVPYNMRVIHFGAGSDKGVAPGTAVLRQWLPTGTLLVDSDLNDFVSDADSTLIGDCATVHTANKWDLIISDMYDPKTKNVTKENDSKEGFFTYICGFIQQKLALGGSVAIKITEHSWNADLYKLMGHFAWWTAFVTNVNASSSEAFLIGCNYLGKPREQIDGYVMHANYIFWRNTNPIQLSSYSLFDMSKFPLKLRGTAVMSLKEGQINDMILSLLSKGRLIIRENNRVVISSDVLVNN*"
gene_AA_dict["S"] = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKWPWYIWLGFIAGLIAIVMVTIMLCCMTSCCSCLKGCCSCGSCCKFDEDDSEPVLKGVKLHYT*"
gene_AA_dict["E"] = "MYSFVSEETGTLIVNSVLLFLAFVVFLLVTLAILTALRLCAYCCNIVNVSLVKPSFYVYSRVKNLNSSRVPDLLV*"
gene_AA_dict["M"] = "MADSNGTITVEELKKLLEQWNLVIGFLFLTWICLLQFAYANRNRFLYIIKLIFLWLLWPVTLACFVLAAVYRINWITGGIAIAMACLVGLMWLSYFIASFRLFARTRSMWSFNPETNILLNVPLHGTILTRPLLESELVIGAVILRGHLRIAGHHLGRCDIKDLPKEITVATSRTLSYYKLGASQRVAGDSGFAAYSRYRIGNYKLNTDHSSSSDNIALLVQ*"
gene_AA_dict["N"] = "MSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLPNNTASWFTALTQHGKEDLKFPRGQGVPINTNSSPDDQIGYYRRATRRIRGGDGKMKDLSPRWYFYYLGTGPEAGLPYGANKDGIIWVATEGALNTPKDHIGTRNPANNAAIVLQLPQGTTLPKGFYAEGSRGGSQASSRSSSRSRNSSRNSTPGSSRGTSPARMAGNGGDAALALLLLDRLNQLESKMSGKGQQQQGQTVTKKSAAEASKKPRQKRTATKAYNVTQAFGRRGPEQTQGNFGDQELIRQGTDYKHWPQIAQFAPSASAFFGMSRIGMEVTPSGTWLTYTGAIKLDDKDPNFKDQVILLNKHIDAYKTFPPTEPKKDKKKKADETQALPQRQKKQQTVTLLPAADLDDFSKQLQQSMSSADSTQA*"
gene_AA_dict["ORF3a"] = "MDLFMRIFTIGTVTLKQGEIKDATPSDFVRATATIPIQASLPFGWLIVGVALLAVFQSASKIITLKKRWQLALSKGVHFVCNLLLLFVTVYSHLLLVAAGLEAPFLYLYALVYFLQSINFVRIIMRLWLCWKCRSKNPLLYDANYFLCWHTNCYDYCIPYNSVTSSIVITSGDGTTSPISEHDYQIGGYTEKWESGVKDCVVLHSYFTSDYYQLYSTQLSTDTGVEHVTFFIYNKIVDEPEEHVQIHTIDGSSGVVNPVMEPIYDEPTTTTSVPL*"
gene_AA_dict["ORF6"] = "MFHLVDFQVTIAEILLIIMRTFKVSIWNLDYIINLIIKNLSKSLTENKYSQLDEEQPMEID*"
gene_AA_dict["ORF7a"] = "MKIILFLALITLATCELYHYQECVRGTTVLLKEPCSSGTYEGNSPFHPLADNKFALTCFSTQFAFACPDGVKHVYQLRARSVSPKLFIRQEEVQELYSPIFLIVAAIVFITLCFTLKRKTE*"
gene_AA_dict["ORF7b"] = "MIELSLIDFYLCFLAFLLFLVLIMLIIFWFSLELQDHNETCHA*"
gene_AA_dict["ORF8"] = "MKFLVFLGIITTVAAFHQECSLQSCTQHQPYVVDDPCPIHFYSKWYIRVGARKSAPLIELCVDEAGSKSPIQYIDIGNYTVSCLPFTINCQEPKLGSLVVRCSFYEDFLEYHDVRVVLDFI*"
gene_AA_dict["ORF9b"] = "MDPKISEMHPALRLVDPQIQLAVTRMENAVGRDQNNVGPKVYPIILRLGSPLSLNMARKTLNSLEDKAFQLTPIAVQMTKLATTEELPDEFVVVTVK*"
######################################################################################################################################
aa_site_to_index = Dict{String, Int}()
aa_index_to_site = Dict{Int, String}()
aa_index = 0
for gene in gene_array
    for i in 1:length(gene_AA_dict[gene])
        aa_index += 1
        gene_n_pos = "$(gene):$(i)"
        aa_site_to_index[gene_n_pos] = aa_index
    end
end
for (genepos, aaindex) in aa_site_to_index
    aa_index_to_site[aaindex] = genepos
end
######################################################################################################################################
NSP_AA_size = Dict{String, Int}("NSP1"=>180, "NSP2"=>638, "NSP3"=>1945, "NSP4"=>500, "NSP5"=>306, "NSP6"=>290, "NSP7"=>83, "NSP8"=>198, "NSP9"=>113, "NSP10"=>139, "NSP11"=>0, "NSP12"=>932, "NSP13"=>601, "NSP14"=>527, "NSP15"=>346, "NSP16"=>299)                                                                # "NSP12"=>BitSet(1:923), 
NSP_ranges = Dict{String, String}("NSP1"=>"ORF1a:1-180", "NSP2"=>"ORF1a:181-818", "NSP3"=>"ORF1a:819-2763", "NSP4"=>"ORF1a:2764-3263", "NSP5"=>"ORF1a:3264-3569", "NSP6"=>"ORF1a:3570-3859", "NSP7"=>"ORF1a:3860-3942", "NSP8"=>"ORF1a:3943-4140", "NSP9"=>"ORF1a:4141-4253", "NSP10"=>"ORF1a:4254-4392", "NSP11"=>"", "NSP12"=>"1a:4393-1b:923", "NSP13"=>"ORF1b:924-1524", "NSP14"=>"ORF1b:1525-2051", "NSP15"=>"ORF1b:2052-2397", "NSP16"=>"ORF1b:2398-2696", "S"=>"S:1-1274", "ORF3a"=>"ORF3a:1-276", "E"=>"E:1-76", "M"=>"M:1-223", "ORF6"=>"ORF6:1-62", "ORF7a"=>"ORF7a:1-122", "ORF7b"=>"ORF7b:1-44", "ORF8"=>"ORF8:1-122", "N"=>"N:1-420", "ORF9b"=>"ORF9b:1-98")
NSP_ranges_num_only = Dict{String, BitSet}("NSP1"=>BitSet(1:180), "NSP2"=>BitSet(181:818), "NSP3"=>BitSet(819:2763), "NSP4"=>BitSet(2764:3263), "NSP5"=>BitSet(3264:3569), "NSP6"=>BitSet(3570:3859), "NSP7"=>BitSet(3860:3942), "NSP8"=>BitSet(3943:4140), "NSP9"=>BitSet(4141:4253), "NSP10"=>BitSet(4254:4392), "NSP11"=>BitSet(), "NSP12"=>BitSet([4393:4401; 1:923]), "NSP13"=>BitSet(924:1524), "NSP14"=>BitSet(1525:2051), "NSP15"=>BitSet(2052:2397), "NSP16"=>BitSet(2398:2696), "S"=>BitSet(1:1274), "ORF3a"=>BitSet(1:276), "E"=>BitSet(1:76), "M"=>BitSet(1:223), "ORF6"=>BitSet(1:62), "ORF7a"=>BitSet(1:122), "ORF7b"=>BitSet(1:44), "ORF8"=>BitSet(1:122), "N"=>BitSet(1:420), "ORF9b"=>BitSet(1:98))
NSP_ranges1a = Dict{Int, BitSet}(1=>BitSet(1:180), 2=>BitSet(181:818), 3=>BitSet(819:2763), 4=>BitSet(2764:3263), 5=>BitSet(3264:3569), 6=>BitSet(3570:3859), 7=>BitSet(3860:3942), 8=>BitSet(3943:4140), 9=>BitSet(4141:4253), 10=>BitSet(4254:4392), 12=>BitSet(4393:4401))
NSP_ranges1b = Dict{Int, BitSet}(12=>BitSet(1:923), 13=>BitSet(924:1524), 14=>BitSet(1525:2051), 15=>BitSet(2052:2397), 16=>BitSet(2398:2696))
NSP1a_add = Dict{Int, Int}(1=>0, 2=>180, 3=>818, 4=>2763, 5=>3263, 6=>3569, 7=>3859, 8=>3942, 9=>4140, 10=>4253, 11=>0, 12=>4392)
NSP1b_add = Dict{Int, Int}(12=>-9, 13=>923, 14=>1524, 15=>2051, 16=>2397)
NSP1ab_add = Dict{Int, Int}(1=>0, 2=>180, 3=>818, 4=>2763, 5=>3263, 6=>3569, 7=>3859, 8=>3942, 9=>4140, 10=>4353, 11=>0, 12=>-9, 13=>923, 14=>1524, 15=>2051, 16=>2397)
######################################################################################################################################
ORF_size_dict = Dict{String, Int}()
for (orf, aaseq) in gene_AA_dict
    orf_len = length(aaseq)
    ORF_size_dict[orf] = orf_len
end
###########################################################################################################################################################################
###########################################################################################################################################################################
AA_residues = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "-", "*", "X"])
aa_pos_comprehensive_dict = Dict{String, Int}()
aa_gene_comprehensive_dict = Dict{String, String}()
aa_gene_and_pos_comprehensive_dict = Dict{String, String}()
refAA_comprehensive_dict = Dict{String, String}()
qryAA_comprehensive_dict = Dict{String, String}()
global nonspike_muts = Set{String}()
for (orf, orf_len) in ORF_size_dict
    for aa1 in AA_residues
        for aa2 in AA_residues
            for i in 1:orf_len
                orf_mut = "$(orf):$(aa1)$(i)$(aa2)"
                orf_pos_mut = "$(orf):$(i)"
                aa_pos_comprehensive_dict[orf_mut] = i
                aa_pos_comprehensive_dict[orf_pos_mut] = i
                aa_gene_comprehensive_dict[orf_mut] = orf
                aa_gene_comprehensive_dict[orf_pos_mut] = orf
                aa_gene_and_pos_comprehensive_dict[orf_mut] = orf_pos_mut
                aa_gene_and_pos_comprehensive_dict[orf_pos_mut] = orf_pos_mut
                if orf ≠ "S"
                    push!(nonspike_muts, orf_mut)
                    push!(nonspike_muts, orf_pos_mut)
                end
                refAA_comprehensive_dict[orf_mut] = aa1
                qryAA_comprehensive_dict[orf_mut] = aa2
            end
        end
    end
end
aa_gene_comprehensive_dict["NTD_disulfide"] = "S"
aa_pos_comprehensive_dict["NTD_disulfide"] = 1
aa_gene_and_pos_comprehensive_dict["NTD_disulfide"] = "S:NTD_disulfide"
refAA_comprehensive_dict["NTD_disulfide"] = "NTDdisulfide"
qryAA_comprehensive_dict["NTD_disulfide"] = "NTD_disulfide"
aa_gene_comprehensive_dict["NTD:disulfide"] = "S"
aa_pos_comprehensive_dict["NTD:disulfide"] = 1
aa_gene_and_pos_comprehensive_dict["NTD:disulfide"] = "S:NTD_disulfide"
refAA_comprehensive_dict["NTD:disulfide"] = "NTDdisulfide"
qryAA_comprehensive_dict["NTD:disulfide"] = "NTD_disulfide"
######################################################## Below: 100% comprehensive ref_nuc & qry_nuc dicts #################################################################
ref_nuc_comprehensive_dict = Dict{String, String}()
qry_nuc_comprehensive_dict = Dict{String, String}()
nuc_mut_int_comprehensive_dict = Dict{String, Int}()
nuc_mut_int_string_comprehensive_dict = Dict{String, String}()
nuc_residues1 = ["T", "C", "A", "G"]
nuc_residues2 = ["T", "C", "A", "G", "Y", "R", "K", "W", "M", "S", "-", "N"]
for nres1 in nuc_residues1
    for nres2 in nuc_residues2
        for i in 1:30000
            mut = "$(nres1)$(i)$(nres2)"
            nucmpos = i
            ref_nuc_comprehensive_dict[mut] = nres1
            qry_nuc_comprehensive_dict[mut] = nres2
            nuc_mut_int_comprehensive_dict[mut] = i
            nuc_mut_int_string_comprehensive_dict[mut] = string(i)
        end
    end
end
##########################################################################################################################################################################
##########################################################################################################################################################################
NSP_muts_pos_dict = Dict{String, Int}()
NSP_muts_gene_dict = Dict{String, String}()
NSP_ref_AA_dict = Dict{String, String}()
NSP_qry_AA_dict = Dict{String, String}()
NSP_set = Set(["NSP1", "NSP2", "NSP3", "NSP4", "NSP5", "NSP6", "NSP7", "NSP8", "NSP9", "NSP10", "NSP12", "NSP13", "NSP14", "NSP15", "NSP16"])
for nsp in NSP_set
    nsp_len = NSP_AA_size[nsp]
    for aa1 in AA_residues
        for aa2 in AA_residues
            for i in 1:nsp_len
                nspmut = "$(nsp):$(aa1)$(i)$(aa2)"
                nsppos = "$(nsp):$(i)"
                NSP_muts_gene_dict[nspmut] = nsp
                NSP_muts_pos_dict[nspmut] = i
                NSP_muts_gene_dict[nsppos] = nsp
                NSP_muts_pos_dict[nsppos] = i
                NSP_ref_AA_dict[nspmut] = aa1
                NSP_qry_AA_dict[nspmut] = aa2  
            end
        end
    end
end
###########################################################################################################################################################################
###########################################################################################################################################################################
AA_triplets = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"X", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"X", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"X", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"X", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"X", "--C"=>"X", "--A"=>"X", "--G"=>"X", "---"=>"X", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
AA_triplet_dels = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"-", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"-", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"-", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"-", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"-", "--C"=>"-", "--A"=>"-", "--G"=>"-", "---"=>"-", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
###########################################################################################################################################################################
EPI_ISL(n) = split(n, "|")[2]
countree(n) = split(n, "/")[2]
US_state(n) = split(split(n, "/")[3], "-")[1]
sequence_date(n) = split(n, "|")[3]
seq_lab(n) = split(n, "/")[3]
nuc_mut_pos(m) = nuc_mut_int_comprehensive_dict[m]
gene_name_fx(m) = aa_gene_comprehensive_dict[m]
AA_pos_int(m) = aa_pos_comprehensive_dict[m]
AA_mut_to_AA_pos(m) = aa_gene_and_pos_comprehensive_dict[m]
################### Below: OLD functions ######################
#nuc_mut_pos(m) = parse(Int, m[2:end-1])
#gene_name_fx(m) = split(m, ":")[1]
#AA_position(m) = split(m, ":")[2][2:end-1]
#AA_pos_int(m) = parse(Int, AA_position(m))
#AA_mut_to_AA_pos(m) = split(m, ":")[1] * ":" * split(m, ":")[2][2:end-1]
######################################################################################################################################
### Turns time in seconds to hours, minutes, & seconds
function seconds_to_hrs_min_sec(t)
    hours = 0
    minutes = 0
    seconds = 0
    hours = t÷3600
    minutes = (t%3600)÷60
    if t > 0.0001
        seconds = (t%3600)%60
    end
    hours_int = Int(hours)
    minutes_int = Int(minutes)
    minutes_str = split(string(minutes_int), ".")[1]
    hours_fin = split(string(hours_int), ".")[1]
    minutes_fin = ""
    minutes_fin = lpad(minutes_str, 2, "0")
    seconds_rd = round(digits=2, seconds)
    seconds_1 = string(split(string(seconds_rd), ".")[1])
    seconds_2 = string(split(string(seconds_rd), ".")[2])
    seconds_left = lpad(seconds_1, 2, "0")
    seconds_right = rpad(seconds_2, 2, "0")
    seconds_fin = "$(seconds_left).$(seconds_right)"
    final_time = "$hours_int:$minutes_int:$seconds_fin"
    final_time2 = "$hours_int hr, $minutes_int min, $seconds_fin sec"
    return final_time, final_time2
end
######################################################################################################################################
ORF_size_dict = Dict{String, Int}()
for (orf, aaseq) in gene_AA_dict
    orf_len = length(aaseq)
    ORF_size_dict[orf] = orf_len
end
######################################################################################################################################
AA_pos_only_pos_dict = Dict{String, Int}()
AA_pos_only_gene_dict = Dict{String, String}()
AA_pos_only_gene_and_pos_dict = Dict{String, String}()
AA_pos_only_set = Set{String}()
####################################################
for (orf, orfsize) in ORF_size_dict
    for i in 1:orfsize
        mut = "$(orf):$(i)"
        push!(AA_pos_only_set, mut)
        AA_pos_only_pos_dict[mut] = i
        AA_pos_only_gene_dict[mut] = orf
        AA_pos_only_gene_and_pos_dict[mut] = mut
    end
end
######################################################################################################################################
#######################################################################################################################################
N3_syn = ["TCT", "TCC", "TCA", "TCG", "CTT", "CTC", "CTA", "CTG", "CCT", "CCC", "CCA", "CCG", "CGT", "CGC", "CGA", "CGG", "ACT", "ACC", "ACA", "ACG", "GTT", "GTC", "GTA", "GTG", "GCT", "GCC", "GCA", "GCG", "GGT", "GGC", "GGA", "GGG"]
N3_tv = ["TTT", "TTC", "TTA", "TTG", "TAT", "TAC", "TAA", "TAG", "AAT", "AAC", "AAA", "AAG", "AGT", "AGC", "AGA", "AGG", "GAT", "GAC", "GAA", "GAG"]
F = ["TTT", "TTC"]; S = ["TTA", "TTG", "TCT", "TCC", "TCA", "TCG", "AGT", "AGC"]; Y = ["TAT", "TAC"]; x = ["TAG", "TAA", "TGA"]; C = ["TGT", "TGC"]; W = ["TGG"]
L = ["TTA", "TTG", "CTT", "CTC", "CTA", "CTG"]; P = ["CCT", "CCC", "CCA", "CCG"]; H = ["CAT", "CAC"]; Q = ["CAA", "CAG"]; R = ["CGT", "CGC", "CGA", "CGG", "AGA", "AGG"]
I = ["ATT", "ATC", "ATA"]; M = ["ATG"]; T = ["ACT", "ACC", "ACA", "ACG"]; N = ["AAT", "AAC"]; K = ["AAA", "AAG"]
V = ["GTT", "GTC", "GTA", "GTG"]; A = ["GCT", "GCC", "GCA", "GCG"]; D = ["GAT", "GAC"]; E = ["GAA", "GAG"]; G = ["GGT", "GGC", "GGA", "GGG"]
#####################################################################################################################################
function mixed2nuc(mix_mut::String)
    nuc_mut = mix_mut
    qrynuc = qry_nuc_comprehensive_dict[mix_mut]
    refnuc = ref_nuc_comprehensive_dict[mix_mut]
    pos_str = nuc_mut_int_string_comprehensive_dict[mix_mut]
    ref_n_pos = refnuc*pos_str
    normal_qry_nucs = Set(["T", "C", "A", "G", "N"])
    if length(mix_mut) ≥ 4
        if !(qrynuc in  normal_qry_nucs)
            if refnuc == "T"
                if qrynuc == "Y"
                    nuc_mut = ref_n_pos*"C"
                elseif qrynuc == "W"
                    nuc_mut = ref_n_pos*"A"
                elseif qrynuc == "K"
                    nuc_mut = ref_n_pos*"G"
                elseif qrynuc == "M"
                    nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)A"
                elseif qrynuc == "S"
                    nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)G"
                elseif qrynuc == "R"
                    nuc_mut = "$(ref_n_pos)A, $(ref_n_pos)G"
                end
            elseif refnuc == "C"
                if qrynuc == "Y"
                    nuc_mut = ref_n_pos*"T"
                elseif qrynuc == "M"
                    nuc_mut = ref_n_pos*"A"
                elseif qrynuc == "S"
                    nuc_mut = ref_n_pos*"G"
                elseif qrynuc == "R"
                    nuc_mut = "$(ref_n_pos)A, $(ref_n_pos)G"
                elseif qrynuc == "W"
                    nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)A"
                elseif qrynuc == "K"
                    nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)G"
                end
            elseif refnuc == "A"
                if qrynuc == "R"
                    nuc_mut = ref_n_pos*"G"
                elseif qrynuc == "W"
                    nuc_mut = ref_n_pos*"T"
                elseif qrynuc == "M"
                    nuc_mut = ref_n_pos*"C"
                elseif qrynuc == "Y"
                    nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)C"
                elseif qrynuc == "K"
                    nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)G"
                elseif qrynuc == "S"
                    nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)G"
                end
            elseif refnuc == "G"
                if qrynuc == "R"
                    nuc_mut = ref_n_pos*"A"
                elseif qrynuc == "K"
                    nuc_mut = ref_n_pos*"T"
                elseif qrynuc == "S"
                    nuc_mut = ref_n_pos*"C"
                elseif qrynuc == "Y"
                    nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)C"
                elseif qrynuc == "W"
                    nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)A"
                elseif qrynuc == "M"
                    nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)A"
                end
            end
        end
    end
    return nuc_mut
end
######################################################################################################################################
function multiepi_to_epis(multi)  
    epi_num_only_pre(n) = split(n, "_")[3]
    epi_num_only_first(n) = parse(Int, split(epi_num_only_pre(n), "-")[1])
    epi_num_only_last(n) = parse(Int, split(epi_num_only_pre(n), "-")[2])
    first = epi_num_only_first(multi)
    last = epi_num_only_last(multi)
    epi_arr = Vector{String}()
    for i in first:last
        i_str = string(i)
        epi = "EPI_ISL_"*i_str
        push!(epi_arr, epi)
    end
    return epi_arr
end
####################################################################
function stringlist_to_strings(txt::String)
    epi_num_only_pre(n) = split(n, "_")[3]
    function epi_sortkey(epi)
        epinum = epi_num_only_pre(epi)
        epi_key = (length(epinum), epinum)
        return epi_key
    end
    arr_of_strings1 = Vector{String}()
    arr_of_strings2 = Vector{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        if '-' in seq
            multis = multiepi_to_epis(seq)
            for mseq in multis
                push!(arr_of_strings2, mseq)
            end
        else 
            push!(arr_of_strings2, seq)
        end
    end
    sort_arr_of_strings2 = sort(collect(arr_of_strings2), by = x -> epi_sortkey(x))       
    return sort_arr_of_strings2
end
####################################################################
function stringlist_to_strings_set(txt::String)
    epi_num_only_pre(n) = split(n, "_")[3]
    function epi_sortkey(epi)
        epinum = epi_num_only_pre(epi)
        epi_key = (length(epinum), epinum)
        return epi_key
    end
    set_of_strings = Set{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        if '-' in seq
            multis = multiepi_to_epis(seq)
            for mseq in multis
                push!(set_of_strings, mseq)
            end
        else 
            push!(set_of_strings, seq)
        end
    end
    return set_of_strings
end   
####################################################################
function multiepi_to_epis_2(multi::String)  
    epi_num_only_pre(n) = split(n, "_")[3]
    epi_num_only_first(n) = parse(Int, split(epi_num_only_pre(n), "-")[1])
    epi_num_only_last(n) = parse(Int, split(epi_num_only_pre(n), "-")[2])
    first = epi_num_only_first(multi)
    last = epi_num_only_last(multi)
    epi_arr = Vector{String}()
    for i in first:last
        i_str = string(i)
        epi = "EPI_ISL_"*i_str
        push!(epi_arr, epi)
    end
    return epi_arr
end
####################################################################################################################################
nonleap_month_day_dict = Dict{Int, Int}(0=>0, 1=>31, 2=>28, 3=>31, 4=>30, 5=>31, 6=>30, 7=>31, 8=>31, 9=>30, 10=>31, 11=>30, 12=>31)
leap_month_day_dict = Dict{Int, Int}(0=>0, 1=>31, 2=>29, 3=>31, 4=>30, 5=>31, 6=>30, 7=>31, 8=>31, 9=>30, 10=>31, 11=>30, 12=>31)
index_to_date = Dict{Int, Tuple{Int, Int, Int}}()
date_to_index = Dict{Tuple{Int, Int, Int}, Int}()
index = 0
for year in 2020:2027
    for month in 1:12
        if year%4 == 0
            month_days = leap_month_day_dict[month]
            for day in 1:month_days
                index += 1
                index_to_date[index] = (year, month, day)
            end
        else
            month_days = nonleap_month_day_dict[month]
            for day in 1:month_days
                index += 1
                index_to_date[index] = (year, month, day)
            end
        end
    end
end
for (index, date) in index_to_date
    date_to_index[date] = index
end
for y in 2020:2027
    date_to_index[(y, 0, 0)] = y*1000000
    index_to_date[y*1000000] = (y, 0, 0)
    for m in 1:12
        date_to_index[(y, m, 0)] = y*1000000 + m*1000
        index_to_date[y*1000000 + m*1000] = (y, m, 0)
    end
end
date_to_index[(0, 0, 0)] = 1000000000
index_to_date[1000000000] = (0, 0, 0)
####################################################################################################################################
function AA_mut_ct_by_date_range_all(date1::Int, date2::Int)
    date1_to_date2_ct = Dict{String, Int}()
    for i in date1:date2
        temp_dict = date_nuc_mut_ct_all[i]
        for (AAmut, ct) in temp_dict
            if !haskey(date1_to_date2_ct, AAmut)
                date1_to_date2_ct[AAmut] = ct
            else
                date1_to_date2_ct[AAmut] += ct
            end
        end
    end
    return date1_to_date2_ct
end
####################################################################################################################################
function AA_mut_ct_no_dels_by_date_range_all(date1::Int, date2::Int)
    date1_to_date2_ct = Dict{String, Int}()
    for i in date1:date2
        temp_dict = date_AA_mut_ct_no_dels_all[i]
        for (AAmut, ct) in temp_dict
            if !haskey(date1_to_date2_ct, AAmut)
                date1_to_date2_ct[AAmut] = ct
            else
                date1_to_date2_ct[AAmut] += ct
            end
        end
    end
    return date1_to_date2_ct
end
####################################################################################################################################
function AA_mut_ct_pos_only_no_dels_by_date_range_all(date1::Int, date2::Int)
    date1_to_date2_ct = Dict{String, Int}()
    for i in date1:date2
        temp_dict = date_AA_mut_ct_pos_only_no_dels_all[i]
        for (AAmut, ct) in temp_dict
            if !haskey(date1_to_date2_ct, AAmut)
                date1_to_date2_ct[AAmut] = ct
            else
                date1_to_date2_ct[AAmut] += ct
            end
        end
    end
    return date1_to_date2_ct
end
####################################################################################################################################
function nuc_mut_ct_by_date_range_all(date1::Int, date2::Int)
    date1_to_date2_ct = Dict{String, Int}()
    for i in date1:date2
        temp_dict = date_nuc_mut_ct_all[i]
        for (nucmut, ct) in temp_dict
            if !haskey(date1_to_date2_ct, nucmut)
                date1_to_date2_ct[nucmut] = ct
            else
                date1_to_date2_ct[nucmut] += ct
            end
        end
    end
    return date1_to_date2_ct
end
####################################################################################################################################
function nuc_mut_ct_no_dels_by_date_range_all(date1::Int, date2::Int)
    date1_to_date2_ct = Dict{String, Int}()
    for i in date1:date2
        temp_dict = date_nuc_mut_ct_no_dels_all[i]
        for (nucmut, ct) in temp_dict
            if !haskey(date1_to_date2_ct, nucmut)
                date1_to_date2_ct[nucmut] = ct
            else
                date1_to_date2_ct[nucmut] += ct
            end
        end
    end
    return date1_to_date2_ct
end
####################################################################################################################################
function AA_mut_ct_by_date_range(date1::Int, date2::Int)
    date1_to_date2_ct = Dict{String, Int}()
    for i in date1:date2
        temp_dict = date_nuc_mut_ct[i]
        for (AAmut, ct) in temp_dict
            if !haskey(date1_to_date2_ct, AAmut)
                date1_to_date2_ct[AAmut] = ct
            else
                date1_to_date2_ct[AAmut] += ct
            end
        end
    end
    return date1_to_date2_ct
end
####################################################################################################################################
function AA_mut_ct_no_dels_by_date_range(date1::Int, date2::Int)
    date1_to_date2_ct = Dict{String, Int}()
    for i in date1:date2
        temp_dict = date_AA_mut_ct_no_dels[i]
        for (AAmut, ct) in temp_dict
            if !haskey(date1_to_date2_ct, AAmut)
                date1_to_date2_ct[AAmut] = ct
            else
                date1_to_date2_ct[AAmut] += ct
            end
        end
    end
    return date1_to_date2_ct
end
####################################################################################################################################
function AA_mut_ct_pos_only_no_dels_by_date_range(date1::Int, date2::Int)
    date1_to_date2_ct = Dict{String, Int}()
    for i in date1:date2
        temp_dict = date_AA_mut_ct_pos_only_no_dels[i]
        for (AAmut, ct) in temp_dict
            if !haskey(date1_to_date2_ct, AAmut)
                date1_to_date2_ct[AAmut] = ct
            else
                date1_to_date2_ct[AAmut] += ct
            end
        end
    end
    return date1_to_date2_ct
end
####################################################################################################################################
function nuc_mut_ct_by_date_range(date1::Int, date2::Int)
    date1_to_date2_ct = Dict{String, Int}()
    for i in date1:date2
        temp_dict = date_nuc_mut_ct[i]
        for (nucmut, ct) in temp_dict
            if !haskey(date1_to_date2_ct, nucmut)
                date1_to_date2_ct[nucmut] = ct
            else
                date1_to_date2_ct[nucmut] += ct
            end
        end
    end
    return date1_to_date2_ct
end
####################################################################################################################################
function nuc_mut_ct_no_dels_by_date_range(date1::Int, date2::Int)
    date1_to_date2_ct = Dict{String, Int}()
    for i in date1:date2
        temp_dict = date_nuc_mut_ct_no_dels[i]
        for (nucmut, ct) in temp_dict
            if !haskey(date1_to_date2_ct, nucmut)
                date1_to_date2_ct[nucmut] = ct
            else
                date1_to_date2_ct[nucmut] += ct
            end
        end
    end
    return date1_to_date2_ct
end
####################################################################################################################################
function read_fasta(filepath::String)
    reader = FASTX.FASTA.Reader(open(filepath, "r"))
    fasta_in = [record for record in reader]
    close(reader)
    return[String(FASTX.FASTX.description(rec)) for rec in fasta_in],
    [uppercase(String(FASTX.FASTA.sequence(rec))) for rec in fasta_in]
end
####################################################################################################################################
headers, seqs = read_fasta("/Users/ryhisner/___pango_consensus_sequences/pango-consensus-sequences_genome-nuc_2025_05_08.fasta")
pango_set = Set{String}()
for head in headers
    push!(pango_set, head)
end
println("length(pango_set) = $(length(pango_set))")
####################################################################################################################################  
print(pwd()); cd("/Users/ryhisner"); print(" | ", pwd()); println()            
######################################################################################################################################
function stringlist_to_strings_nonEPI(txt::String)
    arr_of_strings = Vector{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        push!(arr_of_strings, seq)
    end
    sort_arr_of_strings = sort(collect(arr_of_strings), by = x -> nuc_mut_int_comprehensive_dict[x])  
    return sort_arr_of_strings
end
######################################################################################################################################
function list_to_string_array(txt::String) # similar to stringlist_to_strings but not for EPIs
    no_newlines = replace(txt, "\n" =>" ")
    string_array = string.(split(no_newlines, ", "))
    return string_array
end
######################################################################################################################################
function AA_sub_to_AA_pos(AA_sub)
    gene = aa_gene_comprehensive_dict[AA_sub]
    pos = aa_pos_comprehensive_dict[AA_sub]
    AA_pos = aa_gene_and_pos_comprehensive_dict[AA_sub]
    return AA_pos
end
######################################################################################################################################
function mixed_nucs_filter(mut_arr)
    mixed_nuc_arr = Vector{String}()
    mixed_nuc_res_list = Set(['Y', 'R', 'K', 'M', 'W', 'S'])
    for m in mut_arr
        if m[end] in mixed_nuc_res_list
            push!(mixed_nuc_arr, m)
        end
    end
#    for m in mixed_nuc_arr
#        print(m, ", ")
#    end
    return mixed_nuc_arr
end
######################################################################################################################################
function muts_to_strings(mut_list_in_string_form)
    mut_arr = split(mut_list_in_string_form, ", ")
    mut_array = Vector{String}()
    for st in mut_arr
        if st[end] ≠ '-'
            str = string(st)
            push!(mut_array, str)
        end
    end
    mut_to_int(n) = nuc_mut_int_comprehensive_dict[n]                                            ##################### FUNCTION #####################
    sortKey(n) = (length(nuc_mut_int_comprehensive_dict[n]), nuc_mut_int_comprehensive_dict[n])  ##################### FUNCTION #####################
    mut_array_sort = sort(mut_array, by = x -> sortKey[x])
    mixed_mut_arr = mixed_nucs_filter(mut_arr)
    ct = 0
    return mut_array_sort
end
######################################################################################################################################
function mixed_mut_to_regular_mut(mixed_nuc_muts)            ######## New, 2025-1-26 (entire function)
    ct = 0
    mixed_regular_muts = Vector{String}()
    for i in 1:length(mixed_nuc_muts)
        if mixed_nuc_muts[i][1] == 'T'
            if mixed_nuc_muts[i][end] == 'Y'
                new_end = "C"
                new_mut = mixed_nuc_muts[i][1:end-1]*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if mixed_nuc_muts[i][end] == 'R'
                new_end1 = "A"
                new_end2 = "G"
                new_mut1 = mixed_nuc_muts[i][1:end-1]*new_end1
                new_mut2 = mixed_nuc_muts[i][1:end-1]*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if mixed_nuc_muts[i][end] == 'K'
                new_end = "G"
                new_mut = mixed_nuc_muts[i][1:end-1]*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if mixed_nuc_muts[i][end] == 'M'
                new_end1 = "C"
                new_end2 = "A"
                new_mut1 = mixed_nuc_muts[i][1:end-1]*new_end1
                new_mut2 = mixed_nuc_muts[i][1:end-1]*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if mixed_nuc_muts[i][end] == 'W'
                new_end = "A"
                new_mut = mixed_nuc_muts[i][1:end-1]*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if mixed_nuc_muts[i][end] == 'S'
                new_end1 = "C"
                new_end2 = "G"
                new_mut1 = mixed_nuc_muts[i][1:end-1]*new_end1
                new_mut2 = mixed_nuc_muts[i][1:end-1]*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
        end
        if mixed_nuc_muts[i][1] == 'C'
            if mixed_nuc_muts[i][end] == 'Y'
                new_end = "T"
                new_mut = mixed_nuc_muts[i][1:end-1]*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if mixed_nuc_muts[i][end] == 'R'
                new_end1 = "A"
                new_end2 = "G"
                new_mut1 = mixed_nuc_muts[i][1:end-1]*new_end1
                new_mut2 = mixed_nuc_muts[i][1:end-1]*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if mixed_nuc_muts[i][end] == 'K'
                new_end1 = "T"
                new_end2 = "G"
                new_mut1 = mixed_nuc_muts[i][1:end-1]*new_end1
                new_mut2 = mixed_nuc_muts[i][1:end-1]*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if mixed_nuc_muts[i][end] == 'M'
                new_end = "A"
                new_mut = mixed_nuc_muts[i][1:end-1]*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if mixed_nuc_muts[i][end] == 'W'
                new_end1 = "T"
                new_end2 = "A"
                new_mut1 = mixed_nuc_muts[i][1:end-1]*new_end1
                new_mut2 = mixed_nuc_muts[i][1:end-1]*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if mixed_nuc_muts[i][end] == 'S'
                new_end = "G"
                new_mut = mixed_nuc_muts[i][1:end-1]*new_end
                push!(mixed_regular_muts, new_mut)
            end
        end
        if mixed_nuc_muts[i][1] == 'A'
            if mixed_nuc_muts[i][end] == 'Y'
                new_end1 = "T"
                new_end2 = "C"
                new_mut1 = mixed_nuc_muts[i][1:end-1]*new_end1
                new_mut2 = mixed_nuc_muts[i][1:end-1]*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if mixed_nuc_muts[i][end] == 'R'
                new_end = "G"
                new_mut = mixed_nuc_muts[i][1:end-1]*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if mixed_nuc_muts[i][end] == 'K'
                new_end1 = "T"
                new_end2 = "G"
                new_mut1 = mixed_nuc_muts[i][1:end-1]*new_end1
                new_mut2 = mixed_nuc_muts[i][1:end-1]*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if mixed_nuc_muts[i][end] == 'M'
                new_end = "C"
                new_mut = mixed_nuc_muts[i][1:end-1]*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if mixed_nuc_muts[i][end] == 'W'
                new_end = "T"
                new_mut = mixed_nuc_muts[i][1:end-1]*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if mixed_nuc_muts[i][end] == 'S'
                new_end1 = "C"
                new_end2 = "G"
                new_mut1 = mixed_nuc_muts[i][1:end-1]*new_end1
                new_mut2 = mixed_nuc_muts[i][1:end-1]*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
        end
        if mixed_nuc_muts[i][1] == 'G'
            if mixed_nuc_muts[i][end] == 'Y'
                new_end1 = "T"
                new_end2 = "C"
                new_mut1 = mixed_nuc_muts[i][1:end-1]*new_end1
                new_mut2 = mixed_nuc_muts[i][1:end-1]*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if mixed_nuc_muts[i][end] == 'R'
                new_end = "A"
                new_mut = mixed_nuc_muts[i][1:end-1]*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if mixed_nuc_muts[i][end] == 'K'
                new_end = "T"
                new_mut = mixed_nuc_muts[i][1:end-1]*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if mixed_nuc_muts[i][end] == 'M'
                new_end1 = "C"
                new_end2 = "A"
                new_mut1 = mixed_nuc_muts[i][1:end-1]*new_end1
                new_mut2 = mixed_nuc_muts[i][1:end-1]*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if mixed_nuc_muts[i][end] == 'W'
                new_end1 = "T"
                new_end2 = "A"
                new_mut1 = mixed_nuc_muts[i][1:end-1]*new_end1
                new_mut2 = mixed_nuc_muts[i][1:end-1]*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if mixed_nuc_muts[i][end] == 'S'
                new_end = "C"
                new_mut = mixed_nuc_muts[i][1:end-1]*new_end
                push!(mixed_regular_muts, new_mut)
            end
        end
    end
    return mixed_regular_muts
end
######################################################################################################################################            
function mixed_muts_to_AA(ref_pango::String, muts::String)
    mut_strings = muts_to_strings(muts)
    mixed_muts = mixed_nucs_filter(mut_strings)             ######## New, 2025-1-26
    mixed_muts_regular = mixed_mut_to_regular_mut(mixed_muts)    ######## New, 2025-1-26
    ct = 0
    total_mixed_muts = length(mixed_muts)
    println("Total Mixed Nucs = $(total_mixed_muts)")
    println(); println()
    for i in 1:length(mixed_muts)
        if ct == 0
            print(mixed_muts[i])
            ct = 1
        else
            print(", ", mixed_muts[i])
        end
    end
    ct2 = 0
    println(); println()
    for i in 1:length(mixed_muts_regular)
        if ct2 == 0
            print(mixed_muts_regular[i])
            ct2 = 1
        else
            print(", ", mixed_muts_regular[i])
        end
    end
    println()
    syn_nuc_to_AA_dict_sort, nonsyn_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort = nuc_to_AA(ref_pango, mixed_muts_regular)
    AA__AAprintlen__vec = []
    nonsynnuc__nonsynprintlen__vec = []
    for nuc___AA in nonsyn_nuc_to_AA_dict_sort
        nucmut = nuc___AA[1]
        nucmutlen = length(nucmut)
        AAmut = nuc___AA[2]
        AAmutlen = length(AAmut)
        push!(AA__AAprintlen__vec, [AAmut, AAmutlen])
        push!(nonsynnuc__nonsynprintlen__vec, [nucmut, nucmutlen])
    end
    aa_pad_vec = String[]
    nuc_pad_vec = String[]
    for i in 1:length(AA__AAprintlen__vec)
        aa = AA__AAprintlen__vec[i][1]
        nuc = nonsynnuc__nonsynprintlen__vec[i][1]
        aapad = AA__AAprintlen__vec[i][2]
        nucpad = nonsynnuc__nonsynprintlen__vec[i][2]
        pads = [nucpad, aapad]
        pad = maximum(pads)
        push!(aa_pad_vec, rpad(aa, pad))
        push!(nuc_pad_vec, rpad(nuc, pad))
    end
    aapad_join = join(aa_pad_vec, ", ")
    nucpad_join = join(nuc_pad_vec, ", ")
    println("\n"^1)
    println(aapad_join)
    println(nucpad_join)
    println("\n"^1)
    return syn_nuc_to_AA_dict_sort, nonsyn_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort
end
######################################################################################################################################
function count_nuc_mut_types(mut_strings::Vector{String})
    mut_types_arr = ["TC", "TA", "TG", "CT", "CA", "CG", "AT", "AC", "AG", "GT", "GC", "GA"]
    mut_type_cts = Dict{String, Int}(mut_nuc_type=>0 for mut_nuc_type in mut_types_arr)
    for nuc_mut in mut_strings
        ref = string(nuc_mut[1])
        qry = string(nuc_mut[end])
        if ref == "T"
            if qry == "C"
                mut_type_cts["TC"] += 1
            elseif qry == "A"
                mut_type_cts["TA"] += 1
            elseif qry == "G"
                mut_type_cts["TG"] += 1
            end
        end
        if ref == "C"
            if qry == "T"
                mut_type_cts["CT"] += 1
            elseif qry == "A"
                mut_type_cts["CA"] += 1
            elseif qry == "G"
                mut_type_cts["CG"] += 1
            end
        end 
        if ref == "A"
            if qry == "T"
                mut_type_cts["AT"] += 1
            elseif qry == "C"
                mut_type_cts["AC"] += 1
            elseif qry == "G"
                mut_type_cts["AG"] += 1
            end
        end   
        if ref == "G"
            if qry == "T"
                mut_type_cts["GT"] += 1
            elseif qry == "C"
                mut_type_cts["GC"] += 1
            elseif qry == "A"
                mut_type_cts["GA"] += 1
            end
        end
    end
    mut_type_cts_sort_by_type = sort(collect(mut_type_cts), by = x -> x[1])
    mut_type_cts_sort_by_count = sort(collect(mut_type_cts), by = x -> x[2], rev=true)
    return mut_type_cts_sort_by_count
end
######################################################################################################################################
function all_muts_to_AA(ref_pango::String, muts::String)
    mut_strings = muts_to_strings(muts)
    mut_total = length(mut_strings)
    println()
    mut_total_string = "Total Nuc Mutations = $(mut_total)"
    ct = 0
    println(mut_total_string)
    mut_strings_join = join(mut_strings, ", ")
    println(mut_strings_join)
    mut_types_sorted_by_count = count_nuc_mut_types(stringlist_to_strings_nonEPI(muts))
    synonymous_nuc_AA_sorted_dict, nonsynonymous_nuc_AA_sorted_dict, all_nuc_AA_sorted_dict = nuc_to_AA(ref_pango, mut_strings)
    synonymous_muts_vec = String[]
    nonsynonymous_muts_vec = String[]
    for i in 1:length(synonymous_nuc_AA_sorted_dict)
        push!(synonymous_muts_vec, synonymous_nuc_AA_sorted_dict[i][1])
    end
    for i in 1:length(nonsynonymous_nuc_AA_sorted_dict)
        push!(nonsynonymous_muts_vec, nonsynonymous_nuc_AA_sorted_dict[i][1])
    end
    syn_mut_types_sorted_by_count = count_nuc_mut_types(synonymous_muts_vec)
    nonsyn_mut_types_sorted_by_count = count_nuc_mut_types(nonsynonymous_muts_vec)
    mut_types_sorted_by_count_vector = [mut_types_sorted_by_count, syn_mut_types_sorted_by_count, nonsyn_mut_types_sorted_by_count]
    titles = ["################# Nuc Mut Types Sorted by Count #################", "############# Synonymous Nuc Mut Types Sorted by Count #############", "########## Nonsynonymous Nuc Mut Types Sorted by Count ##########"]      
    for i in 1:length(mut_types_sorted_by_count_vector)
        vec = mut_types_sorted_by_count_vector[i]
        title = titles[i]
        println()
        println(title)
        for type__ct in vec
            type_of_mut = type__ct[1]
            type_ct = type__ct[2]
            if type_ct > 0
                println("                  $(type_of_mut) Count = ", lpad("$(type_ct)", 2))
            end
        end
    end
    print("\n"^1)
end
######################################################################################################################################
######################################################################################################################################
gene_AA_pango_dict = Dict{String, Dict{String, String}}()
nuc_genome_pango_dict = Dict{String, String}()
pango_consensus_set = Set{String}()
headers1a, seqs1a = read_fasta("___pango_consensus_sequences/pango_consensus_AA_ORF1a_2025_06_25_NNL.fasta")
for i in 1:length(headers1a)
    pango = headers1a[i]
    push!(pango_consensus_set, pango)
end
for pango in pango_consensus_set
    gene_AA_pango_dict[pango] = Dict{String, String}()
end
#######################################################################################################################################
gene_array = ["ORF1a", "ORF1b", "S", "ORF3a", "E", "M", "ORF6", "ORF7a", "ORF7b", "ORF8", "N", "ORF9b"]
#######################################################################################################################################
for gene in gene_array
    aa_file = "___pango_consensus_sequences/pango_consensus_AA_$(gene)_2025_06_25_NNL.fasta"
    headers, seqs = read_fasta(aa_file)
    for i in 1:length(headers)
        pango = headers[i]
        aa_seq = seqs[i]
        gene_AA_pango_dict[pango][gene] = aa_seq
    end
end
nuc_file = "___pango_consensus_sequences/pango_consensus_sequences_genome_nuc_2025_06_25_NNL.fasta"
nuc_headers, nuc_seqs = read_fasta(nuc_file)
for i in 1:length(nuc_headers)
    pango = nuc_headers[i]
    nuc_seq = nuc_seqs[i]
    nuc_genome_pango_dict[pango] = nuc_seq
end
######################################################################################################################################
gene_print_order = Dict{String, Int}("S"=>1, "N"=>2, "E"=>3, "M"=>4, "ORF3a"=>5, "ORF6"=>6, "ORF7a"=>7, "ORF7b"=>8, "ORF8"=>9, "ORF9b"=>10, "ORF1a"=>12, "ORF1b"=>13)
######################################################################################################################################
######################################################################################################################################
function nuc_to_AA(ref_pango::String, muts::Vector{String})
    nuc_mut_int_comprehensive_dict[nuc_mut] = nuc_mut_int_comprehensive_dict[nuc_mut] ##################### FUNCTION #####################
    all_muts_sort = sort(muts, by = x -> nuc_pos(x))
    ref_seq = ""; refAA_ORF1a = ""; refAA_ORF1b = ""; refAA_S = ""; refAA_ORF3a = ""; refAA_E = ""; refAA_M = ""
    refAA_ORF6 = ""; refAA_ORF7a = ""; refAA_ORF7b = ""; refAA_ORF8 = ""; refAA_N = ""; refAA_ORF9b = ""
    if ref_pango == "Wuhan"
        ref_seq = "..."
        refAA_ORF1a = "MESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLKDGTCGLVEVEKGVLPQLEQPYVFIKRSDARTAPHGHVMVELVAELEGIQYGRSGETLGVLVPHVGEIPVAYRKVLLRKNGNKGAGGHSYGADLKSFDLGDELGTDPYEDFQENWNTKHSSGVTRELMRELNGGAYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGGAPTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGIFGADPIHSLRVCVDTVRTNVYLAVFDKNLYDKLVSSFLEMKSEKQVEQKIAEIPKEEVKPFITESKPSVEQRKQDDKKIKACVEEVTTTLEETKFLTENLLLYIDINGNLHPDSATLVSDIDITFLKKDAPYIVGDVVQEGVLTAVVIPTKKAGGTTEMLAKALRKVPTDNYITTYPGQGLNGYTVEEAKTVLKKCKSAFYILPSIISNEKQEILGTVSWNLREMLAHAEETRKLMPVCVETKAIVSTIQRKYKGIKIQEGVVDYGARFYFYTSKTTVASLINTLNDLNETLVTMPLGYVTHGLNLEEAARYMRSLKVPATVSVSSPDAVTAYNGYLTSSSKTPEEHFIETISLAGSYKDWSYSGQSTQLGIEFLKRGDKSVYYTSNPTTFHLDGEVITFDNLKTLLSLREVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIKPHNSHEGKTFYVLPNDDTLRVEAFEYYHTTDPSFLGRYMSALNHTKKWKYPQVNGLTSIKWADNNCYLATALLTLQQIELKFNPPALQDAYYRARAGEAANFCALILAYCNKTVGELGDVRETMSYLFQHANLDSCKRVLNVVCKTCGQQQTTLKGVEAVMYMGTLSYEQFKKGVQIPCTCGKQATKYLVQQESPFVMMSAPPAQYELKHGTFTCASEYTGNYQCGHYKHITSKETLYCIDGALLTKSSEYKGPITDVFYKENSYTTTIKPVTYKLDGVVCTEIDPKLDNYYKKDNSYFTEQPIDLVPNQPYPNASFDNFKFVCDNIKFADDLNQLTGYKKPASRELKVTFFPDLNGDVVAIDYKHYTPSFKKGAKLLHKPIVWHVNNATNKATYKPNTWCIRCLWSTKPVETSNSFDVLKSEDAQGMDNLACEDLKPVSEEVVENPTIQKDVLECNVKTTEVVGDIILKPANNSLKITEEVGHTDLMAAYVDNSSLTIKKPNELSRVLGLKTLATHGLAAVNSVPWDTIANYAKPFLNKVVSTTTNIVTRCLNRVCTNYMPYFFTLLLQLCTFTRSTNSRIKASMPTTIAKNTVKSVGKFCLEASFNYLKSPNFSKLINIIIWFLLLSVCLGSLIYSTAALGVLMSNLGMPSYCTGYREGYLNSTNVTIATYCTGSIPCSVCLSGLDSLDTYPSLETIQITISSFKWDLTAFGLVAEWFLAYILFTRFFYVLGLAAIMQLFFSYFAVHFISNSWLMWLIINLVQMAPISAMVRMYIFFASFYYVWKSYVHVVDGCNSSTCMMCYKRNRATRVECTTIVNGVRRSFYVYANGGKGFCKLHNWNCVNCDTFCAGSTFISDEVARDLSLQFKRPINPTDQSSYIVDSVTVKNGSIHLYFDKAGQKTYERHSLSHFVNLDNLRANNTKGSLPINVIVFDGKSKCEESSAKSASVYYSQLMCQPILLLDQALVSDVGDSAEVAVKMFDAYVNTFSSTFNVPMEKLKTLVATAEAELAKNVSLDNVLSTFISAARQGFVDSDVETKDVVECLKLSHQSDIEVTGDSCNNYMLTYNKVENMTPRDLGACIDCSARHINAQVAKSHNIALIWNVKDFMSLSEQLRKQIRSAAKKNNLPFKLTCATTRQVVNVVTTKIALKGGKIVNNWLKQLIKVTLVFLFVAAIFYLITPVHVMSKHTDFSSEIIGYKAIDGGVTRDIASTDTCFANKHADFDTWFSQRGGSYTNDKACPLIAAVITREVGFVVPGLPGTILRTTNGDFLHFLPRVFSAVGNICYTPSKLIEYTDFATSACVLAAECTIFKDASGKPVPYCYDTNVLEGSVAYESLRPDTRYVLMDGSIIQFPNTYLEGSVRVVTTFDSEYCRHGTCERSEAGVCVSTSGRWVLNNDYYRSLPGVFCGVDAVNLLTNMFTPLIQPIGALDISASIVAGGIVAIVVTCLAYYFMRFRRAFGEYSHVVAFNTLLFLMSFTVLCLTPVYSFLPGVYSVIYLYLTFYLTNDVSFLAHIQWMVMFTPLVPFWITIAYIICISTKHFYWFFSNYLKRRVVFNGVSFSTFEEAALCTFLLNKEMYLKLRSDVLLPLTQYNRYLALYNKYKYFSGAMDTTSYREAACCHLAKALNDFSNSGSDVLYQPPQTSITSAVLQSGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQSAVKRTIKGTHHWLLLTILTSLLVLVQSTQWSLFFFLYENAFLPFAMGIIAMSAFAMMFVKHKHAFLCLFLLPSLATVAYFNMVYMPASWVMRIMTWLDMVDTSLSGFKLKDCVMYASAVVLLILMTARTVYDDGARRVWTLMNVLTLVYKVYYGNALDQAISMWALIISVTSNYSGVVTTVMFLARGIVFMCVEYCPIFFITGNTLQCIMLVYCFLGYFCTCYFGLFCLLNRYFRLTLGVYDYLVSTQEFRYMNSQGLLPPKNSIDAFKLNIKLLGVGGKPCIKVATVQSKMSDVKCTSVVLLSVLQQLRVESSSKLWAQCVQLHNDILLAKDTTEAFEKMVSLLSVLLSMQGAVDINKLCEEMLDNRATLQAIASEFSSLPSYAAFATAQEAYEQAVANGDSEVVLKKLKKSLNVAKSEFDRDAAMQRKLEKMADQAMTQMYKQARSEDKRAKVTSAMQTMLFTMLRKLDNDALNNIINNARDGCVPLNIIPLTTAAKLMVVIPDYNTYKNTCDGTTFTYASALWEIQQVVDADSKIVQLSEISMDNSPNLAWPLIVTALRANSAVKLQNNELSPVALRQMSCAAGTTQTACTDDNALAYYNTTKGGRFVLALLSDLQDLKWARFPKSDGTGTIYTELEPPCRFVTDTPKGPKVKYLYFIKGLNNLNRGMVLGSLAATVRLQAGNATEVPANSTVLSFCAFAVDAAKAYKDYLASGGQPITNCVKMLCTHTGTGQAITVTPEANMDQESFGGASCCLYCRCHIDHPNPKGFCDLKGKYVQIPTTCANDPVGFTLKNTVCTVCGMWKGYGCSCDQLREPMLQSADAQSFLNGFAV"
        refAA_ORF1b = "NRVCGVSAARLTPCGTGTSTDVVYRAFDIYNDKVAGFAKFLKTNCCRFQEKDEDDNLIDSYFVVKRHTFSNYQHEETIYNLLKDCPAVAKHDFFKFRIDGDMVPHISRQRLTKYTMADLVYALRHFDEGNCDTLKEILVTYNCCDDDYFNKKDWYDFVENPDILRVYANLGERVRQALLKTVQFCDAMRNAGIVGVLTLDNQDLNGNWYDFGDFIQTTPGSGVPVVDSYYSLLMPILTLTRALTAESHVDTDLTKPYIKWDLLKYDFTEERLKLFDRYFKYWDQTYHPNCVNCLDDRCILHCANFNVLFSTVFPPTSFGPLVRKIFVDGVPFVVSTGYHFRELGVVHNQDVNLHSSRLSFKELLVYAADPAMHAASGNLLLDKRTTCFSVAALTNNVAFQTVKPGNFNKDFYDFAVSKGFFKEGSSVELKHFFFAQDGNAAISDYDYYRYNLPTMCDIRQLLFVVEVVDKYFDCYDGGCINANQVIVNNLDKSAGFPFNKWGKARLYYDSMSYEDQDALFAYTKRNVIPTITQMNLKYAISAKNRARTVAGVSICSTMTNRQFHQKLLKSIAATRGATVVIGTSKFYGGWHNMLKTVYSDVENPHLMGWDYPKCDRAMPNMLRIMASLVLARKHTTCCSLSHRFYRLANECAQVLSEMVMCGGSLYVKPGGTSSGDATTAYANSVFNICQAVTANVNALLSTDGNKIADKYVRNLQHRLYECLYRNRDVDTDFVNEFYAYLRKHFSMMILSDDAVVCFNSTYASQGLVASIKNFKSVLYYQNNVFMSEAKCWTETDLTKGPHEFCSQHTMLVKQGDDYVYLPYPDPSRILGAGCFVDDIVKTDGTLMIERFVSLAIDAYPLTKHPNQEYADVFHLYLQYIRKLHDELTGHMLDMYSVMLTNDNTSRYWEPEFYEAMYTPHTVLQAVGACVLCNSQTSLRCGACIRRPFLCCKCCYDHVISTSHKLVLSVNPYVCNAPGCDVTDVTQLYLGGMSYYCKSHKPPISFPLCANGQVFGLYKNTCVGSDNVTDFNAIATCDWTNAGDYILANTCTERLKLFAAETLKATEETFKLSYGIATVREVLSDRELHLSWEVGKPRPPLNRNYVFTGYRVTKNSKVQIGEYTFEKGDYGDAVVYRGTTTYKLNVGDYFVLTSHTVMPLSAPTLVPQEHYVRITGLYPTLNISDEFSSNVANYQKVGMQKYSTLQGPPGTGKSHFAIGLALYYPSARIVYTACSHAAVDALCEKALKYLPIDKCSRIIPARARVECFDKFKVNSTLEQYVFCTVNALPETTADIVVFDEISMATNYDLSVVNARLRAKHYVYIGDPAQLPAPRTLLTKGTLEPEYFNSVCRLMKTIGPDMFLGTCRRCPAEIVDTVSALVYDNKLKAHKDKSAQCFKMFYKGVITHDVSSAINRPQIGVVREFLTRNPAWRKAVFISPYNSQNAVASKILGLPTQTVDSSQGSEYDYVIFTQTTETAHSCNVNRFNVAITRAKVGILCIMSDRDLYDKLQFTSLEIPRRNVATLQAENVTGLFKDCSKVITGLHPTQAPTHLSVDTKFKTEGLCVDIPGIPKDMTYRRLISMMGFKMNYQVNGYPNMFITREEAIRHVRAWIGFDVEGCHATREAVGTNLPLQLGFSTGVNLVAVPTGYVDTPNNTDFSRVSAKPPPGDQFKHLIPLMYKGLPWNVVRIKIVQMLSDTLKNLSDRVVFVLWAHGFELTSMKYFVKIGPERTCCLCDRRATCFSTASDTYACWHHSIGFDYVYNPFMIDVQQWGFTGNLQSNHDLYCQVHGNAHVASCDAIMTRCLAVHECFVKRVDWTIEYPIIGDELKINAACRKVQHMVVKAALLADKFPVLHDIGNPKAIKCVPQADVEWKFYDAQPCSDKAYKIEELFYSYATHSDKFTDGVCLFWNCNVDRYPANSIVCRFDTRVLSNLNLPGCDGGSLYVNKHAFHTPAFDKSAFVNLKQLPFFYYSDSPCESHGKQVVSDIDYVPLKSATCITRCNLGGAVCRHHANEYRLYLDAYNMMISAGFSLWVYKQFDTYNLWNTFTRLQSLENVAFNVVNKGHFDGQQGEVPVSIINNTVYTKVDGVDVELFENKTTLPVNVAFELWAKRNIKPVPEVKILNNLGVDIAANTVIWDYKRDAPAHISTIGVCSMTDIAKKPTETICAPLTVFFDGRVDGQVDLFRNARNGVLITEGSVKGLQPSVGPKQASLNGVTLIGEAVKTQFNYYKKVDGVVQQLPETYFTQSRNLQEFKPRSQMEIDFLELAMDEFIERYKLEGYAFEHIVYGDFSHSQLGGLHLLIGLAKRFKESPFELEDFIPMDSTVKNYFITDAQTGSSKCVCSVIDLLLDDFVEIIKSQDLSVVSKVVKVTIDYTEISFMLWCKDGHVETFYPKLQSSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVAKYTQLCQYLNTLTLAVPYNMRVIHFGAGSDKGVAPGTAVLRQWLPTGTLLVDSDLNDFVSDADSTLIGDCATVHTANKWDLIISDMYDPKTKNVTKENDSKEGFFTYICGFIQQKLALGGSVAIKITEHSWNADLYKLMGHFAWWTAFVTNVNASSSEAFLIGCNYLGKPREQIDGYVMHANYIFWRNTNPIQLSSYSLFDMSKFPLKLRGTAVMSLKEGQINDMILSLLSKGRLIIRENNRVVISSDVLVNN"
        refAA_S = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKWPWYIWLGFIAGLIAIVMVTIMLCCMTSCCSCLKGCCSCGSCCKFDEDDSEPVLKGVKLHYT"
        refAA_ORF3a = "MDLFMRIFTIGTVTLKQGEIKDATPSDFVRATATIPIQASLPFGWLIVGVALLAVFQSASKIITLKKRWQLALSKGVHFVCNLLLLFVTVYSHLLLVAAGLEAPFLYLYALVYFLQSINFVRIIMRLWLCWKCRSKNPLLYDANYFLCWHTNCYDYCIPYNSVTSSIVITSGDGTTSPISEHDYQIGGYTEKWESGVKDCVVLHSYFTSDYYQLYSTQLSTDTGVEHVTFFIYNKIVDEPEEHVQIHTIDGSSGVVNPVMEPIYDEPTTTTSVPL"
        refAA_E = "MYSFVSEETGTLIVNSVLLFLAFVVFLLVTLAILTALRLCAYCCNIVNVSLVKPSFYVYSRVKNLNSSRVPDLLV"
        refAA_M = "MADSNGTITVEELKKLLEQWNLVIGFLFLTWICLLQFAYANRNRFLYIIKLIFLWLLWPVTLACFVLAAVYRINWITGGIAIAMACLVGLMWLSYFIASFRLFARTRSMWSFNPETNILLNVPLHGTILTRPLLESELVIGAVILRGHLRIAGHHLGRCDIKDLPKEITVATSRTLSYYKLGASQRVAGDSGFAAYSRYRIGNYKLNTDHSSSSDNIALLVQ"
        refAA_ORF6 = "MFHLVDFQVTIAEILLIIMRTFKVSIWNLDYIINLIIKNLSKSLTENKYSQLDEEQPMEID"
        refAA_ORF7a = "MKIILFLALITLATCELYHYQECVRGTTVLLKEPCSSGTYEGNSPFHPLADNKFALTCFSTQFAFACPDGVKHVYQLRARSVSPKLFIRQEEVQELYSPIFLIVAAIVFITLCFTLKRKTE"
        refAA_ORF7b = "MIELSLIDFYLCFLAFLLFLVLIMLIIFWFSLELQDHNETCHA"
        refAA_ORF8 = "MKFLVFLGIITTVAAFHQECSLQSCTQHQPYVVDDPCPIHFYSKWYIRVGARKSAPLIELCVDEAGSKSPIQYIDIGNYTVSCLPFTINCQEPKLGSLVVRCSFYEDFLEYHDVRVVLDFI"
        refAA_N = "MSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLPNNTASWFTALTQHGKEDLKFPRGQGVPINTNSSPDDQIGYYRRATRRIRGGDGKMKDLSPRWYFYYLGTGPEAGLPYGANKDGIIWVATEGALNTPKDHIGTRNPANNAAIVLQLPQGTTLPKGFYAEGSRGGSQASSRSSSRSRNSSRNSTPGSSRGTSPARMAGNGGDAALALLLLDRLNQLESKMSGKGQQQQGQTVTKKSAAEASKKPRQKRTATKAYNVTQAFGRRGPEQTQGNFGDQELIRQGTDYKHWPQIAQFAPSASAFFGMSRIGMEVTPSGTWLTYTGAIKLDDKDPNFKDQVILLNKHIDAYKTFPPTEPKKDKKKKADETQALPQRQKKQQTVTLLPAADLDDFSKQLQQSMSSADSTQA"
        refAA_ORF9b = "MDPKISEMHPALRLVDPQIQLAVTRMENAVGRDQNNVGPKVYPIILRLGSPLSLNMARKTLNSLEDKAFQLTPIAVQMTKLATTEELPDEFVVVTVK"
    else
        ref_seq = nuc_genome_pango_dict[ref_pango]
        refAA_ORF1a = gene_AA_pango_dict[ref_pango]["ORF1a"]
        refAA_ORF1b = gene_AA_pango_dict[ref_pango]["ORF1b"]
        refAA_S = gene_AA_pango_dict[ref_pango]["S"]
        refAA_ORF3a = gene_AA_pango_dict[ref_pango]["ORF3a"]
        refAA_E = gene_AA_pango_dict[ref_pango]["E"]
        refAA_M = gene_AA_pango_dict[ref_pango]["M"]
        refAA_ORF6 = gene_AA_pango_dict[ref_pango]["ORF6"]
        refAA_ORF7a = gene_AA_pango_dict[ref_pango]["ORF7a"]
        refAA_ORF7b = gene_AA_pango_dict[ref_pango]["ORF7b"]
        refAA_ORF8 = gene_AA_pango_dict[ref_pango]["ORF8"]
        refAA_N = gene_AA_pango_dict[ref_pango]["N"]
        refAA_ORF9b = gene_AA_pango_dict[ref_pango]["ORF9b"]
    end
    coding_ranges = BitSet([266:13467..., 13469:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533..., 28284:28577...])
    noncoding_ranges = BitSet([1:265..., 21556:21562..., 25385:25392..., 26221:26244..., 26473:26522..., 27192:27201..., 27388:27393..., 27888:27893..., 28260:28273..., 29534:29830...])
    coding_range_9b = BitSet([28284:28577...])
    gene_nuc_starts = Dict{Int, Int}(0=>263, 1=>13465, 2=>21560, 3=>25390, 4=>26242, 5=>26520, 6=>27199, 7=>27391, 8=>27753, 9=>27891, 10=>28271, 11=>28281)
    ref_AA_genes = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
    gene_strings = Dict{Int, String}(0=>"ORF1a", 1=>"ORF1b", 2=>"S", 3=>"ORF3a", 4=>"E", 5=>"M", 6=>"ORF6", 7=>"ORF7a", 8=>"ORF7b", 9=>"ORF8", 10=>"N", 11=>"ORF9b")
    nuc_gene_num = Dict{Int, Int}()
    nuc_gene_num_9b = Dict{Int, Int}()
    synonymous_nuc_to_AA_dict = Dict{String, String}()
    nonsynonymous_nuc_to_AA_dict = Dict{String, String}()
    all_nuc_to_AA_dict = Dict{String, String}()
    all_nuc_to_context_dict = Dict{String, String}()
    synonymous_nuc_to_context_dict = Dict{String, String}()
    nonsynonymous_nuc_to_context_dict = Dict{String, String}()
    noncoding_nuc_to_context_dict = Dict{String, String}()
    noncoding_range_dict = Dict{Vector{Int}, String}([1, 265]=>"5' UTR", [21556, 21562]=>"Spike TRS", [25385, 25392]=>"ORF3a TRS", [26221, 26234]=>"ORF3a-E UTR", [26235, 26244]=>"E TRS", [26473, 26522]=>"E-M UTR", [27192, 27201]=>"M-ORF6 UTR", [27388, 27393]=>"ORF7a TRS", [27888, 27893]=>"ORF8 TRS", [28260, 28273]=>"N/ORF9b TRS", [29534, 29830]=>"3' UTR", [29794, 29801]=>"Octanucleotide Motif")
################################################
    noncoding_nuc_vector = Vector{String}()
################################################        
    gene_nuc_arr = [[266:13467...], [13469:21555...], [21563:25384...], [25393:26220...], [26245:26472...], [26523:27191...], [27202:27387...], [27394:27755...], [27760:27887...], [27894:28259...], [28274:29533...], [28284:28577...]]
    for i in 1:length(gene_nuc_arr)-1
        for nuc_pos in gene_nuc_arr[i]
            nuc_gene_num[nuc_pos] = i-1
        end
    end
    for nuc_pos in gene_nuc_arr[end]
        nuc_gene_num_9b[nuc_pos] = 11
    end

    rem0_gene = [5, 8, 9, 11]
    rem1_gene = [1, 3, 4, 6, 7]
    rem2_gene = [0, 2, 10]
    rem0 = BitSet([26523:27191..., 27760:27887..., 27894:28259..., 28284:28577...])
    rem1 = BitSet([13469:21555..., 25393:26220..., 26245:26472..., 27202:27387..., 27394:27755...])
    rem2 = BitSet([266:13467..., 21563:25384..., 28274:29533...])
    rem9b = BitSet([28284:28577...])
    rem7ab = BitSet([27756:27759...])

    AA_gene(AA_mut) = aa_gene_comprehensive_dict[AA_mut] ##################### FUNCTION #####################
    AA_pos(AA_mut) = aa_pos_comprehensive_dict[AA_mut] ##################### FUNCTION #####################
    AA_sort_key(AA_mut) = (aa_gene_comprehensive_dict[AA_mut], aa_pos_comprehensive_dict[AA_mut]) ##################### FUNCTION #####################
    
    gene_num(nuc_mut) = nuc_gene_num[nuc_mut_int_comprehensive_dict[nuc_mut]] ##################### FUNCTION #####################
    nuc_to_AA_pos(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - gene_nuc_starts[gene_num(nuc_mut)])÷3) ##################### FUNCTION #####################
    nuc_to_AA_pos_9b(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - 28281)÷3) ##################### FUNCTION #####################
    nuc2AA_ORF1a(nuc_mut, refAA, qryAA) = gene_strings[gene_num(nuc_mut)]*":"*refAA*nuc_to_AA_pos(nuc_mut)*qryAA ##################### FUNCTION #####################
    nuc2AA_ORF9b(nuc_mut, refAA, qryAA) = "ORF9b:"*refAA*nuc_to_AA_pos_9b(nuc_mut)*qryAA
    
    nuc_codon_pos_dict = Dict{Int, Int}()
    for nuc_pos in coding_ranges
        gene_number = nuc_gene_num[nuc_pos]
        gene_start = gene_nuc_starts[gene_number]
        codon_num = (nuc_pos-gene_start)%3 + 1
        nuc_codon_pos_dict[nuc_pos] = codon_num
    end
    nuc_codon_pos_dict_9b = Dict{Int, Int}()
    for nuc_pos in coding_range_9b
        gene_number = 11
        gene_start = gene_nuc_starts[gene_number]
        codon_num = (nuc_pos-gene_start)%3 + 1
        nuc_codon_pos_dict_9b[nuc_pos] = codon_num
    end
#######################################################################################################################################
#######################################################################################################################################
    for nuc_mut in muts
        mut = mixed2nuc(nuc_mut)
        if ',' in mut
            mut1 = string(split(mut, ",")[1])
            mut2 = string(split(mut, ",")[2])
            push!(muts, mut1)
            push!(muts, mut2)
            filter!(x -> !(length(x)>6), muts)
        end
    end
    
    coding_ranges = BitSet([266:13467..., 13468:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533...])
    N_9b_synonymous = Set(["C28379A", "C28394A", "T28406C", "C28475A", "C28535A", "A28547C"])
    AA_mut_set = Set{String}()
    AA_mut = ""
    for nuc_mut in muts
        pos = nuc_mut_int_comprehensive_dict[nuc_mut]
        if pos in coding_ranges
            mut = mixed2nuc(nuc_mut)  
            gene_number = gene_num(mut)
            ref_triple = ""
            qry_triple = ""
            ref_triple_context = ""
            qry_triple_context = ""
            ref2qry_context = ""
            if nuc_codon_pos_dict[pos] == 1
                ref_triple = string(mut[1])*string(ref_seq[pos+1])  *string(ref_seq[pos+2])
                qry_triple = string(mut[end])*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                ref_triple_context = string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])*string(ref_seq[pos+8])
                qry_triple_context = string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])*string(ref_seq[pos+8])
            elseif nuc_codon_pos_dict[pos] == 2
                ref_triple = string(ref_seq[pos-1])*string(mut[1])  *string(ref_seq[pos+1])
                qry_triple = string(ref_seq[pos-1])*string(mut[end])*string(ref_seq[pos+1])
                ref_triple_context = string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])
                qry_triple_context = string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])
            elseif nuc_codon_pos_dict[pos] == 3
                ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*string(mut[1])
                qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*string(mut[end])
                ref_triple_context = string(ref_seq[pos-8])*string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*ref_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])
                qry_triple_context = string(ref_seq[pos-8])*string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*qry_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])
            end

            refAA = AA_triplets[ref_triple]
            qryAA = AA_triplets[qry_triple]
            AA_mut = nuc2AA_ORF1a(mut, refAA, qryAA)
            push!(AA_mut_set, AA_mut)
            ref2qry_context = ref_triple_context*"-->"*qry_triple_context
            all_nuc_to_AA_dict[mut] = AA_mut
            if refAA == qryAA && !(pos in rem9b)
                synonymous_nuc_to_AA_dict[mut] = AA_mut
                synonymous_nuc_to_context_dict[mut] = ref2qry_context
            elseif refAA == qryAA && pos in rem9b && mut in N_9b_synonymous
                synonymous_nuc_to_AA_dict[mut] = AA_mut
                synonymous_nuc_to_context_dict[mut] = ref2qry_context
            else
                nonsynonymous_nuc_to_AA_dict[mut] = AA_mut
                nonsynonymous_nuc_to_context_dict[mut] = ref2qry_context
#                push!(nonsynonymous_nuc_muts, mut)
            end
            all_nuc_to_context_dict[mut] = ref2qry_context
###################################
            for nuc_mut2 in muts
                mut2 = mixed2nuc(nuc_mut2)
                pos2 = nuc_pos(mut2)
                if pos2 in coding_ranges
                    gene_number2 = gene_num(mut2)
                    if mut ≠ mut2 && gene_number == gene_number2 && nuc_to_AA_pos(mut) == nuc_to_AA_pos(mut2)
                        if nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 2
                            ref_triple = string(mut[1])*string(mut2[1])*string(ref_seq[pos+2])
                            qry_triple = string(mut[end])*string(mut2[end])*string(ref_seq[pos+2])
                            ref_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                            qry_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                        elseif nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 3
                            ref_triple = string(mut[1])*string(ref_seq[pos+1])*string(mut2[1])
                            qry_triple = string(mut[end])*string(ref_seq[pos+1])*string(mut2[end])
                            ref_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                            qry_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                        elseif nuc_codon_pos_dict[pos] == 2 && nuc_codon_pos_dict[pos2] == 3
                            ref_triple = string(ref_seq[pos-1])*string(mut[1])*string(mut2[1])
                            qry_triple = string(ref_seq[pos-1])*string(mut[end])*string(mut2[end])
                            ref_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                            qry_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                        elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 2
                            ref_triple = string(mut2[1])*string(mut[1])*string(ref_seq[pos+1])
                            qry_triple = string(mut2[end])*string(mut[end])*string(ref_seq[pos+1])
                            ref_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                            qry_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                        elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 3
                            ref_triple = string(mut2[1])*string(ref_seq[pos-1])*string(mut[1])
                            qry_triple = string(mut2[end])*string(ref_seq[pos-1])*string(mut[end])
                            ref_triple_context = string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*ref_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])
                            qry_triple_context = string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*qry_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])
                        elseif nuc_codon_pos_dict[pos2] == 2 && nuc_codon_pos_dict[pos] == 3
                            ref_triple = string(ref_seq[pos-2])*string(mut2[1])*string(mut[1])
                            qry_triple = string(ref_seq[pos-2])*string(mut2[end])*string(mut[end])
                        end
                        refAA2 = AA_triplets[ref_triple]
                        qryAA2 = AA_triplets[qry_triple]
                        AA_mut2 = nuc2AA_ORF1a(mut2, refAA2, qryAA2)
                        push!(AA_mut_set, AA_mut2)
                        ref2qry_context = ref_triple_context*"-->"*qry_triple_context
                        all_nuc_to_AA_dict[mut2] = AA_mut2
                        all_nuc_to_AA_dict[mut] = AA_mut2
                        if refAA2 == qryAA2 && !(pos2 in rem9b)
                            synonymous_nuc_to_AA_dict[mut2] = AA_mut2
                            synonymous_nuc_to_context_dict[mut2] = ref2qry_context 
                        else
                            nonsynonymous_nuc_to_AA_dict[mut2] = AA_mut2
                            nonsynonymous_nuc_to_context_dict[mut2] = ref2qry_context
                            nonsynonymous_nuc_to_AA_dict[mut] = AA_mut2
                            nonsynonymous_nuc_to_context_dict[mut] = ref2qry_context
                        end
                        all_nuc_to_context_dict[mut] = ref2qry_context
                        all_nuc_to_context_dict[mut2] = ref2qry_context
                    end
                end
            end
        else                  
            npos = nuc_mut_int_comprehensive_dict[nuc_mut]
            qry_nuc = string(nuc_mut[end])
            ref_nc_nuc_context = string(ref_seq[npos-8])*string(ref_seq[npos-7])*string(ref_seq[npos-6])*string(ref_seq[npos-5])*string(ref_seq[npos-4])*string(ref_seq[npos-3])*string(ref_seq[npos-2])*string(ref_seq[npos-1])*string(ref_seq[npos])*string(ref_seq[npos+1])*string(ref_seq[npos+2])*string(ref_seq[npos+3])*string(ref_seq[npos+4])*string(ref_seq[npos+5])*string(ref_seq[npos+6])*string(ref_seq[npos+7])*string(ref_seq[npos+8])
            qry_nc_nuc_context = string(ref_seq[npos-8])*string(ref_seq[npos-7])*string(ref_seq[npos-6])*string(ref_seq[npos-5])*string(ref_seq[npos-4])*string(ref_seq[npos-3])*string(ref_seq[npos-2])*string(ref_seq[npos-1])*qry_nuc*string(ref_seq[npos+1])*string(ref_seq[npos+2])*string(ref_seq[npos+3])*string(ref_seq[npos+4])*string(ref_seq[npos+5])*string(ref_seq[npos+6])*string(ref_seq[npos+7])*string(ref_seq[npos+8])
            full_nc_context = ref_nc_nuc_context*"|"*qry_nc_nuc_context
            noncoding_nuc_to_context_dict[nuc_mut] = full_nc_context
            for (start_end, place) in noncoding_range_dict
                frst = start_end[1]
                last = start_end[2]
                if npos ≥ frst && npos ≤ last
                    mut_vec = [nuc_mut, place]
                    push!(noncoding_nuc_vector, nuc_mut)
                end
            end
        end
    end
#########################################################################################################
    for nuc_mut in muts
        pos_9b = nuc_mut_int_comprehensive_dict[nuc_mut]
        if pos_9b in rem9b
            mut_9b = mixed2nuc(nuc_mut)
            pos_9b = nuc_pos(mut_9b)   
            gene_number_9b = 11
            ref_triple_9b = ""
            qry_triple_9b = ""
            ref_triple_context_9b = ""
            qry_triple_context_9b = ""
            ref2qry_context_9b = ""
            if nuc_codon_pos_dict_9b[pos_9b] == 1
                ref_triple_9b = string(mut_9b[1])*string(ref_seq[pos_9b+1])  *string(ref_seq[pos_9b+2])
                qry_triple_9b = string(mut_9b[end])*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                ref_triple_context_9b = string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])*string(ref_seq[pos_9b+8])
                qry_triple_context_9b = string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])*string(ref_seq[pos_9b+8])
            elseif nuc_codon_pos_dict_9b[pos_9b] == 2
                ref_triple_9b = string(ref_seq[pos_9b-1])*string(mut_9b[1])  *string(ref_seq[pos_9b+1])
                qry_triple_9b = string(ref_seq[pos_9b-1])*string(mut_9b[end])*string(ref_seq[pos_9b+1])
                ref_triple_context_9b = string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])
                qry_triple_context_9b = string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])
            elseif nuc_codon_pos_dict_9b[pos_9b] == 3
                ref_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*string(mut_9b[1])
                qry_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*string(mut_9b[end])
                ref_triple_context_9b = string(ref_seq[pos_9b-8])*string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*ref_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])
                qry_triple_context_9b = string(ref_seq[pos_9b-8])*string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*qry_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])
            end
            refAA_9b = AA_triplets[ref_triple_9b]
            qryAA_9b = AA_triplets[qry_triple_9b]
            AA_mut_9b = nuc2AA_ORF9b(mut_9b, refAA_9b, qryAA_9b)
            push!(AA_mut_set, AA_mut_9b)
            ref2qry_context_9b = ref_triple_context_9b*"-->"*qry_triple_context_9b
            all_nuc_to_AA_dict[mut_9b] = AA_mut_9b
            if refAA_9b == qryAA_9b && nuc_mut in N_9b_synonymous
                synonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                synonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
            end
            if refAA_9b ≠ qryAA_9b
                nonsynonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                nonsynonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
#                push!(nonsynonymous_nuc_muts, mut_9b)
            end
            all_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
###################################
            for nuc_mut2_9b in muts
                mut2_9b = mixed2nuc(nuc_mut2_9b)
                pos2_9b = nuc_pos(mut2_9b)
                if pos2_9b in rem9b
                    gene_number2_9b = 11
                    if mut_9b ≠ mut2_9b && nuc_to_AA_pos_9b(mut_9b) == nuc_to_AA_pos_9b(mut2_9b)
                        if nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 2
                            ref_triple_9b = string(mut_9b[1])*string(mut2_9b[1])*string(ref_seq[pos_9b+2])
                            qry_triple_9b = string(mut_9b[end])*string(mut2_9b[end])*string(ref_seq[pos_9b+2])
                            ref_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                            qry_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                        elseif nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                            ref_triple_9b = string(mut_9b[1])*string(ref_seq[pos_9b+1])*string(mut2_9b[1])
                            qry_triple_9b = string(mut_9b[end])*string(ref_seq[pos_9b+1])*string(mut2_9b[end])
                            ref_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                            qry_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                        elseif nuc_codon_pos_dict_9b[pos_9b] == 2 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                            ref_triple_9b = string(ref_seq[pos_9b-1])*string(mut_9b[1])*string(mut2_9b[1])
                            qry_triple_9b = string(ref_seq[pos_9b-1])*string(mut_9b[end])*string(mut2_9b[end])
                            ref_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                            qry_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 2
                            ref_triple_9b = string(mut2_9b[1])*string(mut_9b[1])*string(ref_seq[pos_9b+1])
                            qry_triple_9b = string(mut2_9b[end])*string(mut_9b[end])*string(ref_seq[pos_9b+1])
                            ref_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                            qry_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 3
                            ref_triple_9b = string(mut2_9b[1])*string(ref_seq[pos_9b-1])*string(mut_9b[1])
                            qry_triple_9b = string(mut2_9b[end])*string(ref_seq[pos_9b-1])*string(mut_9b[end])
                            ref_triple_context_9b = string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*ref_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])
                            qry_triple_context_9b = string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*qry_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 2 && nuc_codon_pos_dict_9b[pos_9b] == 3
                            ref_triple_9b = string(ref_seq[pos_9b-2])*string(mut2_9b[1])*string(mut_9b[1])
                            qry_triple_9b = string(ref_seq[pos_9b-2])*string(mut2_9b[end])*string(mut_9b[end])
                        end
                        refAA2_9b = AA_triplets[ref_triple_9b]
                        qryAA2_9b = AA_triplets[qry_triple_9b]
                        AA_mut2_9b = nuc2AA_ORF9b(mut2_9b, refAA2_9b, qryAA2_9b)
                        push!(AA_mut_set, AA_mut2_9b)
                        ref2qry_context_9b = ref_triple_context_9b*"-->"*qry_triple_context_9b
                        if refAA2_9b == qryAA2_9b && nuc_mut2_9b in N_9b_synonymous 
                            synonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                            synonymous_nuc_to_context_dict[mut2_9b] = ref2qry_context_9b
                            synonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            synonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
                        else
                            nonsynonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                            nonsynonymous_nuc_to_context_dict[mut2_9b] = ref2qry_context_9b
                            nonsynonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            nonsynonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
                        end
                        all_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
                        all_nuc_to_context_dict[mut2_9b] = ref2qry_context_9b
                    end
                end
            end
        end
    end
#####################################################################################################################################
    function AA_order_key(mutation)
        gene = aa_gene_comprehensive_dict[mutation]
        AApos = aa_pos_comprehensive_dict[mutation]
        gene_pos = gene_print_order[gene]
        return (gene_pos, AApos)
    end
    AA_sort = sort(collect(AA_mut_set), by = x -> AA_order_key(x))
    AA_sort2 = Vector{String}()
    for aa in AA_sort
        aa_refqry = string(split(aa, ":")[2])
        if !(string(aa_refqry[1]) == string(aa_refqry[end]))
            push!(AA_sort2, aa)
        end
    end
    print("\n"^2)
    ct = 0
    println("##################### Amino Acid Mutations ######################")
    for i in 1:length(AA_sort2)
        mut = AA_sort2[i]
        gene = string(split(mut, ":")[1])
        non_gene = string(split(mut, ":")[2])
        if i == 1
            print("               ", AA_sort2[i])
        elseif i > 1
            lastmut = AA_sort2[i-1]
            last_gene = string(split(lastmut, ":")[1])
            last_non_gene = string(split(lastmut, ":")[2])
            if gene == last_gene
                print(", $(non_gene)")
            else
                println()
                print("               $(mut)")
            end
        end
    end
    print("\n"^2)
#####################################################################################################################################
    all_nuc_to_AA_dict_sort = sort(collect(all_nuc_to_AA_dict), by = x -> nuc_pos(x[1]))
    all_nuc_to_context_dict_sort = sort(collect(all_nuc_to_context_dict), by = x -> nuc_pos(x[1]))
    nonsynonymous_nuc_to_AA_dict_sort = sort(collect(nonsynonymous_nuc_to_AA_dict), by = x -> nuc_pos(x[1]))
    nonsynonymous_nuc_sort = sort(collect(keys(nonsynonymous_nuc_to_AA_dict)), by = x -> nuc_pos(x))
    nonsynonymous_AA_sort = sort(collect(values(nonsynonymous_nuc_to_AA_dict)), by = x -> AA_sort_key(x))
    nonsynonymous_nuc_to_context_dict_sort = sort(collect(nonsynonymous_nuc_to_context_dict), by = x -> nuc_pos(x[1]))
    synonymous_nuc_to_AA_dict_sort = sort(collect(synonymous_nuc_to_AA_dict), by = x -> nuc_pos(x[1]))
    synonymous_nuc_sort = sort(collect(keys(synonymous_nuc_to_AA_dict)), by = x -> nuc_pos(x))
    synonymous_AA_sort = sort(collect(values(synonymous_nuc_to_AA_dict)), by = x -> AA_sort_key(x))
    synonymous_nuc_to_context_dict_sort = sort(collect(synonymous_nuc_to_context_dict), by = x -> nuc_pos(x[1]))
    noncoding_nuc_to_context_dict_sort = sort(collect(noncoding_nuc_to_context_dict), by = x -> nuc_pos(x[1]))
    noncoding_nuc_vector_sort = sort(collect(noncoding_nuc_vector), by = x -> nuc_pos(x))
############################################################
    println("################## Noncoding Nuc Mutations ##################")
    noncoding_nuc_total = length(keys(noncoding_nuc_to_context_dict))
    println("          Total Number of Noncoding Nucs = $(noncoding_nuc_total)")
    for nc_mut in noncoding_nuc_to_context_dict_sort
        print("               $(nc_mut[1]) ($(nc_mut[2])), ")
    end
    println()
    for i in 1:length(noncoding_nuc_to_context_dict_sort)
        println("$(noncoding_nuc_to_context_dict_sort[i][1]) | $(noncoding_nuc_to_context_dict_sort[i][2])")
        premut_nucseq = split(noncoding_nuc_to_context_dict_sort[i][2], "|")[1]
        postmut_nucseq = split(noncoding_nuc_to_context_dict_sort[i][2], "|")[2]
        println("    $(premut_nucseq)")
        println("    $(postmut_nucseq)")
    end
    println("################# Synonymous Nuc Mutations #################")
    synonymous_nuc_sort_join = join(synonymous_nuc_sort, ", ")
    println("               ", synonymous_nuc_sort_join)
#    for x in synonymous_nuc_sort)
#        print(x, ", ")
#    end
    print("\n"^1)
    synonymous_nuc_total = length(synonymous_nuc_sort)
    println("          Total Number of Synonymous Nuc Muts = $(synonymous_nuc_total)")
#    for i in 1:length(synonymous_nuc_sort)
#        println("$(synonymous_nuc_to_AA_dict_sort[i][1])|$(synonymous_nuc_to_AA_dict_sort[i][2])|$(synonymous_nuc_to_context_dict_sort[i][2])")
#    end
    for i in 1:length(synonymous_nuc_sort)
        println("               $(synonymous_nuc_to_AA_dict_sort[i][1]) | $(synonymous_nuc_to_AA_dict_sort[i][2])")
        premut_nucseq = split(synonymous_nuc_to_context_dict_sort[i][2], "-->")[1]
        postmut_nucseq = split(synonymous_nuc_to_context_dict_sort[i][2], "-->")[2]
        println("                 $(premut_nucseq)")
        println("                 $(postmut_nucseq)")
    end
    print("\n"^1)
    nonsynonymous_nuc_total = length(nonsynonymous_nuc_sort)
    println("        Total Number of Non-synonymous Nuc Muts = $(nonsynonymous_nuc_total)")
    nonsynonymous_nuc_sort_join = join(nonsynonymous_nuc_sort, ", ")
    println("################ Nonsynonymous Nuc Mutations ################")
    println(nonsynonymous_nuc_sort_join)
    print("\n"^1)
    for i in 1:length(nonsynonymous_nuc_sort)
        println("               $(nonsynonymous_nuc_to_AA_dict_sort[i][1]) | $(nonsynonymous_nuc_to_AA_dict_sort[i][2])")
        premut_nucseq = split(nonsynonymous_nuc_to_context_dict_sort[i][2], "-->")[1]
        postmut_nucseq = split(nonsynonymous_nuc_to_context_dict_sort[i][2], "-->")[2]
        println("                 $(premut_nucseq)")
        println("                 $(postmut_nucseq)")
    end
#synonymous_nuc_to_context_dict[mut] = ref_triple_context*"-->"*qry_triple_context
    return synonymous_nuc_to_AA_dict_sort, nonsynonymous_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort
end
########################################################################################################################################################################
########################################################################################################################################################################
date = "2026_02_07"
ndjson_name = "chronics_2026_02_01__6153seq"
date_pct_DQ_thresh = 1
DQ_mut_thresh = 1
abs_min_mut = 1
global all_chronic_seqs_set = load("2026_04_08__chronics_2026_03_22_6186seq_v5__EPCI_8_10_95__HQCS_5_1_5/2026_04_08__chronics_2026_03_22_6186seq_v5__EPCI_8_10_95__HQCS_5_1_5__all_seqs_set.jld2", "all_seqs_set")
println("all_chronic_seqs_set (1_1_1) = $(length(all_chronic_seqs_set)) seqs"); println()
########################################################################################################################################################################
runtime = time() - start; runtime_rd = round(digits=2, runtime); runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds"); println("Runtime v1 = $(runtime1)"); println("Runtime v2 = $(runtime2)"); println()
########################################################################################################################################################################
########################################################################################################################################################################

In [ ]:
### Fx: all_private_muts_04_19_2026 (MAIN FUNCTION) #########################################
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
function all_private_muts_04_19_2026(input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String)        
    date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
    date = "2026_01_06"
    start_time = time()
    no_key_set = Set{String}()
#########################################################################################
    date_regex = r"(\d{4})-(\d{2})-(\d{2})"
##### Explanation of    date_regex = r"(\d{4})-(\d{2})-(\d{2})"   #####################
##### r = regular expression
##### \d = any digit
##### {4} = number of preceding character to match (in this case the number of digits)
##### () = capture groups. Allow you to recall whatever is inside the parentheses later.
#          If date = "2025-09-01", then  date.capture[1] = "2025", date.capture[2] = "09", etc
##### So this checks the j.seqName line for dates in the correct format
#########################################################################################
    function get_full_year_month_day_tuple(seqName::String)
        ymd = match(date_regex, seqName)
        if ymd ≠ nothing
            year = parse(Int64, ymd.captures[1])
            month = parse(Int64, ymd.captures[2])
            day = parse(Int64, ymd.captures[3])
            if year ≥ 2020 && year ≤ 2027 && month ≥ 1 && month ≤ 12 && day ≥ 1 && day ≤ 31
                return (year, month, day)
            end
        end
        return nothing
    end
    function dash_count(str::String)
        return count(c -> c == '-', str)
    end
#########################################################################################
#                          clade     date_index  count
#    clade_date_index_ct = Dict{String, Dict{Int, Int}}()
#    pango_date_index_ct = Dict{String, Dict{Int, Int}}()
#                                     country    date_index   clade   count  
#    country_clade_date_index_ct = Dict{String, Dict{String, Dict{Int, Int}}}()
#    country_pango_date_index_ct = Dict{String, Dict{String, Dict{Int, Int}}}()
#    for clade in clade_set
#        clade_date_index_ct[clade] = Dict{Int, Int}()
#    end
#    for pango in pango_set
#        pango_date_index_ct[pango] = Dict{Int, Int}()
#    end
#    for clade in clade_set
#        for i in 1:2922
#            clade_date_index_ct[clade][i] = 0
#        end
#    end
#    for pango in pango_set
#        for i in 1:2922
#            pango_date_index_ct[pango][i] = 0
#        end
#    end
#    for country in country_set
#        country_clade_date_index_ct[country] = Dict{String, Dict{Int, Int}}()
#        country_pango_date_index_ct[country] = Dict{String, Dict{Int, Int}}()
#    end
#    for country in country_set
#        for clade in clade_set
#            country_clade_date_index_ct[country][clade] = Dict{Int, Int}()
#        end
#        for pango in pango_set
#            country_pango_date_index_ct[country][pango] = Dict{Int, Int}()
#        end
#    end
#    for country in country_set
#        for clade in clade_set
#            for i in 1:2922
#                country_clade_date_index_ct[country][clade][i] = 0
#            end
#        end
#        for pango in pango_set
#            for i in 1:2922
#                country_pango_date_index_ct[country][pango][i] = 0
#            end
#        end
#    end    
####################################################################################################################################
    excluded_pos = BitSet([1:60..., 76:78..., 686:694..., 2453, 3241, 5515, 7926, 11283:11295..., 14960, 15521, 19209:19210..., 19212, 19214, 19217, 21595, 21766, 21772, 21987, 21995:21996..., 22882, 24130, 27291, 28360:28373..., 29632, 29732:29736..., 29757, 29759:29761..., 29830:29900...])
    excluded_AA = Set(["ORF1a:V3917G", "ORF1a:E4388K", "ORF1b:N498I", "ORF1b:F685Y", "S:S27A", "S:K113N", "S:Q115H", "S:D142G", "S:Y145D", "S:I794N", "S:S408R", "S:K440N"])

    gene_mut_density_sort_by_gene = Vector{Pair{String, Float64}}()
    gene_mut_density_sort_by_density = Vector{Pair{String, Float64}}()
    domain_mut_density_sort_by_gene = Vector{Pair{String, Float64}}()
    domain_mut_density_sort_by_density = Vector{Pair{String, Float64}}()
     
    AA_muts_seq = Dict{String, Set{String}}();            sizehint!(AA_muts_seq, 53578)
#    AA_muts_seq_no_dels = Dict{String, Set{String}}()    sizehint!(AA_muts_seq_no_dels, 50280)
    nuc_muts_seq = Dict{String, Set{String}}();           sizehint!(nuc_muts_seq, 50280)
#    nuc_muts_seq_no_dels = Dict{String, Set{String}}()   sizehint!(nuc_muts_seq_no_dels, 50280)

    nonleap_month_day_dict = Dict{Int, Int}(0=>0, 1=>31, 2=>28, 3=>31, 4=>30, 5=>31, 6=>30, 7=>31, 8=>31, 9=>30, 10=>31, 11=>30, 12=>31)
    leap_month_day_dict = Dict{Int, Int}(0=>0, 1=>31, 2=>29, 3=>31, 4=>30, 5=>31, 6=>30, 7=>31, 8=>31, 9=>30, 10=>31, 11=>30, 12=>31)

#### For dicts below: 1st Key = date index; 2nd Key = mutation, value = mutation count on date index
    date_nuc_mut_ct = Dict{Int, Dict{String, Int}}();                 sizehint!(date_nuc_mut_ct, 2500)
    date_nuc_mut_ct_no_dels = Dict{Int, Dict{String, Int}}();         sizehint!(date_nuc_mut_ct_no_dels, 2500)
    date_AA_mut_ct = Dict{Int, Dict{String, Int}}();                  sizehint!(date_AA_mut_ct, 2500)
    date_AA_mut_ct_no_dels = Dict{Int, Dict{String, Int}}();          sizehint!(date_AA_mut_ct_no_dels, 2500)
    date_AA_mut_ct_pos_only_no_dels = Dict{Int, Dict{String, Int}}(); sizehint!(date_AA_mut_ct_pos_only_no_dels, 2500)

    all_seq_ct::Int = 0
    qualifying_seq_ct::Int = 0

    seq_ct_by_year = Dict{Int, Int}(i=>0 for i in 2020:2027);        sizehint!(seq_ct_by_year, 9)
    seq_ct_by_year[0] = 0
    seq_ct_by_year_month = Dict{Tuple{Int, Int}, Int}();             sizehint!(seq_ct_by_year_month, 105)
    for i in 2020:2027
        for j in 0:12
            seq_ct_by_year_month[(i, j)] = 0
        end
    end
    seq_ct_by_year_month[(0, 0)] = 0
    seq_ct_by_year_month_day = Dict{Tuple{Int, Int, Int}, Int}();    sizehint!(seq_ct_by_year_month_day, 3500)
    for i in 2020:2027
        if i%4 == 0
            for j in 0:12
                for k in 0:leap_month_day_dict[j]
                    seq_ct_by_year_month_day[(i, j, k)] = 0
                end
            end
        else
            for j in 0:12
                for k in 0:nonleap_month_day_dict[j]
                    seq_ct_by_year_month_day[(i, j, k)] = 0
                end
            end
        end
    end
    seq_ct_by_year_month_day[(0, 0, 0)] = 0
###################################################################################################################################
#    seq_country = Dict{String, String}()
#    seq_US_state = Dict{String, String}()
#    seq_clade = Dict{String, String}()
#    seq_pango = Dict{String, String}()
#    seq_collection_date = Dict{String, String}()
    seq_date_index = Dict{String, Int}()
    seq_date_tuple = Dict{String, Tuple{Int64, Int64, Int64}}()
#    seq_lab_dict = Dict{String, String}()
#    seq_nuc_del_ranges_ct = Dict{String, Int}()
###################################################################################################################################
    total_nuc_dels::Int = 0
    total_subs::Int = 0
    total_AA_dels::Int = 0
    total_AA_subs::Int = 0
    nuc_muts_ct = Dict{String, Int}();                 sizehint!(nuc_muts_ct, 92000)
    nuc_dels_ct = Dict{String, Int}();                 sizehint!(nuc_dels_ct, 18000)
    nuc_muts_ct_no_dels = Dict{String, Int}();         sizehint!(nuc_muts_ct_no_dels, 74000)
    nuc_muts_ct_pos_only_no_dels = Dict{Int, Int}();
    AA_muts_ct = Dict{String, Int}();                  sizehint!(AA_muts_ct, 58000)
    AA_dels_ct = Dict{String, Int}();                  sizehint!(AA_dels_ct, 6000)
    AA_muts_ct_no_dels = Dict{String, Int}();          sizehint!(AA_muts_ct_no_dels, 52000) 
    AA_muts_ct_pos_only = Dict{String, Int}();
    AA_muts_ct_pos_only_no_dels = Dict{String, Int}();
    AA_muts_ct_WT = Dict{String, Int}();
    AA_muts_ct_WT_no_dels = Dict{String, Int}();
    AA_muts_ct_WT_pos_only_no_dels = Dict{String, Int}()
#####################################################################################################################################
################################### AA Substitution Counts for Select Protein Domains  ##############################################
#####################################################################################################################################
    NSP3_domains = Set(["NSP3_Ubl1", "NSP3_HVR", "NSP3_Mac1", "NSP3_Mac2", "NSP3_Mac3", "NSP3_DPUP", "NSP3_Ubl2", "NSP3_PLpro", "NSP3_NAB", "NSP3_BSM", "NSP3_TM1", "NSP3_Ecto3", "NSP3_TM234HLX", "NSP3_Y1", "NSP3_CoVY"])
    NSP4_domains = Set(["NSP4_TM1", "NSP4_Ecto4", "NSP4_TM2_TM6", "NSP4_CTD"])
    NSP6_domains = Set(["NSP6_AmphHlx", "NSP6_MAE", "NSP6_cyto_CTD"])
    NSP12_domains = Set(["NSP12_NiRAN", "NSP12_intrfce", "NSP12_fingers", "NSP12_palm", "NSP12_palmLnk", "NSP12_thumb"])
    NSP13_domains = Set(["NSP13_ZBD", "NSP13_stalk", "NSP13_1B", "NSP13_RecA1", "NSP13_RecA2"])
    NSP14_domains = Set(["NSP14_nsp10", "NSP14_EXON", "NSP14_hinge1", "NSP14_hinge2", "NSP14_N7MTase"])
    NSP15_domains = Set(["NSP15_NTD", "NSP15_MD", "NSP15_endoU"])
    S_domains = Set(["S_S1", "S_S2", "S_NTD", "S_N2R", "S_RBD", "S_RBM", "S_SD1", "S_SD2", "S_630_loop", "S_FCS_region", "S_Beta1", "S_3H", "S_IL770", "S_FPPR", "S_FP", "S_HR1", "S_CH", "S_CD", "S_Beta2", "S_2turnHelix", "S_HR2", "S_TM", "S_CT"])
    ORF3a_domains = Set(["ORF3a_SignalP", "ORF3a_NTD", "ORF3a_TM1", "ORF3a_TM12Lnk", "ORF3a_TM2", "ORF3a_TM3", "ORF3a_cytosl1", "ORF3a_Loop", "ORF3a_3DB", "ORF3a_CTD"])
    E_domains = Set(["E_TM", "E_cytosol", "E_CTD"])
    N_domains = Set(["N_N1", "N_N2", "N_N3", "N_N4", "N_N5", "N_SR", "N_L_helix", "N_CBP", "N_9b_overlap"])
####################################################################################################################################
#################################### For AA ranges of each domain + sources, see 2 cells below #####################################
####################################################################################################################################
############ "NSP6_loop"=>75,     "NSP6_loop"=>NSP6_loop_ct,    "NSP6_loop"=>NSP6_loop_ct_no_dels,   "NSP6_loop"=>"NSP6:35-40|61-66|88-112|132-138|157-181|205-210",        "NSP6_loop"=>"BitSet(35:40; 61:66; 88:112; 132:138; 157:181; 205-210),  
############ "NSP6_TM"=>124,      "NSP6_TM"=>NSP6_TM_ct,        "NSP6_TM"=>NSP6_TM_ct_no_dels,       "NSP6_TM"=>"NSP6:12-34|41-60|67-87|113-131|139-156|182-204|211-230",       "NSP6_TM"=>BitSet(12:34; 41:60; 67:87; 113:131; 139:156; 182:204; 211:230),
                            ### AA: 10-27, 43-60, 63-80, 111-128, 130-150, 160-172, 176-190, 209-228 (alternate cource for TM: The multiple roles of nsp6 in the molecular pathogenesis of SARS-CoV-2, Antiviral Research; https://doi.org/10.1016/j.antiviral.2023.105590
#    NSP6_TM          ### AA: 12-34, 41-60, 67-87, 113-131, 139-156, 182-204, 211-230    Source: Feng S, O'Brien A, Chen DY, Saeed M, Baker SC. SARS-CoV-2 nonstructural protein 6 from Alpha to Omicron: evolution of a transmembrane protein. mBio. 2023;14(4):e0068823. doi:10.1128/mbio.00688-23
#    NSP6_loop        ### AA: 35-40, 61-66, 88-112, 132-138, 157-181, 205-210           Source: Feng S, O'Brien A, Chen DY, Saeed M, Baker SC. SARS-CoV-2 nonstructural protein 6 from Alpha to Omicron: evolution of a transmembrane protein. mBio. 2023;14(4):e0068823. doi:10.1128/mbio.00688-23
####################################################################################################################################
    gene_protein_length = Dict{String, Int}("NSP1"=>180, "NSP2"=>638, "NSP3"=>1945, "NSP4"=>500, "NSP5"=>306, "NSP6"=>290, "NSP7"=>83, "NSP8"=>198, "NSP9"=>113, "NSP10"=>139, "NSP12"=>932, "NSP13"=>601, "NSP14"=>527, "NSP15"=>346, "NSP16"=>298, "ORF3a"=>275, "ORF6"=>61, "ORF7a"=>121, "ORF7b"=>43, "ORF8"=>121, "ORF9b"=>97, "S"=>1273, "E"=>75, "M"=>222, "N"=>419)
    gene_protein_mut_ct = Dict{String, Int}("NSP1"=>0, "NSP2"=>0, "NSP3"=>0, "NSP4"=>0, "NSP5"=>0, "NSP6"=>0, "NSP7"=>0, "NSP8"=>0, "NSP9"=>0, "NSP10"=>0, "NSP12"=>0, "NSP13"=>0, "NSP14"=>0, "NSP15"=>0, "NSP16"=>0, "ORF3a"=>0, "ORF6"=>0, "ORF7a"=>0, "ORF7b"=>0, "ORF8"=>0, "ORF9b"=>0, "S"=>0, "E"=>0, "M"=>0, "N"=>0)
    gene_protein_order = Dict{String, Int}("NSP1"=>1, "NSP2"=>2, "NSP3"=>3, "NSP4"=>4, "NSP5"=>5, "NSP6"=>6, "NSP7"=>7, "NSP8"=>8, "NSP9"=>9, "NSP10"=>10, "NSP12"=>12, "NSP13"=>13, "NSP14"=>14, "NSP15"=>15, "NSP16"=>16, "ORF3a"=>17, "ORF6"=>18, "ORF7a"=>19, "ORF7b"=>20, "ORF8"=>21, "ORF9b"=>22, "S"=>23, "E"=>24, "M"=>25, "N"=>26)
####################################################################################################################################
    domain_order = Dict{String, Int}("NSP3_Ubl1"=>1, "NSP3_HVR"=>2, "NSP3_Mac1"=>3, "NSP3_Mac2"=>4, "NSP3_Mac3"=>5, "NSP3_DPUP"=>6, "NSP3_Ubl2"=>7, "NSP3_PLpro"=>8, "NSP3_NAB"=>9, "NSP3_BSM"=>10, "NSP3_TM1"=>11, "NSP3_Ecto3"=>12, "NSP3_TM234HLX"=>13, "NSP3_Y1"=>14, "NSP3_CoVY"=>15, "NSP4_TM1"=>16, "NSP4_Ecto4"=>17, "NSP4_TM2_TM6"=>18, "NSP4_CTD"=>19, "NSP6_AmphHlx"=>20, "NSP6_MAE"=>21, "NSP6_cyto_CTD"=>22, "NSP12_NiRAN"=>23, "NSP12_intrfce"=>24, "NSP12_fingers"=>25, "NSP12_palm"=>26, "NSP12_palmLnk"=>27, "NSP12_thumb"=>28, "NSP13_ZBD"=>29, "NSP13_stalk"=>30, "NSP13_1B"=>31, "NSP13_RecA1"=>32, "NSP13_RecA2"=>33, "NSP14_nsp10"=>34, "NSP14_EXON"=>35, "NSP14_hinge1"=>36, "NSP14_hinge2"=>37, "NSP14_N7MTase"=>38, "NSP15_NTD"=>39, "NSP15_MD"=>40, "NSP15_endoU"=>41, "S_S1"=>42, "S_S2"=>43, "S_NTD"=>44, "S_N2R"=>45, "S_RBD"=>46, "S_RBM"=>47, "S_SD1"=>48, "S_SD2"=>49, "S_630_loop"=>50, "S_FCS_region"=>51, "S_Beta1"=>52, "S_3H"=>53, "S_IL770"=>54, "S_FPPR"=>55, "S_FP"=>56, "S_HR1"=>57, "S_CH"=>58, "S_CD"=>59, "S_Beta2"=>60, "S_2turnHelix"=>61, "S_HR2"=>62, "S_TM"=>63, "S_CT"=>64, "ORF3a_SignalP"=>65, "ORF3a_NTD"=>66, "ORF3a_TM1"=>67, "ORF3a_TM12Lnk"=>68, "ORF3a_TM2"=>69, "ORF3a_TM3"=>70, "ORF3a_cytosl1"=>71, "ORF3a_Loop"=>72, "ORF3a_3DB"=>73, "ORF3a_CTD"=>74, "E_TM"=>75, "E_cytosol"=>76, "E_CTD"=>77, "N_N1"=>78, "N_N2"=>79, "N_N3"=>80, "N_N4"=>81, "N_N5"=>82, "N_SR"=>83, "N_L_helix"=>84, "N_CBP"=>85, "N_9b_overlap"=>86)
    domain_length = Dict{String, Int}("NSP3_Ubl1"=>111, "NSP3_HVR"=>95, "NSP3_Mac1"=>168, "NSP3_Mac2"=>176, "NSP3_Mac3"=>123, "NSP3_DPUP"=>70, "NSP3_Ubl2"=>60, "NSP3_PLpro"=>259, "NSP3_NAB"=>151, "NSP3_BSM"=>131, "NSP3_TM1"=>106, "NSP3_Ecto3"=>59, "NSP3_TM234HLX"=>87, "NSP3_Y1"=>179, "NSP3_CoVY"=>181, "NSP4_TM1"=>31, "NSP4_Ecto4"=>240, "NSP4_TM2_TM6"=>139, "NSP4_CTD"=>90, "NSP6_AmphHlx"=>18, "NSP6_MAE"=>40, "NSP6_cyto_CTD"=>14, "NSP12_NiRAN"=>250, "NSP12_intrfce"=>148, "NSP12_fingers"=>183, "NSP12_palm"=>171, "NSP12_palmLnk"=>60, "NSP12_thumb"=>120, "NSP13_ZBD"=>101, "NSP13_stalk"=>49, "NSP13_1B"=>76, "NSP13_RecA1"=>182, "NSP13_RecA2"=>159, "NSP14_nsp10"=>58, "NSP14_EXON"=>216, "NSP14_hinge1"=>15, "NSP14_hinge2"=>24, "NSP14_N7MTase"=>226, "NSP15_NTD"=>64, "NSP15_MD"=>118, "NSP15_endoU"=>141, "S_S1"=>685, "S_S2"=>587, "S_NTD"=>304, "S_N2R"=>29, "S_RBD"=>194, "S_RBM"=>69, "S_SD1"=>63, "S_SD2"=>83, "S_630_loop"=>24, "S_FCS_region"=>17, "S_Beta1"=>12, "S_3H"=>33, "S_IL770"=>62, "S_FPPR"=>35, "S_FP"=>43, "S_HR1"=>76, "S_CH"=>50, "S_CD"=>33, "S_Beta2"=>9, "S_2turnHelix"=>8, "S_HR2"=>48, "S_TM"=>23, "S_CT"=>39, "ORF3a_SignalP"=>14, "ORF3a_NTD"=>19, "ORF3a_TM1"=>23, "ORF3a_TM12Lnk"=>20, "ORF3a_TM2"=>23, "ORF3a_TM3"=>23, "ORF3a_cytosl1"=>34, "ORF3a_Loop"=>14, "ORF3a_3DB"=>53, "ORF3a_CTD"=>53, "E_TM"=>30, "E_cytosol"=>12, "E_CTD"=>7, "N_N1"=>43, "N_N2"=>131, "N_N3"=>71, "N_N4"=>118, "N_N5"=>55, "N_SR"=>31, "N_L_helix"=>21, "N_CBP"=>33, "N_9b_overlap"=>98)
    domain_mut_ct = Dict{String, Int}("NSP3_Ubl1"=>0, "NSP3_HVR"=>0, "NSP3_Mac1"=>0, "NSP3_Mac2"=>0, "NSP3_Mac3"=>0, "NSP3_DPUP"=>0, "NSP3_Ubl2"=>0, "NSP3_PLpro"=>0, "NSP3_NAB"=>0, "NSP3_BSM"=>0, "NSP3_TM1"=>0, "NSP3_Ecto3"=>0, "NSP3_TM234HLX"=>0, "NSP3_Y1"=>0, "NSP3_CoVY"=>0, "NSP4_TM1"=>0, "NSP4_Ecto4"=>0, "NSP4_TM2_TM6"=>0, "NSP4_CTD"=>0, "NSP6_AmphHlx"=>0, "NSP6_MAE"=>0, "NSP6_cyto_CTD"=>0, "NSP12_NiRAN"=>0, "NSP12_intrfce"=>0, "NSP12_fingers"=>0, "NSP12_palm"=>0, "NSP12_palmLnk"=>0, "NSP12_thumb"=>0, "NSP13_ZBD"=>0, "NSP13_stalk"=>0, "NSP13_1B"=>0, "NSP13_RecA1"=>0, "NSP13_RecA2"=>0, "NSP14_nsp10"=>0, "NSP14_EXON"=>0, "NSP14_hinge1"=>0, "NSP14_hinge2"=>0, "NSP14_N7MTase"=>0, "NSP15_NTD"=>0, "NSP15_MD"=>0, "NSP15_endoU"=>0, "S_S1"=>0, "S_S2"=>0, "S_NTD"=>0, "S_N2R"=>0, "S_RBD"=>0, "S_RBM"=>0, "S_SD1"=>0, "S_SD2"=>0, "S_630_loop"=>0, "S_FCS_region"=>0, "S_Beta1"=>0, "S_3H"=>0, "S_IL770"=>0, "S_FPPR"=>0, "S_FP"=>0, "S_HR1"=>0, "S_CH"=>0, "S_CD"=>0, "S_Beta2"=>0, "S_2turnHelix"=>0, "S_HR2"=>0, "S_TM"=>0, "S_CT"=>0, "ORF3a_SignalP"=>0, "ORF3a_NTD"=>0, "ORF3a_TM1"=>0, "ORF3a_TM12Lnk"=>0, "ORF3a_TM2"=>0, "ORF3a_TM3"=>0, "ORF3a_cytosl1"=>0, "ORF3a_Loop"=>0, "ORF3a_3DB"=>0, "ORF3a_CTD"=>0, "E_TM"=>0, "E_cytosol"=>0, "E_CTD"=>0, "N_N1"=>0, "N_N2"=>0, "N_N3"=>0, "N_N4"=>0, "N_N5"=>0, "N_SR"=>0, "N_L_helix"=>0, "N_CBP"=>0, "N_9b_overlap"=>0)
    domain_residues_NSP = Dict{String, String}("NSP3_Ubl1"=>"NSP3:1-111", "NSP3_HVR"=>"NSP3:112-206", "NSP3_Mac1"=>"NSP3:207-374", "NSP3_Mac2"=>"NSP3:375-551", "NSP3_Mac3"=>"NSP3:552-673", "NSP3_DPUP"=>"NSP3:674-743", "NSP3_Ubl2"=>"NSP3:744-803", "NSP3_PLpro"=>"NSP3:804-1052", "NSP3_NAB"=>"NSP3:1053-1203", "NSP3_BSM"=>"NSP3:1204-1334", "NSP3_TM1"=>"NSP3:1335-1440", "NSP3_Ecto3"=>"NSP3:1441-1499", "NSP3_TM234HLX"=>"NSP3:1500-1586", "NSP3_Y1"=>"NSP3:1587-1764", "NSP3_CoVY"=>"NSP3:1765-1945", "NSP4_TM1"=>"NSP4:1-31", "NSP4_Ecto4"=>"NSP4:32-271", "NSP4_TM2_TM6"=>"NSP4:272-410", "NSP4_CTD"=>"NSP4:411-500", "NSP6_AmphHlx"=>"NSP6:219-236", "NSP6_MAE"=>"NSP6:237-276", "NSP6_cyto_CTD"=>"NSP6:277-290", "NSP12_NiRAN"=>"NSP12:1-250", "NSP12_intrfce"=>"NSP12:251-398", "NSP12_fingers"=>"NSP12:399-581", "NSP12_palm"=>"NSP12:582-627", "NSP12_palmLnk"=>"NSP12:628-687", "NSP12_palm2"=>"NSP12:688-812", "NSP12_thumb"=>"NSP12:813-932", "NSP13_ZBD"=>"NSP13:1-101", "NSP13_stalk"=>"NSP13:102-150", "NSP13_1B"=>"NSP13:151-226", "NSP13_RecA1"=>"NSP13:261-442", "NSP13_RecA2"=>"NSP13:443-601", "NSP14_nsp10"=>"NSP14:1-58", "NSP14_EXON"=>"NSP14:67-282 ", "NSP14_hinge1"=>"NSP14:286-300", "NSP14_hinge2"=>"NSP14:411-434", "NSP14_N7MTase"=>"NSP14:302-527", "NSP15_NTD"=>"NSP15:1-64", "NSP15_MD"=>"NSP15:65-182", "NSP15_endoU"=>"NSP15:207-347", "S_S1"=>"S:2-686", "S_S2"=>"S:687-1273", "S_NTD"=>"S:2-305", "S_N2R"=>"S:306-334", "S_RBD"=>"S:335-528", "S_RBM"=>"S:438-506", "S_SD1"=>"S:529-591", "S_SD2"=>"S:592-674", "S_630_loop"=>"S:619-642", "S_FCS_region"=>"S:675-691", "S_Beta1"=>"S:718-729", "S_3H"=>"S:747-769", "S_IL770"=>"S:770-831", "S_FPPR"=>"S:832-866", "S_FP"=>"S:867-909", "S_HR1"=>"S:910-985", "S_CH"=>"S:986-1035", "S_CD"=>"S:1036-1068", "S_Beta2"=>"S:1127-1135", "S_2turnHelix"=>"S:1148-1155", "S_HR2"=>"S:1164-1211", "S_TM"=>"S:1212-1234", "S_CT"=>"S:1235-1273", "ORF3a_SignalP"=>"ORF3a:1-14", "ORF3a_NTD"=>"ORF3a:15-33", "ORF3a_TM1"=>"ORF3a:34-56", "ORF3a_TM12Lnk"=>"ORF3a:57-76", "ORF3a_TM2"=>"ORF3a:77-99", "ORF3a_TM3"=>"ORF3a:103-125", "ORF3a_cytosl1"=>"ORF3a:126-169", "ORF3a_Loop"=>"ORF3a:170-183", "ORF3a_3DB"=>"ORF3a:170-222", "ORF3a_CTD"=>"ORF3a:223-275", "E_TM"=>"E:8-38", "E_cytosol"=>"E:54-65", "E_CTD"=>"E:69-75", "N_N1"=>"N:2-44", "N_N2"=>"N:45-175", "N_N3"=>"N:176-246", "N_N4"=>"N:247-364", "N_N5"=>"N:365-419", "N_SR"=>"N:176-206", "N_L_helix"=>"N:215-235", "N_CBP"=>"N:247-279", "N_9b_overlap"=>"N:4-101")
    domain_residues_NSP_BitSet = Dict{String, BitSet}("NSP3_Ubl1"=>BitSet(1:111), "NSP3_HVR"=>BitSet(112:206), "NSP3_Mac1"=>BitSet(207:374), "NSP3_Mac2"=>BitSet(375:551), "NSP3_Mac3"=>BitSet(552:673), "NSP3_DPUP"=>BitSet(674:743), "NSP3_Ubl2"=>BitSet(744:803), "NSP3_PLpro"=>BitSet(804:1052), "NSP3_NAB"=>BitSet(1053:1203), "NSP3_BSM"=>BitSet(1204:1334), "NSP3_TM1"=>BitSet(1335:1440), "NSP3_Ecto3"=>BitSet(1441:1499), "NSP3_TM234HLX"=>BitSet(1500:1586), "NSP3_Y1"=>BitSet(1587:1764), "NSP3_CoVY"=>BitSet(1765:1945), "NSP4_TM1"=>BitSet(1:31), "NSP4_Ecto4"=>BitSet(32:271), "NSP4_TM2_TM6"=>BitSet(272:410), "NSP4_CTD"=>BitSet(411:500), "NSP6_AmphHlx"=>BitSet(219:236), "NSP6_MAE"=>BitSet(237:276), "NSP6_cyto_CTD"=>BitSet(277:290), "NSP12_NiRAN"=>BitSet(1:250), "NSP12_intrfce"=>BitSet(251:398), "NSP12_fingers"=>BitSet(399:581), "NSP12_palm"=>BitSet(582:627), "NSP12_palmLnk"=>BitSet(628:687), "NSP12_palm2"=>BitSet(688:812), "NSP12_thumb"=>BitSet(813:932), "NSP13_ZBD"=>BitSet(1:101), "NSP13_stalk"=>BitSet(102:150), "NSP13_1B"=>BitSet(151:226), "NSP13_RecA1"=>BitSet(261:442), "NSP13_RecA2"=>BitSet(443:601), "NSP14_nsp10"=>BitSet(1:58), "NSP14_EXON"=>BitSet(67:282), "NSP14_hinge1"=>BitSet(286:300), "NSP14_hinge2"=>BitSet(411:434), "NSP14_N7MTase"=>BitSet(302:527), "NSP15_NTD"=>BitSet(1:64), "NSP15_MD"=>BitSet(65:182), "NSP15_endoU"=>BitSet(207:347), "S_S1"=>BitSet(2:686), "S_S2"=>BitSet(687:1273), "S_NTD"=>BitSet(2:305), "S_N2R"=>BitSet(306:334), "S_RBD"=>BitSet(335:528), "S_RBM"=>BitSet(438:506), "S_SD1"=>BitSet(529:591), "S_SD2"=>BitSet(592:674), "S_630_loop"=>BitSet(619:642), "S_FCS_region"=>BitSet(675:691), "S_Beta1"=>BitSet(718:729), "S_3H"=>BitSet(747:769), "S_IL770"=>BitSet(770:831), "S_FPPR"=>BitSet(832:866), "S_FP"=>BitSet(867:909), "S_HR1"=>BitSet(910:985), "S_CH"=>BitSet(986:1035), "S_CD"=>BitSet(1036:1068), "S_Beta2"=>BitSet(1127:1135), "S_2turnHelix"=>BitSet(1148:1155), "S_HR2"=>BitSet(1164:1211), "S_TM"=>BitSet(1212:1234), "S_CT"=>BitSet(1235:1273), "ORF3a_SignalP"=>BitSet(1:14), "ORF3a_NTD"=>BitSet(15:33), "ORF3a_TM1"=>BitSet(34:56), "ORF3a_TM12Lnk"=>BitSet(57:76), "ORF3a_TM2"=>BitSet(77:99), "ORF3a_TM3"=>BitSet(103:125), "ORF3a_cytosl1"=>BitSet(126:169), "ORF3a_Loop"=>BitSet(170:183), "ORF3a_3DB"=>BitSet(170:222), "ORF3a_CTD"=>BitSet(223:275), "E_TM"=>BitSet(9:38), "E_cytosol"=>BitSet(54:65), "E_CTD"=>BitSet(69:75), "N_N1"=>BitSet(2:44), "N_N2"=>BitSet(45:175), "N_N3"=>BitSet(176:246), "N_N4"=>BitSet(247:364), "N_N5"=>BitSet(365:419), "N_SR"=>BitSet(176:206), "N_L_helix"=>BitSet(215:235), "N_CBP"=>BitSet(247:279), "N_9b_overlap"=>BitSet(4:101) )
    domain_residues_ORF = Dict{String, String}()
    domain_residues_ORF_BitSet = Dict{String, BitSet}()
####################################################################################################################################
    NSP_AA_size = Dict{String, Int}("NSP1"=>180, "NSP2"=>638, "NSP3"=>1945, "NSP4"=>500, "NSP5"=>306, "NSP6"=>290, "NSP7"=>83, "NSP8"=>198, "NSP9"=>113, "NSP10"=>139, "NSP11"=>0, "NSP12"=>932, "NSP13"=>601, "NSP14"=>527, "NSP15"=>346, "NSP16"=>299)                                                                # "NSP12"=>BitSet(1:923), 
    NSP_ranges = Dict{String, String}("NSP1"=>"ORF1a:1-180", "NSP2"=>"ORF1a:181-818", "NSP3"=>"ORF1a:819-2763", "NSP4"=>"ORF1a:2764-3263", "NSP5"=>"ORF1a:3264-3569", "NSP6"=>"ORF1a:3570-3859", "NSP7"=>"ORF1a:3860-3942", "NSP8"=>"ORF1a:3943-4140", "NSP9"=>"ORF1a:4141-4253", "NSP10"=>"ORF1a:4254-4392", "NSP11"=>"", "NSP12"=>"1a:4393-1b:923", "NSP13"=>"ORF1b:924-1524", "NSP14"=>"ORF1b:1525-2051", "NSP15"=>"ORF1b:2052-2397", "NSP16"=>"ORF1b:2398-2696", "S"=>"S:1-1274", "ORF3a"=>"ORF3a:1-276", "E"=>"E:1-76", "M"=>"M:1-223", "ORF6"=>"ORF6:1-62", "ORF7a"=>"ORF7a:1-122", "ORF7b"=>"ORF7b:1-44", "ORF8"=>"ORF8:1-122", "N"=>"N:1-420", "ORF9b"=>"ORF9b:1-98")
    NSP_ranges_num_only = Dict{String, BitSet}("NSP1"=>BitSet(1:180), "NSP2"=>BitSet(181:818), "NSP3"=>BitSet(819:2763), "NSP4"=>BitSet(2764:3263), "NSP5"=>BitSet(3264:3569), "NSP6"=>BitSet(3570:3859), "NSP7"=>BitSet(3860:3942), "NSP8"=>BitSet(3943:4140), "NSP9"=>BitSet(4141:4253), "NSP10"=>BitSet(4254:4392), "NSP11"=>BitSet(), "NSP12"=>BitSet([4393:4401; 1:923]), "NSP13"=>BitSet(924:1524), "NSP14"=>BitSet(1525:2051), "NSP15"=>BitSet(2052:2397), "NSP16"=>BitSet(2398:2696), "S"=>BitSet(1:1274), "ORF3a"=>BitSet(1:276), "E"=>BitSet(1:76), "M"=>BitSet(1:223), "ORF6"=>BitSet(1:62), "ORF7a"=>BitSet(1:122), "ORF7b"=>BitSet(1:44), "ORF8"=>BitSet(1:122), "N"=>BitSet(1:420), "ORF9b"=>BitSet(1:98) )
    NSP_ranges1a = Dict{Int, BitSet}(1=>BitSet(1:180), 2=>BitSet(181:818), 3=>BitSet(819:2763), 4=>BitSet(2764:3263), 5=>BitSet(3264:3569), 6=>BitSet(3570:3859), 7=>BitSet(3860:3942), 8=>BitSet(3943:4140), 9=>BitSet(4141:4253), 10=>BitSet(4254:4392), 12=>BitSet(4393:4401) )
    NSP_ranges1b = Dict{Int, BitSet}(12=>BitSet(1:923), 13=>BitSet(924:1524), 14=>BitSet(1525:2051), 15=>BitSet(2052:2397), 16=>BitSet(2398:2696) )
    NSP1a_add = Dict{Int, Int}(1=>0, 2=>180, 3=>818, 4=>2763, 5=>3263, 6=>3569, 7=>3859, 8=>3942, 9=>4140, 10=>4253, 11=>0, 12=>4392)
    NSP1b_add = Dict{Int, Int}(12=>-9, 13=>923, 14=>1524, 15=>2051, 16=>2397)
    NSP1ab_add = Dict{Int, Int}(1=>0, 2=>180, 3=>818, 4=>2763, 5=>3263, 6=>3569, 7=>3859, 8=>3942, 9=>4140, 10=>4353, 11=>0, 12=>-9, 13=>923, 14=>1524, 15=>2051, 16=>2397)
#####################################################################################################################################
    function NSP_to_ORF(domain::String, NSP::Int)
        ORFadd = NSP1ab_add[NSP]
        first1 = minimum(domain_residues_NSP_BitSet[domain])
        last1 = maximum(domain_residues_NSP_BitSet[domain])
        first = first1 + ORFadd
        last = last1 + ORFadd
        ORF_range_BitSet = BitSet(first:last)
        firstORF = string(first)
        lastORF = string(last)
        ORF_range = ""
        if NSP < 12
            ORF_range = "ORF1a:$(firstORF)-$(lastORF)"
        else
            ORF_range = "ORF1b:$(firstORF)-$(lastORF)"
        end
        return ORF_range, ORF_range_BitSet
    end
#####################################################################################################################################
    NSP_dom_set_minus_NSP12NiRAN = Set(["NSP3_Ubl1", "NSP3_HVR", "NSP3_Mac1", "NSP3_Mac2", "NSP3_Mac3", "NSP3_DPUP", "NSP3_Ubl2", "NSP3_PLpro", "NSP3_NAB", "NSP3_BSM", "NSP3_TM1", "NSP3_Ecto3", "NSP3_TM234HLX", "NSP3_Y1", "NSP3_CoVY", "NSP4_TM1", "NSP4_Ecto4", "NSP4_TM2_TM6", "NSP4_CTD", "NSP6_AmphHlx", "NSP6_MAE", "NSP6_cyto_CTD", "NSP12_intrfce", "NSP12_fingers", "NSP12_palm", "NSP12_palmLnk", "NSP12_thumb", "NSP13_ZBD", "NSP13_stalk", "NSP13_1B", "NSP13_RecA1", "NSP13_RecA2", "NSP14_nsp10", "NSP14_EXON", "NSP14_hinge1", "NSP14_hinge2", "NSP14_N7MTase", "NSP15_NTD", "NSP15_MD", "NSP15_endoU"])
    non_NSP_dom_set = Set(["S_S1", "S_S2", "S_NTD", "S_N2R", "S_RBD", "S_RBM", "S_SD1", "S_SD2", "S_630_loop", "S_FCS_region", "S_Beta1", "S_3H", "S_IL770", "S_FPPR", "S_FP", "S_HR1", "S_CH", "S_CD", "S_Beta2", "S_2turnHelix", "S_HR2", "S_TM", "S_CT", "ORF3a_SignalP", "ORF3a_NTD", "ORF3a_TM1", "ORF3a_TM12Lnk", "ORF3a_TM2", "ORF3a_TM3", "ORF3a_cytosl1", "ORF3a_Loop", "ORF3a_3DB", "ORF3a_CTD", "E_TM", "E_cytosol", "E_CTD", "N_N1", "N_N2", "N_N3", "N_N4", "N_N5", "N_SR", "N_L_helix", "N_CBP", "N_9b_overlap"])   
    domain_residues_ORF["NSP12_NiRAN"] = "1a:4393-1b:241"
    domain_residues_ORF_BitSet["NSP12_NiRAN"] = BitSet([4393:4401; 1:241])
    for (dom, range) in domain_residues_NSP_BitSet
        NSP_part = string(split(dom, "_")[1])
        if dom in NSP_dom_set_minus_NSP12NiRAN
            if length(NSP_part) == 4
                NSP_num = parse(Int, string(NSP_part[4]))
            elseif length(NSP_part) == 5
                NSP_num = parse(Int, string(NSP_part[4:5]))
            end
            ORF, ORF_BitSet = NSP_to_ORF(dom, NSP_num)
            domain_residues_ORF_BitSet[dom] = ORF_BitSet
            domain_residues_ORF[dom] = ORF
        elseif dom in non_NSP_dom_set
            domain_residues_ORF[dom] = domain_residues_NSP[dom]
            domain_residues_ORF_BitSet[dom] = domain_residues_NSP_BitSet[dom]
        end
    end
#####################################################################################################################################    
    half_million_ct::Int64 = 0
    million_ct::Float16 = 0

    max_dels = 1500
    max_frameshifts = 2
    max_stops = 1
    max_SNPclust = 101
    min_coverage = 0.50

    interval_start = time()
    for line in eachline(input_ndjson)
        all_seq_ct += 1
        if all_seq_ct % 250000 == 0
            sequences_complete = rpad(all_seq_ct, 8)
            half_million_ct += 1
#            million_ct = half_million_ct*0.5
#            println("million ct = $(million_ct)")
            interval_time = round(digits=2, time() - interval_start)
            interval_time1, interval_time2 = seconds_to_hrs_min_sec(runtime_elapsed)
            runtime_elapsed = round(digits=2, time() - start_time)
            runtime1_elapsed, runtime2_elapsed = seconds_to_hrs_min_sec(runtime_elapsed)
            println("Seqs Complete = $(sequences_complete) | Total Time Elapsed = $(runtime2_elapsed) | Interval Time = $(interval_time2)")
            interval_start = time()
            nowtime = Dates.format(now(), "I:MM.SS_p"); print(nowtime); println()
        end
############################################################################
        j = JSON3.read(line)
############################################################################
#        if haskey(j, "privateAaMutations") && haskey(j, "qc") && haskey(j, "totalDeletions") && haskey(j, "privateNucMutations")
#        if haskey(j.privateAaMutations, "ORF1a") && haskey(j.privateAaMutations, "ORF1b") && haskey(j.privateAaMutations, "ORF3a") && haskey(j.privateAaMutations, "ORF6") && haskey(j.privateAaMutations, "ORF7a") && haskey(j.privateAaMutations, "ORF7b") && haskey(j.privateAaMutations, "ORF8") && haskey(j.privateAaMutations, "ORF9b") && haskey(j.privateAaMutations, "S") && haskey(j.privateAaMutations, "E") && haskey(j.privateAaMutations, "M") && haskey(j.privateAaMutations, "N")
        nuc_muts = j.privateNucMutations
        if j.totalDeletions ≥ max_dels
            continue
        end
        QC = j.qc
        if QC.overallScore > qc_max
            continue
        end
        if QC.frameShifts.totalFrameShifts - QC.frameShifts.totalFrameShiftsIgnored > max_frameshifts
            continue
        end
        if QC.stopCodons.totalStopCodons - QC.stopCodons.totalStopCodonsIgnored > max_stops
            continue
        end
        if QC.snpClusters.score ≥ max_SNPclust
            continue
        end
        if j.coverage < min_coverage
            continue
        end
        if nuc_muts.totalReversionSubstitutions > revs_thresh
            continue
        end
############################################################################
        name = EPI_ISL(j.seqName)
        del_ranges = nuc_muts.privateDeletionRanges
        private_del_ct = length(nuc_muts.privateDeletionRanges)
        pango = ""
        privAA = j.privateAaMutations
        ORF1a_AA = privAA.ORF1a
        ORF1b_AA = privAA.ORF1b
        ORF3a_AA = privAA.ORF3a
        ORF6_AA = privAA.ORF6
        ORF7a_AA = privAA.ORF7a
        ORF7b_AA = privAA.ORF7b
        ORF8_AA = privAA.ORF8
        ORF9b_AA = privAA.ORF9b
        S_AA = privAA.S
        E_AA = privAA.E
        M_AA = privAA.M
        N_AA = privAA.N
        total_private_AA_muts = ORF1a_AA.totalPrivateSubstitutions + ORF1b_AA.totalPrivateSubstitutions + ORF3a_AA.totalPrivateSubstitutions + ORF6_AA.totalPrivateSubstitutions + ORF7a_AA.totalPrivateSubstitutions + ORF7b_AA.totalPrivateSubstitutions + ORF8_AA.totalPrivateSubstitutions + ORF9b_AA.totalPrivateSubstitutions + S_AA.totalPrivateSubstitutions + E_AA.totalPrivateSubstitutions + M_AA.totalPrivateSubstitutions + N_AA.totalPrivateSubstitutions
        if total_private_AA_muts + private_del_ct ≤ max_AA_mut && !(name in all_chronic_seqs_set)
            qualifying_seq_ct += 1
#            seq_country[name] = countree(j.seqName)
#            seq_US_state[name] = US_state(j.seqName)
###########################################
#            try
#                seq_clade[name] = j.clade
#            catch
#                seq_clade[name] = "Unknown"
#            end
###########################################
#            try
#                seq_pango[name] = j.customNodeAttributes.Nextclade_pango
#            catch
#                seq_pango[name] = "Unknown"
#            end      
######################################################################################
            seq_date = ""
            seq_year = Int64(0)
            seq_month = Int64(0)
            seq_day = Int64(0)
            date_tuple = get_full_year_month_day_tuple(j.seqName)
            if date_tuple == nothing
                try
                    seq_date = string(sequence_date(j.seqName))
                catch
                    seq_date = "0-0-0"
                end
                dash_ct = dash_count(seq_date)
                if dash_ct == 0
                    seq_y = seq_date
                    if all(isdigit, seq_y)
                        seq_y = parse(Int64, seq_y)
                        if seq_y ≥ 2020 && seq_y ≤ 2027
                            seq_year = seq_y
                        end
                    end
                end
                if dash_ct == 1
                    seq_y = string(split(seq_date, "-")[1])
                    seq_m = string(split(seq_date, "-")[2])
                    if all(isdigit, seq_y)
                        seq_y = parse(Int64, seq_y)
                        if seq_y ≥ 2020 && seq_y ≤ 2027
                            seq_year = seq_y
                        end
                    end
                    if all(isdigit, seq_m)
                        seq_m = parse(Int64, seq_m)
                        if seq_m ≥ 1 && seq_m ≤ 12
                            seq_month = seq_m
                        end
                    end
                end
                if dash_ct == 2
                    seq_y = string(split(seq_date, "-")[1])
                    seq_m = string(split(seq_date, "-")[2])
                    seq_d = string(split(seq_date, "-")[3])
                    if all(isdigit, seq_y)
                        seq_y = parse(Int64, seq_y)
                        if seq_y ≥ 2020 && seq_y ≤ 2027
                            seq_year = seq_y
                        end
                    end
                    if all(isdigit, seq_m)
                        seq_m = parse(Int64, seq_m)
                        if seq_m ≥ 1 && seq_m ≤ 12
                            seq_month = seq_m
                        end
                    end
                    if all(isdigit, seq_d)
                        seq_d = parse(Int64, seq_d)
                        if seq_d ≥ 1 && seq_d ≤ 31
                            seq_day = seq_d
                        end
                    end
                end
                date_tuple = (seq_year, seq_month, seq_day)
            end
            date_index = date_to_index[date_tuple]
            seq_date_index[name] = date_index
            seq_date_tuple[name] = date_tuple
####################################################
#            seq_collection_date[name] = seq_date
#            if seq_month == 0 && seq_day == 0
#                seq_collection_date[name] = string(seq_year)*"-0-0"
#            end
#            if seq_month ≠ 0 && seq_day == 0
#                seq_collection_date[name] = string(seq_year)*"-"*string(seq_month)*"-0"
#            end
####################################################
#            clade = seq_clade[name]
#            pango = seq_pango[name]
###################################
#            clade_date_index_ct[clade][date_index] += 1
#            pango_date_index_ct[pango][date_index] += 1
#            country_clade_date_index_ct[country][clade][date_index] += 1
#            country_pango_date_index_ct[country][pango][date_index] += 1
###############################################################################################
#            lab = ""
#            if !isempty(seq_lab(j.seqName))
#                lab = seq_lab(j.seqName)
#            end
#            seq_lab_dict[name] = lab     
#################################################################################################################################
            genes = ["ORF1a", "ORF1b", "ORF3a", "ORF6", "ORF7a", "ORF7b", "ORF8", "ORF9b", "S", "E", "M", "N"]
            del_ranges = nuc_muts.privateDeletionRanges
            private_del_ct = length(nuc_muts.privateDeletionRanges)
            pango = ""
            privAA = j.privateAaMutations
            ORF1a_AA = privAA.ORF1a
            ORF1b_AA = privAA.ORF1b
            ORF3a_AA = privAA.ORF3a
            ORF6_AA = privAA.ORF6
            ORF7a_AA = privAA.ORF7a
            ORF7b_AA = privAA.ORF7b
            ORF8_AA = privAA.ORF8
            ORF9b_AA = privAA.ORF9b
            S_AA = privAA.S
            E_AA = privAA.E
            M_AA = privAA.M
            N_AA = privAA.N
            AA_subs = Set([ORF1a_AA.privateSubstitutions, ORF1b_AA.privateSubstitutions, ORF3a_AA.privateSubstitutions, ORF6_AA.privateSubstitutions, ORF7a_AA.privateSubstitutions, ORF7b_AA.privateSubstitutions, ORF8_AA.privateSubstitutions, ORF9b_AA.privateSubstitutions, S_AA.privateSubstitutions, E_AA.privateSubstitutions, M_AA.privateSubstitutions, N_AA.privateSubstitutions])
            subs = nuc_muts.privateSubstitutions
            dels = nuc_muts.privateDeletions
            seq_ct_by_year[seq_year] += 1
            seq_ct_by_year_month[(seq_year, seq_month)] += 1
            seq_ct_by_year_month_day[(seq_year, seq_month, seq_day)] += 1
#                  seq_nuc_del_ranges_ct[name] = private_del_count
            for k in subs
                if k.pos < 29870
                    pos = k.pos + 1
                    if !(pos in excluded_pos) && k.refNuc ≠ "-" && k.qryNuc ≠ string(ref[pos])
                        nuc_sub = "$(k.refNuc)$(string(pos))$(k.qryNuc)"
##########################################################################
                        nuc_muts_ct[nuc_sub] = get(nuc_muts_ct, nuc_sub, 0) + 1
##########################################################################
######### Note: While much, much less intuitive, the get! form below is equivalent to the three commented-out lines below it.
#########       Apparently, it is much faster doing it this very unintuitive way, which is extremely annoying. 
                        get!(nuc_muts_seq, nuc_sub, Set{String}() )
#                        if !haskey(nuc_muts_seq, nuc_sub)
#                            nuc_muts_seq[nuc_sub] = Set{String}()
#                        end
                        push!(nuc_muts_seq[nuc_sub], name)
##########################################################################
                        get!(date_nuc_mut_ct, date_index, Dict{String, Int64}() )
                        date_nuc_mut_ct[date_index][nuc_sub] = get(date_nuc_mut_ct[date_index], nuc_sub, 0) + 1
##########################################################################
                        if k.qryNuc ≠ '-'
                            nuc_muts_ct_no_dels[nuc_sub] = get(nuc_muts_ct_no_dels, nuc_sub, 0) + 1
                        end
##########################################################################
                    end
                end
            end
            for d in dels
                pos = d.pos + 1
                if !(pos in excluded_pos)
                    total_nuc_dels += 1
                    n_del = "$(d.refNuc)$(string(pos))-"    
##########################################################################
                    nuc_muts_ct[n_del] = get(nuc_muts_ct, n_del, 0) + 1
                    nuc_dels_ct[n_del] = get(nuc_dels_ct, n_del, 0) + 1
##########################################################################
                    get!(date_nuc_mut_ct, date_index, Dict{String, Int64}() ) 
                    date_nuc_mut_ct[date_index][n_del] = get(date_nuc_mut_ct[date_index], n_del, 0) + 1
##########################################################################
                end
            end
#  AA_subs = Set([j.privateAaMutations.ORF1a.privateSubstitutions, j.privateAaMutations.ORF1b.privateSubstitutions... etc] 
#                     j["privateAaMutations"]["gene"]["privateSubstitutions"][1]["refAa"]
#                     j["privateAaMutations"]["gene"]["privateSubstitutions"][1]["cdsName"]
#                     j["privateAaMutations"]["gene"]["privateSubstitutions"][1]["pos"]
#                     j["privateAaMutations"]["gene"]["privateSubstitutions"][1]["qryAa"]
            for i in AA_subs
                for k in i
                    pos = k.pos + 1
                    AA_sub = "$(k.cdsName):$(k.refAa)$(string(pos))$(k.qryAa)"
                    ref_AA_res = string(gene_AA_dict[k.cdsName][pos])
                    if !(k.refAa == "-") && !(AA_sub in excluded_AA) && k.qryAa ≠ ref_AA_res && k.qryAa ≠ '-'
#######################################################################################################
                        total_AA_subs += 1
#######################################################################################################
                        AA_muts_ct[AA_sub] = get(AA_muts_ct, AA_sub, 0) + 1
#######################################################################################################
                        AA_muts_ct_no_dels[AA_sub] = get(AA_muts_ct_no_dels, AA_sub, 0) + 1
#######################################################################################################
                        get!(date_AA_mut_ct, date_index, Dict{String, Int64}() )
                        date_AA_mut_ct[date_index][AA_sub] = get(date_AA_mut_ct[date_index], AA_sub, 0) + 1
#######################################################################################################
                        get!(AA_muts_seq, AA_sub, Set{String}() )
                        push!(AA_muts_seq[AA_sub], name)
#######################################################################################################
                    end
                end
            end
########
##############################################################################################################################################################################
#            j["privateAaMutations"]["S"]["privateDeletions"][3]  ...etc
#            j["privateAaMutations"]["S"]["privateDeletions"][1]["refAa"]
#            j["privateAaMutations"]["S"]["privateDeletions"][1]["cdsName"]
#            j["privateAaMutations"]["S"]["privateDeletions"][1]["pos"]
            AAdels_tuple = ( ("ORF1a", ORF1a_AA.privateDeletions), ("ORF1b", ORF1b_AA.privateDeletions), 
                ("ORF3a", ORF3a_AA.privateDeletions), ("ORF6", ORF6_AA.privateDeletions), ("ORF7a", ORF7a_AA.privateDeletions), 
                ("ORF7b", ORF7b_AA.privateDeletions), ("ORF8", ORF8_AA.privateDeletions), ("ORF9b", ORF9b_AA.privateDeletions), 
                ("S", S_AA.privateDeletions), ("E", E_AA.privateDeletions), ("M", M_AA.privateDeletions), ("N", N_AA.privateDeletions) ) 
            for (orf_str, dels) in AAdels_tuple
                for del in dels
                    del_AA = "$(orf_str):$(del.refAa)$(del.pos + 1)-"
                    AA_muts_ct[del_AA] = get(AA_muts_ct, del_AA, 0) + 1
                    AA_dels_ct[del_AA] = get(AA_dels_ct, del_AA, 0) + 1
                end
            end
        end
    end
####################################################################################################################################          
#           AA_del_set = Set([[AA_dels_ORF1a, "ORF1a"], [AA_dels_ORF1b, "ORF1b"], [AA_dels_ORF3a, "ORF3a"], [AA_dels_ORF6, "ORF6"], [AA_dels_ORF7a, "ORF7a"], [AA_dels_ORF7b, "ORF7b"], [AA_dels_ORF8, "ORF8"], [AA_dels_ORF9b, "ORF9b"], [AA_dels_S, "S"], [AA_dels_E, "E"], [AA_dels_M, "M"], [AA_dels_N, "N"]])
#            for dels__gene in AA_del_set
#                gene_str = dels__gene[2]
#                dels = dels__gene[1]
#                for del in dels
#                    del_AA = "$(gene_str):$(del.refAa)$(del.pos + 1)-"
#                    AA_muts_ct[del_AA] = get(AA_muts_ct, del_AA, 0) + 1
#                end
#            end
####################################################################################################################################          
#            for i in genes
#                AA_dels = private_AA.i.privateDeletions
#                for h in AA_dels
#                    del_AA = "$(i):$(h.refAa)$(h.pos + 1)-"
#                    AA_muts_ct[del_AA] = get(AA_muts_ct, del_AA, 0) + 1
#                end
#            end
#        end
#    end
####################################################################################################################################
nuc_mut_pos(m) = nuc_mut_int_comprehensive_dict[m]
    for (mut, ct) in nuc_muts_ct
        nuc_pos = nuc_mut_int_comprehensive_dict[mut]
        if mut[end] ≠ '-'
            nuc_muts_ct_no_dels[mut] = ct
            nuc_muts_ct_pos_only_no_dels[nuc_pos] = get(nuc_muts_ct_pos_only_no_dels, nuc_pos, 0) + ct
        end
    end
###################################################################################################################################
    for (mut, ct) in AA_muts_ct
        AA_pos = aa_gene_and_pos_comprehensive_dict[mut]
        AA_muts_ct_pos_only[AA_pos] = get(AA_muts_ct_pos_only, AA_pos, 0) + ct
        if mut[end] ≠ '-'
            AA_muts_ct_no_dels[mut] = ct
            AA_muts_ct_pos_only_no_dels[AA_pos] = get(AA_muts_ct_pos_only_no_dels, AA_pos, 0) + ct
        end
    end
#*#        for (mut, ct) in AA_muts_ct_WT
#*#            AA_pos = aa_gene_and_pos_comprehensive_dict[mut]
#*#            AA_muts_ct_WT_pos_only[AA_pos] = get(AA_muts_ct_WT_pos_only, AA_pos, 0) + ct
#*#            if mut[end] ≠ '-'
#*#                AA_muts_ct_WT_no_dels[mut] = get(AA_muts_ct_WT_no_dels, mut, 0) + ct
#*#                AA_muts_ct_WT_pos_only_no_dels[AA_pos] = get(AA_muts_ct_WT_pos_only_no_dels, AA_pos, 0) + ct
#*#            end
#*#        end
####################################################################################################################################   
    AA_sort_key(m) = (aa_gene_comprehensive_dict[m], aa_pos_comprehensive_dict[m])
    AA_sort_key_pos(m) = (aa_gene_comprehensive_dict[m], aa_pos_comprehensive_dict[m])
    AA_muts_ct_sort = sort(collect(AA_muts_ct), by = x -> AA_sort_key(x[1]))
    AA_muts_ct_sort_by_seq_ct = sort(collect(AA_muts_ct), by = x -> x[2], rev = true)
    AA_muts_ct_no_dels_sort = sort(collect(AA_muts_ct_no_dels), by = x -> AA_sort_key(x[1]))
    AA_muts_ct_no_dels_sort_by_seq_ct = sort(collect(AA_muts_ct_no_dels), by = x -> x[2], rev = true)
    AA_muts_ct_pos_only_sort = sort(collect(AA_muts_ct_pos_only), by = x -> AA_sort_key_pos(x[1]))
    AA_muts_ct_pos_only_sort_by_seq_ct = sort(collect(AA_muts_ct_pos_only), by = x -> x[2], rev = true)
    AA_muts_ct_pos_only_no_dels_sort = sort(collect(AA_muts_ct_pos_only_no_dels), by = x -> AA_sort_key_pos(x[1]))
    AA_muts_ct_pos_only_no_dels_sort_by_seq_ct = sort(collect(AA_muts_ct_pos_only_no_dels), by = x -> x[2], rev = true)

    num_from_key(k) = nuc_mut_int_comprehensive_dict[k]
    nuc_muts_ct_sort = sort(collect(nuc_muts_ct), by = x -> num_from_key(x[1]))
    nuc_muts_ct_sort_by_seq_ct = sort(collect(nuc_muts_ct), by = x -> x[2], rev = true)
    nuc_muts_ct_no_dels_sort = sort(collect(nuc_muts_ct_no_dels), by = x -> num_from_key(x[1]))
    nuc_muts_ct_no_dels_sort_by_seq_ct = sort(collect(nuc_muts_ct_no_dels), by = x -> x[2], rev = true)
####################################################################################################################################
    for (mut, AAmutct) in AA_muts_ct_no_dels
        if gene_name_fx(mut) == "M"
            gene_protein_mut_ct["M"] += AAmutct
        end
    end
    for (mut, AAmutct) in AA_muts_ct_no_dels
        if gene_name_fx(mut) == "ORF6"
            gene_protein_mut_ct["ORF6"] += AAmutct
        end
    end
    for (mut, AAmutct) in AA_muts_ct_no_dels
        if gene_name_fx(mut) == "ORF7a"
            gene_protein_mut_ct["ORF7a"] += AAmutct
        end
    end
    for (mut, AAmutct) in AA_muts_ct_no_dels
        if gene_name_fx(mut) == "ORF7b"
            gene_protein_mut_ct["ORF7b"] += AAmutct
        end
    end
    for (mut, AAmutct) in AA_muts_ct_no_dels
        if gene_name_fx(mut) == "ORF8"
            gene_protein_mut_ct["ORF8"] += AAmutct
        end
    end
    for (mut, AAmutct) in AA_muts_ct_no_dels
        if gene_name_fx(mut) == "ORF9b"
            gene_protein_mut_ct["ORF9b"] += AAmutct
        end
    end
####################################################################################################################################
    for (mut, AAmutct) in AA_muts_ct_no_dels
        if gene_name_fx(mut) == "ORF1a"
            if aa_pos_comprehensive_dict[mut] ≤ 180
                gene_protein_mut_ct["NSP1"] += AAmutct
            elseif aa_pos_comprehensive_dict[mut] in BitSet(181:818)        
                gene_protein_mut_ct["NSP2"] += AAmutct
            elseif aa_pos_comprehensive_dict[mut] in BitSet(819:2763)
                gene_protein_mut_ct["NSP3"] += AAmutct
                for dom in NSP3_domains
                    if aa_pos_comprehensive_dict[mut] in domain_residues_ORF_BitSet[dom]
                        domain_mut_ct[dom] += AAmutct
                    end
                end
            elseif aa_pos_comprehensive_dict[mut] in BitSet(2764:3263)
                gene_protein_mut_ct["NSP4"] += AAmutct
                for dom in NSP4_domains
                    if aa_pos_comprehensive_dict[mut] in domain_residues_ORF_BitSet[dom]
                        domain_mut_ct[dom] += AAmutct
                    end
                end
            elseif aa_pos_comprehensive_dict[mut] in BitSet(3264:3569)
                gene_protein_mut_ct["NSP5"] += AAmutct
            elseif aa_pos_comprehensive_dict[mut] in BitSet(3570:3859)
                gene_protein_mut_ct["NSP6"] += AAmutct
                for dom in NSP6_domains
                    if aa_pos_comprehensive_dict[mut] in domain_residues_ORF_BitSet[dom]
                        domain_mut_ct[dom] += AAmutct
                    end
                end
            elseif aa_pos_comprehensive_dict[mut] in BitSet(3860:3942)
                gene_protein_mut_ct["NSP7"] += AAmutct
            elseif aa_pos_comprehensive_dict[mut] in BitSet(3943:4140)
                gene_protein_mut_ct["NSP8"] += AAmutct
            elseif aa_pos_comprehensive_dict[mut] in BitSet(4141:4253)
                gene_protein_mut_ct["NSP9"] += AAmutct
            elseif aa_pos_comprehensive_dict[mut] in BitSet(4254:4392)
                gene_protein_mut_ct["NSP10"] += AAmutct
            elseif aa_pos_comprehensive_dict[mut] in BitSet(4393:4400)
                gene_protein_mut_ct["NSP12"] += AAmutct
                domain_mut_ct["NSP12_NiRAN"] += AAmutct
            end
        end
        if gene_name_fx(mut) == "ORF1b"
            if aa_pos_comprehensive_dict[mut] ≤ 923
                gene_protein_mut_ct["NSP12"] += AAmutct
                for dom in NSP12_domains
                    if aa_pos_comprehensive_dict[mut] in domain_residues_ORF_BitSet[dom]
                        domain_mut_ct[dom] += AAmutct
                    end
                end
            elseif aa_pos_comprehensive_dict[mut] in BitSet(924:1524)
                gene_protein_mut_ct["NSP13"] += AAmutct
                for dom in NSP13_domains
                    if aa_pos_comprehensive_dict[mut] in domain_residues_ORF_BitSet[dom]
                        domain_mut_ct[dom] += AAmutct
                    end
                end
            elseif aa_pos_comprehensive_dict[mut] in BitSet(1525:2051)
                gene_protein_mut_ct["NSP14"] += AAmutct
                for dom in NSP14_domains
                    if aa_pos_comprehensive_dict[mut] in domain_residues_ORF_BitSet[dom]
                        domain_mut_ct[dom] += AAmutct
                    end
                end
            elseif aa_pos_comprehensive_dict[mut] in BitSet(2052:2397)
                gene_protein_mut_ct["NSP15"] += AAmutct
                for dom in NSP15_domains
                    if aa_pos_comprehensive_dict[mut] in domain_residues_ORF_BitSet[dom]
                        domain_mut_ct[dom] += AAmutct
                    end
                end
            elseif aa_pos_comprehensive_dict[mut] ≥ 2398
                gene_protein_mut_ct["NSP16"] += AAmutct
            end
        end
        if gene_name_fx(mut) == "S"
            gene_protein_mut_ct["S"] += AAmutct
            for dom in S_domains
                if aa_pos_comprehensive_dict[mut] in domain_residues_ORF_BitSet[dom]
                    domain_mut_ct[dom] += AAmutct
                end
            end
        end
        if gene_name_fx(mut) == "ORF3a"
            gene_protein_mut_ct["ORF3a"] += AAmutct
            for dom in ORF3a_domains
                if aa_pos_comprehensive_dict[mut] in domain_residues_ORF_BitSet[dom]
                    domain_mut_ct[dom] += AAmutct
                end
            end
        end
        if gene_name_fx(mut) == "E"
            gene_protein_mut_ct["E"] += AAmutct
            for dom in E_domains
                if aa_pos_comprehensive_dict[mut] in domain_residues_ORF_BitSet[dom]
                    domain_mut_ct[dom] += AAmutct
                end
            end
        end
        if gene_name_fx(mut) == "N"
            gene_protein_mut_ct["N"] += AAmutct
            for dom in N_domains
                if aa_pos_comprehensive_dict[mut] in domain_residues_ORF_BitSet[dom]
                    domain_mut_ct[dom] += AAmutct
                end
            end
        end
    end
#####################################################################################################################################
    gene_set = Set(["NSP1", "NSP2", "NSP3", "NSP4", "NSP5", "NSP6", "NSP7", "NSP8", "NSP9", "NSP10", "NSP12", "NSP13", "NSP14", "NSP15", "NSP16", "ORF3a", "ORF6", "ORF7a", "ORF7b", "ORF8", "ORF9b", "S", "E", "M", "N"])
    gene_mut_density = Dict{String, Float64}()
#####################################################################################################################################    
    domain_set = Set(["NSP3_Ubl1", "NSP3_HVR", "NSP3_Mac1", "NSP3_Mac2", "NSP3_Mac3", "NSP3_DPUP", "NSP3_Ubl2", "NSP3_PLpro", "NSP3_NAB", "NSP3_BSM", "NSP3_TM1", "NSP3_Ecto3", "NSP3_TM234HLX", "NSP3_Y1", "NSP3_CoVY", "NSP4_TM1", "NSP4_Ecto4", "NSP4_TM2_TM6", "NSP4_CTD", "NSP6_AmphHlx", "NSP6_MAE", "NSP6_cyto_CTD", "NSP12_NiRAN", "NSP12_intrfce", "NSP12_fingers", "NSP12_palm", "NSP12_palmLnk", "NSP12_thumb", "NSP13_ZBD", "NSP13_stalk", "NSP13_1B", "NSP13_RecA1", "NSP13_RecA2", "NSP14_nsp10", "NSP14_EXON", "NSP14_hinge1", "NSP14_hinge2", "NSP14_N7MTase", "NSP15_NTD", "NSP15_MD", "NSP15_endoU", "S_S1", "S_S2", "S_NTD", "S_N2R", "S_RBD", "S_RBM", "S_SD1", "S_SD2", "S_630_loop", "S_FCS_region", "S_Beta1", "S_3H", "S_IL770", "S_FPPR", "S_FP", "S_HR1", "S_CH", "S_CD", "S_Beta2", "S_2turnHelix", "S_HR2", "S_TM", "S_CT", "ORF3a_SignalP", "ORF3a_NTD", "ORF3a_TM1", "ORF3a_TM12Lnk", "ORF3a_TM2", "ORF3a_TM3", "ORF3a_cytosl1", "ORF3a_Loop", "ORF3a_3DB", "ORF3a_CTD", "E_TM", "E_cytosol", "E_CTD", "N_N1", "N_N2", "N_N3", "N_N4", "N_N5", "N_SR", "N_L_helix", "N_CBP", "N_9b_overlap"])
    domain_mut_density = Dict{String, Float64}()
#####################################################################################################################################    
    gene_sort(n) = gene_protein_order[n]
    protein_dom_sort(n) = domain_order[n]
    open("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_printMin$(print_ct_thresh)_GENE_MUT_DENSITY.txt", "w") do g
        print(g, "\n"^2)
        for geen in gene_set
            gene_mut_density[geen] = round(digits=2, gene_protein_mut_ct[geen]/gene_protein_length[geen])
        end
        for dom in domain_set
            domain_mut_density[dom] = round(digits=2, domain_mut_ct[dom]/domain_length[dom])
        end
        gene_mut_density_sort_by_gene = sort(collect(gene_mut_density), by = x -> gene_sort(x[1]))
        gene_mut_density_sort_by_density = sort(collect(gene_mut_density), by = x -> x[2], rev=true)
        domain_mut_density_sort_by_gene = sort(collect(domain_mut_density), by = x -> protein_dom_sort(x[1]))
        domain_mut_density_sort_by_density = sort(collect(domain_mut_density), by = x -> x[2], rev=true)
        println(g, "*************** Protein Mut Density, Sorted by Gene ***************")
        for (prot, density) in gene_mut_density_sort_by_gene
            println(g, "                 ", rpad("$(prot)", 5), " = ", lpad("$(density)", 5), "   ", rpad("$(NSP_ranges[prot])", 16) )
        end
        println(g)
        println(g, "*************** Protein Mut Density, Sorted by Mut Density ***************")
        for (prot, density) in gene_mut_density_sort_by_density
            println(g, "                  ", rpad("$(prot)", 5), " = ", lpad("$(density)", 5), "   ", rpad("$(NSP_ranges[prot])", 16) )
        end;  print(g, "\n"^2)
#     println(g, "         ", rpad("$(gene)", 17), " = ", lpad("$(ratio)", 5), "   ", lpad("$(domain_residues_NSP[gene])", 16), "  ", rpad("$(domain_residues_ORF[gene])", 17) )
        println(g, "*************** Protein Domain Mut Density, Sorted by Gene, Domain ***************")
        for (domain, density) in domain_mut_density_sort_by_gene
            println(g, "        ", rpad("$(domain)", 17), " = ", lpad("$(density)", 5), "  ", lpad("$(domain_residues_NSP[domain])", 16), "  ", lpad("$(domain_residues_ORF[domain])", 17) )
        end
        println(g)
        println(g, "*************** Protein Domain Mut Density, Sorted by Mut Density ***************")
        for (domain, density) in domain_mut_density_sort_by_density
            println(g, "        ", rpad("$(domain)", 17), " = ", lpad("$(density)", 5), "  ", lpad("$(domain_residues_NSP[domain])", 16), "  ", lpad("$(domain_residues_ORF[domain])", 17) )
        end;  print(g, "\n"^2)
    end
####################################################################################################################################
    for (date, nuc_mut_ct_dict) in date_nuc_mut_ct
        for (mut, ct) in nuc_mut_ct_dict
            get!(date_nuc_mut_ct_no_dels, date, Dict{String, Int}() )
            if mut[end] ≠ '-'
                date_nuc_mut_ct_no_dels[date][mut] = ct
            end
        end
    end
    for (date, AA_mut_ct_dict) in date_AA_mut_ct
        for (mut, ct) in AA_mut_ct_dict
            AA_pos = aa_gene_and_pos_comprehensive_dict[mut]
            if mut[end] ≠ '-'
                get!(date_AA_mut_ct_no_dels, date, Dict{String, Int}() )
                date_AA_mut_ct_no_dels[date][mut] = ct
                get!(date_AA_mut_ct_pos_only_no_dels, date, Dict{String, Int}() )
####################################################
                date_AA_mut_ct_pos_only_no_dels[date][AA_pos] = get(date_AA_mut_ct_pos_only_no_dels[date], AA_pos, 0) + ct
####################################################
            end
        end
    end
#####################################################################################################################################
    date = Dates.format(today(), "yyyy_mm_dd")
    open("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_printMin$(print_ct_thresh).txt", "w") do g
        println(g); println(g, "************** TOP Nuc MUTATIONS ***************"); println(g)
        pad = 7
        for w in nuc_muts_ct_sort_by_seq_ct
            if w[2] ≥ print_ct_thresh
                mutpad = rpad(w[1], 7);  ctpad = lpad(w[2], 6);  println(g, mutpad, " = ", ctpad)
            end
        end;  print(g, "\n"^2)
        println(g, "************** Nuc MUTS Sorted by Position ***************"); println(g)
        for w in nuc_muts_ct_sort
            if w[2] ≥ print_ct_thresh
                mutpad = rpad(w[1], 7);  ctpad = lpad(w[2], 6);  println(g, mutpad, " = ", ctpad)
            end
        end;  print(g, "\n"^2)
        println(g, "************** TOP AA MUTATIONS ***************"); println(g)
        for w in AA_muts_ct_sort_by_seq_ct
            if w[2] ≥ print_ct_thresh
                mutpad = rpad(w[1], 12);  ctpad = lpad(w[2], 6);  println(g, mutpad, " = ", ctpad)
            end
        end;  print(g, "\n"^2)
        println(g, "************** AA MUTS Sorted by Position ***************"); println(g)
        for w in AA_muts_ct_sort
            if w[2] ≥ print_ct_thresh
                mutpad = rpad(w[1], 12);  ctpad = lpad(w[2], 6);  println(g, mutpad, " = ", ctpad)
            end
        end;  print(g, "\n"^2)
        println(g, "************** TOP AA MUTATIONS BY POSITION ONLY ***************"); println(g)
        for w in AA_muts_ct_pos_only_sort_by_seq_ct
            if w[2] ≥ print_ct_thresh
                mutpad = rpad(w[1], 12);  ctpad = lpad(w[2], 6);  println(g, mutpad, " = ", ctpad)
            end
        end
    end
    avg_private_AA_per_circ_seq = total_AA_subs/qualifying_seq_ct
    AA_no_dels_sub_count = 0
    for (mut, AAmutct) in AA_muts_ct_no_dels
        AA_no_dels_sub_count += AAmutct
    end
    avg_private_AA_per_circ_seq2 = AA_no_dels_sub_count/qualifying_seq_ct
####################################################################################################################################
    date_index_seq_ct = Dict{Int, Int}()
    for d_index in values(seq_date_index)
        date_index_seq_ct[d_index] = get!(date_index_seq_ct, d_index, 0) + 1
    end
####################################################################################################################################
#    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_clade_set.jld2", clade_set)
#    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_pango_set.jld2", pango_set)
#    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_country_set.jld2", country_set)
#    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_lab_set.jld2", seq_lab_set)
####################################################################################################################################
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_clade_date_index_ct.jld2", "clade_date_index_ct", clade_date_index_ct)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_pango_date_index_ct.jld2", "pango_date_index_ct", pango_date_index_ct)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_country_clade_date_index_ct.jld2", "country_clade_date_index_ct", country_clade_date_index_ct)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_country_pango_date_index_ct.jld2", "country_pango_date_index_ct", country_pango_date_index_ct)
####################################################################################################################################
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_no_dels_sub_count.jld2", AA_no_dels_sub_count)
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_total_AA_subs.jld2", total_AA_subs)    
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_avg_private_AA_per_circ_seq.jld2", avg_private_AA_per_circ_seq)
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_avg_private_AA_per_circ_seq2.jld2", avg_private_AA_per_circ_seq2)
####################################################################################################################################
#    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_multi_ORF9b_CTD_ct.jld2", multi_ORF9b_CTD_ct)
#    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_qualifying_ORF9b_double_ct.jld2", qualifying_ORF9b_double_ct)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_ORF9b_CTD_muts_seq.jld2", "ORF9b_CTD_muts_seq", ORF9b_CTD_muts_seq)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_multi_ORF9b_CTD.jld2", "multi_ORF9b_CTD", multi_ORF9b_CTD)
####################################################################################################################################
#    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_multi_ORF9b_CTD_ct_relaxed_qc.jld2", multi_ORF9b_CTD_ct_relaxed_qc)
#    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_qualifying_ORF9b_double_ct_relaxed_qc.jld2", qualifying_ORF9b_double_ct_relaxed_qc)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_ORF9b_CTD_muts_seq_relaxed_qc.jld2", "ORF9b_CTD_muts_seq_relaxed_qc", ORF9b_CTD_muts_seq_relaxed_qc)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_multi_ORF9b_CTD_relaxed_qc.jld2", "multi_ORF9b_CTD_relaxed_qc", multi_ORF9b_CTD_relaxed_qc)
####################################################################################################################################
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_ct_by_year.jld2", "seq_ct_by_year", seq_ct_by_year)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_ct_by_year_month.jld2", "seq_ct_by_year_month", seq_ct_by_year_month)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_ct_by_year_month_day.jld2", "seq_ct_by_year_month_day", seq_ct_by_year_month_day)
####################################################################################################################################
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_nuc_mut_ct.jld2", "date_nuc_mut_ct", date_nuc_mut_ct)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_nuc_mut_ct_no_dels.jld2", "date_nuc_mut_ct_no_dels", date_nuc_mut_ct_no_dels)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_AA_mut_ct.jld2", "date_AA_mut_ct", date_AA_mut_ct)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_AA_mut_ct_no_dels.jld2", "date_AA_mut_ct_no_dels", date_AA_mut_ct_no_dels)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_AA_mut_ct_pos_only_no_dels.jld2", "date_AA_mut_ct_pos_only_no_dels", date_AA_mut_ct_pos_only_no_dels)
####################################################################################################################################
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_country.jld2", "seq_country", seq_country)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_US_state.jld2", "seq_US_state", seq_US_state)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_clade.jld2", "seq_clade", seq_clade)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_pango.jld2", "seq_pango", seq_pango)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_collection_date.jld2", "seq_collection_date", seq_collection_date)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_date_index.jld2", "seq_date_index", seq_date_index)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_date_tuple.jld2", "seq_date_tuple", seq_date_tuple)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_lab_dict.jld2", "seq_lab_dict", seq_lab_dict)
####################################################################################################################################    
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_nuc_del_ranges_ct.jld2", "seq_nuc_del_ranges_ct", seq_nuc_del_ranges_ct)
####################################################################################################################################
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_seq.jld2", "AA_muts_seq", AA_muts_seq)
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_all_seq_ct.jld2", all_seq_ct)
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_qualifying_seq_ct.jld2", qualifying_seq_ct)
####################################################################################################################################
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct.jld2", "nuc_muts_ct", nuc_muts_ct)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_dels_ct.jld2", "nuc_dels_ct", nuc_dels_ct)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_no_dels.jld2", "nuc_muts_ct_no_dels", nuc_muts_ct_no_dels)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct.jld2", "AA_muts_ct", AA_muts_ct)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_dels_ct.jld2", "AA_dels_ct", AA_dels_ct)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only.jld2", "AA_muts_ct_pos_only", AA_muts_ct_pos_only)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_no_dels.jld2", "AA_muts_ct_no_dels", AA_muts_ct_no_dels)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_no_dels.jld2", "AA_muts_ct_pos_only_no_dels", AA_muts_ct_pos_only_no_dels)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_gene_mut_density.jld2", "gene_mut_density", gene_mut_density)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_domain_mut_density.jld2", "domain_mut_density", domain_mut_density)
####################################################################################################################################
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_sort.jld2", nuc_muts_ct_sort)
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_sort_by_seq_ct.jld2", nuc_muts_ct_sort_by_seq_ct)    
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_no_dels_sort.jld2", nuc_muts_ct_no_dels_sort)
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_no_dels_sort_by_seq_ct.jld2", nuc_muts_ct_no_dels_sort_by_seq_ct)
####################################################################################################################################
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_sort.jld2", AA_muts_ct_sort)
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_sort_by_seq_ct.jld2", AA_muts_ct_sort_by_seq_ct)
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_no_dels_sort.jld2", AA_muts_ct_no_dels_sort)
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_no_dels_sort_by_seq_ct.jld2", AA_muts_ct_no_dels_sort_by_seq_ct)
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_sort.jld2", AA_muts_ct_pos_only_sort)
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_sort_by_seq_ct.jld2", AA_muts_ct_pos_only_sort_by_seq_ct)
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_no_dels_sort.jld2", AA_muts_ct_pos_only_no_dels_sort)
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_no_dels_sort_by_seq_ct.jld2", AA_muts_ct_pos_only_no_dels_sort_by_seq_ct)
####################################################################################################################################
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_gene_mut_density_sort_by_gene.jld2", gene_mut_density_sort_by_gene)
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_gene_mut_density_sort_by_density.jld2", gene_mut_density_sort_by_density)
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_domain_mut_density_sort_by_gene.jld2", domain_mut_density_sort_by_gene)
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_domain_mut_density_sort_by_density.jld2", domain_mut_density_sort_by_density)
####################################################################################################################################
    println("Dictionaries saved!")
    total_function_time = time() - start_time
    total_function_time_rd = round(digits=1, total_function_time)
    open("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_printMin$(print_ct_thresh)_RUNTIME.txt", "w") do g
        hours = total_function_time÷3600
        hours_int = Int(hours)
        hours_int_fin = split(string(hours_int), ".")[1]
        minutes = (total_function_time%3600)÷60
        minutes_int = Int(minutes)
        minutes_int2 = lpad(minutes_int, 2, "0")
        seconds = (total_function_time%3600)%60
        seconds_rd = round(digits=1, seconds)
        println()
        println("Total Function Run Time = ", total_function_time_rd, " seconds")
        println("$(hours) hours, $(minutes) minutes, $(seconds_rd) seconds")
        println("$(hours).$(minutes).$(seconds_rd)")
        print("\n"^3)
        println(g)
        println(g, "Total Function Run Time = ", total_function_time_rd, " seconds");   println(g)
        println(g, "$(hours_int) hours, $(minutes_int) minutes, $(seconds_rd) seconds");   println(g)
        println(g, "$(hours_int).$(minutes_int2).$(seconds_rd)");   print(g, "\n"^4)
    end
end
################################################################################################################

In [ ]:
### Execute all_private_muts_04_19_2026 | 5, 1, 5 | Runtime = 
start = time()
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
date = Dates.format(today(), "yyyy_mm_dd")
date = "2026_01_06"
### Execute all_private_muts_04_19_2026, 1 hour, 45 minutes ###################### 
all_private_muts_04_19_2026("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 5, 1, 5, 2, "all_private_muts")
#                                                        input_ndjson, ndjson_name,          max_AA_mut, revs_thresh, qc_max, print_ct_thresh,      fx_name
###
### Total Function Run Time = 
### Total Function Run Time = 10774.9 seconds = 2 hr, 59 min, 34.9 sec (2025-11-12)
### Total Function Run Time =  7579.4 seconds = 2 hr,  6 min, 19.4 sec (date??)
### Total Function Run Time =  7139.9 seconds = 1 hr, 58 min, 59.9 sec (2025-09-01)
### Total Function Run Time =  7168.3 seconds = 1 hr, 59 min, 28.3 sec (2025-8-20)
### Total Function Run Time = 16842.4 seconds = 4 hr, 40 min, 42.0 sec = 4:40.42
# input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String        
#####################################################################################################################################

In [ ]:
### Execute all_private_muts_04_19_2026 | 7, 1, 30 | Runtime = 
start = time()
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
date = Dates.format(today(), "yyyy_mm_dd")
date = "2026_01_06"
### Execute all_private_muts_04_19_2026, 1 hour, 45 minutes ###################### 
all_private_muts_04_19_2026("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 7, 1, 30, 1, "all_private_muts")
#                                                        input_ndjson, ndjson_name,          max_AA_mut, revs_thresh, qc_max, print_ct_thresh,      fx_name
#
### Total Function Run Time =  
### Total Function Run Time = 
### Total Function Run Time =  9141.0 seconds =  2 hours, 32 minutes, 21.0 seconds (2025-09-03, QC——7, 1, 10)
### Total Function Run Time = 17392.0 seconds =  4 hours, 49 minutes, 52.0 seconds (2025-09-02, QC——90, 5, 8000)
### Total Function Run Time =  7168.3 seconds =  1 hour,  59 minutes, 28.3 seconds (2025-8-20)
### Total Function Run Time = 16842.4 seconds = 4 hours, 40 minutes, 42 seconds = 4:40.42
# input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String        
#####################################################################################################################################

In [ ]:
### Execute all_private_muts_04_19_2026 | 10, 1, 30 | Runtime = 4.0 hours, 29.0 minutes, 48.1 seconds
start = time()
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
#date = Dates.format(today(), "yyyy_mm_dd")
date = "2026_01_06"
### Execute all_private_muts_04_19_2026, 1 hour, 45 minutes ###################### 
all_private_muts_04_19_2026("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 10, 1, 30, 2, "all_private_muts")
#                                                        input_ndjson, ndjson_name,          max_AA_mut, revs_thresh, qc_max, print_ct_thresh,      fx_name
#
### Total Function Run Time =  
### Total Function Run Time =  
### Total Function Run Time =  9141.0 seconds =  2 hours, 32 minutes, 21.0 seconds (2025-09-03, QC——7, 1, 10)
### Total Function Run Time = 17392.0 seconds =  4 hours, 49 minutes, 52.0 seconds (2025-09-02, QC——90, 5, 8000)
# input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String        
#####################################################################################################################################

In [ ]:
### Execute all_private_muts_04_19_2026 | 90, 5, 8000, 2 | Runtime = 5 hr, 55 min, 16.0 sec (2026-01-07)
start = time()
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
date = Dates.format(today(), "yyyy_mm_dd")
all_private_muts_04_19_2026("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 90, 5, 8000, 2, "all_private_muts")
#                                                        input_ndjson, ndjson_name,          max_AA_mut, revs_thresh, qc_max, print_ct_thresh,      fx_name
### Total Function Run Time = 
### Total Function Run Time = 
### Total Function Run Time = 21316.0 seconds  = 5 hours, 55 minutes, 16.0 seconds (2026-01-07, QC——90, 5, 8000)
### Total Function Run Time = 28544.6 seconds =  7 hours, 55 minutes, 44.6 seconds (2025-11-12, QC——90, 5, 8000)
### Total Function Run Time = 30059.3 seconds =  8 hours, 20 minutes, 59.3 seconds (2025-09-28, QC——90, 5, 8000)
### Total Function Run Time =  9141.0 seconds =  2 hours, 32 minutes, 21.0 seconds (2025-09-03, QC——7, 1, 10)
### Total Function Run Time = 17392.0 seconds =  4 hours, 49 minutes, 52.0 seconds (2025-09-02, QC——90, 5, 8000)
### Total Function Run Time =  7168.3 seconds =  1 hour,  59 minutes, 28.3 seconds (2025-8-20)
### Total Function Run Time = 16842.4 seconds =  4 hours, 40 minutes, 42   seconds = 4:40.42
# input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String        
#####################################################################################################################################

In [ ]:
### Protein Domain ranges & sources######################################################
    #    NSP3_Ubl1      ### NSP3:1-111
    #    NSP3_HVR       ### NSP3:112-206
    #    NSP3_Mac1      ### NSP3:207-374
    #    NSP3_Mac2      ### NSP3:375-551
    #    NSP3_Mac3      ### NSP3:552-673
    #    NSP3_DPUP      ### NSP3:674-743
    #    NSP3_Ubl2      ### NSP3:744-803
    #    NSP3_PLpro     ### NSP3:804-1052
    #    NSP3_NAB       ### NSP3:1053-1203
    #    NSP3_BSM       ### NSP3:1204-1334
    #    NSP3_TM1       ### NSP3:1335-1440
    #    NSP3_Ecto3     ### NSP3:1441-1499
    #    NSP3_TM234HLX  ### NSP3:1500-1586
    #    NSP3_Y1        ### NSP3:1587-1764
    #    NSP3_CoVY      ### NSP3:1765-1945
####################################################################################################################################
############ "NSP6_loop"=>75,     "NSP6_loop"=>NSP6_loop_ct,    "NSP6_loop"=>NSP6_loop_ct_no_dels,   "NSP6_loop"=>"NSP6:35-40|61-66|88-112|132-138|157-181|205-210",        "NSP6_loop"=>"BitSet(35:40; 61:66; 88:112; 132:138; 157:181; 205-210),  
############ "NSP6_TM"=>124,      "NSP6_TM"=>NSP6_TM_ct,        "NSP6_TM"=>NSP6_TM_ct_no_dels,       "NSP6_TM"=>"NSP6:12-34|41-60|67-87|113-131|139-156|182-204|211-230",       "NSP6_TM"=>BitSet(12:34; 41:60; 67:87; 113:131; 139:156; 182:204; 211:230),
                            ### AA: 10-27, 43-60, 63-80, 111-128, 130-150, 160-172, 176-190, 209-228 (alternate cource for TM: The multiple roles of nsp6 in the molecular pathogenesis of SARS-CoV-2, Antiviral Research; https://doi.org/10.1016/j.antiviral.2023.105590
#    NSP6_TM          ### AA: 12-34, 41-60, 67-87, 113-131, 139-156, 182-204, 211-230    Source: Feng S, O'Brien A, Chen DY, Saeed M, Baker SC. SARS-CoV-2 nonstructural protein 6 from Alpha to Omicron: evolution of a transmembrane protein. mBio. 2023;14(4):e0068823. doi:10.1128/mbio.00688-23
#    NSP6_loop        ### AA: 35-40, 61-66, 88-112, 132-138, 157-181, 205-210           Source: Feng S, O'Brien A, Chen DY, Saeed M, Baker SC. SARS-CoV-2 nonstructural protein 6 from Alpha to Omicron: evolution of a transmembrane protein. mBio. 2023;14(4):e0068823. doi:10.1128/mbio.00688-23
####################################################################################################################################
#    NSP6_AmphHlx  ### Amphipathic Helix: 219-236   Source: Ricciardi S, Guarino AM, Giaquinto L, et al. The role of NSP6 in the biogenesis of the SARS-CoV-2 replication organelle. Nature. 2022;606(7915):761-768. doi:10.1038/s41586-022-04835-6
#    NSP6_MAE         ### membrane-associated element: 237-276      Source: Han Y, Yuan Z, Yi Z. Identification of a membrane-associated element (MAE) in the C-terminal region of SARS-CoV-2 nsp6 that is essential for viral replication. J Virol. 2024;98(5):e0034924. doi:10.1128/jvi.00349-24
#    NSP6_cyto_CTD    ### Cytoplasmic CTD: 277-290
####################################################################################################################################
#    NSP12_NiRAN      ### NSP12:1-250    ORF1a:4393-4401 + ORF1b:1-241
#    NSP12_intrfce    ### NSP12:251-398  ORF1b:242-389
#    NSP12_fingers    ### NSP12:399-581  ORF1b:390-572
#    NSP12_palm       ### NSP12:582-627  ORF1b:573-618
#    NSP12_palmLnk    ### NSP12:628-687  ORF1b:619-678
#    NSP12_palm2      ### NSP12:688-812  ORF1b:679-803
#    NSP12_thumb      ### NSP12:813-932  ORF1b:804-923
#####################################################################################################################################
#    NSP13_ZBD     ### NSP13:1-101    ORF1b:924-1024
#    NSP13_stalk   ### NSP13:102-150  ORF1b:1025-1073
#    NSP13_1B      ### NSP13:151-226  ORF1b:1074-1149
#    NSP13_RecA1   ### NSP13:261-442  ORF1b:1174-1365
#    NSP13_RecA2   ### NSP13:443-601  ORF1b:1366-1524
#####################################################################################################################################    
#    NSP14_nsp10  ### NSP14:1-58
#    NSP14_EXON       ### NSP14:67-282 
#    NSP14_hinge1     ### NSP14:286-300
#    NSP14_hinge2     ### NSP14:411-434
#    NSP14_N7MTase    ### NSP14:302-527
##################################################################################################################################### 
#### NSP15 domain source: Pillon MC, Frazier MN, Dillard LB, et al. Cryo-EM structures of the SARS-CoV-2 endoribonuclease Nsp15 reveal insight into nuclease specificity and dynamics. Nat Commun. 2021;12(1):636. Published 2021 Jan 27. doi:10.1038/s41467-020-20608-z
#    NSP15_NTD    ### 1:64
#    NSP15_MD     ### 65:182, middle domain
#    NSP15_endoU  ### 207-347, Endonuclease, poly-U specific
#####################################################################################################################################
#    S_S1          ### S:2-686
#    S_S2          ### S:687-1273  
#    S_NTD         ### S:2-305      N-terminal domain
#    S_N2R         ### S:306-334 =  NTD-to-RBD linker  Source: Structural diversity of the SARS-CoV-2 Omicron spike; Gobeil SMC, et al, Molecular Cell, 2022-6-2; https://doi.org/10.1016/j.molcel.2022.03.028
#    S_RBD         ### S:335-528    receptor-binding domain
#    S_RBM         ### S:438-506    receptor-binding motif
#    S_SD1         ### S:529-591    subdomain 2 (AKA CTD1, RBD-s)
#    S_SD2         ### S:592-674    subdomain 2 (AKA CTD2, NTD-s)
#    S_630_loop    ### S:619-642    Source: Structural diversity of the SARS-CoV-2 Omicron spike; Gobeil SMC, et al, Molecular Cell, 2022-6-2; https://doi.org/10.1016/j.molcel.2022.03.028
#    S_FCS_region  ### S:675-691    furin cleavage site region
######## Source for all regions below: Xing L, Liu Z, Wang X, et al. Early fusion intermediate of ACE2-using coronavirus spike acting as an antiviral target. Cell. 2025;188(5):1297-1314.e24. doi:10.1016/j.cell.2025.01.012
#    S_Beta1        ### S:718-729    beta-strand; 
#    S_3H          ### S:737-769    3-helix; 
#    S_IL770       ### S:770-831    IL770
#    S_FPPR        ### S:832–866    fusion peptide-proximal region
#    S_FP          ### S:867–909    fusion peptide; 
#    S_HR1         ### S:910-985    heptad repeat 1; 
#    S_CH          ### S:986-1035   central helix
#    S_CD          ### S:1036-1068  connector domain
#    S_Beta2       ### S:1127-1135  beta-strand
#    S_2turnHelix  ### S:1148-1155  "two-turn helix"
#    S_HR2         ### S:1164-1211  heptad repeat 2
#    S_TM          ### S:1212-1234  transmembrane
#    S_CT          ### S:1235-1273  cytoplasmic tail
#####################################################################################################################################
## Source for all three E regions: Pearson GJ, Mears HV, Broncel M, Snijders AP, Bauer DLV, Carlton JG. ER-export and ARFRP1/AP-1-dependent delivery of SARS-CoV-2 Envelope to lysosomes controls late stages of viral replication. Sci Adv. 2024;10(14):eadl5012. doi:10.1126/sciadv.adl5012
#    E_TM        ## E:9-38   transmembrane region
#    E_cytosol   ## E:54-65  with YxxØ & quasi FxxFxxxR motifs "responsible for Golgi-to-lysosome trafficking" 
#    E_CTD       ## E:69-75  motif "drives ER export" 
#####################################################################################################################################
## ORF3a TM source: Source: Miao G, Zhao H, Li Y, et al. ORF3a of the COVID-19 virus SARS-CoV-2 blocks HOPS complex-mediated assembly of the SNARE complex required for autolysosome formation. Dev Cell. 2021;56(4):427-442.e5. doi:10.1016/j.devcel.2020.12.010
# "ORF3a_SignalP"=>BitSet(1:14), "ORF3a_NTD"=>BitSet(15:33), "ORF3a_TM1"=>BitSet(34:56), "ORF3a_TM12Lnk"=>BitSet(57:76), "ORF3a_TM2"=>BitSet(77:99), "ORF3a_TM3"=>BitSet(103:125), "ORF3a_cytosl1"=>BitSet(126:169), "ORF3a_Loop"=>BitSet(170:183), "ORF3a_3DB"=>BitSet(170:222), "ORF3a_CTD"=>BitSet(223:275),
#    ORF3a_SignalP   ### ORF3a:1-14
#    ORF3a_NTD       ### ORF3a:15-33
#    ORF3a_TM1       ### ORF3a:34-56
#    ORF3a_TM12Lnk  ### ORF3a:57-76
#    ORF3a_TM2       ### ORF3a:77-99
#    ORF3a_TM3       ### ORF3a:103-125
#    ORF3a_cytosl1  ### ORF3a:126-169
#    ORF3a_Loop      ### ORF3a:170-183
#    ORF3a_3DB       ### ORF3a:170-222
#    ORF3a_CTD       ### ORF3a:223-275
#####################################################################################################################################
#    N_N1          ### N:2-44
#    N_N2          ### N:45-175
#    N_N3          ### N:176-246
#    N_N4          ### N:247-364
#    N_N5          ### N:365-419
#    N_SR          ### N:176-206
#    N_L_helix     ### N:215-235
#    N_CBP         ### N:247-279
#    N_9b_overlap  ### N:4-101
####################################################################################################################################

In [ ]:
### Fx: clade_pango_date_cts
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
function clade_pango_date_cts(input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String)
####################################################################################################################################
    HQCS_qualifying_seqs = Set{String}()
    if max_AA_mut == 5 && revs_thresh == 1 && qc_max == 5
        HQCS_qualifying_seqs = HQCS_qualified_seqs_5_1_5
    elseif max_AA_mut == 10 && revs_thresh == 1 && qc_max == 30
        HQCS_qualifying_seqs = HQCS_qualified_seqs_10_1_30
    elseif max_AA_mut == 90 && revs_thresh == 5 && qc_max == 8000
        HQCS_qualifying_seqs = HQCS_qualified_seqs_90_5_8000
    end
    all_seq_stat_size_hint = 17500000
#    all_seq_stat_size_hint_90_5_8000 = 17128637
    all_seq_stat_size_hint_90_5_8000 = 17103176
    all_seq_stat_size_hint_10_1_30 = 14601802
    all_seq_stat_size_hint_5_1_5 = 10849566
    if max_AA_mut == 90 && qc_max == 8000
        all_seq_stat_size_hint = all_seq_stat_size_hint_90_5_8000
    elseif max_AA_mut == 10 && qc_max == 30
        all_seq_stat_size_hint = all_seq_stat_size_hint_10_1_30
    elseif max_AA_mut == 5 && qc_max == 5
        all_seq_stat_size_hint = all_seq_stat_size_hint_5_1_5
    end
####################################################################################################################################
    #                          clade     date_index  count
#    clade_date_index_ct = Dict{String, Dict{Int, Int}}();            sizehint!(clade_date_index_ct, 3176)
    pango_date_index_ct = Dict{String, Dict{Int, Int}}();             sizehint!(pango_date_index_ct, 3176)
    country_pango_date_index_ct = Dict{String, Dict{String, Dict{Int, Int}}}();            sizehint!(country_pango_date_index_ct, 351)
####################################################################################################################################
    seq_country = Dict{String, String}();                        sizehint!(seq_country, all_seq_stat_size_hint)
    seq_US_state = Dict{String, String}();                       sizehint!(seq_US_state, all_seq_stat_size_hint)
#    seq_clade = Dict{String, String}();                         sizehint!(seq_clade, all_seq_stat_size_hint)
    seq_pango = Dict{String, String}();                          sizehint!(seq_pango, all_seq_stat_size_hint)
    seq_date_index = Dict{String, Int}();                        sizehint!(seq_date_index, all_seq_stat_size_hint)
    seq_date_tuple = Dict{String, Tuple{Int64, Int64, Int64}}(); sizehint!(seq_date_tuple, all_seq_stat_size_hint)
    seq_lab_dict = Dict{String, String}();                       sizehint!(seq_lab_dict, all_seq_stat_size_hint)
####################################################################################################################################
####################################################################################################################################
####################################################################################################################################
####################################################
#        max_dels = 1500
#        max_frameshifts = 2
#        max_stops = 1
#        max_SNPclust = 101
#        min_coverage = 0.50
####################################################
    hundred_thousand_ct = 0
    all_seq_ct = 0
    total_AA_subs = 0
    for line in eachline(input_ndjson)
        all_seq_ct += 1
        if all_seq_ct % 100000 == 0
            hundred_thousand_ct += 1
            println("100000 ct = $(hundred_thousand_ct)")
            GC.gc(true)
        end
############################################################################
        j = JSON3.read(line)
############################################################################
        name = EPI_ISL(j.seqName)
        if name in HQCS_qualifying_seqs
#        del_ranges = j.privateNucMutations.privateDeletionRanges
#        private_del_ct = 0
#        for range in del_ranges
#            private_del_ct += 1
#        end
#        total_private_AA_muts = j.privateAaMutations.ORF1a.totalPrivateSubstitutions + j.privateAaMutations.ORF1b.totalPrivateSubstitutions + j.privateAaMutations.ORF3a.totalPrivateSubstitutions + j.privateAaMutations.ORF6.totalPrivateSubstitutions + j.privateAaMutations.ORF7a.totalPrivateSubstitutions + j.privateAaMutations.ORF7b.totalPrivateSubstitutions + j.privateAaMutations.ORF8.totalPrivateSubstitutions + j.privateAaMutations.ORF9b.totalPrivateSubstitutions + j.privateAaMutations.S.totalPrivateSubstitutions + j.privateAaMutations.E.totalPrivateSubstitutions + j.privateAaMutations.M.totalPrivateSubstitutions + j.privateAaMutations.N.totalPrivateSubstitutions + private_del_ct
#        if j.totalDeletions < 1500 && total_private_AA_muts ≤ max_AA_mut && j.qc.overallScore ≤ qc_max && j.privateNucMutations.totalReversionSubstitutions ≤ revs_thresh && j.qc.frameShifts.totalFrameShifts - j.qc.frameShifts.totalFrameShiftsIgnored == 0
#            seq_country[name] = countree(j.seqName)
#            seq_US_state[name] = US_state(j.seqName)
#            seq_date = ""
####################################################
#            try
#                seq_clade[name] = j.clade
#            catch
#                seq_clade[name] = "Unknown"
#            end
####################################################
            try
                seq_pango[name] = j.customNodeAttributes.Nextclade_pango
            catch
                seq_pango[name] = "Unknown"
            end
####################################################
            seq_date = ""
            try
                seq_date = sequence_date(j.seqName)
            catch
                seq_date = "0-0-0"
            end
####################################################
            lab = ""
            if !isempty(seq_lab(j.seqName))
                lab = seq_lab(j.seqName)
            end
            seq_lab_dict[name] = lab
            seq_year = 0
            seq_month = 0
            seq_day = 0
            seq_yr = join(seq_date[1:4])
            if all(isdigit, seq_yr)
                seq_year = parse(Int, seq_yr)
                if '-' in seq_date
                    seq_mo = split(seq_date, "-")[2]
                    if all(isdigit, seq_mo)
                        seq_month = parse(Int, seq_mo)
                    end
                end
####################################################
                dash_ct = 0
                for i in seq_date
                    if i == '-'
                        dash_ct += 1
                    end
                end
####################################################
                if dash_ct == 2
                    seq_da = split(seq_date, "-")[3]
                    if all(isdigit, seq_da)
                        seq_day = parse(Int, seq_da)
                    end
                end
            end
####################################################
            date_index = date_to_index[date_tuple]
            date_index = date_to_index[date_tuple]
            seq_date_index[name] = date_index
            date_tuple = (seq_year, seq_month, seq_day)
            seq_date_tuple[name] = date_tuple
####################################################
            pango = seq_pango[name]
            pango_unaliased = seq_pango_unaliased[name]
####################################################
            if !haskey(pango_date_index_ct, pango)
                pango_date_index_ct[pango] = Dict{Int, Int}()
            end
            if !haskey(pango_date_index_ct[pango], date_index)
                pango_date_index_ct[pango][date_index] = 1
            else
                pango_date_index_ct[pango][date_index] += 1
            end
####################################################
            country = seq_country[name]
####################################################
            if !haskey(country_pango_date_index_ct, country)
                country_pango_date_index_ct[country] = Dict{String, Dict{Int, Int}}()
            end
            if !haskey(country_pango_date_index_ct[country], pango)
                country_pango_date_index_ct[country][pango] = Dict{Int, Int}()
            end
            if !haskey(country_pango_date_index_ct[country][pango], date_index)
                country_pango_date_index_ct[country][pango][date_index] = 1
            else
                country_pango_date_index_ct[country][pango][date_index] += 1
            end
####################################################
        end
    end
####################################################################################################################################
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_pango_date_index_ct.jld2", "pango_date_index_ct", pango_date_index_ct)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_country_pango_date_index_ct.jld2", "country_pango_date_index_ct", country_pango_date_index_ct)
####################################################################################################################################
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_country.jld2", "seq_country", seq_country)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_US_state.jld2", "seq_US_state", seq_US_state)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_pango.jld2", "seq_pango", seq_pango)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_date_index.jld2", "seq_date_index", seq_date_index)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_date_tuple.jld2", "seq_date_tuple", seq_date_tuple)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_lab_dict.jld2", "seq_lab_dict", seq_lab_dict)
####################################################################################################################################
end

In [ ]:
### Execute clade_pango_date_cts;  Runtime = 40 hrs, 16 min, 19.71 s ************* 5, 1, 5, 2 ***************
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
start_clade_pango_date_ct = time()
clade_pango_date_cts("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 5, 1, 5, 2, "all_private_muts")
finish_clade_pango_date_ct = time() - start_clade_pango_date_ct
time1, time2 = seconds_to_hrs_min_sec(finish_clade_pango_date_ct)
println(time1); println(time2)
# input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String
##  Start: 7:06 AM, 2025-8-4; 10115 at 5:48 PM (8-4); 14500 at 8:52.45 AM (8-5);
### Total Function Run Time = 40 hours, 16 minutes, 19.71 seconds (2025-08-06)
### Total Function Run Time = 16842.4 seconds = 4 hours, 40 minutes, 42 seconds = 4:40.42 (sometime before 2025-08-06)

In [ ]:
### Execute clade_pango_date_cts;  Runtime = 40 hrs, 16 min, 19.71 s ************* 7, 1, 10, 2 ***************
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
start_clade_pango_date_ct = time()
clade_pango_date_cts("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 7, 1, 10, 2, "all_private_muts")
finish_clade_pango_date_ct = time() - start_clade_pango_date_ct
time1, time2 = seconds_to_hrs_min_sec(finish_clade_pango_date_ct)
println(time1); println(time2)
# input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String
##  Start: 7:06 AM, 2025-8-4; 10115 at 5:48 PM (8-4); 14500 at 8:52.45 AM (8-5);
### Total Function Run Time = 40 hours, 16 minutes, 19.71 seconds (2025-08-06)
### Total Function Run Time = 16842.4 seconds = 4 hours, 40 minutes, 42 seconds = 4:40.42 (sometime before 2025-08-06)

In [ ]:
### Execute clade_pango_date_cts;  Runtime = 40 hrs, 16 min, 19.71 s ************* 90, 5, 8000, 2 ***************
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
start_clade_pango_date_ct = time()
clade_pango_date_cts("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 90, 5, 8000, 2, "all_private_muts")
finish_clade_pango_date_ct = time() - start_clade_pango_date_ct
time1, time2 = seconds_to_hrs_min_sec(finish_clade_pango_date_ct)
println(time1); println(time2)
# input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String
##  Start: 7:06 AM, 2025-8-4; 10115 at 5:48 PM (8-4); 14500 at 8:52.45 AM (8-5);
### Total Function Run Time = 40 hours, 16 minutes, 19.71 seconds (2025-08-06)
### Total Function Run Time = 16842.4 seconds = 4 hours, 40 minutes, 42 seconds = 4:40.42 (sometime before 2025-08-06)

In [ ]:
### Stuff to load before running cells below (mainly seq_date_index)
date = "2026_01_06"
fx_name = "all_private_muts"
ndjson_name = "EPI_ISL_400001_20300000"
max_AA_mut = 10
revs_thresh = 1
qc_max = 30
filename = "$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)"
seq_date_index_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_date_index.jld2", "seq_date_index")
#clade_set = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_clade_set.jld2", "clade_set")

In [ ]:
### Fx: clade_date_index_cts ————load cell above first——————
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
date = "2026_01_06"
fx_name = "all_private_muts"
ndjson_name = "EPI_ISL_400001_20300000"
max_AA_mut = 5
revs_thresh = 1
qc_max = 5
function clade_date_index_cts(input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String)
#########################################################################################
    clade_start_time = time()
    half_mill_time_end_dict = Dict{Int, Float64}()
    half_mill_time_complete_dict = Dict{Int, Float64}()
####################################################################################################################################
    HQCS_qualifying_seqs = Set{String}()
    if max_AA_mut == 5 && revs_thresh == 1 && qc_max == 5
        HQCS_qualifying_seqs = HQCS_qualified_seqs_5_1_5
    elseif max_AA_mut == 10 && revs_thresh == 1 && qc_max == 30
        HQCS_qualifying_seqs = HQCS_qualified_seqs_10_1_30
    elseif max_AA_mut == 90 && revs_thresh == 5 && qc_max == 8000
        HQCS_qualifying_seqs = HQCS_qualified_seqs_90_5_8000
    end
    all_seq_stat_size_hint = 17500000
#    all_seq_stat_size_hint_90_5_8000 = 17128637
    all_seq_stat_size_hint_90_5_8000 = 17103176
    all_seq_stat_size_hint_10_1_30 = 14601802
    all_seq_stat_size_hint_5_1_5 = 10849566
    if max_AA_mut == 90 && qc_max == 8000
        all_seq_stat_size_hint = all_seq_stat_size_hint_90_5_8000
    elseif max_AA_mut == 10 && qc_max == 30
        all_seq_stat_size_hint = all_seq_stat_size_hint_10_1_30
    elseif max_AA_mut == 5 && qc_max == 5
        all_seq_stat_size_hint = all_seq_stat_size_hint_5_1_5
    end
####################################################################################################################################
    all_seq_ct = 0
#########################################################################################
    clade_date_index_ct = Dict{String, Dict{Int, Int}}();            sizehint!(clade_date_index_ct, 3176)
    clade_set = Set{String}()
#########################################################################################
    date_regex = r"(\d{4})-(\d{2})-(\d{2})"
##### Explanation of    date_regex = r"(\d{4})-(\d{2})-(\d{2})"   #####################
##### r = regular expression
##### \d = any digit
##### {4} = number of preceding character to match (in this case the number of digits)
##### () = capture groups. Allow you to recall whatever is inside the parentheses later.
#          If date = "2025-09-01", then  date.capture[1] = "2025", date.capture[2] = "09", etc
##### So this checks the j.seqName line for dates in the correct format
#########################################################################################
    function get_full_year_month_day_tuple(seqName::String)
        ymd = match(date_regex, seqName)
        if ymd ≠ nothing
            year = parse(Int64, ymd.captures[1])
            month = parse(Int64, ymd.captures[2])
            day = parse(Int64, ymd.captures[3])
            if year ≥ 2020 && year ≤ 2027 && month ≥ 1 && month ≤ 12 && day ≥ 1 && day ≤ 31
                return (year, month, day)
            end
        end
        return nothing
    end
    function dash_count(str::String)
        return count(c -> c == '-', str)
    end
####################################################################################################################################
    for clade in clade_set_complete
        clade_date_index_ct[clade] = Dict{Int, Int}()
    end
    for clade in clade_set_complete
        for i in 1:2922
            clade_date_index_ct[clade][i] = 0
        end
    end
####################################################################################################################################
    all_seq_stat_size_hint = 17500000
    all_seq_stat_size_hint_90_5_8000 = 17128637
    all_seq_stat_size_hint_10_1_30 = 14631051
    all_seq_stat_size_hint_5_1_5 = 10871228
    if max_AA_mut == 90 && qc_max == 8000
        all_seq_stat_size_hint = all_seq_stat_size_hint_90_5_8000
    elseif max_AA_mut == 10 && qc_max == 30
        all_seq_stat_size_hint = all_seq_stat_size_hint_10_1_30
    elseif max_AA_mut == 5 && qc_max == 5
        all_seq_stat_size_hint = all_seq_stat_size_hint_5_1_5
    end
    seq_clade = Dict{String, String}()            sizehint!(seq_clade, all_seq_stat_size_hint)
####################################################################################################################################
    five_hundred_thousand_ct::Int = 0
    million_ct = 0
    all_seq_ct = 0
    total_AA_subs = 0
    for line in eachline(input_ndjson)
        all_seq_ct += 1
        if all_seq_ct % 500000 == 0
            five_hundred_thousand_ct += 1
            half_mill_time_end_dict[five_hundred_thousand_ct] = time()
            if five_hundred_thousand_ct == 1
                half_mill_time_complete_dict[1] = round(digits=1, time() - clade_start_time)
            else
                half_mill_time_complete_dict[five_hundred_thousand_ct] = round(digits=1, time() - half_mill_time_end_dict[five_hundred_thousand_ct-1])
            end
            million_ct = round(digits=1, five_hundred_thousand_ct/2)
            println("million_ct = $(million_ct)")
            println("Time for last 500,000 = $(half_mill_time_complete_dict[five_hundred_thousand_ct])")
            nowtime = Dates.format(now(), "I:MM.SS_p"); print(nowtime); println()
        end
############################################################################
        j = JSON3.read(line)
############################################################################
        name = EPI_ISL(j.seqName)
        if name in HQCS_qualifying_seqs
            seq_date_index = 0
            clade = ""
####################################################
#        max_dels = 1500
#        max_frameshifts = 2
#        max_stops = 1
#        max_SNPclust = 101
#        min_coverage = 0.50
####################################################
#        if j.totalDeletions ≥ max_dels
#            continue
#        end
#        QC = j.qc
#        if QC.overallScore > qc_max
#            continue
#        end
#        if QC.frameShifts.totalFrameShifts - QC.frameShifts.totalFrameShiftsIgnored > max_frameshifts
#            continue
#        end
#        if QC.stopCodons.totalStopCodons - QC.stopCodons.totalStopCodonsIgnored > max_stops
#            continue
#        end
#        if QC.snpClusters.score ≥ max_SNPclust
#            continue
#        end
#        if j.coverage < min_coverage
#            continue
#        end
#        if j.privateNucMutations.totalReversionSubstitutions > revs_thresh
#            continue
#        end
############################################################################
#        name = EPI_ISL(j.seqName)
#        private_del_ct = length(j.privateNucMutations.privateDeletionRanges)
#        privAA = j.privateAaMutations
#        ORF1a_AA = privAA.ORF1a
#        ORF1b_AA = privAA.ORF1b
#        ORF3a_AA = privAA.ORF3a
#        ORF6_AA = privAA.ORF6
#        ORF7a_AA = privAA.ORF7a
#        ORF7b_AA = privAA.ORF7b
#        ORF8_AA = privAA.ORF8
#        ORF9b_AA = privAA.ORF9b
#        S_AA = privAA.S
#        E_AA = privAA.E
#        M_AA = privAA.M
#        N_AA = privAA.N
#            if haskey(j.privateAaMutations, "ORF1a") && haskey(j.privateAaMutations, "ORF1b") && haskey(j.privateAaMutations, "ORF3a") && haskey(j.privateAaMutations, "ORF6") && haskey(j.privateAaMutations, "ORF7a") && haskey(j.privateAaMutations, "ORF7b") && haskey(j.privateAaMutations, "ORF8") && haskey(j.privateAaMutations, "ORF9b") && haskey(j.privateAaMutations, "S") && haskey(j.privateAaMutations, "E") && haskey(j.privateAaMutations, "M") && haskey(j.privateAaMutations, "N")
#        total_private_AA_muts = ORF1a_AA.totalPrivateSubstitutions + ORF1b_AA.totalPrivateSubstitutions + ORF3a_AA.totalPrivateSubstitutions + ORF6_AA.totalPrivateSubstitutions + ORF7a_AA.totalPrivateSubstitutions + ORF7b_AA.totalPrivateSubstitutions + ORF8_AA.totalPrivateSubstitutions + ORF9b_AA.totalPrivateSubstitutions + S_AA.totalPrivateSubstitutions + E_AA.totalPrivateSubstitutions + M_AA.totalPrivateSubstitutions + N_AA.totalPrivateSubstitutions + private_del_ct
#        seq_date = ""
#        if total_private_AA_muts ≤ max_AA_mut
####################################################
            try
                seq_clade[name] = j.clade
            catch
                seq_clade[name] = "Unknown"
            end
            push!(clade_set, seq_clade[name])
####################################################
            date_tuple = (0, 0, 0)
            if haskey(seq_date_index_all, name)
                seq_date_index = seq_date_index_all[name]
            else
                seq_date = ""
                seq_year = Int64(0)
                seq_month = Int64(0)
                seq_day = Int64(0)
                date_tuple = get_full_year_month_day_tuple(j.seqName)
                if date_tuple == nothing
                    try
                        seq_date = string(sequence_date(j.seqName))
                    catch
                        seq_date = "0-0-0"
                    end
                    dash_ct = dash_count(seq_date)
                    if dash_ct == 0
                        seq_y = seq_date
                        if all(isdigit, seq_y)
                            seq_y = parse(Int64, seq_y)
                            if seq_y ≥ 2020 && seq_y ≤ 2027
                                seq_year = seq_y
                            end
                        end
                    end
                    if dash_ct == 1
                        seq_y = string(split(seq_date, "-")[1])
                        seq_m = string(split(seq_date, "-")[2])
                        if all(isdigit, seq_y)
                            seq_y = parse(Int64, seq_y)
                            if seq_y ≥ 2020 && seq_y ≤ 2027
                                seq_year = seq_y
                            end
                        end
                        if all(isdigit, seq_m)
                            seq_m = parse(Int64, seq_m)
                            if seq_m ≥ 1 && seq_m ≤ 12
                                seq_month = seq_m
                            end
                        end
                    end
                    if dash_ct == 2
                        seq_y = string(split(seq_date, "-")[1])
                        seq_m = string(split(seq_date, "-")[2])
                        seq_d = string(split(seq_date, "-")[3])
                        if all(isdigit, seq_y)
                            seq_y = parse(Int64, seq_y)
                            if seq_y ≥ 2020 && seq_y ≤ 2027
                                seq_year = seq_y
                            end
                        end
                        if all(isdigit, seq_m)
                            seq_m = parse(Int64, seq_m)
                            if seq_m ≥ 1 && seq_m ≤ 12
                                seq_month = seq_m
                            end
                        end
                        if all(isdigit, seq_d)
                            seq_d = parse(Int64, seq_d)
                            if seq_d ≥ 1 && seq_d ≤ 31
                                seq_day = seq_d
                            end
                        end
                    end
                    date_tuple = (seq_year, seq_month, seq_day)
                end
                date_index = get(date_to_index, date_tuple, 0)
                seq_date_index = date_index
            end
            clade = seq_clade[name]
            if seq_date_index ≠ 0
                clade_date_index_ct[clade] = get!(clade_date_index_ct, clade, Dict{Int, Int}())
                clade_date_index_ct[clade][seq_date_index] = get!(clade_date_index_ct[clade], seq_date_index, 0) + 1
            end
        end
####################################################
####################################################
#         pango_date_index_ct[pango] = get!(pango_date_index_ct, pango, Dict{Int, Int}())
#         pango_date_index_ct[pango][seq_date_index] = get!(pango_date_index_ct[pango], seq_date_index, 0) + 1
####################################################
#         pango_unaliased_date_index_ct[pango_unaliased] = get!(pango_unaliased_date_index_ct, pango_unaliased, Dict{Int, Int}())
#         pango_unaliased_date_index_ct[pango_unaliased][seq_date_index] = get!(pango_unaliased_date_index_ct[pango_unaliased], seq_date_index, 0) + 1
#####################################################################################
#         country = seq_country[name]
#         country_clade_date_index_ct[country] = get!(country_clade_date_index_ct, country, Dict{String, Dict{Int, Int}}())
#         country_clade_date_index_ct[country][clade] = get!(country_clade_date_index_ct[country], clade, Dict{Int, Int}())
#         country_clade_date_index_ct[country][clade][seq_date_index] = get!(country_clade_date_index_ct[country][clade], seq_date_index, 0) + 1
####################################################
#         country_pango_date_index_ct[country] = get!(country_pango_date_index_ct, country, Dict{String, Dict{Int, Int}}())
#         country_pango_date_index_ct[country][pango] = get!(country_pango_date_index_ct[country], pango, Dict{Int, Int}())
#         country_pango_date_index_ct[country][pango][seq_date_index] = get!(country_pango_date_index_ct[country][pango], seq_date_index, 0) + 1
####################################################
#         country_pango_unaliased_date_index_ct[country] = get!(country_pango_unaliased_date_index_ct, country, Dict{String, Dict{Int, Int}}())
#         country_pango_unaliased_date_index_ct[country][pango_unaliased] = get!(country_pango_unaliased_date_index_ct[country], pango_unaliased, Dict{Int, Int}())
#         country_pango_unaliased_date_index_ct[country][pango_unaliased][seq_date_index] = get!(country_pango_unaliased_date_index_ct[country][pango_unaliased], seq_date_index, 0) + 1
####################################################
    end
####################################################################################################################################
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_clade_date_index_ct.jld2", "clade_date_index_ct", clade_date_index_ct)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_country_clade_date_index_ct.jld2", "country_clade_date_index_ct", country_clade_date_index_ct)
####################################################################################################################################
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_clade_set.jld2", "clade_set", clade_set)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_clade.jld2", "seq_clade", seq_clade)
####################################################################################################################################
end

In [ ]:
### Execute clade_date_index_cts | 5, 1, 5 | Runtime = 2 hr, 31 min, 17.38 sec
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
start = time()
clade_date_index_cts("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 5, 1, 5, 2, "all_private_muts")
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v1 = $(runtime1)");   println("Runtime v2 = $(runtime2)")
######## Start Time = 2026_02_01__218pm
### (input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String)
######################################################################################################################################################
######################################################################################################################################################

In [ ]:
### Execute clade_date_index_cts | 10, 1, 30
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
start = time()
clade_date_index_cts("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 10, 1, 30, 2, "all_private_muts")
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v1 = $(runtime1)");   println("Runtime v2 = $(runtime2)")
### (input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String)
######################################################################################################################################################
######################################################################################################################################################

In [ ]:
### Execute clade_date_index_cts | 90, 5, 8000
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
start = time()
clade_date_index_cts("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 90, 5, 8000, 2, "all_private_muts")
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v1 = $(runtime1)");   println("Runtime v2 = $(runtime2)")
### (input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String)
######################################################################################################################################################
######################################################################################################################################################

In [ ]:
### Fx: pango_date_index_cts_no_country #######################
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
date = Dates.format(today(), "yyyy_mm_dd")
function pango_date_index_cts_no_country(input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, fx_name::String)
    pango_start_time = time()
####################################################################################################################################
    HQCS_qualifying_seqs = Set{String}()
    if max_AA_mut == 5 && revs_thresh == 1 && qc_max == 5
        HQCS_qualifying_seqs = HQCS_qualified_seqs_5_1_5
    elseif max_AA_mut == 10 && revs_thresh == 1 && qc_max == 30
        HQCS_qualifying_seqs = HQCS_qualified_seqs_10_1_30
    elseif max_AA_mut == 90 && revs_thresh == 5 && qc_max == 8000
        HQCS_qualifying_seqs = HQCS_qualified_seqs_90_5_8000
    end
    all_seq_stat_size_hint = 17500000
#    all_seq_stat_size_hint_90_5_8000 = 17128637
    all_seq_stat_size_hint_90_5_8000 = 17103176
    all_seq_stat_size_hint_10_1_30 = 14601802
    all_seq_stat_size_hint_5_1_5 = 10849566
    if max_AA_mut == 90 && qc_max == 8000
        all_seq_stat_size_hint = all_seq_stat_size_hint_90_5_8000
    elseif max_AA_mut == 10 && qc_max == 30
        all_seq_stat_size_hint = all_seq_stat_size_hint_10_1_30
    elseif max_AA_mut == 5 && qc_max == 5
        all_seq_stat_size_hint = all_seq_stat_size_hint_5_1_5
    end
####################################################################################################################################
    half_mill_time_end_dict = Dict{Int, Float64}()
    half_mill_time_complete_dict = Dict{Int, Float64}()
    all_seq_ct = 0
#########################################################################################
    date_regex = r"(\d{4})-(\d{2})-(\d{2})"
##### Explanation of    date_regex = r"(\d{4})-(\d{2})-(\d{2})"   #####################
##### r = regular expression
##### \d = any digit
##### {4} = number of preceding character to match (in this case the number of digits)
##### () = capture groups. Allow you to recall whatever is inside the parentheses later.
#          If date = "2025-09-01", then  date.capture[1] = "2025", date.capture[2] = "09", etc
##### So this checks the j.seqName line for dates in the correct format
#########################################################################################
    function get_full_year_month_day_tuple(seqName::String)
        ymd = match(date_regex, seqName)
        if ymd ≠ nothing
            year = parse(Int64, ymd.captures[1])
            month = parse(Int64, ymd.captures[2])
            day = parse(Int64, ymd.captures[3])
            if year ≥ 2020 && year ≤ 2027 && month ≥ 1 && month ≤ 12 && day ≥ 1 && day ≤ 31
                return (year, month, day)
            end
        end
        return nothing
    end
    function dash_count(str::String)
        return count(c -> c == '-', str)
    end
#########################################################################################
    pango_date_index_ct = Dict{String, Dict{Int, Int}}()
#    for pango in pango_set
#        pango_date_index_ct[pango] = Dict{Int, Int}()
#        sizehint!(pango_date_index_ct[pango], 3500)
#        for i in 1:2922
#            pango_date_index_ct[pango][i] = 0
#        end
#        for i in [0, 1, 2, 3, 4, 5, 6, 7]
#            for j in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
#		dindex = 2020000000 + 1000000*i + 1000*j
#                pango_date_index_ct[pango][dindex] = 0
#            end
#        end
#        pango_date_index_ct[pango][1000000000] = 0
#    end
#for y in 2020:2027
#    date_to_index[(y, 0, 0)] = y*1000000
#    index_to_date[y*1000000] = (y, 0, 0)
#    for m in 1:12
#        date_to_index[(y, m, 0)] = y*1000000 + m*1000
#        index_to_date[y*1000000 + m*1000] = (y, m, 0)
#    end
#end  
####################################################################################################################################
    five_hundred_thousand_ct::Int = 0
    million_ct = 0
    for line in eachline(input_ndjson)
        all_seq_ct += 1
        if all_seq_ct % 500000 == 0
            five_hundred_thousand_ct += 1
            half_mill_time_end_dict[five_hundred_thousand_ct] = time()
            if five_hundred_thousand_ct == 1
                half_mill_time_complete_dict[1] = round(digits=1, time() - pango_start_time)
            else
                half_mill_time_complete_dict[five_hundred_thousand_ct] = round(digits=1, time() - half_mill_time_end_dict[five_hundred_thousand_ct-1])
            end
            million_ct = round(digits=1, five_hundred_thousand_ct/2)
#            println("Processed $all_seq_ct sequences, forcing garbage collection...")
            println("million_ct = $(million_ct)")
            println("Time for last 500,000 = $(half_mill_time_complete_dict[five_hundred_thousand_ct])")
            nowtime = Dates.format(now(), "I:MM.SS_p"); print(nowtime); println()
        end
############################################################################
        j = JSON3.read(line)
############################################################################
#        if haskey(j, "privateAaMutations") && haskey(j, "qc") && haskey(j, "totalDeletions") && haskey(j, "privateNucMutations")
#        if haskey(j.privateAaMutations, "ORF1a") && haskey(j.privateAaMutations, "ORF1b") && haskey(j.privateAaMutations, "ORF3a") && haskey(j.privateAaMutations, "ORF6") && haskey(j.privateAaMutations, "ORF7a") && haskey(j.privateAaMutations, "ORF7b") && haskey(j.privateAaMutations, "ORF8") && haskey(j.privateAaMutations, "ORF9b") && haskey(j.privateAaMutations, "S") && haskey(j.privateAaMutations, "E") && haskey(j.privateAaMutations, "M") && haskey(j.privateAaMutations, "N")
####################################################
        max_dels = 1500
        max_frameshifts = 2
        max_stops = 1
        max_SNPclust = 101
        min_coverage = 0.50
####################################################
        nuc_muts = j.privateNucMutations
        if j.totalDeletions ≥ max_dels
            continue
        end
        QC = j.qc
        if QC.overallScore > qc_max
            continue
        end
        if QC.frameShifts.totalFrameShifts - QC.frameShifts.totalFrameShiftsIgnored > max_frameshifts
            continue
        end
        if QC.stopCodons.totalStopCodons - QC.stopCodons.totalStopCodonsIgnored > max_stops
            continue
        end
        if QC.snpClusters.score ≥ max_SNPclust
            continue
        end
        if j.coverage < min_coverage
            continue
        end
        if nuc_muts.totalReversionSubstitutions > revs_thresh
            continue
        end
############################################################################
        name = EPI_ISL(j.seqName)
        del_ranges = nuc_muts.privateDeletionRanges
        private_del_ct = length(nuc_muts.privateDeletionRanges)
        pango = ""
        privAA = j.privateAaMutations
        ORF1a_AA = privAA.ORF1a
        ORF1b_AA = privAA.ORF1b
        ORF3a_AA = privAA.ORF3a
        ORF6_AA = privAA.ORF6
        ORF7a_AA = privAA.ORF7a
        ORF7b_AA = privAA.ORF7b
        ORF8_AA = privAA.ORF8
        ORF9b_AA = privAA.ORF9b
        S_AA = privAA.S
        E_AA = privAA.E
        M_AA = privAA.M
        N_AA = privAA.N
        total_private_AA_muts = ORF1a_AA.totalPrivateSubstitutions + ORF1b_AA.totalPrivateSubstitutions + ORF3a_AA.totalPrivateSubstitutions + ORF6_AA.totalPrivateSubstitutions + ORF7a_AA.totalPrivateSubstitutions + ORF7b_AA.totalPrivateSubstitutions + ORF8_AA.totalPrivateSubstitutions + ORF9b_AA.totalPrivateSubstitutions + S_AA.totalPrivateSubstitutions + E_AA.totalPrivateSubstitutions + M_AA.totalPrivateSubstitutions + N_AA.totalPrivateSubstitutions
        if total_private_AA_muts + private_del_ct ≤ max_AA_mut
#            qualifying_seq_ct += 1
#            country = countree(j.seqName)
#            seq_US_state[name] = US_state(j.seqName)
####################################################
            try
                pango = j.customNodeAttributes.Nextclade_pango
            catch
                pango = "Unknown"
            end
#            seq_pango[name] = pango
#####################################################################################################################################
######################################################################################
            seq_date = ""
            seq_year = Int64(0)
            seq_month = Int64(0)
            seq_day = Int64(0)
            date_tuple = get_full_year_month_day_tuple(j.seqName)
            if date_tuple == nothing
                try
                    seq_date = string(sequence_date(j.seqName))
                catch
                    seq_date = "0-0-0"
                end
                dash_ct = dash_count(seq_date)
                if dash_ct == 0
                    seq_y = seq_date
                    if all(isdigit, seq_y)
                        seq_y = parse(Int64, seq_y)
                        if seq_y ≥ 2020 && seq_y ≤ 2027
                            seq_year = seq_y
                        end
                    end
                end
                if dash_ct == 1
                    seq_y = string(split(seq_date, "-")[1])
                    seq_m = string(split(seq_date, "-")[2])
                    if all(isdigit, seq_y)
                        seq_y = parse(Int64, seq_y)
                        if seq_y ≥ 2020 && seq_y ≤ 2027
                            seq_year = seq_y
                        end
                    end
                    if all(isdigit, seq_m)
                        seq_m = parse(Int64, seq_m)
                        if seq_m ≥ 1 && seq_m ≤ 12
                            seq_month = seq_m
                        end
                    end
                end
                if dash_ct == 2
                    seq_y = string(split(seq_date, "-")[1])
                    seq_m = string(split(seq_date, "-")[2])
                    seq_d = string(split(seq_date, "-")[3])
                    if all(isdigit, seq_y)
                        seq_y = parse(Int64, seq_y)
                        if seq_y ≥ 2020 && seq_y ≤ 2027
                            seq_year = seq_y
                        end
                    end
                    if all(isdigit, seq_m)
                        seq_m = parse(Int64, seq_m)
                        if seq_m ≥ 1 && seq_m ≤ 12
                            seq_month = seq_m
                        end
                    end
                    if all(isdigit, seq_d)
                        seq_d = parse(Int64, seq_d)
                        if seq_d ≥ 1 && seq_d ≤ 31
                            seq_day = seq_d
                        end
                    end
                end
                date_tuple = (seq_year, seq_month, seq_day)
            end
            date_index = date_to_index[date_tuple]
#            seq_date_tuple[name] = date_tuple
######################################################################################################################################
#            date_index = date_to_index[date_tuple]
#            date_tuple = (seq_year, seq_month, seq_day)
#            seq_date_index[name] = date_index
#            seq_date_tuple[name] = date_tuple
####################################################
            get!(pango_date_index_ct, pango, Dict{Int, Int}())
            pango_date_index_ct[pango][date_index] = get(pango_date_index_ct[pango], date_index, 0) + 1
####################################################
#            get!(country_pango_date_index_ct, country, Dict{String, Dict{Int, Int}}())
#            get!(country_pango_date_index_ct[country], pango, Dict{Int, Int}())
#            country_pango_date_index_ct[country][pango][date_index] = get(country_pango_date_index_ct[country][pango], date_index, 0) + 1
####################################################
        end
    end
####################################################################################################################################
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_pango_date_index_ct.jld2", "pango_date_index_ct", pango_date_index_ct)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_country_pango_date_index_ct.jld2", "country_pango_date_index_ct", country_pango_date_index_ct)
####################################################################################################################################
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_country.jld2", "seq_country", seq_country)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_US_state.jld2", "seq_US_state", seq_US_state)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_pango.jld2", "seq_pango", seq_pango)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_date_index.jld2", "seq_date_index", seq_date_index)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_date_tuple.jld2", "seq_date_tuple", seq_date_tuple)
####################################################################################################################################
end

In [ ]:
### Execute pango_date_index_cts_no_country |  5, 1, 5 | Runtime: 
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
start_clade_pango_date_ct = time()                                                     ### max_AA_mut=90   revs_thresh=5   qc_max=8000 
pango_date_index_cts_no_country("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 5, 1, 5, "all_private_muts")
finish_clade_pango_date_ct = time() - start_clade_pango_date_ct
time1, time2 = seconds_to_hrs_min_sec(finish_clade_pango_date_ct)
println(time1); println(time2)
#   input_ndjson, ndjson_name, max_AA_mut, revs_thresh, qc_max, print_ct_thresh, fx_name
### Total Function Run Time = 2 hr, 34 min, 39.1 sec (pango_date_index_ct only, 2026-01-07, QC = 5, 1, 5)
### Total Function Run Time = 1 hr, 17 min, 22.7 sec (seq_date_index only, 2025-09-07, QC = 5, 1, 5)
### Total Function Run Time = 2 hours, 48 minutes, 6.28 seconds (pango_date_index_ct, seq_date_index, & seq_pango only——2025-08-25, QC=90,5,8000
### Total Function Run Time = 16842.4 seconds = 4 hours, 40 minutes, 42 seconds = 4:40.42 (was more comprehensive)
###################################

In [ ]:
### Execute pango_date_index_cts_no_country |  7, 1, 10 | Runtime: 
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
################################################                     
start_clade_pango_date_ct = time()                                                     ### max_AA_mut=90   revs_thresh=5   qc_max=8000 
pango_date_index_cts_no_country("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF", "EPI_ISL_400001_20300000", 7, 1, 10, "all_private_muts")
finish_clade_pango_date_ct = time() - start_clade_pango_date_ct
time1, time2 = seconds_to_hrs_min_sec(finish_clade_pango_date_ct)
println(time1); println(time2)
#   input_ndjson, ndjson_name, max_AA_mut, revs_thresh, qc_max, print_ct_thresh, fx_name
#   Start: 
### Total Function Run Time = 2 hours, 48 minutes, 6.28 seconds (pango_date_index, seq_date_index, & seq_pango only——2025-08-25
### Total Function Run Time = 16842.4 seconds = 4 hours, 40 minutes, 42 seconds = 4:40.42 (was more comprehensive)
############################################################################################################################################

In [ ]:
### Execute pango_date_index_cts_no_country | 10, 1, 30 | Runtime: 1 hr, 51 min, 35.08 sec | 2026_02_07
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
################################################                     
start_clade_pango_date_ct = time()                                                     ### max_AA_mut=90   revs_thresh=5   qc_max=8000 
pango_date_index_cts_no_country("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 10, 1, 30, "all_private_muts")
finish_clade_pango_date_ct = time() - start_clade_pango_date_ct
time1, time2 = seconds_to_hrs_min_sec(finish_clade_pango_date_ct)
println(time1); println(time2)
### Total Function Run Time =    (2026-01-07, QC = 5, 1, 5)
### Total Function Run Time = 2 hr, 34 min, 39.1 sec (2026-01-07, QC = 5, 1, 5)
# (input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String)
###################################
################################### start: 10:02 AM, 2026_02_07
############################################################################################################################################
############################################################################################################################################

In [ ]:
### Execute pango_date_index_cts_no_country | 90,5,8000 | Runtime: 
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
start_clade_pango_date_ct = time()                                                     
pango_date_index_cts_no_country("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 90, 5, 8000, "all_private_muts")
finish_clade_pango_date_ct = time() - start_clade_pango_date_ct
time1, time2 = seconds_to_hrs_min_sec(finish_clade_pango_date_ct)
println(time1); println(time2)
#   input_ndjson, ndjson_name, max_AA_mut, revs_thresh, qc_max, print_ct_thresh, fx_name
#   Start: : AM, 2025-5-
### Total Function Run Time = 2 hours, 48 minutes, 6.28 seconds (pango_date_index, seq_date_index, & seq_pango only——2025-08-25, QC=90,5,8000
### Total Function Run Time = 16842.4 seconds = 4 hours, 40 minutes, 42 seconds = 4:40.42 (was more comprehensive)
############################################################################################################################################
############################################################################################################################################

In [ ]:
### Fx: seq_date_tuple_only  ####################################
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
date = Dates.format(today(), "yyyy_mm_dd")
function seq_date_tuple_only(input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, fx_name::String)
    start_time = time()
    all_seq_ct = 0 
############################################################################################################################################################################
####################################################################################################################################
    HQCS_qualifying_seqs = Set{String}()
    if max_AA_mut == 5 && revs_thresh == 1 && qc_max == 5
        HQCS_qualifying_seqs = HQCS_qualified_seqs_5_1_5
    elseif max_AA_mut == 10 && revs_thresh == 1 && qc_max == 30
        HQCS_qualifying_seqs = HQCS_qualified_seqs_10_1_30
    elseif max_AA_mut == 90 && revs_thresh == 5 && qc_max == 8000
        HQCS_qualifying_seqs = HQCS_qualified_seqs_90_5_8000
    end
    all_seq_stat_size_hint = 17500000
#    all_seq_stat_size_hint_90_5_8000 = 17128637
    all_seq_stat_size_hint_90_5_8000 = 17103176
    all_seq_stat_size_hint_10_1_30 = 14601802
    all_seq_stat_size_hint_5_1_5 = 10849566
    if max_AA_mut == 90 && qc_max == 8000
        all_seq_stat_size_hint = all_seq_stat_size_hint_90_5_8000
    elseif max_AA_mut == 10 && qc_max == 30
        all_seq_stat_size_hint = all_seq_stat_size_hint_10_1_30
    elseif max_AA_mut == 5 && qc_max == 5
        all_seq_stat_size_hint = all_seq_stat_size_hint_5_1_5
    end
####################################################################################################################################

    seq_date_tuple = Dict{String, Tuple{Int64, Int64, Int64}}();        sizehint!(seq_date_tuple, all_seq_stat_size_hint)
############################################################################################################################################################################
    hundred_thousand_dict = Dict{Int, Float64}()
    five_hundred_thousand_ct::Int = 0

    date_regex = r"(\d{4})-(\d{2})-(\d{2})"
##### Explanation of    date_regex = r"(\d{4})-(\d{2})-(\d{2})"   #####################
##### r = regular expression
##### \d = any digit
##### {4} = number of preceding character to match (in this case the number of digits)
##### () = capture groups. Allow you to recall whatever is inside the parentheses later.
#          If date = "2025-09-01", then  date.capture[1] = "2025", date.capture[2] = "09", etc
##### So this checks the j.seqName line for dates in the correct format
    function get_full_year_month_day_tuple(seqName::String)
        ymd = match(date_regex, seqName)
        if ymd ≠ nothing
            year = parse(Int64, ymd.captures[1])
            month = parse(Int64, ymd.captures[2])
            day = parse(Int64, ymd.captures[3])
            if year ≥ 2020 && year ≤ 2027 && month ≥ 1 && month ≤ 12 && day ≥ 1 && day ≤ 31
                return (year, month, day)
            end
        end
        return nothing
    end
    function dash_count(str::String)
        return count(c -> c == '-', str)
    end

    max_dels = 1500
    max_frameshifts = 2
    max_stops = 1
    max_SNPclust = 101
    min_coverage = 0.50
    
    for line in eachline(input_ndjson)
        all_seq_ct += 1
        if all_seq_ct % 500000 == 0
            five_hundred_thousand_ct += 1
            million_ct = round(digits=1, five_hundred_thousand_ct/2)
#            println("Processed $all_seq_ct sequences, forcing garbage collection...")
            println("million ct = $(million_ct)")
            nowtime = Dates.format(now(), "I:MM.SS_p"); print(nowtime); println()
        end
############################################################################
        j = JSON3.read(line)
############################################################################
        if j.totalDeletions ≥ max_dels
            continue
        end
        QC = j.qc
        if QC.overallScore > qc_max
            continue
        end
        if QC.frameShifts.totalFrameShifts - QC.frameShifts.totalFrameShiftsIgnored > max_frameshifts
            continue
        end
        if QC.stopCodons.totalStopCodons - QC.stopCodons.totalStopCodonsIgnored > max_stops
            continue
        end
        if QC.snpClusters.score ≥ max_SNPclust
            continue
        end
        if j.coverage < min_coverage
            continue
        end
        if j.privateNucMutations.totalReversionSubstitutions > revs_thresh
            continue
        end
############################################################################
        name = EPI_ISL(j.seqName)
        private_del_ct = length(j.privateNucMutations.privateDeletionRanges)
        privAA = j.privateAaMutations
        ORF1a_AA = privAA.ORF1a
        ORF1b_AA = privAA.ORF1b
        ORF3a_AA = privAA.ORF3a
        ORF6_AA = privAA.ORF6
        ORF7a_AA = privAA.ORF7a
        ORF7b_AA = privAA.ORF7b
        ORF8_AA = privAA.ORF8
        ORF9b_AA = privAA.ORF9b
        S_AA = privAA.S
        E_AA = privAA.E
        M_AA = privAA.M
        N_AA = privAA.N
#            if haskey(j.privateAaMutations, "ORF1a") && haskey(j.privateAaMutations, "ORF1b") && haskey(j.privateAaMutations, "ORF3a") && haskey(j.privateAaMutations, "ORF6") && haskey(j.privateAaMutations, "ORF7a") && haskey(j.privateAaMutations, "ORF7b") && haskey(j.privateAaMutations, "ORF8") && haskey(j.privateAaMutations, "ORF9b") && haskey(j.privateAaMutations, "S") && haskey(j.privateAaMutations, "E") && haskey(j.privateAaMutations, "M") && haskey(j.privateAaMutations, "N")
        total_private_AA_muts = ORF1a_AA.totalPrivateSubstitutions + ORF1b_AA.totalPrivateSubstitutions + ORF3a_AA.totalPrivateSubstitutions + ORF6_AA.totalPrivateSubstitutions + ORF7a_AA.totalPrivateSubstitutions + ORF7b_AA.totalPrivateSubstitutions + ORF8_AA.totalPrivateSubstitutions + ORF9b_AA.totalPrivateSubstitutions + S_AA.totalPrivateSubstitutions + E_AA.totalPrivateSubstitutions + M_AA.totalPrivateSubstitutions + N_AA.totalPrivateSubstitutions + private_del_ct
        seq_date = ""
        if total_private_AA_muts ≤ max_AA_mut
            date_tuple = get_full_year_month_day_tuple(j.seqName)
            if date_tuple == nothing
                try
                    seq_date = string(sequence_date(j.seqName))
                catch
                    seq_date = "0-0-0"
                end
                dash_ct = dash_count(seq_date)
                if dash_ct == 0
                    seq_y = seq_date
                    if all(isdigit, seq_y)
                        seq_y = parse(Int64, seq_y)
                        if seq_y ≥ 2020 && seq_y ≤ 2027
                            seq_year = seq_y
                        end
                    end
                end
                if dash_ct == 1
                    seq_y = string(split(seq_date, "-")[1])
                    seq_m = string(split(seq_date, "-")[2])
                    if all(isdigit, seq_y)
                        seq_y = parse(Int64, seq_y)
                        if seq_y ≥ 2020 && seq_y ≤ 2027
                            seq_year = seq_y
                        end
                    end
                    if all(isdigit, seq_m)
                        seq_m = parse(Int64, seq_m)
                        if seq_m ≥ 1 && seq_m ≤ 12
                            seq_month = seq_m
                        end
                    end
                end
                if dash_ct == 2
                    seq_y = string(split(seq_date, "-")[1])
                    seq_m = string(split(seq_date, "-")[2])
                    seq_d = string(split(seq_date, "-")[3])
                    if all(isdigit, seq_y)
                        seq_y = parse(Int64, seq_y)
                        if seq_y ≥ 2020 && seq_y ≤ 2027
                            seq_year = seq_y
                        end
                    end
                    if all(isdigit, seq_m)
                        seq_m = parse(Int64, seq_m)
                        if seq_m ≥ 1 && seq_m ≤ 12
                            seq_month = seq_m
                        end
                    end
                    if all(isdigit, seq_d)
                        seq_d = parse(Int64, seq_d)
                        if seq_d ≥ 1 && seq_d ≤ 31
                            seq_day = seq_d
                        end
                    end
                end
                date_tuple = (seq_year, seq_month, seq_day)
            end
            seq_date_tuple[name] = date_tuple
####################################################
        end
    end
####################################################################################################################################
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_date_tuple.jld2", "seq_date_tuple", seq_date_tuple)
####################################################################################################################################
end

In [ ]:
### Execute seq_date_tuple_only ***********  90, 5, 8000    Runtime =   ****************
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
################################################                     
start_clade_pango_date_ct = time()                                                     ### max_AA_mut=90   revs_thresh=5   qc_max=8000 
seq_date_tuple_only("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 90, 5, 8000, "all_private_muts")
finish_clade_pango_date_ct = time() - start_clade_pango_date_ct
time1, time2 = seconds_to_hrs_min_sec(finish_clade_pango_date_ct)
println(time1); println(time2)
#   input_ndjson, ndjson_name, max_AA_mut, revs_thresh, qc_max, print_ct_thresh, fx_name
##  Start: : AM, 2025-5-
### Total Function Run Time =     2025-09-01  QC= 90,5,8000
### Total Function Run Time = 
# (input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String)
#####################################################################################################################################
#####################################################################################################################################

In [ ]:
### Fx: seq_pango_all only
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
function seq_pango_all_only(input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String)
    #                          clade     date_index  count 
############################################################################################################################################################################    all_seq_stat_size_hint = 17500000
####################################################################################################################################
    HQCS_qualifying_seqs = Set{String}()
    if max_AA_mut == 5 && revs_thresh == 1 && qc_max == 5
        HQCS_qualifying_seqs = HQCS_qualified_seqs_5_1_5
    elseif max_AA_mut == 10 && revs_thresh == 1 && qc_max == 30
        HQCS_qualifying_seqs = HQCS_qualified_seqs_10_1_30
    elseif max_AA_mut == 90 && revs_thresh == 5 && qc_max == 8000
        HQCS_qualifying_seqs = HQCS_qualified_seqs_90_5_8000
    end
    all_seq_stat_size_hint = 17500000
#    all_seq_stat_size_hint_90_5_8000 = 17128637
    all_seq_stat_size_hint_90_5_8000 = 17103176
    all_seq_stat_size_hint_10_1_30 = 14601802
    all_seq_stat_size_hint_5_1_5 = 10849566
    if max_AA_mut == 90 && qc_max == 8000
        all_seq_stat_size_hint = all_seq_stat_size_hint_90_5_8000
    elseif max_AA_mut == 10 && qc_max == 30
        all_seq_stat_size_hint = all_seq_stat_size_hint_10_1_30
    elseif max_AA_mut == 5 && qc_max == 5
        all_seq_stat_size_hint = all_seq_stat_size_hint_5_1_5
    end
####################################################################################################################################
    seq_pango = Dict{String, String}();                          sizehint!(seq_pango, all_seq_stat_size_hint)
############################################################################################################################################################################
    hundred_thousand_ct = 0
    all_seq_ct = 0
    start_time = time()
    hundred_k_time = time()
    for line in eachline(input_ndjson)
        all_seq_ct += 1
        if all_seq_ct % 100000 == 0
            hundred_thousand_ct += 1
            hundred_k_interval = round(digits=1, time() - hundred_k_time)
            total_time = round(digits=1, time() - start_time)
            println("100000 ct = $(hundred_thousand_ct) | Time for last 100K = $(hundred_k_interval) seconds | Total Time = $(total_time)")
            GC.gc(true)
            hundred_k_time = time()
        end
############################################################################
        j = JSON3.read(line)
############################################################################
        name = EPI_ISL(j.seqName)
        if name in HQCS_qualifying_seqs
#        del_ranges = j.privateNucMutations.privateDeletionRanges
#        private_del_ct = 0
#        for range in del_ranges
#            private_del_ct += 1
#        end
#            if haskey(j.privateAaMutations, "ORF1a") && haskey(j.privateAaMutations, "ORF1b") && haskey(j.privateAaMutations, "ORF3a") && haskey(j.privateAaMutations, "ORF6") && haskey(j.privateAaMutations, "ORF7a") && haskey(j.privateAaMutations, "ORF7b") && haskey(j.privateAaMutations, "ORF8") && haskey(j.privateAaMutations, "ORF9b") && haskey(j.privateAaMutations, "S") && haskey(j.privateAaMutations, "E") && haskey(j.privateAaMutations, "M") && haskey(j.privateAaMutations, "N")
#        total_private_AA_muts = j.privateAaMutations.ORF1a.totalPrivateSubstitutions + j.privateAaMutations.ORF1b.totalPrivateSubstitutions + j.privateAaMutations.ORF3a.totalPrivateSubstitutions + j.privateAaMutations.ORF6.totalPrivateSubstitutions + j.privateAaMutations.ORF7a.totalPrivateSubstitutions + j.privateAaMutations.ORF7b.totalPrivateSubstitutions + j.privateAaMutations.ORF8.totalPrivateSubstitutions + j.privateAaMutations.ORF9b.totalPrivateSubstitutions + j.privateAaMutations.S.totalPrivateSubstitutions + j.privateAaMutations.E.totalPrivateSubstitutions + j.privateAaMutations.M.totalPrivateSubstitutions + j.privateAaMutations.N.totalPrivateSubstitutions + private_del_ct
#        if j.totalDeletions < 1500 && total_private_AA_muts ≤ max_AA_mut && j.qc.overallScore ≤ qc_max && j.privateNucMutations.totalReversionSubstitutions ≤ revs_thresh && j.qc.frameShifts.totalFrameShifts - j.qc.frameShifts.totalFrameShiftsIgnored ≤ 1
            try
                seq_pango[name] = j.customNodeAttributes.Nextclade_pango
            catch
                seq_pango[name] = "Unknown"
            end
        end
    end
####################################################
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_pango.jld2", "seq_pango", seq_pango)
####################################################################################################################################
end

In [ ]:
### Execute seq_pango_all_only
start = time()
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
seq_pango_all_only("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 90, 5, 8000, 1, "all_private_muts")
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v1 = $(runtime1)")
println("Runtime v2 = $(runtime2)")
####################################################################################################################################
####################################################################################################################################

In [ ]:
### Fx: seq_clade_all only
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
function seq_clade_all_only(input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String)
    #                          clade     date_index  count 
############################################################################################################################################################################    all_seq_stat_size_hint = 17500000
####################################################################################################################################
    HQCS_qualifying_seqs = Set{String}()
    if max_AA_mut == 5 && revs_thresh == 1 && qc_max == 5
        HQCS_qualifying_seqs = HQCS_qualified_seqs_5_1_5
    elseif max_AA_mut == 10 && revs_thresh == 1 && qc_max == 30
        HQCS_qualifying_seqs = HQCS_qualified_seqs_10_1_30
    elseif max_AA_mut == 90 && revs_thresh == 5 && qc_max == 8000
        HQCS_qualifying_seqs = HQCS_qualified_seqs_90_5_8000
    end
    all_seq_stat_size_hint = 17500000
#    all_seq_stat_size_hint_90_5_8000 = 17128637
    all_seq_stat_size_hint_90_5_8000 = 17103176
    all_seq_stat_size_hint_10_1_30 = 14601802
    all_seq_stat_size_hint_5_1_5 = 10849566
    if max_AA_mut == 90 && qc_max == 8000
        all_seq_stat_size_hint = all_seq_stat_size_hint_90_5_8000
    elseif max_AA_mut == 10 && qc_max == 30
        all_seq_stat_size_hint = all_seq_stat_size_hint_10_1_30
    elseif max_AA_mut == 5 && qc_max == 5
        all_seq_stat_size_hint = all_seq_stat_size_hint_5_1_5
    end
####################################################################################################################################
    seq_clade = Dict{String, String}();                          sizehint!(seq_clade, all_seq_stat_size_hint)
############################################################################################################################################################################
    hundred_thousand_ct = 0
    all_seq_ct = 0
    start_time = time()
    hundred_k_time = time()
    for line in eachline(input_ndjson)
        all_seq_ct += 1
        if all_seq_ct % 100000 == 0
            hundred_thousand_ct += 1
            hundred_k_interval = round(digits=1, time() - hundred_k_time)
            total_time = round(digits=1, time() - start_time)
            println("100000 ct = $(hundred_thousand_ct) | Time for last 100K = $(hundred_k_interval) seconds | Total Time = $(total_time)")
            GC.gc(true)
            hundred_k_time = time()
        end
############################################################################
        j = JSON3.read(line)
############################################################################
        name = EPI_ISL(j.seqName)
        if name in HQCS_qualifying_seqs
#        del_ranges = j.privateNucMutations.privateDeletionRanges
#        private_del_ct = 0
#        for range in del_ranges
#            private_del_ct += 1
#        end
#            if haskey(j.privateAaMutations, "ORF1a") && haskey(j.privateAaMutations, "ORF1b") && haskey(j.privateAaMutations, "ORF3a") && haskey(j.privateAaMutations, "ORF6") && haskey(j.privateAaMutations, "ORF7a") && haskey(j.privateAaMutations, "ORF7b") && haskey(j.privateAaMutations, "ORF8") && haskey(j.privateAaMutations, "ORF9b") && haskey(j.privateAaMutations, "S") && haskey(j.privateAaMutations, "E") && haskey(j.privateAaMutations, "M") && haskey(j.privateAaMutations, "N")
#        total_private_AA_muts = j.privateAaMutations.ORF1a.totalPrivateSubstitutions + j.privateAaMutations.ORF1b.totalPrivateSubstitutions + j.privateAaMutations.ORF3a.totalPrivateSubstitutions + j.privateAaMutations.ORF6.totalPrivateSubstitutions + j.privateAaMutations.ORF7a.totalPrivateSubstitutions + j.privateAaMutations.ORF7b.totalPrivateSubstitutions + j.privateAaMutations.ORF8.totalPrivateSubstitutions + j.privateAaMutations.ORF9b.totalPrivateSubstitutions + j.privateAaMutations.S.totalPrivateSubstitutions + j.privateAaMutations.E.totalPrivateSubstitutions + j.privateAaMutations.M.totalPrivateSubstitutions + j.privateAaMutations.N.totalPrivateSubstitutions + private_del_ct
#        if j.totalDeletions < 1500 && total_private_AA_muts ≤ max_AA_mut && j.qc.overallScore ≤ qc_max && j.privateNucMutations.totalReversionSubstitutions ≤ revs_thresh && j.qc.frameShifts.totalFrameShifts - j.qc.frameShifts.totalFrameShiftsIgnored ≤ 1
            try
                seq_clade[name] = j.clade
            catch
                seq_clade[name] = "Unknown"
            end
        end
    end
####################################################
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_clade.jld2", "seq_clade", seq_clade)
####################################################################################################################################
end

In [ ]:
### Execute seq_clade_all_only   Runtime: 2 hr, 35 min, 13.35 sec (10-1-30, 2026_04_18)
start = time()
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
seq_clade_all_only("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 10, 1, 30, 1, "all_private_muts")
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v1 = $(runtime1)")
println("Runtime v2 = $(runtime2)")
####################################################################################################################################
####################################################################################################################################

In [ ]:
### Fx: seq_labeled_subs only
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
function seq_labeled_subs_only(input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String)
    #                          clade     date_index  count 
####################################################################################################################################
    HQCS_qualifying_seqs = Set{String}()
    if max_AA_mut == 5 && revs_thresh == 1 && qc_max == 5
        HQCS_qualifying_seqs = HQCS_qualified_seqs_5_1_5
    elseif max_AA_mut == 10 && revs_thresh == 1 && qc_max == 30
        HQCS_qualifying_seqs = HQCS_qualified_seqs_10_1_30
    elseif max_AA_mut == 90 && revs_thresh == 5 && qc_max == 8000
        HQCS_qualifying_seqs = HQCS_qualified_seqs_90_5_8000
    end
    all_seq_stat_size_hint = 17500000
#    all_seq_stat_size_hint_90_5_8000 = 17128637
    all_seq_stat_size_hint_90_5_8000 = 17103176
    all_seq_stat_size_hint_10_1_30 = 14601802
    all_seq_stat_size_hint_5_1_5 = 10849566
    if max_AA_mut == 90 && qc_max == 8000
        all_seq_stat_size_hint = all_seq_stat_size_hint_90_5_8000
    elseif max_AA_mut == 10 && qc_max == 30
        all_seq_stat_size_hint = all_seq_stat_size_hint_10_1_30
    elseif max_AA_mut == 5 && qc_max == 5
        all_seq_stat_size_hint = all_seq_stat_size_hint_5_1_5
    end
####################################################################################################################################
    seq_labeled_subs = Dict{String, Int}();                          sizehint!(seq_labeled_subs, all_seq_stat_size_hint)
############################################################################################################################################################################
    hundred_thousand_ct = 0
    all_seq_ct = 0
    start_time = time()
    hundred_k_time = time()
    for line in eachline(input_ndjson)
        all_seq_ct += 1
        if all_seq_ct % 100000 == 0
            hundred_thousand_ct += 1
            hundred_k_interval = round(digits=1, time() - hundred_k_time)
            total_time = round(digits=1, time() - start_time)
            println("100000 ct = $(hundred_thousand_ct) | Time for last 100K = $(hundred_k_interval) seconds | Total Time = $(total_time)")
            GC.gc(true)
            hundred_k_time = time()
        end
############################################################################
        j = JSON3.read(line)
############################################################################
        name = EPI_ISL(j.seqName)
        if name in HQCS_qualifying_seqs
#        del_ranges = j.privateNucMutations.privateDeletionRanges
#        private_del_ct = 0
#        for range in del_ranges
#            private_del_ct += 1
#        end
#        total_private_AA_muts = j.privateAaMutations.ORF1a.totalPrivateSubstitutions + j.privateAaMutations.ORF1b.totalPrivateSubstitutions + j.privateAaMutations.ORF3a.totalPrivateSubstitutions + j.privateAaMutations.ORF6.totalPrivateSubstitutions + j.privateAaMutations.ORF7a.totalPrivateSubstitutions + j.privateAaMutations.ORF7b.totalPrivateSubstitutions + j.privateAaMutations.ORF8.totalPrivateSubstitutions + j.privateAaMutations.ORF9b.totalPrivateSubstitutions + j.privateAaMutations.S.totalPrivateSubstitutions + j.privateAaMutations.E.totalPrivateSubstitutions + j.privateAaMutations.M.totalPrivateSubstitutions + j.privateAaMutations.N.totalPrivateSubstitutions + private_del_ct
#        if j.totalDeletions < 1500 && total_private_AA_muts ≤ max_AA_mut && j.qc.overallScore ≤ qc_max && j.privateNucMutations.totalReversionSubstitutions ≤ revs_thresh && j.qc.frameShifts.totalFrameShifts - j.qc.frameShifts.totalFrameShiftsIgnored ≤ 2
            seq_labeled_subs[name] = j.privateNucMutations.totalLabeledSubstitutions
        end
    end
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_labeled_subs.jld2", "seq_labeled_subs", seq_labeled_subs)
end
############################################################################################################################################################################
############################################################################################################################################################################

In [ ]:
### Execute seq_labeled_subs_only  | 2 hr, 17 min, 03.47 sec
start = time()
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
seq_labeled_subs_only("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 90, 5, 8000, 1, "all_private_muts")
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v1 = $(runtime1)")
println("Runtime v2 = $(runtime2)")
####################################################################################################################################
####################################################################################################################################

In [ ]:
### Fx: date_nuc_mut_ct_only 
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
function date_nuc_mut_ct_only(input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String)        
    date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
    date = "2026_01_06"
    start_time = time()
    no_key_set = Set{String}()
####################################################################################################################################
    HQCS_qualifying_seqs = Set{String}()
    if max_AA_mut == 5 && revs_thresh == 1 && qc_max == 5
        HQCS_qualifying_seqs = HQCS_qualified_seqs_5_1_5
    elseif max_AA_mut == 10 && revs_thresh == 1 && qc_max == 30
        HQCS_qualifying_seqs = HQCS_qualified_seqs_10_1_30
    elseif max_AA_mut == 90 && revs_thresh == 5 && qc_max == 8000
        HQCS_qualifying_seqs = HQCS_qualified_seqs_90_5_8000
    end
    all_seq_stat_size_hint = 17500000
#    all_seq_stat_size_hint_90_5_8000 = 17128637
    all_seq_stat_size_hint_90_5_8000 = 17103176
    all_seq_stat_size_hint_10_1_30 = 14601802
    all_seq_stat_size_hint_5_1_5 = 10849566
    if max_AA_mut == 90 && qc_max == 8000
        all_seq_stat_size_hint = all_seq_stat_size_hint_90_5_8000
    elseif max_AA_mut == 10 && qc_max == 30
        all_seq_stat_size_hint = all_seq_stat_size_hint_10_1_30
    elseif max_AA_mut == 5 && qc_max == 5
        all_seq_stat_size_hint = all_seq_stat_size_hint_5_1_5
    end
####################################################################################################################################
    date_regex = r"(\d{4})-(\d{2})-(\d{2})"
##### Explanation of    date_regex = r"(\d{4})-(\d{2})-(\d{2})"   #####################
##### r = regular expression
##### \d = any digit
##### {4} = number of preceding character to match (in this case the number of digits)
##### () = capture groups. Allow you to recall whatever is inside the parentheses later.
#          If date = "2025-09-01", then  date.capture[1] = "2025", date.capture[2] = "09", etc
##### So this checks the j.seqName line for dates in the correct format
#########################################################################################
    function get_full_year_month_day_tuple(seqName::String)
        ymd = match(date_regex, seqName)
        if ymd ≠ nothing
            year = parse(Int64, ymd.captures[1])
            month = parse(Int64, ymd.captures[2])
            day = parse(Int64, ymd.captures[3])
            if year ≥ 2020 && year ≤ 2027 && month ≥ 1 && month ≤ 12 && day ≥ 1 && day ≤ 31
                return (year, month, day)
            end
        end
        return nothing
    end
    function dash_count(str::String)
        return count(c -> c == '-', str)
    end
####################################################################################################################################
    excluded_pos = BitSet([1:60..., 76:78..., 686:694..., 2453, 3241, 5515, 7926, 11283:11295..., 14960, 15521, 19209:19210..., 19212, 19214, 19217, 21595, 21766, 21772, 21987, 21995:21996..., 22882, 24130, 27291, 28360:28373..., 29632, 29732:29736..., 29757, 29759:29761..., 29830:29900...])
    excluded_AA = Set(["ORF1a:V3917G", "ORF1a:E4388K", "ORF1b:N498I", "ORF1b:F685Y", "S:S27A", "S:K113N", "S:Q115H", "S:D142G", "S:Y145D", "S:I794N", "S:S408R", "S:K440N"])

#### For dicts below: 1st Key = date index; 2nd Key = mutation, value = mutation count on date index
    date_nuc_mut_ct = Dict{Int, Dict{String, Int}}();                 sizehint!(date_nuc_mut_ct, 2500)
    date_nuc_mut_ct_no_dels = Dict{Int, Dict{String, Int}}();         sizehint!(date_nuc_mut_ct_no_dels, 2500)

    all_seq_ct::Int = 0
    qualifying_seq_ct::Int = 0
###################################################################################################################################
    total_nuc_dels::Int = 0
    total_subs::Int = 0
#####################################################################################################################################    
    
    max_dels = 1500
    max_frameshifts = 2
    max_stops = 1
    max_SNPclust = 101
    min_coverage = 0.50

    interval_start = time()
    for line in eachline(input_ndjson)
        all_seq_ct += 1
        if all_seq_ct % 250000 == 0
            sequences_complete = rpad(all_seq_ct, 8)
            interval_time = round(digits=2, time() - interval_start)
            interval_time1, interval_time2 = seconds_to_hrs_min_sec(interval_time)
            runtime_elapsed = round(digits=2, time() - start_time)
            runtime1_elapsed, runtime2_elapsed = seconds_to_hrs_min_sec(runtime_elapsed)
            println("Seqs Complete = $(sequences_complete) | Total Time Elapsed = $(runtime2_elapsed) | Interval Time = $(interval_time2)")
            interval_start = time()
            nowtime = Dates.format(now(), "I:MM.SS_p"); print(nowtime); println()
        end
############################################################################
        j = JSON3.read(line)
############################################################################
        name = EPI_ISL(j.seqName)
        if name in HQCS_qualifying_seqs
#        if haskey(j, "privateAaMutations") && haskey(j, "qc") && haskey(j, "totalDeletions") && haskey(j, "privateNucMutations")
#        if haskey(j.privateAaMutations, "ORF1a") && haskey(j.privateAaMutations, "ORF1b") && haskey(j.privateAaMutations, "ORF3a") && haskey(j.privateAaMutations, "ORF6") && haskey(j.privateAaMutations, "ORF7a") && haskey(j.privateAaMutations, "ORF7b") && haskey(j.privateAaMutations, "ORF8") && haskey(j.privateAaMutations, "ORF9b") && haskey(j.privateAaMutations, "S") && haskey(j.privateAaMutations, "E") && haskey(j.privateAaMutations, "M") && haskey(j.privateAaMutations, "N")
            nuc_muts = j.privateNucMutations
#        if j.totalDeletions ≥ max_dels
#            continue
#        end
#        QC = j.qc
#        if QC.overallScore > qc_max
#            continue
#        end
#        if QC.frameShifts.totalFrameShifts - QC.frameShifts.totalFrameShiftsIgnored > max_frameshifts
#            continue
#        end
#        if QC.stopCodons.totalStopCodons - QC.stopCodons.totalStopCodonsIgnored > max_stops
#            continue
#        end
#        if QC.snpClusters.score ≥ max_SNPclust
#            continue
#        end
#        if j.coverage < min_coverage
#            continue
#        end
#        if nuc_muts.totalReversionSubstitutions > revs_thresh
#            continue
#        end
############################################################################
#        name = EPI_ISL(j.seqName)
#        del_ranges = nuc_muts.privateDeletionRanges
#        private_del_ct = length(nuc_muts.privateDeletionRanges)
#        pango = ""
#        privAA = j.privateAaMutations
#        ORF1a_AA = privAA.ORF1a
#        ORF1b_AA = privAA.ORF1b
#        ORF3a_AA = privAA.ORF3a
#        ORF6_AA = privAA.ORF6
#        ORF7a_AA = privAA.ORF7a
#        ORF7b_AA = privAA.ORF7b
#        ORF8_AA = privAA.ORF8
#        ORF9b_AA = privAA.ORF9b
#        S_AA = privAA.S
#        E_AA = privAA.E
#        M_AA = privAA.M
#        N_AA = privAA.N
#        total_private_AA_muts = ORF1a_AA.totalPrivateSubstitutions + ORF1b_AA.totalPrivateSubstitutions + ORF3a_AA.totalPrivateSubstitutions + ORF6_AA.totalPrivateSubstitutions + ORF7a_AA.totalPrivateSubstitutions + ORF7b_AA.totalPrivateSubstitutions + ORF8_AA.totalPrivateSubstitutions + ORF9b_AA.totalPrivateSubstitutions + S_AA.totalPrivateSubstitutions + E_AA.totalPrivateSubstitutions + M_AA.totalPrivateSubstitutions + N_AA.totalPrivateSubstitutions
#        if total_private_AA_muts + private_del_ct ≤ max_AA_mut && !(name in all_chronic_seqs_set)
            qualifying_seq_ct += 1
######################################################################################
            seq_date = ""
            seq_year = Int64(0)
            seq_month = Int64(0)
            seq_day = Int64(0)
            date_tuple = get_full_year_month_day_tuple(j.seqName)
            if date_tuple == nothing
                try
                    seq_date = string(sequence_date(j.seqName))
                catch
                    seq_date = "0-0-0"
                end
                dash_ct = dash_count(seq_date)
                if dash_ct == 0
                    seq_y = seq_date
                    if all(isdigit, seq_y)
                        seq_y = parse(Int64, seq_y)
                        if seq_y ≥ 2020 && seq_y ≤ 2027
                            seq_year = seq_y
                        end
                    end
                end
                if dash_ct == 1
                    seq_y = string(split(seq_date, "-")[1])
                    seq_m = string(split(seq_date, "-")[2])
                    if all(isdigit, seq_y)
                        seq_y = parse(Int64, seq_y)
                        if seq_y ≥ 2020 && seq_y ≤ 2027
                            seq_year = seq_y
                        end
                    end
                    if all(isdigit, seq_m)
                        seq_m = parse(Int64, seq_m)
                        if seq_m ≥ 1 && seq_m ≤ 12
                            seq_month = seq_m
                        end
                    end
                end
                if dash_ct == 2
                    seq_y = string(split(seq_date, "-")[1])
                    seq_m = string(split(seq_date, "-")[2])
                    seq_d = string(split(seq_date, "-")[3])
                    if all(isdigit, seq_y)
                        seq_y = parse(Int64, seq_y)
                        if seq_y ≥ 2020 && seq_y ≤ 2027
                            seq_year = seq_y
                        end
                    end
                    if all(isdigit, seq_m)
                        seq_m = parse(Int64, seq_m)
                        if seq_m ≥ 1 && seq_m ≤ 12
                            seq_month = seq_m
                        end
                    end
                    if all(isdigit, seq_d)
                        seq_d = parse(Int64, seq_d)
                        if seq_d ≥ 1 && seq_d ≤ 31
                            seq_day = seq_d
                        end
                    end
                end
                date_tuple = (seq_year, seq_month, seq_day)
            end
            date_index = date_to_index[date_tuple]
#################################################################################################################################
            subs = nuc_muts.privateSubstitutions
            dels = nuc_muts.privateDeletions
#                  seq_nuc_del_ranges_ct[name] = private_del_count
            for k in subs
                if k.pos < 29870
                    pos = k.pos + 1
                    if !(pos in excluded_pos) && k.refNuc ≠ "-" && k.qryNuc ≠ string(ref[pos])
                        nuc_sub = "$(k.refNuc)$(string(pos))$(k.qryNuc)"
##########################################################################
                        get!(date_nuc_mut_ct, date_index, Dict{String, Int64}() )
                        date_nuc_mut_ct[date_index][nuc_sub] = get(date_nuc_mut_ct[date_index], nuc_sub, 0) + 1
##########################################################################
                    end
                end
            end
            for d in dels
                pos = d.pos + 1
                if !(pos in excluded_pos)
                    total_nuc_dels += 1
                    n_del = "$(d.refNuc)$(string(pos))-"    
##########################################################################
                    get!(date_nuc_mut_ct, date_index, Dict{String, Int64}() ) 
                    date_nuc_mut_ct[date_index][n_del] = get(date_nuc_mut_ct[date_index], n_del, 0) + 1
##########################################################################
                end
            end
        end
    end          
#####################################################################################################################################  
    for (date, nuc_mut_ct_dict) in date_nuc_mut_ct
        for (mut, ct) in nuc_mut_ct_dict
            get!(date_nuc_mut_ct_no_dels, date, Dict{String, Int}() )
            if mut[end] ≠ '-'
                date_nuc_mut_ct_no_dels[date][mut] = ct
            end
        end
    end
####################################################################################################################################
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_nuc_mut_ct.jld2", "date_nuc_mut_ct", date_nuc_mut_ct)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_nuc_mut_ct_no_dels.jld2", "date_nuc_mut_ct_no_dels", date_nuc_mut_ct_no_dels)
####################################################################################################################################
    println("Dictionaries saved!"); println()
    date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now); println()
end
################################################################################################################

In [ ]:
### Execute date_nuc_mut_ct_only   Runtime:  (10-1-30, 2026_04_18)
start = time()
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
date_nuc_mut_ct_only("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 10, 1, 30, 1, "all_private_muts")
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v1 = $(runtime1)")
println("Runtime v2 = $(runtime2)")
####################################################################################################################################
####################################################################################################################################

In [ ]:
### Fx: nuc_muts_seq_only 
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
function nuc_muts_seq_only(input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String)        
    date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
    date = "2026_01_06"
    start_time = time()
####################################################################################################################################
    HQCS_qualifying_seqs = Set{String}()
    if max_AA_mut == 5 && revs_thresh == 1 && qc_max == 5
        HQCS_qualifying_seqs = HQCS_qualified_seqs_5_1_5
    elseif max_AA_mut == 10 && revs_thresh == 1 && qc_max == 30
        HQCS_qualifying_seqs = HQCS_qualified_seqs_10_1_30
    elseif max_AA_mut == 90 && revs_thresh == 5 && qc_max == 8000
        HQCS_qualifying_seqs = HQCS_qualified_seqs_90_5_8000
    end
    all_seq_stat_size_hint = 17500000
#    all_seq_stat_size_hint_90_5_8000 = 17128637
    all_seq_stat_size_hint_90_5_8000 = 17103176
    all_seq_stat_size_hint_10_1_30 = 14601802
    all_seq_stat_size_hint_5_1_5 = 10849566
    if max_AA_mut == 90 && qc_max == 8000
        all_seq_stat_size_hint = all_seq_stat_size_hint_90_5_8000
    elseif max_AA_mut == 10 && qc_max == 30
        all_seq_stat_size_hint = all_seq_stat_size_hint_10_1_30
    elseif max_AA_mut == 5 && qc_max == 5
        all_seq_stat_size_hint = all_seq_stat_size_hint_5_1_5
    end
####################################################################################################################################
    excluded_pos = BitSet([1:60..., 76:78..., 686:694..., 2453, 3241, 5515, 7926, 11283:11295..., 14960, 15521, 19209:19210..., 19212, 19214, 19217, 21595, 21766, 21772, 21987, 21995:21996..., 22882, 24130, 27291, 28360:28373..., 29632, 29732:29736..., 29757, 29759:29761..., 29830:29900...])

    nuc_muts_seq = Dict{String, Set{String}}();           sizehint!(nuc_muts_seq, 50280)
    nuc_muts_seq_no_dels = Dict{String, Set{String}}();   sizehint!(nuc_muts_seq_no_dels, 50280)

    all_seq_ct::Int = 0
    qualifying_seq_ct::Int = 0

    qualified_seqs_90_5_8000 = Set{String}()
###################################################################################################################################
    total_nuc_dels::Int = 0
    total_subs::Int = 0
#####################################################################################################################################    

    max_dels = 1500
    max_frameshifts = 2
    max_stops = 1
    max_SNPclust = 101
    min_coverage = 0.50

    interval_start = time()
    for line in eachline(input_ndjson)
        all_seq_ct += 1
        if all_seq_ct % 250000 == 0
            sequences_complete = rpad(all_seq_ct, 8)
            interval_time = round(digits=2, time() - interval_start)
            interval_time1, interval_time2 = seconds_to_hrs_min_sec(interval_time)
            runtime_elapsed = round(digits=2, time() - start_time)
            runtime1_elapsed, runtime2_elapsed = seconds_to_hrs_min_sec(runtime_elapsed)
            println("Seqs Complete = $(sequences_complete) | Total Time Elapsed = $(runtime2_elapsed) | Interval Time = $(interval_time2)")
            interval_start = time()
            nowtime = Dates.format(now(), "I:MM.SS_p"); print(nowtime); println()
        end
############################################################################
        j = JSON3.read(line)
############################################################################
        name = EPI_ISL(j.seqName)
        if name in HQCS_qualifying_seqs
#        if haskey(j, "privateAaMutations") && haskey(j, "qc") && haskey(j, "totalDeletions") && haskey(j, "privateNucMutations")
#        if haskey(j.privateAaMutations, "ORF1a") && haskey(j.privateAaMutations, "ORF1b") && haskey(j.privateAaMutations, "ORF3a") && haskey(j.privateAaMutations, "ORF6") && haskey(j.privateAaMutations, "ORF7a") && haskey(j.privateAaMutations, "ORF7b") && haskey(j.privateAaMutations, "ORF8") && haskey(j.privateAaMutations, "ORF9b") && haskey(j.privateAaMutations, "S") && haskey(j.privateAaMutations, "E") && haskey(j.privateAaMutations, "M") && haskey(j.privateAaMutations, "N")
#        nuc_muts = j.privateNucMutations
#        if j.totalDeletions ≥ max_dels
#            continue
#        end
#        QC = j.qc
#        if QC.overallScore > qc_max
#            continue
#        end
#        if QC.frameShifts.totalFrameShifts - QC.frameShifts.totalFrameShiftsIgnored > max_frameshifts
#            continue
#        end
#        if QC.stopCodons.totalStopCodons - QC.stopCodons.totalStopCodonsIgnored > max_stops
#            continue
#        end
#        if QC.snpClusters.score ≥ max_SNPclust
#            continue
#        end
#        if j.coverage < min_coverage
#            continue
#        end
#        if nuc_muts.totalReversionSubstitutions > revs_thresh
#            continue
#        end
############################################################################
#        name = EPI_ISL(j.seqName)
#            del_ranges = nuc_muts.privateDeletionRanges
#            private_del_ct = length(nuc_muts.privateDeletionRanges)
#            pango = ""
#        privAA = j.privateAaMutations
#        ORF1a_AA = privAA.ORF1a
#        ORF1b_AA = privAA.ORF1b
#        ORF3a_AA = privAA.ORF3a
#        ORF6_AA = privAA.ORF6
#        ORF7a_AA = privAA.ORF7a
#        ORF7b_AA = privAA.ORF7b
#        ORF8_AA = privAA.ORF8
#        ORF9b_AA = privAA.ORF9b
#        S_AA = privAA.S
#        E_AA = privAA.E
#        M_AA = privAA.M
#        N_AA = privAA.N
#        total_private_AA_muts = ORF1a_AA.totalPrivateSubstitutions + ORF1b_AA.totalPrivateSubstitutions + ORF3a_AA.totalPrivateSubstitutions + ORF6_AA.totalPrivateSubstitutions + ORF7a_AA.totalPrivateSubstitutions + ORF7b_AA.totalPrivateSubstitutions + ORF8_AA.totalPrivateSubstitutions + ORF9b_AA.totalPrivateSubstitutions + S_AA.totalPrivateSubstitutions + E_AA.totalPrivateSubstitutions + M_AA.totalPrivateSubstitutions + N_AA.totalPrivateSubstitutions
#        if total_private_AA_muts + private_del_ct ≤ max_AA_mut && !(name in all_chronic_seqs_set)
            qualifying_seq_ct += 1
#################################################################################################################################
            genes = ["ORF1a", "ORF1b", "ORF3a", "ORF6", "ORF7a", "ORF7b", "ORF8", "ORF9b", "S", "E", "M", "N"]
            subs = nuc_muts.privateSubstitutions
            dels = nuc_muts.privateDeletions
            for k in subs
                if k.pos < 29870
                    pos = k.pos + 1
                    if !(pos in excluded_pos) && k.refNuc ≠ "-" && k.qryNuc ≠ string(ref[pos])
                        nuc_sub = "$(k.refNuc)$(string(pos))$(k.qryNuc)"
                        get!(nuc_muts_seq, nuc_sub, Set{String}() )
                        push!(nuc_muts_seq[nuc_sub], name)
                        get!(nuc_muts_seq_no_dels, nuc_sub, Set{String}() )
                        push!(nuc_muts_seq_no_dels[nuc_sub], name)
                    end
                end
            end
            for k in dels
                if k.pos < 29870
                    pos = k.pos + 1
                    if !(pos in excluded_pos)
                        nuc_sub = "$(k.refNuc)$(string(pos))$(k.qryNuc)"
                        get!(nuc_muts_seq, nuc_sub, Set{String}() )
                        push!(nuc_muts_seq[nuc_sub], name)
                    end
                end
            end
        end
    end
####################################################################################################################################   
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_seq.jld2", "nuc_muts_seq", nuc_muts_seq)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_seq_no_dels.jld2", "nuc_muts_seq_no_dels", nuc_muts_seq_no_dels)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_qualified_seqs_90_5_8000.jld2", "qualified_seqs_90_5_8000", qualified_seqs_90_5_8000)
####################################################################################################################################
    println("Dictionaries saved!"); println()
    date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now); println()
end
####################################################################################################################################
####################################################################################################################################

In [ ]:
### Execute nuc_muts_seq_only   Runtime: 2 hr, 48 min, 08.50 sec (90-5-8000, 2026_04_18)
start = time()
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
nuc_muts_seq_only("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 90, 5, 8000, 1, "all_private_muts")
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v1 = $(runtime1)")
println("Runtime v2 = $(runtime2)")
####################################################################################################################################
####################################################################################################################################

In [ ]:
### Fx: AA_dels_seq_only 
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
function AA_dels_seq_only(input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String)        
    date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
    date = "2026_01_06"
    start_time = time()
############################################################################################################################################################################
    HQCS_qualifying_seqs = Set{String}()
    if max_AA_mut == 5 && revs_thresh == 1 && qc_max == 5
        HQCS_qualifying_seqs = HQCS_qualified_seqs_5_1_5
    elseif max_AA_mut == 10 && revs_thresh == 1 && qc_max == 30
        HQCS_qualifying_seqs = HQCS_qualified_seqs_10_1_30
    elseif max_AA_mut == 90 && revs_thresh == 5 && qc_max == 8000
        HQCS_qualifying_seqs = HQCS_qualified_seqs_90_5_8000
    end
    all_seq_stat_size_hint = 17500000
#    all_seq_stat_size_hint_90_5_8000 = 17128637
    all_seq_stat_size_hint_90_5_8000 = 17103176
    all_seq_stat_size_hint_10_1_30 = 14601802
    all_seq_stat_size_hint_5_1_5 = 10849566
    if max_AA_mut == 90 && qc_max == 8000
        all_seq_stat_size_hint = all_seq_stat_size_hint_90_5_8000
    elseif max_AA_mut == 10 && qc_max == 30
        all_seq_stat_size_hint = all_seq_stat_size_hint_10_1_30
    elseif max_AA_mut == 5 && qc_max == 5
        all_seq_stat_size_hint = all_seq_stat_size_hint_5_1_5
    end
    
    max_dels = 1500
    max_frameshifts = 2
    max_stops = 1
    max_SNPclust = 101
    min_coverage = 0.50

    qualifying_seq_ct = 0

####################################################################################################################################
    AA_dels_seq = Dict{String, Set{String}}();         # sizehint!(AA_muts_seq, 53578)
###################################################################################################################################
    all_seq_ct = 0
    interval_start = time()
    for line in eachline(input_ndjson)
        all_seq_ct += 1
        if all_seq_ct % 250000 == 0
            sequences_complete = rpad(all_seq_ct, 8)
            interval_time = round(digits=2, time() - interval_start)
            interval_time1, interval_time2 = seconds_to_hrs_min_sec(interval_time)
            runtime_elapsed = round(digits=2, time() - start_time)
            runtime1_elapsed, runtime2_elapsed = seconds_to_hrs_min_sec(runtime_elapsed)
            println("Seqs Complete = $(sequences_complete) | Total Time Elapsed = $(runtime2_elapsed) | Interval Time = $(interval_time2)")
            interval_start = time()
            nowtime = Dates.format(now(), "I:MM.SS_p"); print(nowtime); println()
        end
############################################################################
        j = JSON3.read(line)
############################################################################
#        if haskey(j, "privateAaMutations") && haskey(j, "qc") && haskey(j, "totalDeletions") && haskey(j, "privateNucMutations")
#        if haskey(j.privateAaMutations, "ORF1a") && haskey(j.privateAaMutations, "ORF1b") && haskey(j.privateAaMutations, "ORF3a") && haskey(j.privateAaMutations, "ORF6") && haskey(j.privateAaMutations, "ORF7a") && haskey(j.privateAaMutations, "ORF7b") && haskey(j.privateAaMutations, "ORF8") && haskey(j.privateAaMutations, "ORF9b") && haskey(j.privateAaMutations, "S") && haskey(j.privateAaMutations, "E") && haskey(j.privateAaMutations, "M") && haskey(j.privateAaMutations, "N")
        nuc_muts = j.privateNucMutations
        if j.totalDeletions ≥ max_dels
            continue
        end
        QC = j.qc
        if QC.overallScore > qc_max
            continue
        end
        if QC.frameShifts.totalFrameShifts - QC.frameShifts.totalFrameShiftsIgnored > max_frameshifts
            continue
        end
        if QC.stopCodons.totalStopCodons - QC.stopCodons.totalStopCodonsIgnored > max_stops
            continue
        end
        if QC.snpClusters.score ≥ max_SNPclust
            continue
        end
        if j.coverage < min_coverage
            continue
        end
        if nuc_muts.totalReversionSubstitutions > revs_thresh
            continue
        end
############################################################################
#        del_ranges = nuc_muts.privateDeletionRanges
#        private_del_ct = length(nuc_muts.privateDeletionRanges)
#        pango = ""
#        privAA = j.privateAaMutations
#        ORF1a_AA = privAA.ORF1a
#        ORF1b_AA = privAA.ORF1b
#        ORF3a_AA = privAA.ORF3a
#        ORF6_AA = privAA.ORF6
#        ORF7a_AA = privAA.ORF7a
#        ORF7b_AA = privAA.ORF7b
#        ORF8_AA = privAA.ORF8
#        ORF9b_AA = privAA.ORF9b
#        S_AA = privAA.S
#        E_AA = privAA.E
#        M_AA = privAA.M
#        N_AA = privAA.N
#        total_private_AA_muts = ORF1a_AA.totalPrivateSubstitutions + ORF1b_AA.totalPrivateSubstitutions + ORF3a_AA.totalPrivateSubstitutions + ORF6_AA.totalPrivateSubstitutions + ORF7a_AA.totalPrivateSubstitutions + ORF7b_AA.totalPrivateSubstitutions + ORF8_AA.totalPrivateSubstitutions + ORF9b_AA.totalPrivateSubstitutions + S_AA.totalPrivateSubstitutions + E_AA.totalPrivateSubstitutions + M_AA.totalPrivateSubstitutions + N_AA.totalPrivateSubstitutions
#        if total_private_AA_muts + private_del_ct ≤ max_AA_mut && !(name in all_chronic_seqs_set)
        name = EPI_ISL(j.seqName)
        if name in HQCS_qualifying_seqs
            push!(HQCS_qualified_seqs_10_1_30, name)
            qualifying_seq_ct += 1
#################################################################################################################################
            genes = ["ORF1a", "ORF1b", "ORF3a", "ORF6", "ORF7a", "ORF7b", "ORF8", "ORF9b", "S", "E", "M", "N"]
            pango = ""
            privAA = j.privateAaMutations
            ORF1a_AA = privAA.ORF1a
            ORF1b_AA = privAA.ORF1b
            ORF3a_AA = privAA.ORF3a
            ORF6_AA = privAA.ORF6
            ORF7a_AA = privAA.ORF7a
            ORF7b_AA = privAA.ORF7b
            ORF8_AA = privAA.ORF8
            ORF9b_AA = privAA.ORF9b
            S_AA = privAA.S
            E_AA = privAA.E
            M_AA = privAA.M
            N_AA = privAA.N
#            j["privateAaMutations"]["S"]["privateDeletions"][3]  ...etc
#            j["privateAaMutations"]["S"]["privateDeletions"][1]["refAa"]
#            j["privateAaMutations"]["S"]["privateDeletions"][1]["cdsName"]
#            j["privateAaMutations"]["S"]["privateDeletions"][1]["pos"]
            AAdels_tuple = ( ("ORF1a", ORF1a_AA.privateDeletions), ("ORF1b", ORF1b_AA.privateDeletions), 
                ("ORF3a", ORF3a_AA.privateDeletions), ("ORF6", ORF6_AA.privateDeletions), ("ORF7a", ORF7a_AA.privateDeletions), 
                ("ORF7b", ORF7b_AA.privateDeletions), ("ORF8", ORF8_AA.privateDeletions), ("ORF9b", ORF9b_AA.privateDeletions), 
                ("S", S_AA.privateDeletions), ("E", E_AA.privateDeletions), ("M", M_AA.privateDeletions), ("N", N_AA.privateDeletions) ) 
            for (orf_str, dels) in AAdels_tuple
                for del in dels
                    del_AA = "$(orf_str):$(del.refAa)$(del.pos + 1)-"
                    AA_dels_seq[del_AA] = get(AA_dels_seq, del_AA, Set{String}())
                    push!(AA_dels_seq[del_AA], name)
                end
            end
        end
    end
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_dels_seq.jld2", "AA_dels_seq", AA_dels_seq)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_HQCS_qualified_seqs_10_1_30.jld2", "HQCS_qualified_seqs_10_1_30", HQCS_qualified_seqs_10_1_30)
####################################################################################################################################
    println("Dictionaries saved!"); print("\n"^1)
end
################################################################################################################

In [ ]:
### Execute AA_dels_seq  | Runtime: 2 hr, 49 min, 28.68 sec (5-1-5, 2026_04_29) | Runtime: 3 hr, 19 min, 00.78 sec (10-1-13) | 
start = time()
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
AA_dels_seq_only("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 10, 1, 30, 1, "all_private_muts")
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v2 = $(runtime2)"); print("\n"^1)
####################################################################################################################################
####################################################################################################################################

In [ ]:
### Fx: seq_country_HQCS only
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
function seq_country_HQCS_only(input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String)
    #                          clade     date_index  count 
############################################################################################################################################################################
    HQCS_qualifying_seqs = Set{String}()
    if max_AA_mut == 5 && revs_thresh == 1 && qc_max == 5
        HQCS_qualifying_seqs = HQCS_qualified_seqs_5_1_5
    elseif max_AA_mut == 10 && revs_thresh == 1 && qc_max == 30
        HQCS_qualifying_seqs = HQCS_qualified_seqs_10_1_30
    elseif max_AA_mut == 90 && revs_thresh == 5 && qc_max == 8000
        HQCS_qualifying_seqs = HQCS_qualified_seqs_90_5_8000
    end
    all_seq_stat_size_hint = 17500000
#    all_seq_stat_size_hint_90_5_8000 = 17128637
    all_seq_stat_size_hint_90_5_8000 = 17103176
    all_seq_stat_size_hint_10_1_30 = 14601802
    all_seq_stat_size_hint_5_1_5 = 10849566
    if max_AA_mut == 90 && qc_max == 8000
        all_seq_stat_size_hint = all_seq_stat_size_hint_90_5_8000
    elseif max_AA_mut == 10 && qc_max == 30
        all_seq_stat_size_hint = all_seq_stat_size_hint_10_1_30
    elseif max_AA_mut == 5 && qc_max == 5
        all_seq_stat_size_hint = all_seq_stat_size_hint_5_1_5
    end
    HQCS_qualified_seqs_90_5_8000 = Set{String}()
    seq_country_HQCS = Dict{String, String}();                          sizehint!(seq_country_HQCS, all_seq_stat_size_hint)
############################################################################################################################################################################
    start_time = time()
    all_seq_ct = 0
    interval_start = time()
    for line in eachline(input_ndjson)
        all_seq_ct += 1
        if all_seq_ct % 250000 == 0
            sequences_complete = rpad(all_seq_ct, 8)
            interval_time = round(digits=2, time() - interval_start)
            interval_time1, interval_time2 = seconds_to_hrs_min_sec(interval_time)
            runtime_elapsed = round(digits=2, time() - start_time)
            runtime1_elapsed, runtime2_elapsed = seconds_to_hrs_min_sec(runtime_elapsed)
            println("Seqs Complete = $(sequences_complete) | Total Time Elapsed = $(runtime2_elapsed) | Interval Time = $(interval_time2)")
            interval_start = time()
            nowtime = Dates.format(now(), "I:MM.SS_p"); print(nowtime); println()
        end
        j = JSON3.read(line)
        name = EPI_ISL(j.seqName)
        if name in HQCS_qualifying_seqs
            country = countree(j.seqName)
            seq_country_HQCS[name] = country
        end
    end
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_country_HQCS.jld2", "seq_country_HQCS", seq_country_HQCS)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_HQCS_qualified_seqs_90_5_8000.jld2", "HQCS_qualified_seqs_90_5_8000", HQCS_qualified_seqs_90_5_8000)
end
############################################################################################################################################################################
############################################################################################################################################################################

In [ ]:
### Execute seq_country_HQCS_only  |  Runtime: 3 hr, 15 min, 43.25 sec (90-5-8000)
start = time()
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
seq_country_HQCS_only("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 90, 5, 8000, 1, "all_private_muts")
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v1 = $(runtime1)")
println("Runtime v2 = $(runtime2)")
####################################################################################################################################
####################################################################################################################################

In [ ]:
### Fx: country_seq_HQCS only
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
function country_seq_HQCS_only(input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String)
    #                          clade     date_index  count 
############################################################################################################################################################################
####################################################################################################################################
    HQCS_qualifying_seqs = Set{String}()
    if max_AA_mut == 5 && revs_thresh == 1 && qc_max == 5
        HQCS_qualifying_seqs = HQCS_qualified_seqs_5_1_5
    elseif max_AA_mut == 10 && revs_thresh == 1 && qc_max == 30
        HQCS_qualifying_seqs = HQCS_qualified_seqs_10_1_30
    elseif max_AA_mut == 90 && revs_thresh == 5 && qc_max == 8000
        HQCS_qualifying_seqs = HQCS_qualified_seqs_90_5_8000
    end
    all_seq_stat_size_hint = 17500000
#    all_seq_stat_size_hint_90_5_8000 = 17128637
    all_seq_stat_size_hint_90_5_8000 = 17103176
    all_seq_stat_size_hint_10_1_30 = 14601802
    all_seq_stat_size_hint_5_1_5 = 10849566
    if max_AA_mut == 90 && qc_max == 8000
        all_seq_stat_size_hint = all_seq_stat_size_hint_90_5_8000
    elseif max_AA_mut == 10 && qc_max == 30
        all_seq_stat_size_hint = all_seq_stat_size_hint_10_1_30
    elseif max_AA_mut == 5 && qc_max == 5
        all_seq_stat_size_hint = all_seq_stat_size_hint_5_1_5
    end
####################################################################################################################################
    country_seq_HQCS = Dict{String, String}();                          
############################################################################################################################################################################
    start_time = time()
    all_seq_ct = 0
    interval_start = time()
    for line in eachline(input_ndjson)
        all_seq_ct += 1
        if all_seq_ct % 250000 == 0
            sequences_complete = rpad(all_seq_ct, 8)
            interval_time = round(digits=2, time() - interval_start)
            interval_time1, interval_time2 = seconds_to_hrs_min_sec(interval_time)
            runtime_elapsed = round(digits=2, time() - start_time)
            runtime1_elapsed, runtime2_elapsed = seconds_to_hrs_min_sec(runtime_elapsed)
            println("Seqs Complete = $(sequences_complete) | Total Time Elapsed = $(runtime2_elapsed) | Interval Time = $(interval_time2)")
            interval_start = time()
            nowtime = Dates.format(now(), "I:MM.SS_p"); print(nowtime); println()
        end
        j = JSON3.read(line)
        name = EPI_ISL(j.seqName)
        if name in HQCS_qualifying_seqs
#            country = countree(j.seqName)
            country = country_seq_HQCS[name]
            country_seq_HQCS[country] = get(country_seq_HQCS, country, Set{String}())
            push!(country_seq_HQCS[country], name)
#            push!(HQCS_qualified_seqs_90_5_8000, name)
        end
    end
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_country_HQCS.jld2", "seq_country_HQCS", seq_country_HQCS)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_HQCS_qualified_seqs_90_5_8000.jld2", "HQCS_qualified_seqs_90_5_8000", HQCS_qualified_seqs_90_5_8000)
end
############################################################################################################################################################################
############################################################################################################################################################################

In [ ]:
### Fx: country_clade_pango_date_index_cts ####################################
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
countree(n) = split(n, "/")[2]
US_state(n) = split(split(n, "/")[3], "-")[1]
function country_clade_pango_date_index_cts(input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String)
####################################################################################################################################
    HQCS_qualifying_seqs = Set{String}()
    if max_AA_mut == 5 && revs_thresh == 1 && qc_max == 5
        HQCS_qualifying_seqs = HQCS_qualified_seqs_5_1_5
    elseif max_AA_mut == 10 && revs_thresh == 1 && qc_max == 30
        HQCS_qualifying_seqs = HQCS_qualified_seqs_10_1_30
    elseif max_AA_mut == 90 && revs_thresh == 5 && qc_max == 8000
        HQCS_qualifying_seqs = HQCS_qualified_seqs_90_5_8000
    end
    all_seq_stat_size_hint = 17500000
#    all_seq_stat_size_hint_90_5_8000 = 17128637
    all_seq_stat_size_hint_90_5_8000 = 17103176
    all_seq_stat_size_hint_10_1_30 = 14601802
    all_seq_stat_size_hint_5_1_5 = 10849566
    if max_AA_mut == 90 && qc_max == 8000
        all_seq_stat_size_hint = all_seq_stat_size_hint_90_5_8000
    elseif max_AA_mut == 10 && qc_max == 30
        all_seq_stat_size_hint = all_seq_stat_size_hint_10_1_30
    elseif max_AA_mut == 5 && qc_max == 5
        all_seq_stat_size_hint = all_seq_stat_size_hint_5_1_5
    end
####################################################################################################################################
#                                     country    clade     date_index   count  
#    country_clade_date_index_ct = Dict{String, Dict{String, Dict{Int, Int}}}()
    country_pango_date_index_ct = Dict{String, Dict{String, Dict{Int, Int}}}()
#    country_pango_unaliased_date_index_ct = Dict{String, Dict{String, Dict{Int, Int}}}()
####################################################################################################################################
####################################################################################################################################
#   hundred_thousand_ct = 0
    for line in eachline(input_ndjson)
#        all_seq_ct += 1
#        hundred_thousand_ct = 0
#        if all_seq_ct % 100000 == 0
#            println("Processed $all_seq_ct sequences, forcing garbage collection...")
#            println("100000 ct = $(hundred_thousand_ct)")
#            GC.gc(true)
#        end
        j = JSON3.read(line)
        name = EPI_ISL(j.seqName)
        del_ranges = j.privateNucMutations.privateDeletionRanges
        private_del_ct = 0
        country = countree(j.seqName)
        USA_state = US_state(j.seqName)
        seq_date = ""
#        clade = ""
        pango = ""
#        pango_unaliased = ""
        for range in del_ranges
            private_del_ct += 1
        end
#            if haskey(j.privateAaMutations, "ORF1a") && haskey(j.privateAaMutations, "ORF1b") && haskey(j.privateAaMutations, "ORF3a") && haskey(j.privateAaMutations, "ORF6") && haskey(j.privateAaMutations, "ORF7a") && haskey(j.privateAaMutations, "ORF7b") && haskey(j.privateAaMutations, "ORF8") && haskey(j.privateAaMutations, "ORF9b") && haskey(j.privateAaMutations, "S") && haskey(j.privateAaMutations, "E") && haskey(j.privateAaMutations, "M") && haskey(j.privateAaMutations, "N")
        total_private_AA_muts = j.privateAaMutations.ORF1a.totalPrivateSubstitutions + j.privateAaMutations.ORF1b.totalPrivateSubstitutions + j.privateAaMutations.ORF3a.totalPrivateSubstitutions + j.privateAaMutations.ORF6.totalPrivateSubstitutions + j.privateAaMutations.ORF7a.totalPrivateSubstitutions + j.privateAaMutations.ORF7b.totalPrivateSubstitutions + j.privateAaMutations.ORF8.totalPrivateSubstitutions + j.privateAaMutations.ORF9b.totalPrivateSubstitutions + j.privateAaMutations.S.totalPrivateSubstitutions + j.privateAaMutations.E.totalPrivateSubstitutions + j.privateAaMutations.M.totalPrivateSubstitutions + j.privateAaMutations.N.totalPrivateSubstitutions + private_del_ct
        if j.totalDeletions < 1500 && total_private_AA_muts ≤ max_AA_mut && j.qc.overallScore ≤ qc_max && j.privateNucMutations.totalReversionSubstitutions ≤ revs_thresh && j.qc.frameShifts.totalFrameShifts - j.qc.frameShifts.totalFrameShiftsIgnored == 0
####################################################
#            try
#                clade = j.clade
#            catch
#                clade = "Unknown"
#            end
####################################################
            try
                pango = j.customNodeAttributes.Nextclade_pango
            catch
                pango = "Unknown"
            end
####################################################
#            try
#                pango_unaliased = j.customNodeAttributes.partiallyAliased
#            catch
#                pango_unaliased = "Unknown"
#            end
####################################################
            seq_date = ""
            try
                seq_date = sequence_date(j.seqName)
            catch
                seq_date = "0-0-0"
            end
####################################################
            seq_year = 0
            seq_month = 0
            seq_day = 0
            seq_yr = join(seq_date[1:4])
            if all(isdigit, seq_yr)
                seq_year = parse(Int, seq_yr)
                if '-' in seq_date
                    seq_mo = split(seq_date, "-")[2]
                    if all(isdigit, seq_mo)
                        seq_month = parse(Int, seq_mo)
                    end
                end
####################################################
                dash_ct = 0
                for i in seq_date
                    if i == '-'
                        dash_ct += 1
                    end
                end
####################################################
                if dash_ct == 2
                    seq_da = split(seq_date, "-")[3]
                    if all(isdigit, seq_da)
                        seq_day = parse(Int, seq_da)
                    end
                end
            end
####################################################
            date_index = date_to_index[(seq_year, seq_month, seq_day)]
            date_tuple = (seq_year, seq_month, seq_day)
####################################################
#            if !haskey(country_clade_date_index_ct, country)
#                country_clade_date_index_ct[country] = Dict{String, Dict{Int, Int}}()
#            end
#            if !haskey(country_clade_date_index_ct[country], clade)
#                country_clade_date_index_ct[country][clade] = Dict{Int, Int}()
#            end
#            if !haskey(country_clade_date_index_ct[country][clade], date_index)
#                country_clade_date_index_ct[country][clade][date_index] = 1
#            else
#                country_clade_date_index_ct[country][clade][date_index] += 1
#            end
####################################################
            if !haskey(country_pango_date_index_ct, country)
                country_pango_date_index_ct[country] = Dict{String, Dict{Int, Int}}()
            end
            if !haskey(country_pango_date_index_ct[country], pango)
                country_pango_date_index_ct[country][pango] = Dict{Int, Int}()
            end
            if !haskey(country_pango_date_index_ct[country][pango], date_index)
                country_pango_date_index_ct[country][pango][date_index] = 1
            else
                country_pango_date_index_ct[country][pango][date_index] += 1
            end
####################################################
#            if !haskey(country_pango_unaliased_date_index_ct, country)
#                country_pango_unaliased_date_index_ct[country] = Dict{String, Dict{Int, Int}}()
#            end
#            if !haskey(country_pango_unaliased_date_index_ct[country], pango_unaliased)
#                country_pango_unaliased_date_index_ct[country][pango_unaliased] = Dict{Int, Int}()
#            end
#            if !haskey(country_pango_unaliased_date_index_ct[country][pango_unaliased], date_index)
#                country_pango_unaliased_date_index_ct[country][pango_unaliased][date_index] = 1
#            else
#                country_pango_unaliased_date_index_ct[country][pango_unaliased][date_index] += 1
#            end
####################################################
        end
    end
####################################################################################################################################
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_country_clade_date_index_ct.jld2", "country_clade_date_index_ct", country_clade_date_index_ct)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_country_pango_date_index_ct.jld2", "country_pango_date_index_ct", country_pango_date_index_ct)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_country_pango_unaliased_date_index_ct.jld2", "country_pango_unaliased_date_index_ct", country_pango_unaliased_date_index_ct)
####################################################################################################################################
end

In [ ]:
### Execute country_clade_pango_date_index_cts ###########################
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
start_country_clade_pango_date_ct = time()
country_clade_pango_date_index_cts("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 5, 1, 5, 2, "all_private_muts")
finish_country_clade_pango_date_ct = time() - start_country_clade_pango_date_ct
country_time1, country_time2 = seconds_to_hrs_min_sec(finish_country_clade_pango_date_ct)
println(country_time1); println(country_time2)

# 2025_06_11_136PM
# 1:23:11.64
# 1 hours, 23 minutes, 11.64 seconds
#   input_ndjson, ndjson_name, max_AA_mut, revs_thresh, qc_max, print_ct_thresh, fx_name
##  Start: 4:51 PM, 2025-5-13 
### Total Function Run Time =  seconds
### Total Function Run Time = 16842.4 seconds = 4 hours, 40 minutes, 42 seconds = 4:40.42
#   all_private_muts_10_06_2024("test2000.ndjson", "test2000", 5, 1, 5, 2, "test2000")

In [ ]:
### Fx: country_pango_date_index_cts
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
function country_pango_date_index_cts(input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String)
    #                          clade     date_index  count
#    clade_date_index_ct = Dict{String, Dict{Int, Int}}()
#    pango_date_index_ct = Dict{String, Dict{Int, Int}}()
#    pango_unaliased_date_index_ct = Dict{String, Dict{Int, Int}}()
#                                     country    date_index   clade   count  
#    country_clade_date_index_ct = Dict{String, Dict{String, Dict{Int, Int}}}()
    country_pango_date_index_ct = Dict{String, Dict{String, Dict{Int, Int}}}()
#    country_pango_unaliased_date_index_ct = Dict{String, Dict{String, Dict{Int, Int}}}()
####################################################################################################################################
#    for clade in clade_set
#        clade_date_index_ct[clade] = Dict{Int, Int}()
#    end
#    for pango in pango_set
#        pango_date_index_ct[pango] = Dict{Int, Int}()
#    end
#    for pango_unaliased in pango_unaliased_set
#        pango_unaliased_date_index_ct[pango_unaliased] = Dict{Int, Int}()
#    end
#    for clade in clade_set
#        for i in 1:2922
#            clade_date_index_ct[clade][i] = 0
#        end
#        for i in ["0", "1", "2", "3", "4", "5", "6", "7"]
#            for j in ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
#                dindex = parse(Int, "202"*i*"00"*j*"000")
#                clade_date_index_ct[clade][dindex] = 0
#            end
#            for j in ["10", "11", "12"]
#                dindex = parse(Int, "202"*i*"0"*j*"000")
#                clade_date_index_ct[clade][dindex] = 0
#            end 
#        end
#        clade_date_index_ct[clade][1000000000] = 0
#    end
#    for pango in pango_set
#        for i in 1:2922
#            pango_date_index_ct[pango][i] = 0
#        end
#        for i in ["0", "1", "2", "3", "4", "5", "6", "7"]
#            for j in ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
#                dindex = parse(Int, "202"*i*"00"*j*"000")
#                pango_date_index_ct[pango][dindex] = 0
#            end
#            for j in ["10", "11", "12"]
#                dindex = parse(Int, "202"*i*"0"*j*"000")
#                pango_date_index_ct[pango][dindex] = 0
#            end 
#        end
#        pango_date_index_ct[pango][1000000000] = 0
#    end
#for y in 2020:2027
#    date_to_index[(y, 0, 0)] = y*1000000
#    index_to_date[y*1000000] = (y, 0, 0)
#    for m in 1:12
#        date_to_index[(y, m, 0)] = y*1000000 + m*1000
#        index_to_date[y*1000000 + m*1000] = (y, m, 0)
#    end
#end
#    for pango_unaliased in pango_unaliased_set
#        for i in 1:2922
#            pango_unaliased_date_index_ct[pango_unaliased][i] = 0
#        end
#        for i in ["0", "1", "2", "3", "4", "5", "6", "7"]
#            for j in ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
#                dindex = parse(Int, "202"*i*"00"*j*"000")
#                pango_unaliased_date_index_ct[pango_unaliased][dindex] = 0
#            end
#            for j in ["10", "11", "12"]
#                dindex = parse(Int, "202"*i*"0"*j*"000")
#                pango_unaliased_date_index_ct[pango_unaliased][dindex] = 0
#            end 
#        end
#        pango_unaliased_date_index_ct[pango_unaliased][1000000000] = 0
#    end
    for country in country_set
#        country_clade_date_index_ct[country] = Dict{String, Dict{Int, Int}}()
        country_pango_date_index_ct[country] = Dict{String, Dict{Int, Int}}()
#        country_pango_unaliased_date_index_ct[country] = Dict{String, Dict{Int, Int}}()
    end
    for country in country_set
#        for clade in clade_set
#            country_clade_date_index_ct[country][clade] = Dict{Int, Int}()
#        end
        for pango in pango_set
            country_pango_date_index_ct[country][pango] = Dict{Int, Int}()
        end
#        for pango_unaliased in pango_unaliased_set
#            country_pango_unaliased_date_index_ct[country][pango_unaliased] = Dict{Int, Int}()
#        end
    end
    for country in country_set
#        for clade in clade_set
#            for i in 1:2922
#                country_clade_date_index_ct[country][clade][i] = 0
#            end
#            for i in ["0", "1", "2", "3", "4", "5", "6", "7"]
#                for j in ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
#                    dindex = parse(Int, "202"*i*"00"*j*"000")
#                    country_clade_date_index_ct[clade][dindex] = 0
#                end
#                for j in ["10", "11", "12"]
#                    dindex = parse(Int, "202"*i*"0"*j*"000")
#                    country_clade_date_index_ct[clade][dindex] = 0
#                end 
#            end
#        end
        for pango in pango_set
            for i in 1:2922
                country_pango_date_index_ct[country][pango][i] = 0
            end
            for i in ["0", "1", "2", "3", "4", "5", "6", "7"]
                for j in ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
                    dindex = parse(Int, "202"*i*"00"*j*"000")
                    country_pango_date_index_ct[pango][dindex] = 0
                end
                for j in ["10", "11", "12"]
                    dindex = parse(Int, "202"*i*"0"*j*"000")
                    country_pango_date_index_ct[pango][dindex] = 0
                end 
            end
            country_pango_date_index_ct[pango][1000000000] = 0
        end
#        for pango_unaliased in pango_unaliased_set
#            for i in 1:2922
#                country_pango_unaliased_date_index_ct[country][pango_unaliased][i] = 0
#            end
#            for i in ["0", "1", "2", "3", "4", "5", "6", "7"]
#                for j in ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
#                    dindex = parse(Int, "202"*i*"00"*j*"000")
#                    country_pango_unaliased_date_index_ct[pango_unaliased][dindex] = 0
#                end
#                for j in ["10", "11", "12"]
#                    dindex = parse(Int, "202"*i*"0"*j*"000")
#                    country_pango_unaliased_date_index_ct[pango_unaliased][dindex] = 0
#                end 
#            end
#        end
    end   
####################################################################################################################################
#    seq_country = Dict{String, String}()
#    seq_US_state = Dict{String, String}()
#    seq_clade = Dict{String, String}()
#    seq_pango = Dict{String, String}()
#    seq_pango_unaliased = Dict{String, String}()
#    seq_collection_date = Dict{String, String}()
#    seq_date_index = Dict{String, Int}()
#    seq_date_tuple = Dict{String, Tuple{Int64, Int64, Int64}}()
#    seq_lab_dict = Dict{String, String}()
####################################################################################################################################
####################################################################################################################################
####################################################################################################################################
#   hundred_thousand_ct = 0
    for line in eachline(input_ndjson)
#        all_seq_ct += 1
#        if all_seq_ct % 100000 == 0
#            hundred_thousand_ct += 1
#            println("Processed $all_seq_ct sequences, forcing garbage collection...")
#            println("100000 ct = $(hundred_thousand_ct)")
#            GC.gc(true)
#        end
        j = JSON3.read(line)
        name = EPI_ISL(j.seqName)
        del_ranges = j.privateNucMutations.privateDeletionRanges
        private_del_ct = 0
#        clade = ""
        pango = ""
#        pango_unaliased = ""
        for range in del_ranges
            private_del_ct += 1
        end
#            if haskey(j.privateAaMutations, "ORF1a") && haskey(j.privateAaMutations, "ORF1b") && haskey(j.privateAaMutations, "ORF3a") && haskey(j.privateAaMutations, "ORF6") && haskey(j.privateAaMutations, "ORF7a") && haskey(j.privateAaMutations, "ORF7b") && haskey(j.privateAaMutations, "ORF8") && haskey(j.privateAaMutations, "ORF9b") && haskey(j.privateAaMutations, "S") && haskey(j.privateAaMutations, "E") && haskey(j.privateAaMutations, "M") && haskey(j.privateAaMutations, "N")
        total_private_AA_muts = j.privateAaMutations.ORF1a.totalPrivateSubstitutions + j.privateAaMutations.ORF1b.totalPrivateSubstitutions + j.privateAaMutations.ORF3a.totalPrivateSubstitutions + j.privateAaMutations.ORF6.totalPrivateSubstitutions + j.privateAaMutations.ORF7a.totalPrivateSubstitutions + j.privateAaMutations.ORF7b.totalPrivateSubstitutions + j.privateAaMutations.ORF8.totalPrivateSubstitutions + j.privateAaMutations.ORF9b.totalPrivateSubstitutions + j.privateAaMutations.S.totalPrivateSubstitutions + j.privateAaMutations.E.totalPrivateSubstitutions + j.privateAaMutations.M.totalPrivateSubstitutions + j.privateAaMutations.N.totalPrivateSubstitutions + private_del_ct
        if j.totalDeletions < 1500 && total_private_AA_muts ≤ max_AA_mut && j.qc.overallScore ≤ qc_max && j.privateNucMutations.totalReversionSubstitutions ≤ revs_thresh && j.qc.frameShifts.totalFrameShifts - j.qc.frameShifts.totalFrameShiftsIgnored == 0
#            total_AA_subs += total_private_AA_muts
            country = countree(j.seqName)
            USA_state = US_state(j.seqName)
            seq_date = ""
####################################################
#            try
#                clade = j.clade
#            catch
#                clade = "Unknown"
#            end
####################################################
            try
                pango = j.customNodeAttributes.Nextclade_pango
            catch
                pango = "Unknown"
            end
####################################################
#            try
#                pango_unaliased = j.customNodeAttributes.partiallyAliased
#            catch
#                pango_unaliased = "Unknown"
#            end
####################################################
            seq_date = ""
            try
                seq_date = sequence_date(j.seqName)
            catch
                seq_date = "0-0-0"
            end
####################################################
#            lab = ""
#            if !isempty(seq_lab(j.seqName))
#                lab = seq_lab(j.seqName)
#            end
#            seq_lab_dict[name] = lab
            seq_year = 0
            seq_month = 0
            seq_day = 0
            seq_yr = join(seq_date[1:4])
            if all(isdigit, seq_yr)
                seq_year = parse(Int, seq_yr)
                if '-' in seq_date
                    seq_mo = split(seq_date, "-")[2]
                    if all(isdigit, seq_mo)
                        seq_month = parse(Int, seq_mo)
                    end
                end
####################################################
                dash_ct = 0
                for i in seq_date
                    if i == '-'
                        dash_ct += 1
                    end
                end
####################################################
                if dash_ct == 2
                    seq_da = split(seq_date, "-")[3]
                    if all(isdigit, seq_da)
                        seq_day = parse(Int, seq_da)
                    end
                end
            end
####################################################
#            seq_collection_date[name] = seq_date
#            if seq_month == 0 && seq_day == 0
#                seq_collection_date[name] = string(seq_year)*"-0-0"
#            end
#           if seq_month ≠ 0 && seq_day == 0
#                seq_collection_date[name] = string(seq_year)*"-"*string(seq_month)*"-0"
#            end
            date_index = date_to_index[(seq_year, seq_month, seq_day)]
#            seq_date_index[name] = date_index
#            date_tuple = (seq_year, seq_month, seq_day)
#            seq_date_tuple[name] = date_tuple
####################################################
#            if !haskey(pango_unaliased_date_index_ct[date_index], [seq_clade[name]])
#                pango_unaliased_date_index_ct[date_index][seq_pango_unaliased[name]] = 1
#            else
#                pango_unaliased_date_index_ct[date_index][seq_pango_unaliased[name]] += 1
#            end
###################################
#            clade = seq_clade[name]
#            pango = seq_pango[name]
#            pango_unaliased = seq_pango_unaliased[name]
###################################
#            clade_date_index_ct[clade][date_index] += 1
#            pango_date_index_ct[pango][date_index] += 1
#            pango_unaliased_date_index_ct[pango_unaliased][date_index] += 1
#            country_clade_date_index_ct[country][clade][date_index] += 1
            country_pango_date_index_ct[country][pango][date_index] += 1
#            country_pango_unaliased_date_index_ct[country][pango_unaliased][date_index] += 1
####################################################
        end
    end
####################################################################################################################################
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_clade_date_index_ct.jld2", "clade_date_index_ct", clade_date_index_ct)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_pango_date_index_ct.jld2", "pango_date_index_ct", pango_date_index_ct)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_pango_unaliased_date_index_ct.jld2", "pango_unaliased_date_index_ct", pango_unaliased_date_index_ct)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_country_clade_date_index_ct.jld2", "country_clade_date_index_ct", country_clade_date_index_ct)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_country_pango_date_index_ct.jld2", "country_pango_date_index_ct", country_pango_date_index_ct)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_country_pango_unaliased_date_index_ct.jld2", "country_pango_unaliased_date_index_ct", country_pango_unaliased_date_index_ct)
####################################################################################################################################
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_country.jld2", "seq_country", seq_country)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_US_state.jld2", "seq_US_state", seq_US_state)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_clade.jld2", "seq_clade", seq_clade)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_pango.jld2", "seq_pango", seq_pango)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_pango_unaliased.jld2", "seq_pango_unaliased", seq_pango_unaliased)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_collection_date.jld2", "seq_collection_date", seq_collection_date)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_date_index.jld2", "seq_date_index", seq_date_index)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_date_tuple.jld2", "seq_date_tuple", seq_date_tuple)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_lab_dict.jld2", "seq_lab_dict", seq_lab_dict)
####################################################################################################################################
end

In [ ]:
### Execute country_clade_pango_date_index_cts  ***********  5, 1, 5, 2 ****************
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
start_pango = time()
country_pango_date_index_cts("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 5, 1, 5, 2, "all_private_muts")
final_pango = time() - start_pango
pango_time1, pango_time2 = seconds_to_hrs_min_sec(final_pango)
println(pango_time1); println(pango_time2)
#   input_ndjson, ndjson_name, max_AA_mut, revs_thresh, qc_max, print_ct_thresh, fx_name
##  Start: 7:23 AM, 2025-5-10 
### Total Function Run Time =  seconds
### Total Function Run Time = 16842.4 seconds = 4 hours, 40 minutes, 42 seconds = 4:40.42
################################################################################################################

In [ ]:
### Execute country_clade_pango_date_index_cts  ***********  7, 1, 10, 2 ****************
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
start_pango = time()
country_pango_date_index_cts("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 7, 1, 10, 2, "all_private_muts")
final_pango = time() - start_pango
pango_time1, pango_time2 = seconds_to_hrs_min_sec(final_pango)
println(pango_time1); println(pango_time2)
#   input_ndjson, ndjson_name, max_AA_mut, revs_thresh, qc_max, print_ct_thresh, fx_name
##  Start: 
### Total Function Run Time =  seconds
### Total Function Run Time = 16842.4 seconds = 4 hours, 40 minutes, 42 seconds = 4:40.42
################################################################################################################

In [ ]:
### Execute country_clade_pango_date_index_cts  ***********  10, 1, 30, 2 ****************
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
start_pango = time()
country_pango_date_index_cts("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 10, 1, 30, 2, "all_private_muts")
final_pango = time() - start_pango
pango_time1, pango_time2 = seconds_to_hrs_min_sec(final_pango)
println(pango_time1); println(pango_time2)
#   input_ndjson, ndjson_name, max_AA_mut, revs_thresh, qc_max, print_ct_thresh, fx_name
### Total Function Run Time =  seconds
### Total Function Run Time = 16842.4 seconds = 4 hours, 40 minutes, 42 seconds = 4:40.42
################################################################################################################

In [ ]:
### Execute country_clade_pango_date_index_cts  ***********  90, 5, 8000, 2 ****************
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
start_pango = time()
country_pango_date_index_cts("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 90, 5, 8000, 2, "all_private_muts")
final_pango = time() - start_pango
pango_time1, pango_time2 = seconds_to_hrs_min_sec(final_pango)
println(pango_time1); println(pango_time2)
#   input_ndjson, ndjson_name, max_AA_mut, revs_thresh, qc_max, print_ct_thresh, fx_name
##  Start: 
### Total Function Run Time =  seconds
### Total Function Run Time = 16842.4 seconds = 4 hours, 40 minutes, 42 seconds = 4:40.42
################

In [ ]:
### Fx: country_clade_date_index_cts #####################
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
function country_clade_date_index_cts(input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String)
#                              clade     date_index  count
    clade_date_index_ct = Dict{String, Dict{Int, Int}}()
    pango_date_index_ct = Dict{String, Dict{Int, Int}}()
    pango_unaliased_date_index_ct = Dict{String, Dict{Int, Int}}()
#                                     country       clade    date_index  count  
    country_clade_date_index_ct = Dict{String, Dict{String, Dict{Int, Int}}}()
    country_pango_date_index_ct = Dict{String, Dict{String, Dict{Int, Int}}}()
    country_pango_unaliased_date_index_ct = Dict{String, Dict{String, Dict{Int, Int}}}()
####################################################################################################################################
#    for clade in clade_set
#        clade_date_index_ct[clade] = Dict{Int, Int}()
#    end
#    for pango in pango_set
#        pango_date_index_ct[pango] = Dict{Int, Int}()
#    end
#    for pango_unaliased in pango_unaliased_set
#        pango_unaliased_date_index_ct[pango_unaliased] = Dict{Int, Int}()
#    end
#    for clade in clade_set
#        for i in 1:2922
#            clade_date_index_ct[clade][i] = 0
#        end
#        for i in ["0", "1", "2", "3", "4", "5", "6", "7"]
#            for j in ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
#                dindex = parse(Int, "202"*i*"00"*j*"000")
#                clade_date_index_ct[clade][dindex] = 0
#            end
#            for j in ["10", "11", "12"]
#                dindex = parse(Int, "202"*i*"0"*j*"000")
#                clade_date_index_ct[clade][dindex] = 0
#            end 
#        end
#        clade_date_index_ct[clade][1000000000] = 0
#    end
#    for pango in pango_set
#        for i in 1:2922
#            pango_date_index_ct[pango][i] = 0
#        end
#        for i in ["0", "1", "2", "3", "4", "5", "6", "7"]
#            for j in ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
#                dindex = parse(Int, "202"*i*"00"*j*"000")
#                pango_date_index_ct[pango][dindex] = 0
#            end
#            for j in ["10", "11", "12"]
#                dindex = parse(Int, "202"*i*"0"*j*"000")
#                pango_date_index_ct[pango][dindex] = 0
#            end 
#        end
#        pango_date_index_ct[pango][1000000000] = 0
#    end
#for y in 2020:2027
#    date_to_index[(y, 0, 0)] = y*1000000
#    index_to_date[y*1000000] = (y, 0, 0)
#    for m in 1:12
#        date_to_index[(y, m, 0)] = y*1000000 + m*1000
#        index_to_date[y*1000000 + m*1000] = (y, m, 0)
#    end
#end
#    for pango_unaliased in pango_unaliased_set
#        for i in 1:2922
#            pango_unaliased_date_index_ct[pango_unaliased][i] = 0
#        end
#        for i in ["0", "1", "2", "3", "4", "5", "6", "7"]
#            for j in ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
#                dindex = parse(Int, "202"*i*"00"*j*"000")
#                pango_unaliased_date_index_ct[pango_unaliased][dindex] = 0
#            end
#            for j in ["10", "11", "12"]
#                dindex = parse(Int, "202"*i*"0"*j*"000")
#                pango_unaliased_date_index_ct[pango_unaliased][dindex] = 0
#            end 
#        end
#        pango_unaliased_date_index_ct[pango_unaliased][1000000000] = 0
#    end
#    for country in country_set
#        country_clade_date_index_ct[country] = Dict{String, Dict{Int, Int}}()
#        country_pango_date_index_ct[country] = Dict{String, Dict{Int, Int}}()
#        country_pango_unaliased_date_index_ct[country] = Dict{String, Dict{Int, Int}}()
#    end
#    for country in country_set
#        for clade in clade_set
#            country_clade_date_index_ct[country][clade] = Dict{Int, Int}()
#        end
#        for pango in pango_set
#            country_pango_date_index_ct[country][pango] = Dict{Int, Int}()
#        end
#        for pango_unaliased in pango_unaliased_set
#            country_pango_unaliased_date_index_ct[country][pango_unaliased] = Dict{Int, Int}()
#        end
#    end
#    for country in country_set
#        for clade in clade_set
#            for i in 1:2922
#                country_clade_date_index_ct[country][clade][i] = 0
#            end
#            for i in ["0", "1", "2", "3", "4", "5", "6", "7"]
#                for j in ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
#                    dindex = parse(Int, "202"*i*"00"*j*"000")
#                    country_clade_date_index_ct[clade][dindex] = 0
#                end
#                for j in ["10", "11", "12"]
#                    dindex = parse(Int, "202"*i*"0"*j*"000")
#                    country_clade_date_index_ct[clade][dindex] = 0
#                end 
#            end
#        end
#        for pango in pango_set
#            for i in 1:2922
#                country_pango_date_index_ct[country][pango][i] = 0
#            end
#            for i in ["0", "1", "2", "3", "4", "5", "6", "7"]
#                for j in ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
#                    dindex = parse(Int, "202"*i*"00"*j*"000")
#                    country_pango_date_index_ct[pango][dindex] = 0
#                end
#                for j in ["10", "11", "12"]
#                    dindex = parse(Int, "202"*i*"0"*j*"000")
#                    country_pango_date_index_ct[pango][dindex] = 0
#                end 
#            end
#            country_pango_date_index_ct[pango][1000000000] = 0
#        end
#        for pango_unaliased in pango_unaliased_set
#            for i in 1:2922
#                country_pango_unaliased_date_index_ct[country][pango_unaliased][i] = 0
#            end
#            for i in ["0", "1", "2", "3", "4", "5", "6", "7"]
#                for j in ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
#                    dindex = parse(Int, "202"*i*"00"*j*"000")
#                    country_pango_unaliased_date_index_ct[pango_unaliased][dindex] = 0
#                end
#                for j in ["10", "11", "12"]
#                    dindex = parse(Int, "202"*i*"0"*j*"000")
#                    country_pango_unaliased_date_index_ct[pango_unaliased][dindex] = 0
#                end 
#            end
#        end
#    end   
####################################################################################################################################
    seq_country = Dict{String, String}()
    seq_US_state = Dict{String, String}()
    seq_clade = Dict{String, String}()
    seq_pango = Dict{String, String}()
    seq_pango_unaliased = Dict{String, String}()
    seq_collection_date = Dict{String, String}()
    seq_date_index = Dict{String, Int}()
    seq_date_tuple = Dict{String, Tuple{Int64, Int64, Int64}}()
    seq_lab_dict = Dict{String, String}()
####################################################################################################################################
####################################################################################################################################
####################################################################################################################################
#   hundred_thousand_ct = 0
    for line in eachline(input_ndjson)
#        all_seq_ct += 1
#        if all_seq_ct % 100000 == 0
#            hundred_thousand_ct += 1
#            println("Processed $all_seq_ct sequences, forcing garbage collection...")
#            println("100000 ct = $(hundred_thousand_ct)")
#            GC.gc(true)
#        end
        j = JSON3.read(line)
        name = EPI_ISL(j.seqName)
        del_ranges = j.privateNucMutations.privateDeletionRanges
        private_del_ct = 0
        clade = ""
        pango = ""
        pango_unaliased = ""
        for range in del_ranges
            private_del_ct += 1
        end
#            if haskey(j.privateAaMutations, "ORF1a") && haskey(j.privateAaMutations, "ORF1b") && haskey(j.privateAaMutations, "ORF3a") && haskey(j.privateAaMutations, "ORF6") && haskey(j.privateAaMutations, "ORF7a") && haskey(j.privateAaMutations, "ORF7b") && haskey(j.privateAaMutations, "ORF8") && haskey(j.privateAaMutations, "ORF9b") && haskey(j.privateAaMutations, "S") && haskey(j.privateAaMutations, "E") && haskey(j.privateAaMutations, "M") && haskey(j.privateAaMutations, "N")
        total_private_AA_muts = j.privateAaMutations.ORF1a.totalPrivateSubstitutions + j.privateAaMutations.ORF1b.totalPrivateSubstitutions + j.privateAaMutations.ORF3a.totalPrivateSubstitutions + j.privateAaMutations.ORF6.totalPrivateSubstitutions + j.privateAaMutations.ORF7a.totalPrivateSubstitutions + j.privateAaMutations.ORF7b.totalPrivateSubstitutions + j.privateAaMutations.ORF8.totalPrivateSubstitutions + j.privateAaMutations.ORF9b.totalPrivateSubstitutions + j.privateAaMutations.S.totalPrivateSubstitutions + j.privateAaMutations.E.totalPrivateSubstitutions + j.privateAaMutations.M.totalPrivateSubstitutions + j.privateAaMutations.N.totalPrivateSubstitutions + private_del_ct
        if j.totalDeletions < 1500 && total_private_AA_muts ≤ max_AA_mut && j.qc.overallScore ≤ qc_max && j.privateNucMutations.totalReversionSubstitutions ≤ revs_thresh && j.qc.frameShifts.totalFrameShifts - j.qc.frameShifts.totalFrameShiftsIgnored == 0
#            total_AA_subs += total_private_AA_muts
            country = country(j.seqName)
            USA_state = US_state(j.seqName)
            seq_date = ""
####################################################
            try
                clade = j.clade
            catch
                clade = "Unknown"
            end
####################################################
            try
                pango = j.customNodeAttributes.Nextclade_pango
            catch
                pango = "Unknown"
            end
####################################################
            try
                pango_unaliased = j.customNodeAttributes.partiallyAliased
            catch
                pango_unaliased = "Unknown"
            end
####################################################
            seq_date = ""
            try
                seq_date = sequence_date(j.seqName)
            catch
                seq_date = "0-0-0"
            end
####################################################
#            lab = ""
#            if !isempty(seq_lab(j.seqName))
#                lab = seq_lab(j.seqName)
#            end
#            seq_lab_dict[name] = lab
            seq_year = 0
            seq_month = 0
            seq_day = 0
            seq_yr = join(seq_date[1:4])
            if all(isdigit, seq_yr)
                seq_year = parse(Int, seq_yr)
                if '-' in seq_date
                    seq_mo = split(seq_date, "-")[2]
                    if all(isdigit, seq_mo)
                        seq_month = parse(Int, seq_mo)
                    end
                end
####################################################
                dash_ct = 0
                for i in seq_date
                    if i == '-'
                        dash_ct += 1
                    end
                end
####################################################
                if dash_ct == 2
                    seq_da = split(seq_date, "-")[3]
                    if all(isdigit, seq_da)
                        seq_day = parse(Int, seq_da)
                    end
                end
            end
####################################################
            seq_collection_date[name] = seq_date
            if seq_month == 0 && seq_day == 0
                seq_collection_date[name] = string(seq_year)*"-0-0"
            end
            if seq_month ≠ 0 && seq_day == 0
                seq_collection_date[name] = string(seq_year)*"-"*string(seq_month)*"-0"
            end
            date_index = date_to_index[(seq_year, seq_month, seq_day)]
            seq_date_index[name] = date_index
            date_tuple = (seq_year, seq_month, seq_day)
            seq_date_tuple[name] = date_tuple
####################################################
            clade = seq_clade[name]
            pango = seq_pango[name]
            pango_unaliased = seq_pango_unaliased[name]
####################################################
            if !haskey(clade_date_index_ct, clade)
                clade_date_index_ct[clade] = Dict{Int, Int}()
            end
            if !haskey(pango_date_index_ct, pango)
                pango_date_index_ct[pango] = Dict{Int, Int}()
            end
            if !haskey(pango_unaliased_date_index_ct, pango_unaliased)
                pango_unaliased_date_index_ct[pango_unaliased] = Dict{Int, Int}()
            end
################################
            if !haskey(clade_date_index_ct[clade], date_index)
                clade_date_index_ct[clade][date_index] = 1
            else
                clade_date_index_ct[clade][date_index] += 1
            end
###################
            if !haskey(pango_date_index_ct[pango], date_index)
                pango_date_index_ct[pango][date_index] = 1
            else
                pango_date_index_ct[pango][date_index] += 1
            end
###################   
            if !haskey(pango_unaliased_date_index_ct[pango_unaliased], date_index)
                pango_unaliased_date_index_ct[pango_unaliased][date_index] = 1
            else
                pango_unaliased_date_index_ct[pango_unaliased][date_index] += 1
            end
##########################################################
            if !haskey(country_clade_date_index_ct, country)
                country_clade_date_index_ct[country] = Dict{String, Dict{Int, Int}}()
            end
            if !haskey(country_pango_date_index_ct, country)
                country_pango_date_index_ct[country] = Dict{String, Dict{Int, Int}}()  
            end
            if !haskey(country_pango_unaliased_date_index_ct, country)
                country_pango_unaliased_date_index_ct[country] = Dict{String, Dict{Int, Int}}()
            end
#################################
            if !haskey(country_clade_date_index_ct[country], clade)
                country_clade_date_index_ct[country][clade] = Dict{Int, Int}()
            end
            if !haskey(country_pango_date_index_ct[country], pango)
                country_pango_date_index_ct[country][pango] = Dict{Int, Int}()
            end
            if !haskey(country_pango_unaliased_date_index_ct[country], pango_unaliased)
                country_pango_unaliased_date_index_ct[country][pango_unaliased] = Dict{Int, Int}()
            end
#################################
            if !haskey(country_clade_date_index_ct[country][clade], date_index)
                country_clade_date_index_ct[country][clade][date_index] = 1
            else
                country_clade_date_index_ct[country][clade][date_index] += 1
            end
###################           
            if !haskey(country_pango_date_index_ct[country][pango], date_index)
                country_pango_date_index_ct[country][pango][date_index] = 1
            else
                country_pango_date_index_ct[country][pango][date_index] += 1
            end
###################  
            if !haskey(country_pango_unaliased_date_index_ct[country][pango_unaliased], date_index)
                country_pango_unaliased_date_index_ct[country][pango_unaliased][date_index] = 1
            else
                country_pango_unaliased_date_index_ct[country][pango_unaliased][date_index] += 1
            end
        end
    end
####################################################################################################################################
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_clade_date_index_ct.jld2", "clade_date_index_ct", clade_date_index_ct)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_pango_date_index_ct.jld2", "pango_date_index_ct", pango_date_index_ct)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_pango_unaliased_date_index_ct.jld2", "pango_unaliased_date_index_ct", pango_unaliased_date_index_ct)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_country_clade_date_index_ct.jld2", "country_clade_date_index_ct", country_clade_date_index_ct)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_country_pango_date_index_ct.jld2", "country_pango_date_index_ct", country_pango_date_index_ct)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_country_pango_unaliased_date_index_ct.jld2", "country_pango_unaliased_date_index_ct", country_pango_unaliased_date_index_ct)
####################################################################################################################################
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_country.jld2", "seq_country", seq_country)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_US_state.jld2", "seq_US_state", seq_US_state)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_clade.jld2", "seq_clade", seq_clade)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_pango.jld2", "seq_pango", seq_pango)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_pango_unaliased.jld2", "seq_pango_unaliased", seq_pango_unaliased)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_collection_date.jld2", "seq_collection_date", seq_collection_date)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_date_index.jld2", "seq_date_index", seq_date_index)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_date_tuple.jld2", "seq_date_tuple", seq_date_tuple)
#    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_lab_dict.jld2", "seq_lab_dict", seq_lab_dict)
####################################################################################################################################
end

In [ ]:
### Execute country_clade_date_index_cts   ***********  5, 1, 5, 2 ****************
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
start_clade = time()
country_clade_date_index_cts("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 5, 1, 5, 2, "all_private_muts")
final_clade = time() - start_clade
clade_time1, clade_time2 = seconds_to_hrs_min_sec(final_clade)
println(clade_time1); println(clade_time2)
#   input_ndjson, ndjson_name, max_AA_mut, revs_thresh, qc_max, print_ct_thresh, fx_name
##  Start: 
### Total Function Run Time =  seconds
### Total Function Run Time = 16842.4 seconds = 4 hours, 40 minutes, 42 seconds = 4:40.42
################################################################################################################

In [ ]:
### Execute country_clade_date_index_cts   ***********  90, 5, 8000, 2 ****************
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
start_clade = time()
country_clade_date_index_cts("/Volumes/PRO2/EPI_ISL_EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 90, 5, 8000, 2, "all_private_muts")
final_clade = time() - start_clade
clade_time1, clade_time2 = seconds_to_hrs_min_sec(final_clade)
println(clade_time1); println(clade_time2)
#   input_ndjson, ndjson_name, max_AA_mut, revs_thresh, qc_max, print_ct_thresh, fx_name
##  Start: 
### Total Function Run Time =  seconds
### Total Function Run Time = 16842.4 seconds = 4 hours, 40 minutes, 42 seconds = 4:40.42
################################################################################################################

In [ ]:
### Fx: sequences_lacking_keys_shortv2  | Used to find and eliminate sequences lacking NDJSON keys (which causes errors)
start_out = time()
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
function sequences_lacking_keys_shortv2(ndjson::String)
    start = time()
    privAAct = 0
    hundredK_time_dict = Dict{Int, Float64}()
    for i in 1:250
        hundredK_time_dict[i] = 0
    end
    key_set2 = Set([:ORF1a, :ORF1b, :ORF3a, :ORF6, :ORF7a, :ORF7b, :ORF8, :ORF9b, :S, :E, :M, :N])
#    missing_keys_ct = Dict{String, Int}()
    seq_ct = 0
    hundred_k_ct = 0
#    for key in key_set2
#        missing_keys_ct[key] = 0
#    end
    bad_seqs2 = Set{String}()
    for line in eachline(ndjson) 
        seq_ct += 1
        j = JSON3.read(line)
        name = split(j.seqName, "|")[2]
        if !haskey(j, :privateAaMutations)
            push!(bad_seqs2, name)
            privAAct += 1
        else
            for key in key_set2
                if !haskey(j.privateAaMutations, key)
#                    missing_keys_ct[key] +=1
                    push!(bad_seqs2, name)
                    break
                end
            end
        end
        if seq_ct%100000 == 0
            hundred_k_ct +=  1
            time_100000 = round(digits=1, time() - start)
            if hundred_k_ct == 1
                hundredK_time_dict[1] = time_100000
            else
                hundredK_time_dict[hundred_k_ct] = round(digits=1, time() - hundredK_time_dict[hundred_k_ct-1])
            end
            time_100000 = round(digits=1, time_100000)
            hundredK_rd = round(digits=1, hundredK_time_dict[hundred_k_ct])
            println("Time to $(hundred_k_ct)00000 Sequences = $(time_100000) sec; last 100K time = $(hundredK_rd)")
            println("          Sequences Removed so Far = $(length(bad_seqs2))")
        end
    end
    open("sequences_lacking_2nd_keys_to_exclude_from_functions_$(date_now).txt", "w") do g
        print("\n"^2); print(g, "\n"^3)
#        for (key, ct) in missing_keys_ct
#            println("$(key) Count = $(ct)")
#            println(g, "$(key) Count = $(ct)")
#        end
        println("Seqs Missing privateAaMutations = $(privAAct)")
        println(g, "Seqs Missing privateAaMutations = $(privAAct)")
        print("\n"^6); print(g, "\n"^6)
        for seq in bad_seqs2
            print("$(seq), ")
            println(g, "$(seq)")
        end
        tot = length(bad_seqs2)
        print("\n"^3); print(g, "\n"^3)
        println("Total Number of Seqs Lacking Keys = $(tot)")
        print("\n"^2); print(g, "\n"^2)
        total_time = round(digits=1, time() - start)
        println("Total Function Time = $(total_time)")
        println(g, "Total Function Time = $(total_time)")
        print("\n"^2); print(g, "\n"^2)
    end
end
total_time = time() - start_out
time1, time2 = seconds_to_hrs_min_sec(total_time)
println(time1); println(time2)

In [ ]:
### Execute function sequences_lacking_keys_shortv2; Runtime = Total Function Time = 5564.9; 1 hr 32 min 44.9 sec ############
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
start = time()
sequences_lacking_keys_shortv2("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson")
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v1 = $(runtime1)")
println("Runtime v2 = $(runtime2)")
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
###########################################################################

EPI_ISL_12576383, EPI_ISL_17033526, EPI_ISL_4449321, EPI_ISL_13098076, EPI_ISL_9421819, EPI_ISL_2953170, EPI_ISL_15198318, EPI_ISL_6319333, EPI_ISL_12910384, EPI_ISL_15372825, EPI_ISL_9424930, EPI_ISL_15197303, EPI_ISL_15591142, EPI_ISL_4736616, EPI_ISL_3739118, EPI_ISL_14451898, EPI_ISL_15198415, EPI_ISL_5249487, EPI_ISL_7473236, EPI_ISL_5326240, EPI_ISL_5249180, EPI_ISL_15548313, EPI_ISL_16822158, EPI_ISL_12793087, EPI_ISL_5434749, EPI_ISL_2449680, EPI_ISL_12793633, EPI_ISL_14012422, EPI_ISL_18349197, EPI_ISL_7473457, EPI_ISL_9420908, EPI_ISL_13403356, EPI_ISL_9423747, EPI_ISL_15745539, EPI_ISL_2567319, EPI_ISL_12483892, EPI_ISL_13663522, EPI_ISL_1772448, EPI_ISL_12908969, EPI_ISL_10682861, EPI_ISL_2950586, EPI_ISL_15593226, EPI_ISL_14591294, EPI_ISL_3980499, EPI_ISL_3235914, EPI_ISL_500806, EPI_ISL_13369441, EPI_ISL_15914454, EPI_ISL_17367044, EPI_ISL_18141259, EPI_ISL_14309403, EPI_ISL_14821733, EPI_ISL_3936475, EPI_ISL_13448626, EPI_ISL_15461523, EPI_ISL_4074172, EPI_ISL_15995067,

EPI_ISL_14450591, EPI_ISL_16519691, EPI_ISL_14821175, EPI_ISL_14997437, EPI_ISL_17166659, EPI_ISL_2979391, EPI_ISL_14821541, EPI_ISL_5326778, EPI_ISL_1773817, EPI_ISL_19456124, EPI_ISL_12794124, EPI_ISL_14297533, EPI_ISL_2950320, EPI_ISL_9419787, EPI_ISL_16164427, EPI_ISL_4574211, EPI_ISL_482780, EPI_ISL_1770853, EPI_ISL_12762930, EPI_ISL_16421271, EPI_ISL_16874334, EPI_ISL_15262869, EPI_ISL_12798105, EPI_ISL_17597448, EPI_ISL_17235371, EPI_ISL_10039185, EPI_ISL_14704539, EPI_ISL_13853044, EPI_ISL_10039295, EPI_ISL_13347568, EPI_ISL_15459288, EPI_ISL_12231697, EPI_ISL_10039801, EPI_ISL_13664805, EPI_ISL_13852624, EPI_ISL_4573146, EPI_ISL_10038409, EPI_ISL_10941818, EPI_ISL_3058687, EPI_ISL_14013156, EPI_ISL_12796539, EPI_ISL_1773626, EPI_ISL_1893603, EPI_ISL_11382949, EPI_ISL_15289610, EPI_ISL_2712233, EPI_ISL_1778291, EPI_ISL_17234656, EPI_ISL_15994869, EPI_ISL_7883867, EPI_ISL_13662774, EPI_ISL_10037311, EPI_ISL_14451500, EPI_ISL_1772745, EPI_ISL_12498991, EPI_ISL_15719395, EPI_ISL_1

EPI_ISL_1764504, EPI_ISL_15263026, EPI_ISL_15915966, EPI_ISL_15746881, EPI_ISL_14658191, EPI_ISL_1768270, EPI_ISL_15197089, EPI_ISL_12793083, EPI_ISL_12909011, EPI_ISL_13348227, EPI_ISL_18348508, EPI_ISL_13852636, EPI_ISL_5983686, EPI_ISL_2952433, EPI_ISL_6509388, EPI_ISL_2948775, EPI_ISL_5658001, EPI_ISL_4574263, EPI_ISL_14297637, EPI_ISL_13139383, EPI_ISL_16138134, EPI_ISL_14013659, EPI_ISL_16262428, EPI_ISL_15915160, EPI_ISL_7734062, EPI_ISL_2945340, EPI_ISL_13663576, EPI_ISL_4573970, EPI_ISL_14539699, EPI_ISL_16874122, EPI_ISL_15461173, EPI_ISL_3936368, EPI_ISL_3738623, EPI_ISL_14723740, EPI_ISL_3146798, EPI_ISL_16686147, EPI_ISL_17843932, EPI_ISL_13139461, EPI_ISL_1767620, EPI_ISL_14297545, EPI_ISL_2979070, EPI_ISL_10682986, EPI_ISL_13096737, EPI_ISL_2949197, EPI_ISL_17011747, EPI_ISL_2287784, EPI_ISL_3908528, EPI_ISL_7473746, EPI_ISL_15289049, EPI_ISL_13139564, EPI_ISL_5326846, EPI_ISL_2943674, EPI_ISL_3936397, EPI_ISL_2950764, EPI_ISL_2948872, EPI_ISL_16163668, EPI_ISL_13448722,

EPI_ISL_13473042, EPI_ISL_5249599, EPI_ISL_12795883, EPI_ISL_16422133, EPI_ISL_5656763, EPI_ISL_14452312, EPI_ISL_1774885, EPI_ISL_3936176, EPI_ISL_15290795, EPI_ISL_15719530, EPI_ISL_13348899, EPI_ISL_17224772, EPI_ISL_3738724, EPI_ISL_13139051, EPI_ISL_7473554, EPI_ISL_15745990, EPI_ISL_4449911, EPI_ISL_17366536, EPI_ISL_1764852, EPI_ISL_1772921, EPI_ISL_15593440, EPI_ISL_15747530, EPI_ISL_18032460, EPI_ISL_9423761, EPI_ISL_15473947, EPI_ISL_15372181, EPI_ISL_1646804, EPI_ISL_3059898, EPI_ISL_2287733, EPI_ISL_15371642, EPI_ISL_5326759, EPI_ISL_1784578, EPI_ISL_1764318, EPI_ISL_3059066, EPI_ISL_10683691, EPI_ISL_2979107, EPI_ISL_14012870, EPI_ISL_3388317, EPI_ISL_2979219, EPI_ISL_13850953, EPI_ISL_5984828, EPI_ISL_14453311, EPI_ISL_17282028, EPI_ISL_1766117, EPI_ISL_2288516, EPI_ISL_14310428, EPI_ISL_2947498, EPI_ISL_1777505, EPI_ISL_3253657, EPI_ISL_2107497, EPI_ISL_4966525, EPI_ISL_12761900, EPI_ISL_13447422, EPI_ISL_12073122, EPI_ISL_13473359, EPI_ISL_13753581, EPI_ISL_14308748, EP

EPI_ISL_7474396, EPI_ISL_17812088, EPI_ISL_4573594, EPI_ISL_1782618, EPI_ISL_13851854, EPI_ISL_15548877, EPI_ISL_14308701, EPI_ISL_15662498, EPI_ISL_13097273, EPI_ISL_14310094, EPI_ISL_15719630, EPI_ISL_15309286, EPI_ISL_13348225, EPI_ISL_14453021, EPI_ISL_2449657, EPI_ISL_1767795, EPI_ISL_12910330, EPI_ISL_1779963, EPI_ISL_12799110, EPI_ISL_16513256, EPI_ISL_15591565, EPI_ISL_16519303, EPI_ISL_15914182, EPI_ISL_1766095, EPI_ISL_2953709, EPI_ISL_12797397, EPI_ISL_408998, EPI_ISL_14014512, EPI_ISL_15549335, EPI_ISL_3374823, EPI_ISL_15549079, EPI_ISL_17010500, EPI_ISL_11162024, EPI_ISL_1773815, EPI_ISL_9420171, EPI_ISL_6323051, EPI_ISL_12482703, EPI_ISL_14309063, EPI_ISL_16943034, EPI_ISL_1779690, EPI_ISL_16052863, EPI_ISL_2193362, EPI_ISL_3059089, EPI_ISL_14012947, EPI_ISL_13139111, EPI_ISL_12576171, EPI_ISL_12909464, EPI_ISL_17235670, EPI_ISL_2943681, EPI_ISL_14592617, EPI_ISL_2952868, EPI_ISL_4074194, EPI_ISL_12761965, EPI_ISL_12797198, EPI_ISL_6323567, EPI_ISL_10683068, EPI_ISL_14218

...

EPI_ISL_13852057, EPI_ISL_13448853, EPI_ISL_4573664, EPI_ISL_15197507, EPI_ISL_15916010, EPI_ISL_10682964, EPI_ISL_13250523, EPI_ISL_2954787, EPI_ISL_16942679, EPI_ISL_15371278, EPI_ISL_16379186, EPI_ISL_4737427, EPI_ISL_8191857, EPI_ISL_15289543, EPI_ISL_10683844, EPI_ISL_15290863, EPI_ISL_1395225, EPI_ISL_1780854, EPI_ISL_14704427, EPI_ISL_15548920, EPI_ISL_8421522, EPI_ISL_9424735, EPI_ISL_10682793, EPI_ISL_15590802, EPI_ISL_15663555, EPI_ISL_15719536, EPI_ISL_17082639, EPI_ISL_15197222, EPI_ISL_13095148, EPI_ISL_15479245, EPI_ISL_15461088, EPI_ISL_7733541, EPI_ISL_5984294, EPI_ISL_15747067, EPI_ISL_11383495, EPI_ISL_8199038, EPI_ISL_13472406, EPI_ISL_13663393, EPI_ISL_4450776, EPI_ISL_5984888, EPI_ISL_10190809, EPI_ISL_12230195, EPI_ISL_14218089, EPI_ISL_3278107, EPI_ISL_15593227, EPI_ISL_10936040, EPI_ISL_13139462, EPI_ISL_1784016, EPI_ISL_11678709, EPI_ISL_1280058, EPI_ISL_14012442, EPI_ISL_15995148, EPI_ISL_10683711, EPI_ISL_14049902, EPI_ISL_13447479, EPI_ISL_16942287, EPI_ISL_

EPI_ISL_8011018, EPI_ISL_11382789, EPI_ISL_2947612, EPI_ISL_1780501, EPI_ISL_2950919, EPI_ISL_5332692, EPI_ISL_15915253, EPI_ISL_16421910, EPI_ISL_17597503, EPI_ISL_2946348, EPI_ISL_3150599, EPI_ISL_7884727, EPI_ISL_14310464, EPI_ISL_10683428, EPI_ISL_9050227, EPI_ISL_12073715, EPI_ISL_15087664, EPI_ISL_14820178, EPI_ISL_15549059, EPI_ISL_5331036, EPI_ISL_1776281, EPI_ISL_10038122, EPI_ISL_13851197, EPI_ISL_16064558, EPI_ISL_12231423, EPI_ISL_4449528, EPI_ISL_10683309, EPI_ISL_2019838, EPI_ISL_15547753, EPI_ISL_15087169, EPI_ISL_2449619, EPI_ISL_8486011, EPI_ISL_11875788, EPI_ISL_1395394, EPI_ISL_8422492, EPI_ISL_3236014, EPI_ISL_15817631, EPI_ISL_11160768, EPI_ISL_17366938, EPI_ISL_4450076, EPI_ISL_10941697, EPI_ISL_15086454, EPI_ISL_1058419, EPI_ISL_15661907, EPI_ISL_1781725, EPI_ISL_14450361, EPI_ISL_16163107, EPI_ISL_16262271, EPI_ISL_2949427, EPI_ISL_7850032, EPI_ISL_7850350, EPI_ISL_2943671, EPI_ISL_2711006, EPI_ISL_3551726, EPI_ISL_13094957, EPI_ISL_17282893, EPI_ISL_1936067, EP

EPI_ISL_12074585, EPI_ISL_9051168, EPI_ISL_18141312, EPI_ISL_7474850, EPI_ISL_14218118, EPI_ISL_13348109, EPI_ISL_9424032, EPI_ISL_19137678, EPI_ISL_12771040, EPI_ISL_15915291, EPI_ISL_1779946, EPI_ISL_7474368, EPI_ISL_1782029, EPI_ISL_14012958, EPI_ISL_1770105, EPI_ISL_14453701, EPI_ISL_9424317, EPI_ISL_10036653, EPI_ISL_2954571, EPI_ISL_4075135, EPI_ISL_13097752, EPI_ISL_15662293, EPI_ISL_2287067, EPI_ISL_2947703, EPI_ISL_6322506, EPI_ISL_426412, EPI_ISL_16335883, EPI_ISL_12793596, EPI_ISL_16422080, EPI_ISL_4574282, EPI_ISL_16519814, EPI_ISL_15198653, EPI_ISL_17083694, EPI_ISL_14538977, EPI_ISL_11679271, EPI_ISL_10191302, EPI_ISL_9420380, EPI_ISL_1324083, EPI_ISL_18349471, EPI_ISL_16518335, EPI_ISL_2979456, EPI_ISL_16821248, EPI_ISL_10940820, EPI_ISL_15460982, EPI_ISL_1764545, EPI_ISL_15289281, EPI_ISL_1783956, EPI_ISL_11677842, EPI_ISL_14309288, EPI_ISL_15548904, EPI_ISL_14894309, EPI_ISL_14894970, EPI_ISL_5656853, EPI_ISL_16238205, EPI_ISL_15915078, EPI_ISL_16164601, EPI_ISL_905066

Excessive output truncated after 524289 bytes.

EPI_ISL_11303087, EPI_ISL_14822333, EPI_ISL_13097671, EPI_ISL_13095636, EPI_ISL_14820334, EPI_ISL_12762154, EPI_ISL_15461365, EPI_ISL_4738020, EPI_ISL_1767362, EPI_ISL_16942808, EPI_ISL_15816703, EPI_ISL_10190332, EPI_ISL_15086845, EPI_ISL_5324918, EPI_ISL_4461330, EPI_ISL_10683209, EPI_ISL_16820457, EPI_ISL_12798853, EPI_ISL_3737758, EPI_ISL_12910054, EPI_ISL_10037334, EPI_ISL_18348308, EPI_ISL_15416846, EPI_ISL_13663394, EPI_ISL_12793810, EPI_ISL_1780409, EPI_ISL_18570664, EPI_ISL_5980690, EPI_ISL_10683102, EPI_ISL_15817364, EPI_ISL_13403621, EPI_ISL_16875043, EPI_ISL_15996261, EPI_ISL_14450623, EPI_ISL_11161590, EPI_ISL_17235386, EPI_ISL_1778918, EPI_ISL_10941006, EPI_ISL_14895219, EPI_ISL_1771904, EPI_ISL_2288442, EPI_ISL_2019647, EPI_ISL_583539, EPI_ISL_2948937, EPI_ISL_14217980, EPI_ISL_1775250, 

In [ ]:
### Execute function sequences_lacking_keys_shortv2; Runtime = Total Function Time = 5564.9; 1 hr 32 min 44.9 sec ############
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
start = time()
sequences_lacking_keys_shortv2("EPI_ISL_2008001_20300000_NNL_CFF.ndjson")
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v1 = $(runtime1)")
println("Runtime v2 = $(runtime2)")
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
####################################################################################################################
####################################################################################################################

In [ ]:
# Missing Key Counts
# ORF9b Count = 263845
# ORF1a Count = 259595
# ORF7a Count = 263012
# ORF6 Count =  262726
# M Count =     262170
# N Count =     263618
# ORF1b Count =  68342
# ORF7b Count = 263603
# ORF3a Count = 259564
# ORF8 Count =  263240
# E Count =     262027
# S Count =       4919

In [ ]:
### Fx: date_AA_muts_ct_only #########################################
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
date = Dates.format(today(), "yyyy_mm_dd")
nonleap_month_day_dict = Dict{Int, Int}(0=>0, 1=>31, 2=>28, 3=>31, 4=>30, 5=>31, 6=>30, 7=>31, 8=>31, 9=>30, 10=>31, 11=>30, 12=>31)
leap_month_day_dict = Dict{Int, Int}(0=>0, 1=>31, 2=>29, 3=>31, 4=>30, 5=>31, 6=>30, 7=>31, 8=>31, 9=>30, 10=>31, 11=>30, 12=>31)
index_to_date = Dict{Int, Tuple{Int, Int, Int}}()
date_to_index = Dict{Tuple{Int, Int, Int}, Int}()
index = 0
for year in 2020:2027
    for month in 1:12
        if year%4 == 0
            month_days = leap_month_day_dict[month]
            for day in 1:month_days
                index += 1
                index_to_date[index] = (year, month, day)
            end
        else
            month_days = nonleap_month_day_dict[month]
            for day in 1:month_days
                index += 1
                index_to_date[index] = (year, month, day)
            end
        end
    end
end
for (index, date) in index_to_date
    date_to_index[date] = index
end
for y in 2020:2027
    date_to_index[(y, 0, 0)] = y*1000000
    index_to_date[y*1000000] = (y, 0, 0)
    for m in 1:12
        date_to_index[(y, m, 0)] = y*1000000 + m*1000
        index_to_date[y*1000000 + m*1000] = (y, m, 0)
    end
end
date_to_index[(0, 0, 0)] = 1000000000
index_to_date[1000000000] = (0, 0, 0)
#date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
function date_AA_muts_ct_only(input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String)           
#    date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
    date = Dates.format(today(), "yyyy_mm_dd")
    start_time = time()
#########################################################################################
    date_regex = r"(\d{4})-(\d{2})-(\d{2})"
##### Explanation of    date_regex = r"(\d{4})-(\d{2})-(\d{2})"   #####################
##### r = regular expression
##### \d = any digit
##### {4} = number of preceding character to match (in this case the number of digits)
##### () = capture groups. Allow you to recall whatever is inside the parentheses later.
#          If date = "2025-09-01", then  date.capture[1] = "2025", date.capture[2] = "09", etc
##### So this checks the j.seqName line for dates in the correct format
#########################################################################################
    function get_full_year_month_day_tuple(seqName::String)
        ymd = match(date_regex, seqName)
        if ymd ≠ nothing
            year = parse(Int64, ymd.captures[1])
            month = parse(Int64, ymd.captures[2])
            day = parse(Int64, ymd.captures[3])
            if year ≥ 2020 && year ≤ 2027 && month ≥ 1 && month ≤ 12 && day ≥ 1 && day ≤ 31
                return (year, month, day)
            end
        end
        return nothing
    end
    function dash_count(str::String)
        return count(c -> c == '-', str)
    end
####################################################################################################################################
    excluded_pos = BitSet([1:60..., 76:78..., 686:694..., 2453, 3241, 5515, 7926, 11283:11295..., 14960, 15521, 19209:19210..., 19212, 19214, 19217, 21595, 21766, 21772, 21987, 21995:21996..., 22882, 24130, 27291, 28360:28373..., 29632, 29732:29736..., 29757, 29759:29761..., 29830:29900...])
    excluded_AA = Set(["S:K113N", "S:Q115H", "S:I794N", "ORF1b:F685Y", "ORF1b:N498I", "S:S27A", "S:D142G", "S:Y145D", "S:S408R", "S:K440N"])
    AA_muts_seq = Dict{String, Set{String}}();            sizehint!(AA_muts_seq, 53578)
    nuc_muts_seq = Dict{String, Set{String}}();           sizehint!(nuc_muts_seq, 50280)
    nonleap_month_day_dict = Dict{Int, Int}(0=>0, 1=>31, 2=>28, 3=>31, 4=>30, 5=>31, 6=>30, 7=>31, 8=>31, 9=>30, 10=>31, 11=>30, 12=>31)
    leap_month_day_dict = Dict{Int, Int}(0=>0, 1=>31, 2=>29, 3=>31, 4=>30, 5=>31, 6=>30, 7=>31, 8=>31, 9=>30, 10=>31, 11=>30, 12=>31)
#### For dicts below: 1st Key = date index; 2nd Key = mutation, value = mutation count on date index
    date_nuc_mut_ct = Dict{Int, Dict{String, Int}}();                 sizehint!(date_nuc_mut_ct, 2042)
    date_nuc_mut_ct_no_dels = Dict{Int, Dict{String, Int}}();         sizehint!(date_nuc_mut_ct_no_dels, 2042)
    date_AA_mut_ct = Dict{Int, Dict{String, Int}}();                  sizehint!(date_AA_mut_ct, 2040)
    date_AA_mut_ct_no_dels = Dict{Int, Dict{String, Int}}();          sizehint!(date_AA_mut_ct_no_dels, 2040)
    date_AA_mut_ct_pos_only_no_dels = Dict{Int, Dict{String, Int}}(); sizehint!(date_AA_mut_ct_pos_only_no_dels, 2040)
    all_seq_ct::Int = 0
    qualifying_seq_ct::Int = 0
    seq_ct_by_year = Dict{Int, Int}(i=>0 for i in 2020:2027);        sizehint!(seq_ct_by_year, 9)
    seq_ct_by_year[0] = 0
    seq_ct_by_year_month = Dict{Tuple{Int, Int}, Int}();             sizehint!(seq_ct_by_year_month, 105)
    for i in 2020:2027
        for j in 0:12
            seq_ct_by_year_month[(i, j)] = 0
        end
    end
    seq_ct_by_year_month[(0, 0)] = 0
    seq_ct_by_year_month_day = Dict{Tuple{Int, Int, Int}, Int}();    sizehint!(seq_ct_by_year_month_day, 3500)
    for i in 2020:2027
        if i%4 == 0
            for j in 0:12
                for k in 0:leap_month_day_dict[j]
                    seq_ct_by_year_month_day[(i, j, k)] = 0
                end
            end
        else
            for j in 0:12
                for k in 0:nonleap_month_day_dict[j]
                    seq_ct_by_year_month_day[(i, j, k)] = 0
                end
            end
        end
    end
    seq_ct_by_year_month_day[(0, 0, 0)] = 0
###################################################################################################################################
    seq_collection_date = Dict{String, String}()
    seq_date_index = Dict{String, Int}()
    seq_date_tuple = Dict{String, Tuple{Int64, Int64, Int64}}()
#####################################################################################################################################    
    half_million_ct::Int64 = 0
    million_ct::Float16 = 0
    max_dels = 1500
    max_frameshifts = 2
    max_stops = 1
    max_SNPclust = 101
    min_coverage = 0.50
    for line in eachline(input_ndjson)
        all_seq_ct += 1
        if all_seq_ct % 500000 == 0
            half_million_ct += 1
            million_ct = half_million_ct*0.5
            println("million ct = $(million_ct)")
            nowtime = Dates.format(now(), "I:MM.SS_p"); print(nowtime); println()
        end
############################################################################
        j = JSON3.read(line)
############################################################################
        nuc_muts = j.privateNucMutations
        if j.totalDeletions ≥ max_dels
            continue
        end
        QC = j.qc
        if QC.overallScore > qc_max
            continue
        end
        if QC.frameShifts.totalFrameShifts - QC.frameShifts.totalFrameShiftsIgnored > max_frameshifts
            continue
        end
        if QC.stopCodons.totalStopCodons - QC.stopCodons.totalStopCodonsIgnored > max_stops
            continue
        end
        if QC.snpClusters.score ≥ max_SNPclust
            continue
        end
        if j.coverage < min_coverage
            continue
        end
        if nuc_muts.totalReversionSubstitutions > revs_thresh
            continue
        end
############################################################################
        name = EPI_ISL(j.seqName)
        del_ranges = nuc_muts.privateDeletionRanges
        private_del_ct = length(nuc_muts.privateDeletionRanges)
        pango = ""
        privAA = j.privateAaMutations
        ORF1a_AA = privAA.ORF1a
        ORF1b_AA = privAA.ORF1b
        ORF3a_AA = privAA.ORF3a
        ORF6_AA = privAA.ORF6
        ORF7a_AA = privAA.ORF7a
        ORF7b_AA = privAA.ORF7b
        ORF8_AA = privAA.ORF8
        ORF9b_AA = privAA.ORF9b
        S_AA = privAA.S
        E_AA = privAA.E
        M_AA = privAA.M
        N_AA = privAA.N
        total_private_AA_muts = ORF1a_AA.totalPrivateSubstitutions + ORF1b_AA.totalPrivateSubstitutions + ORF3a_AA.totalPrivateSubstitutions + ORF6_AA.totalPrivateSubstitutions + ORF7a_AA.totalPrivateSubstitutions + ORF7b_AA.totalPrivateSubstitutions + ORF8_AA.totalPrivateSubstitutions + ORF9b_AA.totalPrivateSubstitutions + S_AA.totalPrivateSubstitutions + E_AA.totalPrivateSubstitutions + M_AA.totalPrivateSubstitutions + N_AA.totalPrivateSubstitutions
        if total_private_AA_muts + private_del_ct ≤ max_AA_mut
            qualifying_seq_ct += 1
            seq_date = ""
            seq_year = Int64(0)
            seq_month = Int64(0)
            seq_day = Int64(0)
            date_tuple = get_full_year_month_day_tuple(j.seqName)
            if date_tuple == nothing
                try
                    seq_date = string(sequence_date(j.seqName))
                catch
                    seq_date = "0-0-0"
                end
                dash_ct = dash_count(seq_date)
                if dash_ct == 0
                    seq_y = seq_date
                    if all(isdigit, seq_y)
                        seq_y = parse(Int64, seq_y)
                        if seq_y ≥ 2020 && seq_y ≤ 2027
                            seq_year = seq_y
                        end
                    end
                end
                if dash_ct == 1
                    seq_y = string(split(seq_date, "-")[1])
                    seq_m = string(split(seq_date, "-")[2])
                    if all(isdigit, seq_y)
                        seq_y = parse(Int64, seq_y)
                        if seq_y ≥ 2020 && seq_y ≤ 2027
                            seq_year = seq_y
                        end
                    end
                    if all(isdigit, seq_m)
                        seq_m = parse(Int64, seq_m)
                        if seq_m ≥ 1 && seq_m ≤ 12
                            seq_month = seq_m
                        end
                    end
                end
                if dash_ct == 2
                    seq_y = string(split(seq_date, "-")[1])
                    seq_m = string(split(seq_date, "-")[2])
                    seq_d = string(split(seq_date, "-")[3])
                    if all(isdigit, seq_y)
                        seq_y = parse(Int64, seq_y)
                        if seq_y ≥ 2020 && seq_y ≤ 2027
                            seq_year = seq_y
                        end
                    end
                    if all(isdigit, seq_m)
                        seq_m = parse(Int64, seq_m)
                        if seq_m ≥ 1 && seq_m ≤ 12
                            seq_month = seq_m
                        end
                    end
                    if all(isdigit, seq_d)
                        seq_d = parse(Int64, seq_d)
                        if seq_d ≥ 1 && seq_d ≤ 31
                            seq_day = seq_d
                        end
                    end
                end
                date_tuple = (seq_year, seq_month, seq_day)
            end
            date_index = date_to_index[date_tuple]
            seq_date_index[name] = date_index
            seq_date_tuple[name] = date_tuple
#################################################################################################################################
            AA_subs = Set([ORF1a_AA.privateSubstitutions, ORF1b_AA.privateSubstitutions, ORF3a_AA.privateSubstitutions, ORF6_AA.privateSubstitutions, ORF7a_AA.privateSubstitutions, ORF7b_AA.privateSubstitutions, ORF8_AA.privateSubstitutions, ORF9b_AA.privateSubstitutions, S_AA.privateSubstitutions, E_AA.privateSubstitutions, M_AA.privateSubstitutions, N_AA.privateSubstitutions])
            subs = nuc_muts.privateSubstitutions
            dels = nuc_muts.privateDeletions
            for d in dels
                pos = d.pos + 1
                if !(pos in excluded_pos)
                    n_del = "$(d.refNuc)$(string(pos))-"    
##########################################################################
                    get!(date_nuc_mut_ct, date_index, Dict{String, Int64}() ) 
                    date_nuc_mut_ct[date_index][n_del] = get(date_nuc_mut_ct[date_index], n_del, 0) + 1
##########################################################################
                end
            end
            for i in AA_subs
                for k in i
                    pos = k.pos + 1
                    AA_sub = "$(k.cdsName):$(k.refAa)$(string(pos))$(k.qryAa)"
                    ref_AA_res = string(gene_AA_dict[k.cdsName][pos])
                    if !(k.refAa == "-") && !(AA_sub in excluded_AA) && k.qryAa ≠ ref_AA_res && k.qryAa ≠ '-'
                        get!(date_AA_mut_ct, date_index, Dict{String, Int64}() )
                        date_AA_mut_ct[date_index][AA_sub] = get(date_AA_mut_ct[date_index], AA_sub, 0) + 1
                    end
                end
            end
        end
    end   
#####################################################################################################################################    
    for (date, nuc_mut_ct_dict) in date_nuc_mut_ct
        for (mut, ct) in nuc_mut_ct_dict
            get!(date_nuc_mut_ct_no_dels, date, Dict{String, Int}() )
            if mut[end] ≠ '-'
                date_nuc_mut_ct_no_dels[date][mut] = ct
            end
        end
    end
    for date in keys(date_AA_mut_ct)
        for (mut, ct) in date_AA_mut_ct[date]
            AA_pos = aa_gene_and_pos_comprehensive_dict[mut]
            if mut[end] ≠ '-'
                get!(date_AA_mut_ct_no_dels, date, Dict{String, Int}() )
                date_AA_mut_ct_no_dels[date][mut] = ct
                get!(date_AA_mut_ct_pos_only_no_dels, date, Dict{String, Int}() )
####################################################
                date_AA_mut_ct_pos_only_no_dels[date][AA_pos] = get(date_AA_mut_ct_pos_only_no_dels[date], AA_pos, 0) + ct
####################################################
            end
        end
    end
####################################################################################################################################
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_nuc_mut_ct.jld2", "date_nuc_mut_ct", date_nuc_mut_ct)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_nuc_mut_ct_no_dels.jld2", "date_nuc_mut_ct_no_dels", date_nuc_mut_ct_no_dels)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_AA_mut_ct.jld2", "date_AA_mut_ct", date_AA_mut_ct)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_AA_mut_ct_no_dels.jld2", "date_AA_mut_ct_no_dels", date_AA_mut_ct_no_dels)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_AA_mut_ct_pos_only_no_dels.jld2", "date_AA_mut_ct_pos_only_no_dels", date_AA_mut_ct_pos_only_no_dels)
####################################################################################################################################
    println("Dictionaries saved!")
    total_function_time = time() - start_time
    total_function_time_rd = round(digits=1, total_function_time)
    open("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_printMin$(print_ct_thresh)_RUNTIME.txt", "w") do g
        hours = total_function_time÷3600
        hours_int = Int(hours)
        hours_int_fin = split(string(hours_int), ".")[1]
        minutes = (total_function_time%3600)÷60
        minutes_int = Int(minutes)
        minutes_int2 = lpad(minutes_int, 2, "0")
        seconds = (total_function_time%3600)%60
        seconds_rd = round(digits=1, seconds)
        println()
        println("Total Function Run Time = ", total_function_time_rd, " seconds")
        println("$(hours) hours, $(minutes) minutes, $(seconds_rd) seconds")
        println("$(hours).$(minutes).$(seconds_rd)")
        print("\n"^3)
        println(g)
        println(g, "Total Function Run Time = ", total_function_time_rd, " seconds");   println(g)
        println(g, "$(hours_int) hours, $(minutes_int) minutes, $(seconds_rd) seconds");   println(g)
        println(g, "$(hours_int).$(minutes_int2).$(seconds_rd)");   print(g, "\n"^4)
    end
    return date_AA_mut_ct
end
################################################################################################################

In [ ]:
### Execute date_AA_muts_ct_only | 5, 1, 5
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
date_AA_muts_ct_only("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 5, 1, 5, 2, "all_private_muts")
#### Total Function Run Time = 8378.4 seconds    2 hours, 19 minutes, 38.4 seconds     2025_09_28_839PM    20, 2, 100, 2
################################################################################################################################
################################################################################################################################

In [ ]:
### Execute date_AA_muts_ct_only | 10, 1, 30
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
date_AA_muts_ct_only("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 10, 1, 30, 2, "all_private_muts")
###### Total Function Run Time = 7324.6 seconds        2 hours, 2 minutes, 4.6 seconds     2025_09_28_1059PM    7, 1, 10, 2
################################################################################################################################
################################################################################################################################

In [ ]:
### Execute date_AA_muts_ct_only | 90, 5, 8000
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
date_AA_muts_ct_only("/Volumes/PRO2/EPI_ISL_400001_20300000_NNL_CFF.ndjson", "EPI_ISL_400001_20300000", 90, 5, 8000, 2, "all_private_muts")
###### Total Function Run Time = 7324.6 seconds        2 hours, 2 minutes, 4.6 seconds     2025_09_28_1059PM    7, 1, 10, 2
################################################################################################################################
################################################################################################################################

In [ ]:
### Fx: all_private_muts_04_19_2026__AA_muts_seq__and__nuc_muts_seq_only #################################
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
function all_private_muts_04_19_2026__AA_seq_muts__and__nuc_muts_seq_only(input_ndjson::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int, print_ct_thresh::Int, fx_name::String)        
    date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
    date = Dates.format(today(), "yyyy_mm_dd")
    start_time = time()
    no_key_set = Set{String}()
####################################################################################################################################
####################################################################################################################################
    excluded_pos = BitSet([1:60..., 76:78..., 686:694..., 2453, 3241, 5515, 7926, 11283:11295..., 14960, 15521, 19209:19210..., 19212, 19214, 19217, 21595, 21766, 21772, 21987, 21995:21996..., 22882, 24130, 27291, 28360:28373..., 29632, 29732:29736..., 29757, 29759:29761..., 29830:29900...])
    excluded_AA = Set(["S:K113N", "S:Q115H", "S:I794N", "ORF1b:F685Y", "ORF1b:N498I", "S:S27A", "S:D142G", "S:Y145D", "S:S408R", "S:K440N"])
    total_AA_subs::Int = 0
    AA_muts_seq = Dict{String, Set{String}}();            sizehint!(AA_muts_seq, 53578)
    nuc_muts_seq = Dict{String, Set{String}}()           #sizehint!(nuc_muts_seq, 50280)
#    AA_muts_seq_no_dels = Dict{String, Set{String}}()    #sizehint!(AA_muts_seq_no_dels, 50280)
#    nuc_muts_seq_no_dels = Dict{String, Set{String}}()   #sizehint!(nuc_muts_seq_no_dels, 50280)
    all_seq_ct::Int = 0
    qualifying_seq_ct::Int = 0
###################################################################################################################################
    total_nuc_dels::Int = 0
    total_subs::Int = 0
    total_AA_dels::Int = 0
    total_AA_subs::Int = 0
#####################################################################################################################################    
    half_million_ct::Int = 0
    million_ct::Float16 = 0
    for line in eachline(input_ndjson)
        all_seq_ct += 1
        if all_seq_ct % 500000 == 0
            half_million_ct += 1
            million_ct = half_million_ct*0.5
            println("million ct = $(million_ct)")
            nowtime = Dates.format(now(), "I:MM.SS_p"); print(nowtime); println()
        end
        j = JSON3.read(line)
        name = EPI_ISL(j.seqName)
        nuc_muts = j.privateNucMutations
        del_ranges = nuc_muts.privateDeletionRanges
        private_del_ct::Int = 0
        for range in del_ranges
            private_del_ct += 1
        end
#        if haskey(j.privateAaMutations, "ORF1a") && haskey(j.privateAaMutations, "ORF1b") && haskey(j.privateAaMutations, "ORF3a") && haskey(j.privateAaMutations, "ORF6") && haskey(j.privateAaMutations, "ORF7a") && haskey(j.privateAaMutations, "ORF7b") && haskey(j.privateAaMutations, "ORF8") && haskey(j.privateAaMutations, "ORF9b") && haskey(j.privateAaMutations, "S") && haskey(j.privateAaMutations, "E") && haskey(j.privateAaMutations, "M") && haskey(j.privateAaMutations, "N")
        private_AA = j.privateAaMutations
        ORF1a_AA = private_AA.ORF1a; ORF1b_AA = private_AA.ORF1b; ORF3a_AA = private_AA.ORF3a; ORF6_AA = private_AA.ORF6
        ORF7a_AA = private_AA.ORF7a; ORF7b_AA = private_AA.ORF7b; ORF8_AA = private_AA.ORF8;   ORF9b_AA = private_AA.ORF9b
        S_AA = private_AA.S; E_AA = private_AA.E; M_AA = private_AA.M; N_AA = private_AA.N
#        private_AA_tuple = (ORF1a_AA, ORF1b_AA, ORF3a_AA, ORF6_AA, ORF7a_AA, ORF7b_AA, ORF8_AA, ORF9b_AA, S_AA, E_AA, M_AA, N_AA)
        private_AA_tuple_total_AAsubs = (ORF1a_AA.totalPrivateSubstitutions, ORF1b_AA.totalPrivateSubstitutions, ORF3a_AA.totalPrivateSubstitutions, ORF6_AA.totalPrivateSubstitutions, ORF7a_AA.totalPrivateSubstitutions, ORF7b_AA.totalPrivateSubstitutions, ORF8_AA.totalPrivateSubstitutions, ORF9b_AA.totalPrivateSubstitutions, S_AA.totalPrivateSubstitutions, E_AA.totalPrivateSubstitutions, M_AA.totalPrivateSubstitutions, N_AA.totalPrivateSubstitutions)
        total_private_AA_muts = sum(private_AA_tuple_total_AAsubs)
        if j.totalDeletions < 1500 && nuc_muts.totalReversionSubstitutions ≤ revs_thresh && j.qc.frameShifts.totalFrameShifts - j.qc.frameShifts.totalFrameShiftsIgnored == 0  && j.qc.overallScore ≤ qc_max && total_private_AA_muts ≤ max_AA_mut 
            total_AA_subs += total_private_AA_muts
####################################################
            genes = ["ORF1a", "ORF1b", "ORF3a", "ORF6", "ORF7a", "ORF7b", "ORF8", "ORF9b", "S", "E", "M", "N"]
            AA_subs = Set([ORF1a_AA.privateSubstitutions, ORF1b_AA.privateSubstitutions, ORF3a_AA.privateSubstitutions, ORF6_AA.privateSubstitutions, ORF7a_AA.privateSubstitutions, ORF7b_AA.privateSubstitutions, ORF8_AA.privateSubstitutions, ORF9b_AA.privateSubstitutions, S_AA.privateSubstitutions, E_AA.privateSubstitutions, M_AA.privateSubstitutions, N_AA.privateSubstitutions])
            subs = nuc_muts.privateSubstitutions
            dels = nuc_muts.privateDeletions
            qualifying_seq_ct += 1
            @inbounds for k in subs
                if k.pos < 29870
                    pos = k.pos + 1
                    if !(pos in excluded_pos) && k.refNuc ≠ "-" && k.qryNuc ≠ string(ref[pos])
                        nuc_sub = "$(k.refNuc)$(string(pos))$(k.qryNuc)"
##########################################################################
                        if !haskey(nuc_muts_seq, nuc_sub)
                            nuc_muts_seq[nuc_sub] = Set{String}()
                        end
                        push!(nuc_muts_seq[nuc_sub], name)
##########################################################################
                    end
                end
            end
            for d in dels
                pos = d.pos + 1
                if !(pos in excluded_pos)
                    total_nuc_dels += 1
                    n_del = "$(d.refNuc)$(string(pos))-"    
                    if !haskey(nuc_muts_seq, n_del)
                        nuc_muts_seq[n_del] = Set{String}()
                    end
                    push!(nuc_muts_seq[n_del], name)
                end
            end
            for i in AA_subs
                for k in i
                    pos = k.pos + 1
                    AA_sub = "$(k.cdsName):$(k.refAa)$(string(pos))$(k.qryAa)"
                    ref_AA_res = string(gene_AA_dict[k.cdsName][pos])
                    if !(k.refAa == "-") && !(AA_sub in excluded_AA) && k.qryAa ≠ ref_AA_res && k.qryAa ≠ '-'
#######################################################################################################
                        if !haskey(AA_muts_seq, AA_sub)
                            AA_muts_seq[AA_sub] = Set{String}()
                        end
                        push!(AA_muts_seq[AA_sub], name)
#######################################################################################################
                    end
                end
            end
            AAdels_tuple = ( ("ORF1a", ORF1a_AA.privateDeletions), ("ORF1b", ORF1b_AA.privateDeletions), 
                ("ORF3a", ORF3a_AA.privateDeletions), ("ORF6", ORF6_AA.privateDeletions), ("ORF7a", ORF7a_AA.privateDeletions), 
                ("ORF7b", ORF7b_AA.privateDeletions), ("ORF8", ORF8_AA.privateDeletions), ("ORF9b", ORF9b_AA.privateDeletions), 
                ("S", S_AA.privateDeletions), ("E", E_AA.privateDeletions), ("M", M_AA.privateDeletions), ("N", N_AA.privateDeletions) ) 
            for (orf_str, dels) in AAdels_tuple
                for del in dels
                    del_AA = "$(orf_str):$(del.refAa)$(del.pos + 1)-"
                    if !haskey(AA_muts_seq, del_AA)
                        AA_muts_seq[del_AA] = Set{String}()
                    end
                    push!(AA_muts_seq[del_AA], name)
                end
            end
        end
    end
#####################################################################################################################################
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_seq.jld2", "AA_muts_seq", AA_muts_seq)
    save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_seq.jld2", "nuc_muts_seq", nuc_muts_seq)
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_all_seq_ct.jld2", all_seq_ct)
    @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_qualifying_seq_ct.jld2", qualifying_seq_ct)
####################################################################################################################################
#        @save("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_.jld2", seq) 
####################################################################################################################################
    println("Dictionaries saved!")
    total_function_time = time() - start_time
    total_function_time_rd = round(digits=1, total_function_time)
    open("$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_printMin$(print_ct_thresh)_RUNTIME.txt", "w") do g
        hours = total_function_time÷3600
        hours_int = Int(hours)
        hours_int_fin = split(string(hours_int), ".")[1]
        minutes = (total_function_time%3600)÷60
        minutes_int = Int(minutes)
        minutes_int2 = lpad(minutes_int, 2, "0")
        seconds = (total_function_time%3600)%60
        seconds_rd = round(digits=1, seconds)
        println()
        println("Total Function Run Time = ", total_function_time_rd, " seconds")
        println("$(hours) hours, $(minutes) minutes, $(seconds_rd) seconds")
        println("$(hours).$(minutes).$(seconds_rd)")
        print("\n"^3)
        println(g)
        println(g, "Total Function Run Time = ", total_function_time_rd, " seconds");   println(g)
        println(g, "$(hours_int) hours, $(minutes_int) minutes, $(seconds_rd) seconds");   println(g)
        println(g, "$(hours_int).$(minutes_int2).$(seconds_rd)");   print(g, "\n"^4)
    end
end